# Thai Shrine Classifier — โค้ดโครงการใน notebook เดียว

เวอร์ชันจัดแพ็กวันที่ 23 ตุลาคม 2026 · protocol `architecture-mode-5trial-v2`

Notebook นี้รวม **Python ทั้ง 27 ไฟล์ พร้อม JavaScript/PowerShell สำหรับรายงานและ requirements รวม 45 source files** ของโครงการปัจจุบัน ไม่ต้องดาวน์โหลดไฟล์ Python แยกเพื่อเริ่มต้น

เมื่อรันเซลล์ source จะสร้างโมดูลที่อยู่ใน notebook ลงในโฟลเดอร์โครงการอัตโนมัติ เพื่อให้ imports, CLI, Windows multiprocessing และ code fingerprints ใช้โครงสร้างเดิม ไฟล์ที่สร้างนี้เก็บเฉพาะในเครื่องและไม่ต้อง push ขึ้น Git แต่ละ source cell แสดงโค้ดจริงให้อ่านและแก้ได้ ไม่ได้ดึงโค้ดจาก repository ขณะรัน

**ลำดับงาน:** ตั้งค่า → เตรียม source → ตรวจข้อมูล → preprocessing/imbalance → tuning → lock → final → evaluation → Grad-CAM/eyeball → ResNet ablation → รายงาน

- ค่าเริ่มต้นแสดงผลที่บันทึกแล้ว และ **ไม่เริ่ม training, tests หรือ inference** เมื่อ Run All
- การทดลองหลักเสร็จแล้ว: 75 tuning trials + 75 final runs; ResNet ablation เพิ่ม 10 runs ผลรวม 160 runs / 2,906 epochs
- Validation reference คือ **MobileNetV3-Large/Partial**; รายงานเลือก **ResNet50/Full** ภายหลังพิจารณา test จึงเป็น post-hoc/exploratory บน split ที่เคยตรวจดูแล้ว
- ใช้ notebook ตัวอย่าง v11 เป็นแนวทางการจัดหน้าเท่านั้น ไม่ย้ายผลเก่า, Bayesian search, CV หรือ preprocessing คนละ protocol มาปะปน
- โค้ดใน notebook ยังไม่ได้ถูก execute ในการจัดทำครั้งนี้ ทุก code cell เริ่มด้วย execution_count=null และ outputs ว่าง

ข้อมูลภาพ, canonical split, ผลการทดลอง และ checkpoints เป็นไฟล์ประกอบภายนอกตามปกติ ไม่ถูกฝังลงใน notebook นี้ อ่านรายละเอียดผลใน README เดิมซึ่งไม่ได้ถูกแก้โดยการจัดแพ็กครั้งนี้


## 0. ตั้งค่า project และเลือกขั้นตอนที่จะรัน

เปิด notebook จากโฟลเดอร์โครงการ หรือแก้ PROJECT_ROOT ให้ตรงตำแหน่งที่มี dataset_cleaned และ outputs หากใช้ Colab ให้ mount Drive เองแล้วกำหนด path ที่ต้องการ ไม่มีการ hard-code Drive ของผู้เขียน

ค่าเริ่มต้นใช้ outputs_modes_v3_td สำหรับอ่านผลเดิม หากเริ่มการทดลองคนละ protocol ต้องใช้ output directory ใหม่และรักษา canonical split เดิม การเปิด flag เป็น True หมายถึงผู้ใช้เลือกให้รันขั้นตอนนั้นเอง ไม่ต้องรันการทดลองใหม่เพื่ออ่านรายงาน


In [ ]:
from pathlib import Path
import os
import sys
import json
import csv
import hashlib
import shlex
import subprocess
import importlib.util
from collections import Counter
from IPython.display import display, Markdown, Image, SVG
from IPython.core.magic import register_cell_magic

PROJECT_ROOT = Path.cwd().resolve()  # แก้ path ตรงนี้หากเปิดจากโฟลเดอร์อื่น
DATA_ROOT = PROJECT_ROOT / "dataset_cleaned"
BASE_OUT_DIR = PROJECT_ROOT / "outputs"
OUT_DIR = PROJECT_ROOT / "outputs_modes_v3_td"
WANDB_PROJECT = "thai-shrine-classifier-td"
WANDB_MODE = "online"
DEVICE = "auto"
NUM_WORKERS = 0
SEEDS = [42, 123, 456, 789, 1024]
ARCHITECTURES = ["resnet50", "efficientnet_b0", "mobilenet_v3_large", "densenet121", "convnext_tiny"]
MODES = ["full", "frozen", "partial"]
CLASSES = ["san_phraphrom", "san_phraphum", "san_tayay", "tee_ju_eia"]
RESUME = True

INSTALL_DEPENDENCIES = False
ALLOW_SOURCE_CHANGES = False  # เปิดเฉพาะเมื่อจงใจแก้ source/แทนไฟล์ local
ALLOW_README_UPDATE = False  # report tools บางตัวเขียน README; ปิดไว้เป็นค่าเริ่มต้น
RUN = {stage: False for stage in (
    "prepare", "smoke", "tune", "lock", "final", "analyze", "gradcam", "eyeball", "html_report",
    "resnet_gradcam", "ablation_lock", "ablation", "ablation_analyze", "readme_export", "learning_curves"
)}
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Output:", OUT_DIR)
print("Enabled stages:", [name for name, enabled in RUN.items() if enabled] or "none (read saved results)")


### 0.1 Dependencies

เช็กว่ามีแพ็กเกจหลักหรือไม่ด้วย find_spec โดยไม่ฝึกโมเดล การติดตั้งปิดไว้เป็นค่าเริ่มต้น PyTorch/torchvision ต้องเป็นคู่ที่เข้ากันและเหมาะกับ runtime ของผู้ใช้ ส่วน pandas ใช้สำหรับ learning-curve utility

Environment ของผลเดิมบันทึก Python 3.13.5, torch 2.13.0+cu130, torchvision 0.28.0+cu130 และ RTX 5060 การติดตั้งบน runtime ใหม่ไม่ได้ยืนยันว่าจะให้ผลตัวเลขตรงเดิมทุกประการ


In [ ]:
required_imports = {"numpy": "numpy", "Pillow": "PIL", "scipy": "scipy", "scikit-learn": "sklearn",
                    "matplotlib": "matplotlib", "wandb": "wandb", "pandas": "pandas", "torch": "torch", "torchvision": "torchvision"}
missing = [name for name, module in required_imports.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing or "none")
if INSTALL_DEPENDENCIES:
    packages = ["numpy>=1.26", "Pillow>=10.0", "scipy>=1.11", "scikit-learn>=1.4", "matplotlib>=3.8", "wandb>=0.18", "pandas>=2.0", "torch", "torchvision"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)


## 1. ตัวบันทึก source จาก notebook

เซลล์ต่อไปลงทะเบียน cell magic ชื่อ project_source เซลล์ source แต่ละอันจะบันทึกข้อความที่เห็นเป็นไฟล์ในเครื่อง โดยไม่ execute เนื้อหาโมดูล SHA256, newline และท้ายไฟล์ถูกรักษาให้เหมือนต้นฉบับ จึงไม่เปลี่ยน fingerprint เพียงเพราะจัดแพ็กเป็น notebook

หากมีไฟล์ local ที่เนื้อหาต่างกัน ระบบจะหยุดก่อนเขียนทับ หากต้องการแก้โค้ดใน notebook โดยเจตนาให้ตั้ง ALLOW_SOURCE_CHANGES=True และถือเป็นการเปลี่ยน code identity ตามกติกา resume เดิม ไม่แก้ locks หรือผลทดลองเพื่อให้ผ่านการตรวจ


In [ ]:
BUNDLE_MANIFEST = json.loads(r'''{
  "pipeline.py": {
    "sha256": "8218c07a7504c99a52c93856284f510bfff00fc899910239c72f3f43aa40d6a4",
    "bytes": 18533,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "pipeline_extensions.py": {
    "sha256": "bb6f2adaec20c60f2c565281d210d29ecda3b7bf134d08ad09129527a4f3feb4",
    "bytes": 29511,
    "newline": "crlf",
    "terminal_newlines": 1,
    "bom": false
  },
  "pipeline_modes.py": {
    "sha256": "72bd76b4e9fe3766113216f0c318cf0699e91ec09791b562c7c47f18c0c8ebcc",
    "bytes": 70094,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "plot_learning_curves.py": {
    "sha256": "7aebb2a2049f4f9efdb130c3c6a88d67a8c079f93f31e9ba1d1fd17775328bd3",
    "bytes": 16469,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/build_report.ps1": {
    "sha256": "d6b0fcf0614640e82988b36b4073212277c0a7ffbea02a5a9e88e6edd324e62f",
    "bytes": 13374,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/copy_learning_curves.ps1": {
    "sha256": "fdccc52158d26847296aa56815d9d5057b4fdfa5eb91d1620516e4533458d1b5",
    "bytes": 5987,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/figure_theme.js": {
    "sha256": "4d59082ee830418f65e3c5028196b374ad89cbe8069c3dd95e0fe611432843f6",
    "bytes": 1238,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_context_figures.js": {
    "sha256": "a74d9fbfb358a0f5e71b07d4da24b4dfe6da9cef1ffe78d98d844791aff568cf",
    "bytes": 5676,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_followup_figures.js": {
    "sha256": "6f1cbc045cd0d0789a09289b98310b96a1add31661ab14e24f030e277ac7edd1",
    "bytes": 7548,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_latest_figures.js": {
    "sha256": "70fd85267e07afa8b1beb567a082be781e9dbbb27edd35effb5e88d465342323",
    "bytes": 17760,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_learning_report.js": {
    "sha256": "dc14faa373a82c0a409e55ef88da02d1af672407192e2ada57b55baa4657fbcf",
    "bytes": 21525,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_matched_mobilenet_cam.py": {
    "sha256": "1f529ad509885c56364d230ddf2fd3e3cd79fa2107ff9636eb73730220e97bf1",
    "bytes": 7806,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_preprocessing_examples.js": {
    "sha256": "588465463ff0368cdac3ad8261f80deee557a56251e6d958797e33970eff384f",
    "bytes": 12480,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/generate_selected_model_calibration.js": {
    "sha256": "fa3cc83c6dc7eecf2e895cf64057553b8c5bab11060c5a03216a3f6e0f7867a2",
    "bytes": 4503,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/render_preview.js": {
    "sha256": "ab2e734e61de00baff84a55b89b4a10d3f8a1be75e400a4390d019a5623c2c3f",
    "bytes": 4303,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/render_readme.js": {
    "sha256": "d3e5cdb325449be60040186bd2bd7a4a7fb76ba191d2453a55e8641a98a3be2a",
    "bytes": 11534,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/report_utils.js": {
    "sha256": "ff2bde30739ffcf0a9baf16e8bc8bd944030a441525e7b1332e0c8349923fcfb",
    "bytes": 1771,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/revise_readme_figures.js": {
    "sha256": "6bf465a50de2dcc751344edbf6cf3777a8c3afb2d6552c456038968e0d144435",
    "bytes": 7801,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/validate_report_presentation.js": {
    "sha256": "b79f2ba4bb489cfa6983e11596dfc72e5c4cd6cdf6262c491f2cbb685a6862ad",
    "bytes": 4322,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/verify_readme_revision.js": {
    "sha256": "73448dbe90b5239749112fe7b1d7a3b23fb57962c18432e91f54d0b56eee8d17",
    "bytes": 7095,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/verify_resnet_followup.js": {
    "sha256": "4a6107744520cda8ed9d37f1eb9a0cb28570cda91469b8e5197a6dee50026880",
    "bytes": 8950,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "report/write_report.ps1": {
    "sha256": "4e9d62316ac9a2bb1f01be2df630ff7cd1fad0e1998d4d126d2fdef61b619b66",
    "bytes": 11001,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "requirements.txt": {
    "sha256": "a3b4c2c1e7e7f852e067d7454a22327fbba34cb213abe0d3ccc4dcd22363977b",
    "bytes": 83,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "resnet_followup.py": {
    "sha256": "50071435f77a5c41cc48a37acc6fc0dd021cd0a8f80af64d2c8f7112330ea721",
    "bytes": 39708,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/__init__.py": {
    "sha256": "95cd33d80e21a62eeef762de25138cb7514de0722b92364d0463cb559ee30fdc",
    "bytes": 86,
    "newline": "lf",
    "terminal_newlines": 2,
    "bom": false
  },
  "shrine_classifier/analysis.py": {
    "sha256": "b7e9bc7f8a22d2c61e530e4124bcbbcc427c2f8ca3bfd8a73d1bcc44477b8baa",
    "bytes": 8276,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/baseline.py": {
    "sha256": "30285775aeeeb260ce43e96cb87fb80c026764086a1f702ad57c6aeb3b3c6d72",
    "bytes": 3920,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/data.py": {
    "sha256": "896e2606a668cbf471e2618490380dece777a52f5419e6fb6af9471dbd96aadb",
    "bytes": 24108,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/experiments.py": {
    "sha256": "03e880638a95ad8132edbdd01cd1bb08859c786c6f8551b939d8a526e14ca74c",
    "bytes": 21318,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/models.py": {
    "sha256": "1694f6e237106e3ae55b7d2d1f57051e551b1ed476aac0ff584a7082ac229a2b",
    "bytes": 2290,
    "newline": "lf",
    "terminal_newlines": 2,
    "bom": false
  },
  "shrine_classifier/tracking.py": {
    "sha256": "192eab0fe509cc49a23e80fd164874cef10df789afc24bc4e0664f701a6c825b",
    "bytes": 4604,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/training.py": {
    "sha256": "178ddc5b4fb7b11c7d105c38152700c6bcad2332bce7090108a63360e68bc5fe",
    "bytes": 21111,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier/visualization.py": {
    "sha256": "e27190fea77a8c8a746126d93e84d500c1de23dc03a2e61dec36831ce637da4d",
    "bytes": 10589,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/__init__.py": {
    "sha256": "c06878382aa6f3219df3180ccf9414880ce2df0fad87d348a88ecc8c0196623b",
    "bytes": 110,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/confidence_analysis.py": {
    "sha256": "58499c07b229c75650d053051be833521feee971dc35de9f3f89de929aef78f0",
    "bytes": 19453,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/diagnostics_analysis.py": {
    "sha256": "eebb2072438031796354795a819affeb89d2e9eda95e540ff5f2a281b5956dd7",
    "bytes": 19342,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/extension_experiments.py": {
    "sha256": "ed46b85d1126823b95de1fbc05a4a7864c5333ceb18eb9070372d50581db5695",
    "bytes": 30419,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/extension_report.py": {
    "sha256": "7ad360dc410e5bc99a15b76f9ad813a69261b2ec1350fd314d3bba794a0913d5",
    "bytes": 10113,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "shrine_classifier_ext/finetuning_modes.py": {
    "sha256": "e94b61939ae95a00efe1f94449e7be3f7ba83cd7cd3a676964ef5fa4cebe3d77",
    "bytes": 13472,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_analysis.py": {
    "sha256": "665f652a8a0d4ab70d2dd5dce51f74f6b84b88f5e5209566d63a08c04d824061",
    "bytes": 2340,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_data.py": {
    "sha256": "fbc7815d919c45148fb74b39f4e9f1333c5771358baa306735c266a42a87d03e",
    "bytes": 2335,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_extensions.py": {
    "sha256": "a08a5e013e35e20810faed05488b019c1c94153c2ceb3114735296e54d01f39f",
    "bytes": 23372,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_mode_protocol_v2.py": {
    "sha256": "89fb770611d7ec0c14494fb5703d1a8c0c61ee9fd3723d9ae6fc2a5a92f10b1a",
    "bytes": 5953,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_pipeline_smoke.py": {
    "sha256": "fd1c4e8cbb6302eef88dc9b96284fa4c590415aaf332d944140dd1184e195fb6",
    "bytes": 1099,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  },
  "tests/test_retired_protocol.py": {
    "sha256": "3e3671c26c995979c5c4bf2c75e8268076a7466c5249e739d734b815db1cd159",
    "bytes": 1489,
    "newline": "lf",
    "terminal_newlines": 1,
    "bom": false
  }
}''')
MATERIALIZED = {}

@register_cell_magic
def project_source(line, cell):
    relative = line.strip()
    if relative not in BUNDLE_MANIFEST:
        raise ValueError("Unknown bundled source: " + relative)
    spec = BUNDLE_MANIFEST[relative]
    target = (PROJECT_ROOT / relative).resolve()
    target.relative_to(PROJECT_ROOT)
    normalized = cell.replace("\r\n", "\n").replace("\r", "\n").rstrip("\n")
    normalized += "\n" * spec["terminal_newlines"]
    restored = normalized.replace("\n", "\r\n") if spec["newline"] == "crlf" else normalized
    payload = (b"\xef\xbb\xbf" if spec["bom"] else b"") + restored.encode("utf-8")
    digest = hashlib.sha256(payload).hexdigest()
    if digest != spec["sha256"] and not ALLOW_SOURCE_CHANGES:
        raise ValueError("Notebook source changed: " + relative + "; review changes before enabling ALLOW_SOURCE_CHANGES")
    if target.exists():
        existing = target.read_bytes()
        if existing != payload and not ALLOW_SOURCE_CHANGES:
            raise ValueError("Local source differs: " + relative + "; preserve local edits before replacing")
        if existing == payload:
            MATERIALIZED[relative] = {"sha256": digest, "state": "unchanged"}
            return
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".notebook-tmp")
    temporary.write_bytes(payload)
    temporary.replace(target)
    MATERIALIZED[relative] = {"sha256": digest, "state": "written"}
    print("Prepared", relative)


## 2. Data, models, training และ metrics — source code

โค้ดแกนกลาง: data.py ตรวจ/เตรียมภาพและ split; models.py สร้าง pretrained models; training.py ดูแล DataLoader, weighted loss, checkpoint และ early stopping; analysis.py คำนวณ metrics/Grad-CAM; tracking.py บันทึกผล โค้ด experiments/visualization รุ่นเดิมยังรวมไว้เพื่อรักษา imports และ fingerprint แต่ workflow ปัจจุบันใช้ pipeline_modes


In [ ]:
%%project_source shrine_classifier/__init__.py
"""Reproducible Thai shrine image-classification pipeline."""

__version__ = "0.1.0"


In [ ]:
%%project_source shrine_classifier/analysis.py
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np
from scipy.stats import ttest_rel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

from .data import CLASSES


def classification_metrics(y_true, y_pred) -> dict[str, Any]:
    p, r, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(CLASSES))), zero_division=0)
    report = classification_report(y_true, y_pred, labels=list(range(len(CLASSES))),
                                   target_names=CLASSES, output_dict=True, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_precision": float(np.mean(p)), "macro_recall": float(np.mean(r)),
            "macro_f1": float(np.mean(f1)),
            "per_class": {name: {"precision": float(p[i]), "recall": float(r[i]),
                                  "f1": float(f1[i]), "support": int(support[i])}
                          for i, name in enumerate(CLASSES)},
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES)))).tolist(),
            "classification_report": report}


def summarize_runs(rows: list[dict[str, Any]], metrics: tuple[str, ...] = ("test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1")) -> list[dict[str, Any]]:
    summaries = []
    architectures = list(dict.fromkeys(r["architecture"] for r in rows))
    for arch in architectures:
        subset = [r for r in rows if r["architecture"] == arch and r.get("status", "completed") == "completed"]
        result: dict[str, Any] = {"architecture": arch, "completed_runs": len(subset)}
        for metric in metrics:
            values = [float(r[metric]) for r in subset if r.get(metric) is not None]
            result[metric + "_mean"] = float(np.mean(values)) if values else None
            result[metric + "_sd"] = float(np.std(values, ddof=1)) if len(values) >= 2 else None
        summaries.append(result)
    return summaries


def paired_holm_tests(rows: list[dict[str, Any]], reference_architecture: str,
                      *, metrics: tuple[str, ...] = ("test_acc", "test_macro_f1"), alpha: float = .05,
                      architecture_order: tuple[str, ...] | list[str] | None = None) -> list[dict[str, Any]]:
    by_arch_seed: dict[str, dict[int, dict[str, Any]]] = {}
    for row in rows:
        if row.get("status", "completed") != "completed" or row.get("seed") is None:
            continue
        by_arch_seed.setdefault(row["architecture"], {})[int(row["seed"])] = row
    if reference_architecture not in by_arch_seed:
        raise ValueError(f"Reference architecture {reference_architecture!r} has no completed runs")
    records = []
    for arch, seed_map in by_arch_seed.items():
        if arch == reference_architecture:
            continue
        shared = sorted(set(seed_map) & set(by_arch_seed[reference_architecture]))
        for metric in metrics:
            if not shared or any(metric not in seed_map[s] or metric not in by_arch_seed[reference_architecture][s] for s in shared):
                records.append({"reference": reference_architecture, "competitor": arch, "metric": metric,
                                "paired_n": len(shared), "mean_difference": None, "t_statistic": None,
                                "p_value": None, "status": "incomplete_pairs"})
                continue
            diffs = np.array([float(by_arch_seed[reference_architecture][s][metric]) - float(seed_map[s][metric]) for s in shared])
            if not np.all(np.isfinite(diffs)) or len(diffs) < 2 or np.std(diffs, ddof=1) == 0:
                records.append({"reference": reference_architecture, "competitor": arch, "metric": metric,
                                "paired_n": len(shared), "mean_difference": float(np.mean(diffs)) if np.all(np.isfinite(diffs)) else None,
                                "t_statistic": None, "p_value": None, "status": "undefined_zero_variance_or_nonfinite"})
                continue
            test = ttest_rel([by_arch_seed[reference_architecture][s][metric] for s in shared],
                             [seed_map[s][metric] for s in shared], alternative="two-sided")
            records.append({"reference": reference_architecture, "competitor": arch, "metric": metric,
                            "paired_n": len(shared), "mean_difference": float(np.mean(diffs)),
                            "t_statistic": float(test.statistic), "p_value": float(test.pvalue), "status": "ok"})
    valid = sorted((r for r in records if r["p_value"] is not None), key=lambda r: r["p_value"])
    remaining = len(metrics) * max(0, len(architecture_order or list(by_arch)) - 1)
    running_max = 0.0
    for row in valid:
        running_max = max(running_max, row["p_value"] * remaining)
        row["p_holm"] = min(1.0, running_max)
        row["significant_holm"] = bool(row["p_holm"] < alpha)
        remaining -= 1
    for row in records:
        row.setdefault("p_holm", None)
        row.setdefault("significant_holm", None)
        row["alpha"] = alpha
    return records


def save_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def save_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(k for row in rows for k in row)) or ["status"]
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def save_confusion_matrix(path: Path, values: list[list[int]]) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    image = ax.imshow(np.asarray(values), cmap="Blues")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
    for i, row in enumerate(values):
        for j, value in enumerate(row):
            ax.text(j, i, str(value), ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax); fig.tight_layout(); path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160); plt.close(fig)


def gradcam(model, architecture: str, tensor, class_index: int):
    """Compute a normalized spatial Grad-CAM map; caller aligns it with content_bbox."""
    import torch
    from .models import cam_layer
    activations: list[Any] = []
    gradients: list[Any] = []
    layer = cam_layer(model, architecture)
    forward = layer.register_forward_hook(lambda _m, _x, output: activations.append(output))
    try:
        model.zero_grad(set_to_none=True)
        # Frozen modes have no trainable backbone path. Requiring input
        # gradients keeps the CAM target activations differentiable anyway.
        if not tensor.requires_grad:
            tensor = tensor.detach().requires_grad_(True)
        logits = model(tensor)
        score = logits[:, class_index].sum()
        if not score.requires_grad:
            raise RuntimeError("Grad-CAM target has no gradient")
        activations[-1].register_hook(lambda grad: gradients.append(grad))
        score.backward()
        if not gradients:
            raise RuntimeError("CAM layer produced no gradient")
        activation = activations[-1].detach()
        weights = gradients[-1].detach().mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * activation).sum(dim=1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam - cam.amin(dim=(2, 3), keepdim=True)
        cam = cam / cam.amax(dim=(2, 3), keepdim=True).clamp_min(1e-8)
        result = cam[:, 0]
        if not torch.isfinite(result).all():
            raise RuntimeError("Grad-CAM contains non-finite values")
        return result
    finally:
        forward.remove()


In [ ]:
%%project_source shrine_classifier/baseline.py
from __future__ import annotations

import csv
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import DataLoader
from torchvision.models import ResNet50_Weights, resnet50

from .data import CLASSES
from .training import ImageDataset, load_split
from .analysis import save_csv, save_json


def imagenet_predict(image: Image.Image, model=None, weights=None) -> tuple[str, float]:
    if weights is None:
        weights = ResNet50_Weights.DEFAULT
    if model is None:
        model = resnet50(weights=weights)
        model.eval()
    tensor = weights.transforms()(image.convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        probs = model(tensor).softmax(dim=1)[0]
    index = int(probs.argmax())
    return str(weights.meta["categories"][index]), float(probs[index])


def run_baseline(args, out: Path) -> dict:
    rows = [r for r in load_split(out / "splits.csv") if r["split"] == "train"]
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights).to(args.device if args.device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()
    dataset = ImageDataset(args.data_root, rows, training=False, size=args.input_size)
    loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=args.num_workers)
    results = []
    with torch.no_grad():
        for tensors, labels, ids, _ in loader:
            logits = model(tensors.to(next(model.parameters()).device)).softmax(dim=1)
            for i, image_id in enumerate(ids):
                pred = int(logits[i].argmax())
                row = next(x for x in rows if x["image_id"] == image_id)
                results.append({"image_id": image_id, "true_custom_class": row["label"],
                                "imagenet_top1": weights.meta["categories"][pred],
                                "imagenet_confidence": float(logits[i, pred])})
    outdir = out / "baseline"
    save_csv(outdir / "pretrained_predictions.csv", results)
    save_json(outdir / "baseline_metadata.json", {"model": "resnet50", "weights": "ResNet50_Weights.DEFAULT",
                                                   "interpretation": "qualitative 1000-class label-space mismatch only; no shrine-class metrics"})
    _grid(args.data_root, rows, {r["image_id"]: r for r in results}, outdir / "pretrained_examples.png")
    try:
        import wandb
        run = wandb.init(project=args.wandb_project, group="baseline", name="baseline_imagenet_resnet50",
                         entity=args.wandb_entity, mode=args.wandb_mode, reinit=True)
        run.log({"baseline_examples": wandb.Image(str(outdir / "pretrained_examples.png"))})
        run.finish()
    except Exception as exc:
        save_json(outdir / "wandb_status.json", {"mode": args.wandb_mode, "error": f"{type(exc).__name__}: {exc}", "local_saved": True})
    return {"images": len(results), "output": str(outdir)}


def _grid(root: Path, rows, results, path: Path):
    from .data import open_rgb
    path.parent.mkdir(parents=True, exist_ok=True)
    canvas = Image.new("RGB", (4 * 330, 3 * 260), "white")
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", 14)
    except Exception:
        font = ImageFont.load_default()
    selected = []
    for cls in CLASSES:
        selected.extend([r for r in rows if r["label"] == cls][:3])
    for i, row in enumerate(selected[:12]):
        x, y = (i % 4) * 330, (i // 4) * 260
        im = open_rgb(root / row["path"]); im.thumbnail((310, 205), Image.Resampling.LANCZOS)
        canvas.paste(im, (x + 8, y + 4))
        pred = results[row["image_id"]]
        draw.text((x + 6, y + 212), f"true: {row['label']}", fill="black", font=font)
        draw.text((x + 6, y + 231), f"ImageNet: {pred['imagenet_top1']} ({pred['imagenet_confidence']:.2f})", fill="black", font=font)
    canvas.save(path)


In [ ]:
%%project_source shrine_classifier/data.py
from __future__ import annotations

import csv
import hashlib
import json
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any
from urllib.parse import urlsplit

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps
from sklearn.model_selection import GroupShuffleSplit

CLASSES = ["san_phraphrom", "san_phraphum", "san_tayay", "tee_ju_eia"]
CLASS_TH = ["ศาลพระพรหม", "ศาลพระภูมิ", "ศาลตายาย", "ศาลตี่จู้เอี๊ยะ"]
NEUTRAL = (124, 116, 104)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".gif"}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def open_rgb(path: Path) -> Image.Image:
    """Open one image/frame, orient it, and composite transparency neutrally."""
    with Image.open(path) as image:
        image.seek(0)
        oriented = ImageOps.exif_transpose(image)
        if oriented.mode == "CMYK":
            converted = oriented.convert("RGB")
        elif "A" in oriented.getbands() or "transparency" in oriented.info:
            rgba = oriented.convert("RGBA")
            background = Image.new("RGBA", rgba.size, NEUTRAL + (255,))
            converted = Image.alpha_composite(background, rgba).convert("RGB")
        else:
            converted = oriented.convert("RGB")
        converted.load()
        return converted.copy()


def letterbox(image: Image.Image, size: int = 224) -> tuple[Image.Image, tuple[int, int, int, int]]:
    image = image.convert("RGB")
    scale = min(size / image.width, size / image.height)
    width = max(1, round(image.width * scale))
    height = max(1, round(image.height * scale))
    resized = image.resize((width, height), Image.Resampling.LANCZOS)
    x = (size - width) // 2
    y = (size - height) // 2
    canvas = Image.new("RGB", (size, size), NEUTRAL)
    canvas.paste(resized, (x, y))
    return canvas, (x, y, x + width, y + height)


def _phash(image: Image.Image) -> int:
    from scipy.fft import dctn

    gray = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:8, :8].flatten()[1:]
    median = float(np.median(low))
    value = 0
    for coefficient in low:
        value = (value << 1) | int(coefficient > median)
    return value


def _distance(left: int, right: int) -> int:
    return (left ^ right).bit_count()


def _load_metadata(root: Path, *, required: bool) -> tuple[dict[tuple[str, str], dict[str, str]], dict[str, Any]]:
    path = root / "metadata_cleaned.csv"
    if not path.exists():
        if not required:
            return {}, {"path": None, "rows": 0, "rows_with_url": 0}
        raise FileNotFoundError(f"Required provenance file not found: {path}")
    by_id: dict[tuple[str, str], dict[str, str]] = {}
    rows = 0
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        required = {"filename", "category", "source_url"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"{path} must contain columns {sorted(required)}")
        for row in reader:
            rows += 1
            key = (row["category"].strip(), Path(row["filename"].strip()).stem)
            if key in by_id:
                raise ValueError(f"Duplicate metadata image ID: {key}")
            by_id[key] = {k: (v or "").strip() for k, v in row.items() if k}
    return by_id, {"path": str(path), "sha256": sha256_file(path), "rows": rows,
                   "rows_with_url": sum(bool(r.get("source_url")) for r in by_id.values())}


class _UnionFind:
    def __init__(self) -> None:
        self.parent: dict[str, str] = {}

    def find(self, x: str) -> str:
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


def audit_dataset(root: Path, out: Path, *, near_threshold: int = 6,
                  require_metadata: bool = True) -> dict[str, Any]:
    root, out = root.resolve(), out.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset root not found: {root}")
    out.mkdir(parents=True, exist_ok=True)
    metadata, metadata_summary = _load_metadata(root, required=require_metadata)
    paths = sorted((p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS),
                   key=lambda p: p.relative_to(root).as_posix().casefold())
    rows: list[dict[str, Any]] = []
    errors: list[dict[str, str]] = []
    unexpected = [p.relative_to(root).as_posix() for p in root.rglob("*")
                  if p.is_file() and p.suffix.lower() not in EXTENSIONS and p.name != "metadata_cleaned.csv"]
    for path in paths:
        rel = path.relative_to(root)
        label = rel.parts[0] if rel.parts else ""
        if label not in CLASSES:
            errors.append({"path": rel.as_posix(), "error": f"unknown class directory: {label}"})
            continue
        key = (label, path.stem)
        try:
            image = open_rgb(path)
            with Image.open(path) as raw_image:
                original_mode = raw_image.mode
                original_format = raw_image.format or "unknown"
                transparency = "A" in raw_image.getbands() or "transparency" in raw_image.info
                frame_count = int(getattr(raw_image, "n_frames", 1))
            raw = sha256_file(path)
            pixels = hashlib.sha256(f"{image.width}x{image.height}:".encode() + image.tobytes()).hexdigest()
            meta = metadata.get(key, {})
            if meta and meta.get("category") != label:
                raise ValueError(f"metadata class {meta.get('category')} conflicts with path class {label}")
            url = meta.get("source_url", "")
            domain = urlsplit(url).hostname or "" if url else ""
            pixels_array = np.asarray(image, dtype=np.uint8)
            nearblack = np.all(pixels_array <= 10, axis=2)
            band_x, band_y = max(1, image.width // 50), max(1, image.height // 50)
            edge = np.zeros(nearblack.shape, dtype=bool)
            edge[:band_y] = edge[-band_y:] = True
            edge[:, :band_x] = edge[:, -band_x:] = True
            rows.append({
                "image_id": f"{label}/{path.stem}", "path": rel.as_posix(), "label": label,
                "label_index": CLASSES.index(label), "width": image.width, "height": image.height,
                "mode": original_mode, "format": original_format,
                "has_alpha": int(transparency), "animated_frames": frame_count,
                "byte_sha256": raw, "pixel_sha256": pixels, "source_url": url, "source_domain": domain,
                "phash": f"{_phash(image):016x}", "file_bytes": path.stat().st_size,
                "nearblack_fraction": float(nearblack.mean()),
                "edge_nearblack_fraction": float(nearblack[edge].mean()),
            })
        except Exception as exc:
            errors.append({"path": rel.as_posix(), "error": f"{type(exc).__name__}: {exc}"})

    # Detect metadata/image mismatches without treating .jpg/.png as different source IDs.
    actual_ids = {(r["label"], Path(r["path"]).stem) for r in rows}
    metadata_only = sorted(f"{k[0]}/{k[1]}" for k in set(metadata) - actual_ids)
    image_only = sorted(f"{k[0]}/{k[1]}" for k in actual_ids - set(metadata))
    id_collisions: dict[str, list[str]] = defaultdict(list)
    for row in rows:
        id_collisions[row["image_id"]].append(row["path"])
    collisions = {k: v for k, v in id_collisions.items() if len(v) > 1}

    exact: dict[str, list[dict[str, Any]]] = defaultdict(list)
    url_groups: dict[str, list[str]] = defaultdict(list)
    for row in rows:
        exact[row["pixel_sha256"]].append(row)
        if row["source_url"]:
            url_groups[row["source_url"]].append(row["image_id"])
    exact_groups = [v for v in exact.values() if len(v) > 1]
    repeated_urls = [v for v in url_groups.values() if len(v) > 1]
    if require_metadata and (metadata_only or image_only or collisions):
        errors.append({"path": "metadata_cleaned.csv", "error":
                       f"metadata join mismatch: metadata_only={len(metadata_only)}, image_only={len(image_only)}, ID collisions={len(collisions)}"})
    missing_classes = {c: sum(r["label"] == c for r in rows) for c in CLASSES}
    for c, count in missing_classes.items():
        if count == 0:
            errors.append({"path": str(root), "error": f"missing/empty class: {c}"})

    hashes = [(r, int(r["phash"], 16)) for r in rows]
    candidates: list[dict[str, Any]] = []
    for i, (a, ha) in enumerate(hashes):
        for b, hb in hashes[i + 1:]:
            distance = _distance(ha, hb)
            if distance <= near_threshold:
                candidates.append({"path_a": a["path"], "path_b": b["path"],
                                   "id_a": a["image_id"], "id_b": b["image_id"],
                                   "cross_class": int(a["label"] != b["label"]),
                                   "hamming_distance": distance, "review_status": "pending"})
    candidates.sort(key=lambda r: (r["hamming_distance"], r["path_a"], r["path_b"]))

    review_path = out / "near_duplicate_review_decisions.csv"
    review_by_pair: dict[tuple[str, str], dict[str, str]] = {}
    stale_review_rows = []
    if review_path.exists():
        with review_path.open("r", encoding="utf-8-sig", newline="") as stream:
            for reviewed in csv.DictReader(stream):
                pair = tuple(sorted((reviewed.get("id_a", ""), reviewed.get("id_b", ""))))
                if pair not in {tuple(sorted((r["id_a"], r["id_b"]))) for r in candidates}:
                    stale_review_rows.append(reviewed)
                    continue
                review_by_pair[pair] = reviewed
        for candidate in candidates:
            prior = review_by_pair.get(tuple(sorted((candidate["id_a"], candidate["id_b"]))))
            if prior:
                candidate["review_status"] = prior.get("review_status", "pending")

    _write_csv(out / "image_properties.csv", rows)
    _write_csv(out / "near_duplicate_candidates.csv", candidates)
    _write_csv(out / "duplicate_groups.csv", [
        {"group_type": "exact_pixels", "image_ids": ";".join(r["image_id"] for r in group),
         "paths": ";".join(r["path"] for r in group), "review_status": "confirmed_exact"}
        for group in exact_groups
    ] + [
        {"group_type": "source_url", "image_ids": ";".join(ids), "paths": "", "review_status": "confirmed_url"}
        for ids in repeated_urls
    ] + [
        {"group_type": "visual_near_duplicate", "image_ids": ";".join((r["id_a"], r["id_b"])),
         "paths": ";".join((r["path_a"], r["path_b"])), "review_status": "confirmed_same_source"}
        for r in candidates if r["review_status"] == "confirmed_same_source"
    ])
    _write_csv(out / "metadata_join_issues.csv", [
        *({"issue": "metadata_only", "image_id": x} for x in metadata_only),
        *({"issue": "image_only", "image_id": x} for x in image_only),
    ])
    _write_csv(out / "audit_errors.csv", errors)
    summary = {
        "dataset_root": str(root), "audit_time_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
        "class_order": CLASSES, "num_images": len(rows), "num_read_errors_or_identity_issues": len(errors),
        "class_counts": missing_classes, "mode_counts": dict(Counter(r["mode"] for r in rows)),
        "format_counts": dict(Counter(r["format"] for r in rows)),
        "distinct_dimensions": len({(r["width"], r["height"]) for r in rows}),
        "dimension_range": {"width": [min((r["width"] for r in rows), default=0), max((r["width"] for r in rows), default=0)],
                            "height": [min((r["height"] for r in rows), default=0), max((r["height"] for r in rows), default=0)]},
        "alpha_or_transparency_count": sum(r["has_alpha"] for r in rows),
        "all_or_nearly_all_black_files": [r["path"] for r in rows if r["nearblack_fraction"] >= 0.98],
        "mean_nearblack_fraction_by_class": {c: float(np.mean([r["nearblack_fraction"] for r in rows if r["label"] == c])) for c in CLASSES},
        "mean_edge_nearblack_fraction_by_class": {c: float(np.mean([r["edge_nearblack_fraction"] for r in rows if r["label"] == c])) for c in CLASSES},
        "animated_file_count": sum(r["animated_frames"] > 1 for r in rows),
        "exact_decoded_pixel_duplicate_groups": [[r["image_id"] for r in g] for g in exact_groups],
        "repeated_source_url_groups": repeated_urls, "num_near_duplicate_candidates": len(candidates),
        "near_duplicate_review": {"file": review_path.name if review_path.exists() else None,
                                  "confirmed": sum(r["review_status"] == "confirmed_same_source" for r in candidates),
                                  "rejected": sum(r["review_status"] == "rejected_different_image" for r in candidates),
                                  "pending": sum(r["review_status"] == "pending" for r in candidates),
                                  "stale_rows": len(stale_review_rows)},
        "phash_hamming_threshold": near_threshold,
        "metadata": {**metadata_summary, "status": "checked" if require_metadata else "not_supplied_for_this_version"},
        "source_domain_counts_by_class": {c: dict(Counter(r["source_domain"] or "unknown" for r in rows if r["label"] == c)) for c in CLASSES},
        "metadata_only_ids": metadata_only, "image_only_ids": image_only if require_metadata else [],
        "unexpected_files": unexpected, "errors": errors,
        "limitations": ["Perceptual hash rows are candidates, not confirmed duplicates.",
                        "Visual/source-site shortcut review remains a human review task."],
    }
    (out / "dataset_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    _save_distribution(out / "class_distribution.png", missing_classes)
    _save_sample_grid(root, rows, out / "sample_grid.png")
    _save_candidate_grid(root, candidates[:100], out / "near_duplicate_review.png")
    _save_preprocessing_grid(root, rows, out / "preprocessing_examples.png")
    return summary


def _write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    fields = list(rows[0]) if rows else ["issue", "image_id"]
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def _font(size: int = 18):
    try:
        return ImageFont.truetype("C:/Windows/Fonts/arial.ttf", size)
    except Exception:
        return ImageFont.load_default()


def _save_distribution(path: Path, counts: dict[str, int]) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(CLASSES, [counts[c] for c in CLASSES], color=["#4178be", "#47a881", "#f0ad4e", "#d95d6a"])
    ax.set_ylabel("Verified image count")
    ax.set_title("Thai Shrine Dataset - cleaned source")
    ax.grid(axis="y", alpha=.25)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def _save_sample_grid(root: Path, rows: list[dict[str, Any]], path: Path) -> None:
    cell_w, thumb = 260, (242, 170)
    canvas = Image.new("RGB", (4 * cell_w, 690), "white")
    draw = ImageDraw.Draw(canvas)
    for col, label in enumerate(CLASSES):
        draw.text((col * cell_w + 8, 4), label, fill="black", font=_font(16))
        chosen = [r for r in rows if r["label"] == label][:3]
        for ridx, row in enumerate(chosen):
            image = open_rgb(root / row["path"])
            image.thumbnail(thumb, Image.Resampling.LANCZOS)
            x = col * cell_w + (cell_w - image.width) // 2
            y = 30 + ridx * 215
            canvas.paste(image, (x, y))
            draw.text((col * cell_w + 5, y + 174), row["image_id"], fill="black", font=_font(11))
    canvas.save(path)


def _save_preprocessing_grid(root: Path, rows: list[dict[str, Any]], path: Path) -> None:
    canvas = Image.new("RGB", (4 * 440, 285), "white")
    draw = ImageDraw.Draw(canvas)
    for idx, label in enumerate(CLASSES):
        row = next((r for r in rows if r["label"] == label), None)
        if not row:
            continue
        original = open_rgb(root / row["path"])
        original.thumbnail((205, 220), Image.Resampling.LANCZOS)
        fitted, bbox = letterbox(open_rgb(root / row["path"]), 224)
        x = idx * 440
        canvas.paste(original, (x + 4, 30))
        canvas.paste(fitted, (x + 218, 30))
        draw.text((x + 4, 5), f"{label} - original", fill="black", font=_font(14))
        draw.text((x + 218, 5), f"neutral letterbox, bbox={bbox}", fill="black", font=_font(12))
        draw.text((x + 4, 255), row["path"], fill="black", font=_font(12))
    canvas.save(path)


def _save_candidate_grid(root: Path, candidates: list[dict[str, Any]], path: Path) -> None:
    cell_w, cell_h, thumb = 420, 150, (180, 115)
    n = max(1, len(candidates))
    canvas = Image.new("RGB", (2 * cell_w, ((n + 1) // 2) * cell_h), "white")
    draw = ImageDraw.Draw(canvas)
    for idx, item in enumerate(candidates):
        x, y = (idx % 2) * cell_w, (idx // 2) * cell_h
        for offset, key in ((0, "path_a"), (200, "path_b")):
            img = open_rgb(root / item[key])
            img.thumbnail(thumb, Image.Resampling.LANCZOS)
            canvas.paste(img, (x + offset, y))
        draw.text((x, y + 118), f"d={item['hamming_distance']} {item['id_a']} <> {item['id_b']}", fill="black", font=_font(11))
    canvas.save(path)


def groups_from_audit(rows: list[dict[str, str]], reviewed: Path | None = None) -> dict[str, str]:
    uf = _UnionFind()
    by_pixels: dict[str, list[str]] = defaultdict(list)
    by_url: dict[str, list[str]] = defaultdict(list)
    labels = {r["image_id"]: r["label"] for r in rows}
    for r in rows:
        uf.find(r["image_id"])
        by_pixels[r["pixel_sha256"]].append(r["image_id"])
        if r.get("source_url"):
            by_url[r["source_url"]].append(r["image_id"])
    for collection in (by_pixels.values(), by_url.values()):
        for ids in collection:
            for other in ids[1:]:
                uf.union(ids[0], other)
    if reviewed and reviewed.exists():
        with reviewed.open("r", encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                if row.get("review_status") == "confirmed_same_source":
                    a, b = row["id_a"], row["id_b"]
                    if a not in labels or b not in labels:
                        raise ValueError(f"Duplicate review references unknown IDs: {a}, {b}")
                    uf.union(a, b)
    members: dict[str, list[str]] = defaultdict(list)
    for image_id in labels:
        members[uf.find(image_id)].append(image_id)
    conflict = [ids for ids in members.values() if len({labels[i] for i in ids}) > 1]
    if conflict:
        raise ValueError(f"Confirmed duplicate group has conflicting labels: {conflict[0]}")
    return {image_id: min(ids) for ids in members.values() for image_id in ids}


def create_split(data_root: Path, audit_dir: Path, out_csv: Path, *, split_seed: int = 42,
                 review_csv: Path | None = None) -> dict[str, Any]:
    if out_csv.exists():
        raise FileExistsError(f"Refusing to overwrite canonical split {out_csv}; inspect it or choose a new --out-dir")
    props = _read_csv(audit_dir / "image_properties.csv")
    if not props:
        raise ValueError("Run --stage audit before --stage prep")
    groups = groups_from_audit(props, review_csv)
    ids = np.array([r["image_id"] for r in props])
    y = np.array([r["label"] for r in props])
    g = np.array([groups[i] for i in ids])
    classes = np.array(CLASSES)
    # Deterministic repeated grouped holdouts; choose the candidate closest to all targets.
    outer = GroupShuffleSplit(n_splits=256, test_size=0.15, random_state=split_seed)
    best: tuple[float, np.ndarray, np.ndarray, np.ndarray] | None = None
    for temp_idx, test_idx in outer.split(ids, y, groups=g):
        inner = GroupShuffleSplit(n_splits=12, test_size=0.15 / 0.85, random_state=split_seed)
        for train_rel, val_rel in inner.split(ids[temp_idx], y[temp_idx], groups=g[temp_idx]):
            train_idx, val_idx = temp_idx[train_rel], temp_idx[val_rel]
            splits = (train_idx, val_idx, test_idx)
            if any(set(y[ix]) != set(CLASSES) for ix in splits):
                continue
            if any(set(g[splits[i]]) & set(g[splits[j]]) for i, j in ((0, 1), (0, 2), (1, 2))):
                continue
            score = 0.0
            for ix, target in zip(splits, (0.70, 0.15, 0.15)):
                score += ((len(ix) / len(ids) - target) / target) ** 2
                for cls in CLASSES:
                    observed = np.sum(y[ix] == cls) / np.sum(y == cls)
                    score += 0.35 * ((observed - target) / target) ** 2
            if best is None or score < best[0]:
                best = (score, train_idx, val_idx, test_idx)
    if best is None:
        raise ValueError("Could not build a group-stratified split with all four classes; review duplicate groups or dataset labels")
    _, train_idx, val_idx, test_idx = best
    assignment = {ids[i]: split for split, ix in (("train", train_idx), ("validation", val_idx), ("test", test_idx)) for i in ix}
    lookup = {r["image_id"]: r for r in props}
    records = [{"image_id": image_id, "path": lookup[image_id]["path"], "label": lookup[image_id]["label"],
                "label_index": int(lookup[image_id]["label_index"]), "group_id": groups[image_id], "split": assignment[image_id]}
               for image_id in ids]
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    _write_csv(out_csv, records)
    manifest_digest = hashlib.sha256("\n".join(f"{r['image_id']}:{r['pixel_sha256']}" for r in props).encode()).hexdigest()
    group_digest = hashlib.sha256("\n".join(f"{k}:{groups[k]}" for k in sorted(groups)).encode()).hexdigest()
    split_digest = hashlib.sha256("\n".join(f"{r['image_id']}:{r['split']}" for r in sorted(records, key=lambda x: x["image_id"])).encode()).hexdigest()
    summary = {"split_seed": split_seed, "method": "256x12 deterministic GroupShuffleSplit candidates; minimize split-size and per-class deviations",
               "target_ratios": {"train": .70, "validation": .15, "test": .15},
               "counts": {name: {"n": sum(r["split"] == name for r in records),
                                 "by_class": {c: sum(r["split"] == name and r["label"] == c for r in records) for c in CLASSES}}
                          for name in ("train", "validation", "test")},
               "group_overlap": False, "num_groups": len(set(groups.values())), "class_order": CLASSES,
               "manifest_sha256": manifest_digest, "group_sha256": group_digest, "split_sha256": split_digest,
               "duplicate_review_sha256": sha256_file(review_csv) if review_csv and review_csv.exists() else None,
               "human_near_duplicate_review": str(review_csv) if review_csv else "not supplied; perceptual candidates remain unconfirmed"}
    meta_path = out_csv.parent / "split_metadata.json"
    meta_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary


def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


In [ ]:
%%project_source shrine_classifier/experiments.py
from __future__ import annotations

import csv
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np

from .analysis import paired_holm_tests, save_csv, save_json, summarize_runs
from .data import CLASSES
from .models import ARCHITECTURES
from .training import fit_run, load_split

SCREEN = {"optimizer": "adamw", "learning_rate": 1e-4, "weight_decay": 1e-4,
          "batch_size": 16, "epochs": 30, "patience": 7, "input_size": 224}
SEARCH_SPACE = {
    "method": "random", "metric": {"name": "best_val_macro_f1", "goal": "maximize"},
    "parameters": {"learning_rate": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-3},
                   "weight_decay": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-2},
                   "optimizer": {"values": ["adamw", "adam", "sgd"]},
                   "batch_size": {"values": [16, 32]}},
}


def default_config(args) -> dict[str, Any]:
    return {**SCREEN, "epochs": int(args.epochs), "input_size": int(args.input_size),
            "black_threshold": int(args.black_threshold)}


def final_seed_list(args) -> list[int]:
    if args.seeds:
        seeds = [int(x.strip()) for x in args.seeds.split(",") if x.strip()]
    else:
        run_count = args.runs or 5
        base = [42, 123, 456, 789, 1024]
        seeds = base[:min(run_count, len(base))]
        rng = np.random.default_rng(args.seed)
        while len(seeds) < run_count:
            candidate = int(rng.integers(10000, 2**31 - 1))
            if candidate not in seeds:
                seeds.append(candidate)
    if not 3 <= len(seeds) <= 10 or len(set(seeds)) != len(seeds):
        raise ValueError("Final run seeds must contain 3-10 distinct integers")
    if args.runs is not None and args.runs != len(seeds):
        raise ValueError(f"--runs={args.runs} disagrees with {len(seeds)} seed values")
    return seeds


def _append_result(path: Path, result: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    current = []
    if path.exists():
        with path.open("r", encoding="utf-8-sig", newline="") as stream:
            current = list(csv.DictReader(stream))
    current.append(result)
    save_csv(path, current)


def run_screen(args, out: Path) -> list[dict[str, Any]]:
    split = out / "splits.csv"
    rows = load_split(split)
    results = []
    for arch in args.architectures:
        try:
            result = fit_run(root=args.data_root, split_csv=split, out=out, architecture=arch,
                             seed=args.seed, config={**default_config(args), "augmentation": True,
                             "use_class_weights": True}, stage="screen", wandb_project=args.wandb_project,
                             wandb_entity=args.wandb_entity, wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers)
        except Exception as exc:
            result = {"architecture": arch, "seed": args.seed, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
        results.append(result)
        _append_result(out / "screening" / "screening_results.csv", result)
    save_json(out / "screening" / "screening_results.json", results)
    return results


def _sample_local_configs(trials: int, seed: int):
    rng = np.random.default_rng(seed)
    optimizers = ("adamw", "adam", "sgd")
    batches = (16, 32)
    for _ in range(trials):
        yield {"learning_rate": float(10 ** rng.uniform(-5, -3)),
               "weight_decay": float(10 ** rng.uniform(-6, -2)),
               "optimizer": str(rng.choice(optimizers)), "batch_size": int(rng.choice(batches))}


def _search_objective(args, out: Path, arch: str, trial_no: int, config: dict[str, Any], active_run=None,
                      tracking_mode: str | None = None) -> dict[str, Any]:
    trial_id = f"sweep_{arch}_trial{trial_no:03d}_{int(time.time())}"
    try:
        result = fit_run(root=args.data_root, split_csv=out / "splits.csv", out=out, architecture=arch,
                         seed=args.seed, config={**default_config(args), **config, "augmentation": True,
                         "use_class_weights": True, "stage": "tune", "trial": trial_no},
                         stage="tuning", wandb_project=args.wandb_project,
                         wandb_entity=args.wandb_entity,
                         wandb_mode=tracking_mode or args.wandb_mode, device=args.device,
                         workers=args.num_workers, active_wandb_run=active_run)
        result.update({"trial": trial_no, "trial_id": trial_id, **config})
        if active_run is not None:
            active_run.summary.update({"best_epoch": result["best_epoch"],
                                       "best_val_acc": result["best_val_acc"],
                                       "best_val_macro_f1": result["best_val_macro_f1"]})
    except Exception as exc:
        result = {"architecture": arch, "trial": trial_no, "trial_id": trial_id, "status": "failed",
                  **config, "error": f"{type(exc).__name__}: {exc}"}
    if active_run is not None:
        try:
            active_run.finish(exit_code=0 if result.get("status") == "completed" else 1)
        except Exception:
            pass
    _append_result(out / "tuning" / "sweep_summary.csv", result)
    return result


def run_tuning(args, out: Path) -> dict[str, Any]:
    load_split(out / "splits.csv")  # no test loader is constructed during search
    tuning_dir = out / "tuning"
    tuning_dir.mkdir(parents=True, exist_ok=True)
    all_results: list[dict[str, Any]] = []
    backends: dict[str, str] = {}
    sweep_meta: dict[str, Any] = {"project": args.wandb_project, "trials_per_architecture": args.trials,
                                  "common_recipe": default_config(args), "search_space": SEARCH_SPACE, "sweeps": {}}
    for ai, arch in enumerate(args.architectures):
        arch_results: list[dict[str, Any]] = []
        sweep_id = None
        wandb_module = None
        if args.wandb_mode == "online":
            try:
                import wandb as wandb_module
                config = {"method": "random", "metric": {"name": "best_val_macro_f1", "goal": "maximize"},
                          "parameters": {**SEARCH_SPACE["parameters"], "architecture": {"value": arch},
                                         "seed": {"value": args.seed}, "stage": {"value": "tune"}}}
                sweep_id = wandb_module.sweep(config, project=args.wandb_project, entity=args.wandb_entity)
                backends[arch] = "wandb_sweep_online"
            except Exception as exc:
                sweep_meta["sweeps"][arch] = {"status": "unavailable", "error": f"{type(exc).__name__}: {exc}"}
                wandb_module, sweep_id = None, None
        if sweep_id and wandb_module:
            trial_count = 0
            def objective():
                nonlocal trial_count
                trial_count += 1
                run = wandb_module.init(project=args.wandb_project, group="sweep",
                                        entity=args.wandb_entity, name=f"sweep_{arch}_trial{trial_count:03d}", reinit=True)
                params = {k: run.config[k] for k in ("learning_rate", "weight_decay", "optimizer", "batch_size")}
                arch_results.append(_search_objective(args, out, arch, trial_count, params, active_run=run, tracking_mode="online"))
            try:
                wandb_module.agent(sweep_id, function=objective, count=args.trials,
                                   project=args.wandb_project, entity=args.wandb_entity)
                sweep_meta["sweeps"][arch] = {"status": "completed", "sweep_id": sweep_id, "trials": trial_count}
            except Exception as exc:
                sweep_meta["sweeps"][arch] = {"status": "interrupted", "sweep_id": sweep_id,
                                              "completed_trials": trial_count, "error": f"{type(exc).__name__}: {exc}"}
            remaining = args.trials - len(arch_results)
            fallback_mode = "offline" if wandb_module is not None else "disabled"
            if remaining > 0:
                backends[arch] += f"+local_random_{fallback_mode}"
                for no, params in enumerate(_sample_local_configs(remaining, args.seed + ai * 1009 + trial_count), trial_count + 1):
                    arch_results.append(_search_objective(args, out, arch, no, params, tracking_mode=fallback_mode))
        else:
            available = False
            try:
                import wandb  # noqa: F401
                available = True
            except Exception:
                pass
            fallback_mode = "offline" if args.wandb_mode == "offline" or (args.wandb_mode == "online" and available) else "disabled"
            backends[arch] = f"local_random_{fallback_mode}"
            for no, params in enumerate(_sample_local_configs(args.trials, args.seed + ai * 1009), 1):
                arch_results.append(_search_objective(args, out, arch, no, params, tracking_mode=fallback_mode))
            if args.wandb_mode == "disabled":
                backends[arch] = "local_random_disabled"
        if not arch_results:
            raise RuntimeError(f"Tuning produced no recorded trials for {arch}")
        all_results.extend(arch_results)
        complete = [r for r in arch_results if r.get("status") == "completed" and r.get("best_val_macro_f1") is not None]
        if not complete:
            raise RuntimeError(f"All {len(arch_results)} tuning trials failed for {arch}")
        arch_results.sort(key=lambda r: (-float(r["best_val_macro_f1"]), -float(r.get("best_val_acc", 0)), float(r.get("best_val_loss", float("inf"))), int(r["trial"])))
        sweep_meta["sweeps"].setdefault(arch, {"status": "completed_locally"})
        sweep_meta["sweeps"][arch]["backend"] = backends[arch]
        sweep_meta["sweeps"][arch]["successful_trials"] = len(complete)
        sweep_meta["sweeps"][arch]["failed_trials"] = len(arch_results) - len(complete)
        best = arch_results[0]
        sweep_meta["sweeps"][arch]["best_trial"] = {k: best.get(k) for k in ("trial", "learning_rate", "weight_decay", "optimizer", "batch_size", "best_val_acc", "best_val_macro_f1", "best_val_loss")}

    # Fixed architecture order is a stable last-resort tie-break; no test scores are present here.
    by_arch = {}
    for arch in args.architectures:
        candidates = [r for r in all_results if r["architecture"] == arch and r.get("status") == "completed"]
        candidates.sort(key=lambda r: (-r["best_val_macro_f1"], -r["best_val_acc"], r["best_val_loss"], r["trial"]))
        best = candidates[0]
        by_arch[arch] = {"architecture": arch, "learning_rate": best["learning_rate"], "weight_decay": best["weight_decay"],
                         "optimizer": best["optimizer"], "batch_size": best["batch_size"],
                         "best_trial": best["trial"], "best_val_macro_f1": best["best_val_macro_f1"],
                         "best_val_acc": best["best_val_acc"], "best_val_loss": best["best_val_loss"]}
    arch_order = {a: i for i, a in enumerate(ARCHITECTURES)}
    reference = sorted(by_arch.values(), key=lambda x: (-x["best_val_macro_f1"], -x["best_val_acc"], x["best_val_loss"], arch_order[x["architecture"]]))[0]["architecture"]
    seeds = final_seed_list(args)
    selection = {"created_utc": datetime.now(timezone.utc).isoformat(), "selection_metric": "best tuned validation macro F1",
                 "reference_architecture": reference, "representative_seed": 42 if 42 in seeds else seeds[0],
                 "architectures": args.architectures, "seeds": seeds, "split_sha256": json.loads((out / "split_metadata.json").read_text(encoding="utf-8"))["split_sha256"],
                 "test_metrics_read": False}
    save_json(tuning_dir / "best_params_by_arch.json", by_arch)
    save_json(out / "selection.json", selection)
    save_json(tuning_dir / "sweep_summary.json", sweep_meta)
    save_csv(tuning_dir / "all_trials.csv", all_results)
    return {"selection": selection, "best_params": by_arch, "sweeps": sweep_meta}


def run_final(args, out: Path) -> dict[str, Any]:
    selection_path, params_path = out / "selection.json", out / "tuning" / "best_params_by_arch.json"
    if not selection_path.exists() or not params_path.exists():
        raise FileNotFoundError("Final testing is locked behind tuning artifacts. Run: python pipeline.py --stage tune")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    if args.architectures != selection["architectures"] or final_seed_list(args) != selection["seeds"]:
        raise ValueError("Final architecture list/seeds differ from selection.json; refusing test access")
    split = load_split(out / "splits.csv")
    split_meta = json.loads((out / "split_metadata.json").read_text(encoding="utf-8"))
    if split_meta["split_sha256"] != selection["split_sha256"]:
        raise ValueError("Canonical split changed after model selection; refusing test access")
    params = json.loads(params_path.read_text(encoding="utf-8"))
    results = []
    resume_log = []
    split_sha256 = __import__("hashlib").sha256((out / "splits.csv").read_bytes()).hexdigest()
    for arch in selection["architectures"]:
        if arch not in params:
            raise ValueError(f"Missing locked hyperparameters for {arch}")
        config = {**default_config(args), **params[arch], "augmentation": True, "use_class_weights": True}
        for seed in selection["seeds"]:
            resumed = _find_resumable_final_run(out, arch, int(seed), config, split_sha256) if args.resume else None
            if resumed is not None:
                results.append(resumed)
                resume_log.append({"architecture": arch, "seed": seed, "action": "reused_completed_run",
                                   "run_dir": resumed["run_dir"], "checkpoint": resumed["checkpoint"]})
                continue
            try:
                result = fit_run(root=args.data_root, split_csv=out / "splits.csv", out=out, architecture=arch,
                                 seed=seed, config=config, stage="final", wandb_project=args.wandb_project,
                                 wandb_entity=args.wandb_entity, wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers,
                                 evaluate_test=True)
            except Exception as exc:
                result = {"architecture": arch, "seed": seed, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            results.append(result)
            _append_result(out / "final" / "run_metrics.csv", result)
            resume_log.append({"architecture": arch, "seed": seed,
                               "action": "trained" if result.get("status") == "completed" else "failed",
                               "run_dir": result.get("run_dir"), "error": result.get("error")})
    completed = [r for r in results if r.get("status") == "completed"]
    summaries = summarize_runs(results)
    tests = paired_holm_tests(results, selection["reference_architecture"], architecture_order=selection["architectures"])
    save_json(out / "final" / "run_metrics.json", results)
    save_csv(out / "final" / "architecture_summary.csv", summaries)
    save_csv(out / "final" / "statistical_tests.csv", tests)
    per_class_rows = []
    for arch in selection["architectures"]:
        for label in CLASSES:
            per_run = [r.get("per_class", {}).get(label, {}) for r in completed if r["architecture"] == arch]
            if not per_run:
                continue
            for metric in ("precision", "recall", "f1"):
                values = [float(r[metric]) for r in per_run if metric in r]
                per_class_rows.append({"architecture": arch, "class": label, "metric": metric,
                                       "mean": float(np.mean(values)), "sd": float(np.std(values, ddof=1)) if len(values) >= 2 else None})
            per_class_rows.append({"architecture": arch, "class": label, "metric": "support",
                                   "mean": per_run[0].get("support"), "sd": 0.0})
    save_csv(out / "final" / "per_class_summary.csv", per_class_rows)
    save_json(out / "final" / "resume_decisions.json", resume_log)
    return {"completed_runs": len(completed), "expected_runs": len(selection["architectures"]) * len(selection["seeds"]),
            "selection": selection, "architecture_summary": summaries, "statistical_tests": tests}


def _find_resumable_final_run(out: Path, architecture: str, seed: int,
                              expected: dict[str, Any], split_sha256: str) -> dict[str, Any] | None:
    """Reuse only a complete final result whose saved configuration and split are identical."""
    required = {"optimizer", "learning_rate", "weight_decay", "batch_size", "epochs",
                "input_size", "black_threshold", "augmentation", "use_class_weights"}
    for config_path in sorted((out / "final").glob("*/config.json")):
        try:
            cfg = json.loads(config_path.read_text(encoding="utf-8"))
            if cfg.get("stage") != "final" or cfg.get("architecture") != architecture or int(cfg.get("seed", -1)) != seed:
                continue
            if cfg.get("split_sha256") != split_sha256 or any(cfg.get(key) != expected.get(key) for key in required):
                continue
            run_dir = config_path.parent
            metrics_path, test_path = run_dir / "run_metrics.json", run_dir / "test_metrics.json"
            checkpoint = run_dir / f"best_{architecture}_seed{seed}.pt"
            if not (metrics_path.is_file() and test_path.is_file() and checkpoint.is_file()):
                continue
            result = json.loads(metrics_path.read_text(encoding="utf-8"))
            if result.get("status") != "completed" or not all(k in result for k in
                    ("test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1", "per_class")):
                continue
            result["run_dir"], result["checkpoint"] = str(run_dir), str(checkpoint)
            result["resumed"] = True
            return result
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            continue
    return None


def run_ablation(args, out: Path) -> list[dict[str, Any]]:
    selection = json.loads((out / "selection.json").read_text(encoding="utf-8"))
    arch, seed = selection["reference_architecture"], int(selection["representative_seed"])
    final_runs = json.loads((out / "final" / "run_metrics.json").read_text(encoding="utf-8"))
    baseline = next((r for r in final_runs if r["architecture"] == arch and int(r["seed"]) == seed and r.get("status") == "completed"), None)
    if baseline is None:
        raise FileNotFoundError("A matched full-pipeline final run is required before ablation")
    best_params = json.loads((out / "tuning" / "best_params_by_arch.json").read_text(encoding="utf-8"))[arch]
    base_cfg = {**default_config(args), **best_params, "augmentation": True, "use_class_weights": True}
    variants = [
        ("ablation_no_augmentation", args.data_root, base_cfg | {"augmentation": False}, out / "splits.csv"),
        ("ablation_no_class_weights", args.data_root, base_cfg | {"use_class_weights": False}, out / "splits.csv"),
    ]
    padded_properties = out / "dataset_audit_padding" / "image_properties.csv"
    if padded_properties.exists():
        props = list(csv.DictReader(padded_properties.open(encoding="utf-8-sig", newline="")))
        padding_lookup = {(Path(r["path"]).parent.name, Path(r["path"]).stem): r["path"] for r in props}
        clean_split = load_split(out / "splits.csv")
        padded_split = []
        for row in clean_split:
            key = (Path(row["path"]).parent.name, Path(row["path"]).stem)
            if key not in padding_lookup:
                raise ValueError(f"No paired padded image for {row['image_id']}")
            padded_split.append(row | {"path": padding_lookup[key]})
        pad_csv = out / "ablation" / "paired_padding_split.csv"
        save_csv(pad_csv, padded_split)
        variants.append(("ablation_original_black_padding", args.padded_root, base_cfg, pad_csv))
    results = []
    for stage_name, root, config, split_csv in variants:
        try:
            result = fit_run(root=root, split_csv=split_csv, out=out, architecture=arch, seed=seed,
                             config=config, stage=stage_name, wandb_project=args.wandb_project,
                             wandb_entity=args.wandb_entity,
                             wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers,
                             evaluate_test=True)
            result["variant"] = stage_name
            result["delta_accuracy_vs_full"] = result["test_acc"] - float(baseline["test_acc"])
            result["delta_macro_f1_vs_full"] = result["test_macro_f1"] - float(baseline["test_macro_f1"])
        except Exception as exc:
            result = {"architecture": arch, "seed": seed, "variant": stage_name, "status": "failed",
                      "error": f"{type(exc).__name__}: {exc}"}
        results.append(result)
    save_json(out / "ablation" / "ablation_results.json", results)
    save_csv(out / "ablation" / "ablation_results.csv", results)
    return results


In [ ]:
%%project_source shrine_classifier/models.py
from __future__ import annotations

ARCHITECTURES = ("resnet50", "efficientnet_b0", "mobilenet_v3_large", "densenet121", "convnext_tiny")

CAM_LAYER_PATH = {
    "resnet50": "layer4[-1]",
    "efficientnet_b0": "features[-1]",
    "mobilenet_v3_large": "features[-1]",
    "densenet121": "features.denseblock4",
    "convnext_tiny": "features[-1]",
}


def build_model(architecture: str, *, pretrained: bool = True):
    """Return a torchvision CNN with exactly four output logits and weight metadata."""
    try:
        import torch.nn as nn
        from torchvision import models
    except Exception as exc:
        raise RuntimeError("Install compatible PyTorch and torchvision to construct CNNs") from exc
    if architecture not in ARCHITECTURES:
        raise ValueError(f"Unsupported architecture {architecture!r}; choose from {ARCHITECTURES}")
    ctor = getattr(models, architecture)
    enum_name = {
        "resnet50": "ResNet50_Weights", "efficientnet_b0": "EfficientNet_B0_Weights",
        "mobilenet_v3_large": "MobileNet_V3_Large_Weights", "densenet121": "DenseNet121_Weights",
        "convnext_tiny": "ConvNeXt_Tiny_Weights",
    }[architecture]
    weight_enum = getattr(models, enum_name)
    weights = weight_enum.DEFAULT if pretrained else None
    model = ctor(weights=weights)
    if architecture == "resnet50":
        model.fc = nn.Linear(model.fc.in_features, 4)
    elif architecture in ("efficientnet_b0", "mobilenet_v3_large", "convnext_tiny"):
        head = model.classifier[-1]
        model.classifier[-1] = nn.Linear(head.in_features, 4)
    else:
        model.classifier = nn.Linear(model.classifier.in_features, 4)
    return model, {"architecture": architecture,
                   "weights": f"{enum_name}.DEFAULT" if pretrained else None,
                   "cam_layer": CAM_LAYER_PATH[architecture],
                   "num_classes": 4, "pretrained": bool(pretrained)}


def cam_layer(model, architecture: str):
    if architecture == "resnet50":
        return model.layer4[-1]
    if architecture in ("efficientnet_b0", "mobilenet_v3_large", "convnext_tiny"):
        return model.features[-1]
    if architecture == "densenet121":
        return model.features.denseblock4
    raise ValueError(f"No CAM layer configured for {architecture}")


In [ ]:
%%project_source shrine_classifier/tracking.py
from __future__ import annotations

import csv
from pathlib import Path
from typing import Any


class RunLogger:
    """Append local history first; optionally mirror it to W&B."""
    def __init__(self, history_path: Path, *, mode: str, project: str, group: str,
                 name: str, config: dict[str, Any], entity: str | None = None, active_run=None) -> None:
        self.history_path, self.mode = history_path, mode
        self.project, self.group, self.name, self.config, self.entity = project, group, name, config, entity
        self.run = active_run
        self.owns_run = False
        self.tracking_error: str | None = None
        self._fallback_no = 0
        self.history_path.parent.mkdir(parents=True, exist_ok=True)
        if active_run is None and mode != "disabled":
            try:
                import wandb
                self.run = wandb.init(project=project, group=group, name=name, config=config,
                                      entity=entity, mode=mode, reinit=True)
                self.owns_run = True
            except Exception as exc:
                self.tracking_error = f"{type(exc).__name__}: {exc}"
                try:
                    if mode == "online":
                        import wandb
                        self.run = wandb.init(project=project, group=group, name=name, config=config,
                                              entity=entity, mode="offline", reinit=True)
                        self.owns_run = True
                        self.mode = "offline"
                    else:
                        self.mode = "disabled"
                except Exception as fallback_exc:
                    self.tracking_error += f"; offline fallback: {type(fallback_exc).__name__}: {fallback_exc}"
                    self.mode = "disabled"

    def log(self, values: dict[str, Any]) -> None:
        base_fields = ["epoch", "train_loss", "train_acc", "train_macro_f1", "val_loss", "val_acc",
                       "val_macro_f1", "learning_rate", "best_epoch", "best_val_acc", "best_val_macro_f1",
                       "test_loss", "test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1"]
        class_fields = [f"{metric}_{cls}" for metric in ("precision", "recall", "f1", "support")
                        for cls in ("san_phraphrom", "san_phraphum", "san_tayay", "tee_ju_eia")]
        fields = base_fields + class_fields
        exists = self.history_path.exists()
        with self.history_path.open("a", encoding="utf-8-sig", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
            if not exists:
                writer.writeheader()
            writer.writerow({k: values.get(k, "") for k in fields})
        if self.run is not None:
            try:
                self.run.log(values, step=int(values["epoch"]) if values.get("epoch") is not None else None)
            except Exception as exc:
                self.tracking_error = f"{type(exc).__name__}: {exc}"
                old_run, self.run = self.run, None
                try:
                    if self.owns_run:
                        old_run.finish(exit_code=1)
                except Exception as finish_exc:
                    self.tracking_error += f"; failed to finish online run: {type(finish_exc).__name__}: {finish_exc}"
                self.owns_run = False
                if self.mode == "online":
                    try:
                        import wandb
                        self._fallback_no += 1
                        self.run = wandb.init(project=self.project, group=self.group,
                            name=f"{self.name}_offline{self._fallback_no}", config=self.config,
                            entity=self.entity, mode="offline", reinit=True)
                        self.owns_run = True
                        self.mode = "offline"
                    except Exception as fallback_exc:
                        self.tracking_error += f"; offline fallback failed: {type(fallback_exc).__name__}: {fallback_exc}"
                        self.mode = "disabled"

    def finish(self, *, exit_code: int = 0) -> None:
        if self.owns_run and self.run is not None:
            try:
                self.run.finish(exit_code=exit_code)
            except Exception as exc:
                message=f"{type(exc).__name__}: {exc}"
                self.tracking_error=f"{self.tracking_error}; finish: {message}" if self.tracking_error else message
                self.mode="disabled"
            finally:
                self.owns_run=False


In [ ]:
%%project_source shrine_classifier/training.py
from __future__ import annotations

import json
import random
import time
import uuid
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from .analysis import classification_metrics, save_csv, save_json
from .data import CLASSES, IMAGENET_MEAN, IMAGENET_STD, letterbox, open_rgb
from .models import build_model
from .tracking import RunLogger
from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode, enforce_mode_per_epoch, log_mode_stats


class ImageDataset(Dataset):
    def __init__(self, root: Path, records: list[dict[str, str]], *, training: bool, size: int = 224,
                 augmentation: bool = True):
        self.root, self.records, self.training, self.size = root, records, training, size
        aug = [transforms.RandomRotation(12, fill=(124, 116, 104)),
               transforms.RandomAffine(degrees=0, translate=(.05, .05), scale=(.92, 1.08), fill=(124, 116, 104)),
               transforms.RandomPerspective(distortion_scale=.10, p=.20, fill=(124, 116, 104)),
               transforms.RandomHorizontalFlip(.5), transforms.ColorJitter(.10, .10, .10, .02)]
        base = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
        if training and augmentation:
            neutral_normalized = [((pixel / 255.0) - mean) / std for pixel, mean, std in
                                  zip((124, 116, 104), IMAGENET_MEAN, IMAGENET_STD)]
            self.transform = transforms.Compose(aug + base + [transforms.RandomErasing(
                p=.05, scale=(.02, .08), value=neutral_normalized)])
        else:
            self.transform = transforms.Compose(base)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        image = open_rgb(self.root / row["path"])
        canvas, bbox = letterbox(image, self.size)
        return self.transform(canvas), int(row["label_index"]), row["image_id"], bbox


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def worker_init(worker_id: int) -> None:
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed + worker_id)
    np.random.seed(seed + worker_id)


def load_split(path: Path) -> list[dict[str, str]]:
    import csv
    if not path.exists():
        raise FileNotFoundError(f"Missing canonical split {path}; run: python pipeline.py --stage prep")
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    required = {"image_id", "path", "label", "label_index", "group_id", "split"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError(f"Split must include {sorted(required)}")
    if any(r["label"] not in CLASSES or int(r["label_index"]) != CLASSES.index(r["label"]) for r in rows):
        raise ValueError("Split label/index does not match fixed CLASSES order")
    group_split: dict[str, str] = {}
    for r in rows:
        previous = group_split.setdefault(r["group_id"], r["split"])
        if previous != r["split"]:
            raise ValueError(f"Duplicate group crosses split boundary: {r['group_id']}")
    if {r["split"] for r in rows} != {"train", "validation", "test"}:
        raise ValueError("Canonical split must contain train, validation, and test")
    if len({r["image_id"] for r in rows}) != len(rows):
        raise ValueError("Duplicate image IDs in canonical split")
    for split in ("train", "validation", "test"):
        labels = {r["label"] for r in rows if r["split"] == split}
        if labels != set(CLASSES):
            raise ValueError(f"All four classes are required in {split}; found {sorted(labels)}")
    return rows


def split_records(rows: list[dict[str, str]]) -> dict[str, list[dict[str, str]]]:
    return {s: [r for r in rows if r["split"] == s] for s in ("train", "validation", "test")}


def make_loaders(root: Path, rows: list[dict[str, str]], *, size: int, batch_size: int,
                 seed: int, workers: int, include_test: bool = False,
                 augmentation: bool = True, class_weighted: bool = True):
    parts = split_records(rows)
    gen = torch.Generator().manual_seed(seed)
    loaders = {}
    for split in ("train", "validation") + (("test",) if include_test else ()):
        dataset = ImageDataset(root, parts[split], training=(split == "train"), size=size,
                               augmentation=augmentation)
        loaders[split] = DataLoader(dataset, batch_size=batch_size, shuffle=(split == "train"),
                                    num_workers=workers, pin_memory=torch.cuda.is_available(),
                                    worker_init_fn=worker_init, generator=gen,
                                    persistent_workers=workers > 0, drop_last=False)
    counts = np.bincount([int(r["label_index"]) for r in parts["train"]], minlength=4)
    if (counts == 0).any():
        raise ValueError("Every class needs training examples to compute class weights")
    weights = len(parts["train"]) / (len(CLASSES) * counts.astype(np.float64))
    if not class_weighted:
        weights = np.ones(len(CLASSES), dtype=np.float64)
    return loaders, weights.astype(np.float32), parts


def _phase(model, loader, criterion, optimizer, device, *, training: bool,
           architecture: str | None = None, mode: str | None = None):
    model.train(training)
    if training and mode is not None:
        # The shared phase resets train/eval flags. Reapply the mode here so
        # frozen BatchNorm and dropout modules never drift back into train().
        enforce_mode_per_epoch(model, architecture, mode)
    loss_sum, weight_sum = 0.0, 0.0
    y_true, y_pred, probabilities = [], [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels, _ids, _bbox in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            per_item = criterion(logits, labels)
            if training:
                per_item.mean().backward()
                optimizer.step()
            loss_sum += float(per_item.detach().sum().cpu())
            weight_sum += len(labels)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.detach().argmax(1).cpu().tolist())
            if not training:
                probabilities.extend(torch.softmax(logits.detach(), dim=1).cpu().tolist())
    return {"loss": loss_sum / max(weight_sum, 1),
            "acc": float(accuracy_score(y_true, y_pred)),
            "macro_f1": classification_metrics(y_true, y_pred)["macro_f1"],
            "y_true": y_true, "y_pred": y_pred, "probabilities": probabilities}


def _atomic_checkpoint(path: Path, state: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(state, tmp)
    tmp.replace(path)


def fit_run(*, root: Path, split_csv: Path, out: Path, architecture: str, seed: int,
            config: dict[str, Any], stage: str, wandb_project: str,
            wandb_entity: str | None = None,
            wandb_mode: str = "online", device: str = "auto", workers: int = 0,
            evaluate_test: bool = False, active_wandb_run=None,
            mode: str | None = None, run_identity: str | None = None) -> dict[str, Any]:
    seed_everything(seed)
    rows = load_split(split_csv)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("--device cuda requested but CUDA is unavailable")
    device_obj = torch.device(device)
    batch_size = int(config.get("batch_size", 16))
    # Test remains a split manifest row set only until a selected validation
    # checkpoint has been restored below; no test DataLoader exists during fit.
    loaders, weights, parts = make_loaders(root, rows, size=int(config.get("input_size", 224)),
                                           batch_size=batch_size, seed=seed, workers=workers,
                                           include_test=False,
                                           augmentation=bool(config.get("augmentation", True)),
                                           class_weighted=bool(config.get("use_class_weights", True)))
    model, model_meta = build_model(architecture, pretrained=True)
    model.to(device_obj)
    mode_stats = apply_finetuning_mode(model, architecture, mode) if mode else None
    weight_tensor = torch.tensor(weights, dtype=torch.float32, device=device_obj)
    criterion = torch.nn.CrossEntropyLoss(weight=weight_tensor, reduction="none")
    optimizer_name = str(config.get("optimizer", "adamw")).lower()
    lr, decay = float(config.get("learning_rate", 1e-4)), float(config.get("weight_decay", 1e-4))
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    if not trainable_parameters:
        raise RuntimeError(f"Mode {mode!r} left {architecture} with no trainable parameters")
    if optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(trainable_parameters, lr=lr, weight_decay=decay)
    elif optimizer_name == "adam":
        optimizer = torch.optim.Adam(trainable_parameters, lr=lr, weight_decay=decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(trainable_parameters, lr=lr, momentum=.9, nesterov=True, weight_decay=decay)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=3)
    identity_suffix = run_identity or f"seed{seed}"
    mode_token = f"_{mode}" if mode else ""
    run_id = f"{stage}_{architecture}{mode_token}_{identity_suffix}_seed{seed}_attempt{int(time.time())}_{uuid.uuid4().hex[:6]}"
    run_dir = out / stage / architecture / mode / run_id if mode else out / stage / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    cfg = {**config, **model_meta, "stage": stage, "mode": mode, "mode_stats": mode_stats,
           "run_identity": run_identity, "seed": seed, "optimizer": optimizer_name,
           "learning_rate": lr, "weight_decay": decay, "batch_size": batch_size,
           "epochs": int(config.get("epochs", 30)), "early_stopping_patience": int(config.get("patience", 7)),
           "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=3)", "device": str(device_obj),
           "class_weights": weights.tolist(), "split_csv": str(split_csv.resolve()),
           "split_sha256": __import__("hashlib").sha256(split_csv.read_bytes()).hexdigest(),
           "black_threshold": int(config.get("black_threshold", 10)),
           "padding_policy": "neutral ImageNet mean; cleaned source",
           "class_weight_policy": "inverse train frequency" if config.get("use_class_weights", True) else "uniform loss",
           "augmentation_policy": "mild train-only" if config.get("augmentation", True) else "disabled"}
    save_json(run_dir / "config.json", cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    cfg["trainable_parameters"] = trainable
    save_json(run_dir / "config.json", cfg)
    group_name = "sweep" if stage in {"tune", "tuning"} else ("ablation" if stage.startswith("ablation") else stage)
    logger = RunLogger(run_dir / "history.csv", mode=wandb_mode, project=wandb_project,
                       group=group_name, name=run_id, config=cfg, entity=wandb_entity,
                       active_run=active_wandb_run)
    history = []
    best_score, best_loss, best_epoch, stale = -1.0, float("inf"), 0, 0
    max_epochs, patience = int(config.get("epochs", 30)), int(config.get("patience", 7))
    checkpoint = run_dir / f"best_{architecture}{mode_token}_seed{seed}.pt"
    try:
        for epoch in range(1, max_epochs + 1):
            train = _phase(model, loaders["train"], criterion, optimizer, device_obj, training=True,
                           architecture=architecture, mode=mode)
            val = _phase(model, loaders["validation"], criterion, optimizer, device_obj, training=False)
            lr_now = float(optimizer.param_groups[0]["lr"])
            scheduler.step(val["macro_f1"])
            row = {"epoch": epoch, "train_loss": train["loss"], "train_acc": train["acc"],
                   "train_macro_f1": train["macro_f1"], "val_loss": val["loss"],
                   "val_acc": val["acc"], "val_macro_f1": val["macro_f1"], "learning_rate": lr_now}
            history.append(row)
            logger.log(row)
            improvement = val["macro_f1"] > best_score + float(config.get("min_delta", 1e-4))
            checkpoint_better = val["macro_f1"] > best_score or (val["macro_f1"] == best_score and val["loss"] < best_loss)
            if checkpoint_better:
                best_score, best_loss, best_epoch = val["macro_f1"], val["loss"], epoch
                _atomic_checkpoint(checkpoint, {"model_state": model.state_dict(), "architecture": architecture,
                                                "mode": mode, "seed": seed, "epoch": epoch, "best_val_macro_f1": best_score,
                                                "best_val_loss": best_loss, "config": cfg})
            if improvement:
                stale = 0
            else:
                stale += 1
            if stale >= patience:
                break
        if best_epoch == 0:
            raise RuntimeError("No finite validation checkpoint was selected")
        state = torch.load(checkpoint, map_location=device_obj, weights_only=False)
        model.load_state_dict(state["model_state"])
        result: dict[str, Any] = {"architecture": architecture, "mode": mode, "seed": seed, "status": "completed",
                                  "trial": config.get("trial"), "run_identity": run_identity,
                                  "best_epoch": best_epoch, "best_val_acc": next(r["val_acc"] for r in history if r["epoch"] == best_epoch),
                                  "best_val_macro_f1": best_score, "best_val_loss": best_loss,
                                  "checkpoint": str(checkpoint), "run_dir": str(run_dir), "epochs_completed": len(history),
                                  "tracking_mode": logger.mode, "tracking_error": logger.tracking_error}
        try:
            result["wandb_run_id"] = logger.run.id if logger.run is not None else None
            result["wandb_run_url"] = logger.run.url if logger.run is not None else None
        except Exception:
            result["wandb_run_id"] = None
            result["wandb_run_url"] = None
        logger.log({"epoch": len(history), "best_epoch": best_epoch,
                    "best_val_acc": result["best_val_acc"], "best_val_macro_f1": best_score})
        if evaluate_test:
            from torch.utils.data import DataLoader
            test_dataset = ImageDataset(root, parts["test"], training=False, size=int(config.get("input_size", 224)),
                                        augmentation=False)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                num_workers=workers, pin_memory=torch.cuda.is_available(), worker_init_fn=worker_init,
                generator=torch.Generator().manual_seed(seed), persistent_workers=workers > 0, drop_last=False)
            test = _phase(model, test_loader, criterion, optimizer, device_obj, training=False)
            metrics = classification_metrics(test["y_true"], test["y_pred"])
            result.update({"test_loss": test["loss"], "test_acc": metrics["accuracy"], "test_macro_precision": metrics["macro_precision"],
                           "test_macro_recall": metrics["macro_recall"], "test_macro_f1": metrics["macro_f1"],
                           "per_class": metrics["per_class"]})
            save_json(run_dir / "test_metrics.json", metrics)
            prediction_rows = []
            for record, label, pred, probs in zip(parts["test"], test["y_true"], test["y_pred"], test["probabilities"]):
                prediction_rows.append({"image_id": record["image_id"], "true_label": CLASSES[label],
                                        "predicted_label": CLASSES[pred], "confidence": float(max(probs)),
                                        "architecture": architecture, "mode": mode, "seed": seed,
                                        "run_identity": run_identity,
                                        **{f"prob_{c}": float(probs[i]) for i, c in enumerate(CLASSES)}})
            save_csv(run_dir / "test_predictions.csv", prediction_rows)
            from .analysis import save_confusion_matrix
            save_confusion_matrix(run_dir / "confusion_matrix.png", metrics["confusion_matrix"])
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            matrix=np.asarray(metrics["confusion_matrix"],dtype=np.float64)
            row_totals=matrix.sum(axis=1,keepdims=True)
            normalized=np.divide(matrix,row_totals,out=np.zeros_like(matrix),where=row_totals>0)
            fig,ax=plt.subplots(figsize=(6.5,5.5));image=ax.imshow(normalized,cmap="Blues",vmin=0,vmax=1)
            ax.set_xticks(range(len(CLASSES)),CLASSES,rotation=30,ha="right");ax.set_yticks(range(len(CLASSES)),CLASSES)
            ax.set_xlabel("Predicted");ax.set_ylabel("True");ax.set_title("Row-normalized confusion matrix")
            for i,row_values in enumerate(normalized):
                for j,value in enumerate(row_values):ax.text(j,i,f"{value:.2f}",ha="center",va="center",color="black")
            fig.colorbar(image,ax=ax);fig.tight_layout();fig.savefig(run_dir/"confusion_matrix_normalized.png",dpi=160);plt.close(fig)
            final_log = {"epoch": len(history), **{k: v for k, v in result.items() if k.startswith("test_")}}
            logger.log(final_log)
            for cls, values in metrics["per_class"].items():
                logger.log({"epoch": len(history), f"precision_{cls}": values["precision"],
                            f"recall_{cls}": values["recall"], f"f1_{cls}": values["f1"],
                            f"support_{cls}": values["support"]})
        logger.finish()
        result["tracking_mode"], result["tracking_error"] = logger.mode, logger.tracking_error
        save_json(run_dir / "run_metrics.json", result)
        _save_curves(run_dir / "curves.png", history)
        save_json(run_dir / "checkpoint_metadata.json", {"checkpoint": checkpoint.name, **result,
                                                           "mode_stats": mode_stats, "class_order": CLASSES,
                                                           "split_sha256": cfg["split_sha256"]})
        return result
    except Exception as exc:
        if run_dir.exists():
            try:
                save_json(run_dir / "failure.json", {"architecture": architecture, "mode": mode,
                    "seed": seed, "run_identity": run_identity, "error": f"{type(exc).__name__}: {exc}",
                    "config": cfg, "finished_utc": datetime.now(timezone.utc).isoformat()})
            except Exception:
                pass
        logger.finish(exit_code=1)
        raise
    finally:
        logger.finish()


def _save_test_predictions(path: Path, records, model, loader, device):
    import csv
    rows = []
    model.eval()
    with torch.no_grad():
        for images, labels, ids, _bbox in loader:
            probs = torch.softmax(model(images.to(device)), dim=1).cpu().numpy()
            for image_id, label, p in zip(ids, labels.tolist(), probs):
                pred = int(np.argmax(p))
                rows.append({"image_id": image_id, "true_label": CLASSES[label], "predicted_label": CLASSES[pred],
                             "confidence": float(p[pred]), **{f"prob_{c}": float(p[i]) for i, c in enumerate(CLASSES)}})
    save_csv(path, rows)


def _save_curves(path: Path, history):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    for ax, keys, title in zip(axes, [("train_loss", "val_loss"), ("train_acc", "val_acc"),
                                      ("train_macro_f1", "val_macro_f1")], ("Loss", "Accuracy", "Macro F1")):
        for key in keys:
            ax.plot([r["epoch"] for r in history], [r[key] for r in history], label=key)
        ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=.25); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


In [ ]:
%%project_source shrine_classifier/visualization.py
from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

from .analysis import gradcam, save_csv, save_json
from .baseline import imagenet_predict
from .data import CLASSES, letterbox, open_rgb
from .models import build_model
from .training import ImageDataset, load_split


def make_analysis_grids(args, out: Path, *, mode: str = "both") -> dict:
    selection_path = out / "selection.json"
    final_path = out / "final" / "run_metrics.json"
    if not selection_path.exists() or not final_path.exists():
        raise FileNotFoundError("Run tuning and final evaluation before analysis; use --stage tune, then --stage final")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    run_rows = json.loads(final_path.read_text(encoding="utf-8"))
    arch, seed = selection["reference_architecture"], int(selection["representative_seed"])
    run = next((r for r in run_rows if r["architecture"] == arch and int(r["seed"]) == seed and r.get("status") == "completed"), None)
    if not run:
        raise FileNotFoundError(f"No completed selected checkpoint for {arch}, seed {seed}")
    device = args.device if args.device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")
    model, _ = build_model(arch, pretrained=False)
    state = torch.load(run["checkpoint"], map_location=device, weights_only=False)
    model.load_state_dict(state["model_state"]); model.to(device).eval()
    rows = [r for r in load_split(out / "splits.csv") if r["split"] == "test"]
    dataset = ImageDataset(args.data_root, rows, training=False, size=args.input_size)
    predictions = list(csv.DictReader(Path(run["run_dir"], "test_predictions.csv").open(encoding="utf-8-sig", newline="")))
    by_id = {r["image_id"]: r for r in predictions}
    selected = []
    for cls in CLASSES:
        class_rows = [r for r in rows if r["label"] == cls and r["image_id"] in by_id]
        correct = sorted((r for r in class_rows if by_id[r["image_id"]]["true_label"] == by_id[r["image_id"]]["predicted_label"]),
                         key=lambda r: float(by_id[r["image_id"]]["confidence"]))
        wrong = sorted((r for r in class_rows if by_id[r["image_id"]]["true_label"] != by_id[r["image_id"]]["predicted_label"]),
                       key=lambda r: -float(by_id[r["image_id"]]["confidence"]))
        if wrong:
            selected.append((wrong[0], "misclassified_test_example"))
        if correct:
            selected.append((correct[0], "lowest_confidence_correct_test_example"))
        elif class_rows:
            selected.append((min(class_rows, key=lambda r: float(by_id[r["image_id"]]["confidence"])), "lowest_confidence_available_test_example"))
    visdir = out / "gradcam"; visdir.mkdir(parents=True, exist_ok=True)
    records = []
    if mode in {"both", "gradcam"}:
        for row, reason in selected:
            idx = next(i for i, record in enumerate(rows) if record["image_id"] == row["image_id"])
            tensor, label_index, image_id, bbox = dataset[idx]
            logits = model(tensor.unsqueeze(0).to(device))
            pred_idx = int(logits.argmax(1).item())
            conf = float(logits.softmax(1)[0, pred_idx].item())
            cam = gradcam(model, arch, tensor.unsqueeze(0).to(device), pred_idx)[0].detach().cpu().numpy()
            x1, y1, x2, y2 = map(int, bbox)
            cam_crop = cam[y1:y2, x1:x2]
            original = open_rgb(args.data_root / row["path"])
            heat = _heat_image(cam_crop, original.size)
            overlay = Image.blend(original.convert("RGBA"), heat.convert("RGBA"), .42).convert("RGB")
            stem = row["image_id"].replace("/", "__")
            original.save(visdir / f"{stem}_original.jpg")
            heat.save(visdir / f"{stem}_heatmap.png")
            overlay.save(visdir / f"{stem}_overlay.jpg")
            records.append({"image_id": image_id, "true_label": CLASSES[label_index], "predicted_label": CLASSES[pred_idx],
                            "target_class": CLASSES[pred_idx], "confidence": conf, "selection_reason": reason,
                            "original": f"{stem}_original.jpg", "heatmap": f"{stem}_heatmap.png", "overlay": f"{stem}_overlay.jpg"})
        _cam_grid(visdir / "gradcam_grid.png", records, visdir)
    imagenet_model = None
    if mode in {"both", "eyeball"}:
        from torchvision.models import ResNet50_Weights, resnet50
        imagenet_weights = ResNet50_Weights.DEFAULT
        imagenet_model = resnet50(weights=imagenet_weights).to(device).eval()
        eye_dir = out / "eyeball"; eye_dir.mkdir(parents=True, exist_ok=True)
        imagenet_rows = []
        for row, reason in selected:
            orig = open_rgb(args.data_root / row["path"])
            label_info = by_id[row["image_id"]]
            generic, generic_conf = imagenet_predict(orig, model=imagenet_model, weights=imagenet_weights)
            imagenet_rows.append({"image_id": row["image_id"], "path": row["path"], "true_label": row["label"],
                                  "imagenet_top1": generic, "imagenet_confidence": generic_conf,
                                  "finetuned_prediction": label_info["predicted_label"],
                                  "finetuned_confidence": float(label_info["confidence"]),
                                  "correct": int(label_info["true_label"] == label_info["predicted_label"]),
                                  "selection_reason": reason})
        save_csv(eye_dir / "error_analysis.csv", imagenet_rows)
        _eyeball_grid(eye_dir / "eyeball_grid.png", imagenet_rows, args.data_root)
    _log_analysis_outputs(args, out, run, mode)
    save_json(visdir / "example_manifest.json", records)
    return {"architecture": arch, "seed": seed, "examples": len(selected), "gradcam": mode in {"both", "gradcam"}, "eyeball": mode in {"both", "eyeball"}}


def _heat_image(array: np.ndarray, size: tuple[int, int]) -> Image.Image:
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import colormaps
    values = np.nan_to_num(array, nan=0.0, posinf=0.0, neginf=0.0)
    rgba = (colormaps["jet"](np.clip(values, 0, 1))[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgba, "RGB").resize(size, Image.Resampling.BILINEAR)


def _font(size=13):
    try:
        return ImageFont.truetype("C:/Windows/Fonts/arial.ttf", size)
    except Exception:
        return ImageFont.load_default()


def _cam_grid(path: Path, rows, directory: Path):
    cell_w, cell_h = 690, 260
    canvas = Image.new("RGB", (cell_w * 3, cell_h * max(1, len(rows))), "white")
    draw = ImageDraw.Draw(canvas)
    for i, row in enumerate(rows):
        y = i * cell_h
        for j, key in enumerate(("original", "heatmap", "overlay")):
            im = Image.open(directory / row[key]).convert("RGB"); im.thumbnail((cell_w - 12, 215), Image.Resampling.LANCZOS)
            canvas.paste(im, (j * cell_w + 5, y + 2))
            draw.text((j * cell_w + 5, y + 222), f"{key}: {row['image_id']}", fill="black", font=_font(12))
        draw.text((5, y + 242), f"true={row['true_label']} predicted={row['predicted_label']} confidence={row['confidence']:.3f} ({row['selection_reason']})", fill="black", font=_font(12))
    canvas.save(path)


def _eyeball_grid(path: Path, rows, root: Path):
    width, height = 450, 285
    canvas = Image.new("RGB", (width * 3, height * max(1, (len(rows) + 2) // 3)), "white")
    draw = ImageDraw.Draw(canvas)
    for i, row in enumerate(rows):
        x, y = (i % 3) * width, (i // 3) * height
        im = open_rgb(root / row["path"]); im.thumbnail((width - 12, 195), Image.Resampling.LANCZOS)
        canvas.paste(im, (x + 5, y + 3))
        draw.text((x + 5, y + 205), f"true={row['true_label']} | ImageNet={row['imagenet_top1']}", fill="black", font=_font(12))
        draw.text((x + 5, y + 223), f"tuned={row['finetuned_prediction']} ({row['finetuned_confidence']:.3f})", fill="black", font=_font(12))
        draw.text((x + 5, y + 242), f"{'correct' if row['correct'] else 'ERROR'} | {row['selection_reason']}", fill="black", font=_font(11))
    canvas.save(path)


def _log_analysis_outputs(args, out: Path, selected_run: dict, mode: str) -> None:
    status = {"requested_mode": args.wandb_mode, "logged": False}
    if args.wandb_mode == "disabled":
        status["reason"] = "W&B disabled by request"
    else:
        try:
            import wandb
            arch, seed = selected_run["architecture"], selected_run["seed"]
            tables = {}
            for name, file in (("architecture_summary", out / "final" / "architecture_summary.csv"),
                               ("statistical_tests", out / "final" / "statistical_tests.csv"),
                               ("per_class_summary", out / "final" / "per_class_summary.csv")):
                if file.exists():
                    with file.open("r", encoding="utf-8-sig", newline="") as stream:
                        records = list(csv.DictReader(stream))
                    if records:
                        columns = list(records[0])
                        tables[name] = wandb.Table(columns=columns, data=[[r.get(k, "") for k in columns] for r in records])
            run = wandb.init(project=args.wandb_project, group="final", name=f"final_visual_analysis_{arch}_seed{seed}",
                             entity=args.wandb_entity, mode=args.wandb_mode, reinit=True,
                             config={"stage": "final_analysis", "architecture": arch, "seed": seed,
                                     "visuals": mode})
            artifacts = {}
            if mode in {"both", "gradcam"} and (out / "gradcam" / "gradcam_grid.png").exists():
                artifacts["gradcam_grid"] = wandb.Image(str(out / "gradcam" / "gradcam_grid.png"))
            if mode in {"both", "eyeball"} and (out / "eyeball" / "eyeball_grid.png").exists():
                artifacts["eyeball_grid"] = wandb.Image(str(out / "eyeball" / "eyeball_grid.png"))
            confusion = Path(selected_run["run_dir"]) / "confusion_matrix.png"
            if confusion.exists():
                artifacts["confusion_matrix"] = wandb.Image(str(confusion))
            artifacts.update(tables)
            if artifacts:
                run.log(artifacts)
            run.finish()
            status.update({"logged": True, "run_url": getattr(run, "url", None)})
        except Exception as exc:
            status["error"] = f"{type(exc).__name__}: {exc}"
    save_json(out / "report_assets" / "analysis_wandb_status.json", status)


## 3. Fine-tuning policies และ diagnostics — source code

finetuning_modes.py กำหนด Full/Frozen/Partial และสถานะ BatchNorm/dropout ส่วน confidence_analysis.py กับ diagnostics_analysis.py ดู calibration และ training dynamics โค้ด extension experiment/report รุ่นเก่ารวมไว้เพื่อความเข้ากันได้ ไม่เปิดการทดลองที่เลิกใช้แล้ว


In [ ]:
%%project_source shrine_classifier_ext/__init__.py
"""Extension package for shrine classifier: confidence analysis, diagnostics, ablation, fine-tuning modes."""


In [ ]:
%%project_source shrine_classifier_ext/confidence_analysis.py
"""Item 1 — Confidence / reliability analysis.

Reads canonical final test_predictions.csv files (one per architecture × seed),
validates probability integrity, computes ECE and bin metrics, and generates
reliability diagrams and confidence histograms.

No model inference is performed here; only cached CSV predictions are loaded.
"""

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np

from shrine_classifier.data import CLASSES
from shrine_classifier.analysis import save_csv, save_json

N_CLASSES = len(CLASSES)
BIN_EDGES = np.linspace(0.0, 1.0, 11)  # 10 equal-width bins [0,.1),...,[.9,1]


# ---------------------------------------------------------------------------
# Loading and validation
# ---------------------------------------------------------------------------

def _load_predictions_csv(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return rows


def _validate_predictions(rows: list[dict[str, Any]], source: str) -> list[dict[str, Any]]:
    """Validate a predictions CSV and return enriched records.

    Raises ValueError describing the first integrity failure found.
    """
    issues: list[str] = []
    seen_ids: set[str] = set()
    clean: list[dict[str, Any]] = []
    prob_cols = [f"prob_{c}" for c in CLASSES]

    for idx, row in enumerate(rows):
        image_id = row.get("image_id", "")
        if not image_id:
            issues.append(f"row {idx}: missing image_id")
            continue
        if image_id in seen_ids:
            issues.append(f"row {idx}: duplicate image_id '{image_id}'")
            continue
        seen_ids.add(image_id)

        true_label = row.get("true_label", "")
        if true_label not in CLASSES:
            issues.append(f"row {idx}: unknown true_label '{true_label}'")
            continue

        predicted_label = row.get("predicted_label", "")
        if predicted_label not in CLASSES:
            issues.append(f"row {idx}: unknown predicted_label '{predicted_label}'")
            continue

        try:
            confidence = float(row.get("confidence", "nan"))
        except (ValueError, TypeError):
            issues.append(f"row {idx}: non-numeric confidence")
            continue

        probs: list[float] = []
        for col in prob_cols:
            try:
                v = float(row.get(col, "nan"))
            except (ValueError, TypeError):
                issues.append(f"row {idx}: non-numeric {col}")
                v = float("nan")
            probs.append(v)

        if any(np.isnan(p) or not np.isfinite(p) or p < 0 or p > 1 for p in probs):
            issues.append(f"row {idx}: probabilities not all finite in [0,1]")
            continue

        prob_sum = sum(probs)
        if abs(prob_sum - 1.0) > 1e-6:
            issues.append(f"row {idx}: probabilities sum to {prob_sum:.8f} (tolerance 1e-6)")
            continue

        # Check argmax consistency with saved predicted_label and confidence
        argmax_idx = int(np.argmax(probs))
        argmax_label = CLASSES[argmax_idx]
        if argmax_label != predicted_label:
            issues.append(
                f"row {idx}: saved predicted_label='{predicted_label}' does not match "
                f"argmax='{argmax_label}'"
            )
            continue
        max_prob = probs[argmax_idx]
        if abs(max_prob - confidence) > 1e-6:
            issues.append(
                f"row {idx}: saved confidence={confidence:.8f} does not match max_prob={max_prob:.8f}"
            )
            continue

        true_idx = CLASSES.index(true_label)
        correct = int(argmax_idx == true_idx)
        clean.append({
            "image_id": image_id,
            "true_label": true_label,
            "true_index": true_idx,
            "predicted_label": predicted_label,
            "predicted_index": argmax_idx,
            "confidence": confidence,
            "correct": correct,
            "probs": probs,
        })

    if issues:
        summary = "\n".join(issues[:20])
        extra = len(issues) - 20
        if extra > 0:
            summary += f"\n... and {extra} more issues"
        raise ValueError(f"Prediction integrity failure in {source}:\n{summary}")

    return clean


# ---------------------------------------------------------------------------
# Bin metrics and ECE
# ---------------------------------------------------------------------------

def _bin_metrics(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Compute reliability diagram bin stats over a list of validated records."""
    confidences = np.array([r["confidence"] for r in records])
    corrects = np.array([r["correct"] for r in records])
    n_total = len(records)

    rows: list[dict[str, Any]] = []
    for i in range(len(BIN_EDGES) - 1):
        lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]
        if i == len(BIN_EDGES) - 2:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        n_bin = int(mask.sum())
        if n_bin == 0:
            rows.append({
                "bin_lower": round(lo, 2), "bin_upper": round(hi, 2),
                "count": 0, "mean_confidence": None, "accuracy": None,
                "abs_gap": None,
            })
        else:
            mean_conf = float(confidences[mask].mean())
            acc = float(corrects[mask].mean())
            rows.append({
                "bin_lower": round(lo, 2), "bin_upper": round(hi, 2),
                "count": n_bin, "mean_confidence": mean_conf, "accuracy": acc,
                "abs_gap": abs(acc - mean_conf),
            })
    return rows


def _ece(bin_rows: list[dict[str, Any]], n_total: int) -> float | None:
    if n_total == 0:
        return None
    ece = 0.0
    for b in bin_rows:
        if b["count"] > 0 and b["abs_gap"] is not None:
            ece += (b["count"] / n_total) * b["abs_gap"]
    return ece


# ---------------------------------------------------------------------------
# High-confidence errors
# ---------------------------------------------------------------------------

def _high_confidence_errors(records: list[dict[str, Any]],
                             threshold: float = 0.9) -> list[dict[str, Any]]:
    return [
        {
            "image_id": r["image_id"],
            "true_label": r["true_label"],
            "predicted_label": r["predicted_label"],
            "confidence": r["confidence"],
        }
        for r in records
        if r["correct"] == 0 and r["confidence"] >= threshold
    ]


# ---------------------------------------------------------------------------
# Per-group summaries
# ---------------------------------------------------------------------------

def _group_stats(records: list[dict[str, Any]]) -> dict[str, Any]:
    correct_confs = [r["confidence"] for r in records if r["correct"] == 1]
    wrong_confs = [r["confidence"] for r in records if r["correct"] == 0]
    return {
        "n_total": len(records),
        "n_correct": len(correct_confs),
        "n_wrong": len(wrong_confs),
        "mean_confidence_correct": float(np.mean(correct_confs)) if correct_confs else None,
        "mean_confidence_wrong": float(np.mean(wrong_confs)) if wrong_confs else None,
    }


# ---------------------------------------------------------------------------
# Figures
# ---------------------------------------------------------------------------

def _plot_reliability(bin_rows: list[dict[str, Any]], title: str, path: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    xs = [b["mean_confidence"] for b in bin_rows if b["count"] > 0 and b["mean_confidence"] is not None]
    ys = [b["accuracy"] for b in bin_rows if b["count"] > 0 and b["accuracy"] is not None]
    counts = [b["count"] for b in bin_rows if b["count"] > 0]

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    scatter = ax.scatter(xs, ys, c=counts, cmap="YlOrRd", s=80, zorder=3, label="Bins (sized by count)")
    ax.set_xlabel("Mean max probability (confidence)")
    ax.set_ylabel("Accuracy of predicted class")
    ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    plt.colorbar(scatter, ax=ax, label="Bin count")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160)
    plt.close(fig)


def _plot_confidence_hist(records_correct: list[float], records_wrong: list[float],
                           title: str, path: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(records_correct, bins=BIN_EDGES, alpha=0.65, label=f"Correct (n={len(records_correct)})",
            color="#2196F3", edgecolor="white")
    ax.hist(records_wrong, bins=BIN_EDGES, alpha=0.65, label=f"Wrong (n={len(records_wrong)})",
            color="#F44336", edgecolor="white")
    ax.set_xlabel("Confidence (max softmax probability)")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160)
    plt.close(fig)


def _plot_reliability_pooled(pooled_bin_rows: list[dict[str, Any]],
                              arch: str, n_seeds: int, path: Path) -> None:
    """Plot reliability diagram for per-seed pooled predictions.

    Label clearly that this pools repeated predictions of the same 85 images,
    not independent samples.
    """
    title = (f"Reliability — {arch}\n"
             f"Pooled over {n_seeds} seeds × 85 images (not independent samples)")
    _plot_reliability(pooled_bin_rows, title, path)


# ---------------------------------------------------------------------------
# Discover canonical final prediction CSVs
# ---------------------------------------------------------------------------

def _discover_runs(final_dir: Path) -> list[tuple[str, int, Path]]:
    """Return (architecture, seed, prediction_csv_path) for completed runs."""
    runs: list[tuple[str, int, Path]] = []
    for config_path in sorted(final_dir.glob("*/config.json")):
        try:
            cfg = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if cfg.get("stage") != "final":
            continue
        arch = cfg.get("architecture", "")
        seed = cfg.get("seed", None)
        if not arch or seed is None:
            continue
        pred_path = config_path.parent / "test_predictions.csv"
        if pred_path.is_file():
            runs.append((arch, int(seed), pred_path))
    return runs


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def run_confidence_analysis(base_out_dir: Path, out_dir: Path,
                             selection: dict[str, Any]) -> dict[str, Any]:
    """Analyse confidence/reliability from canonical final test predictions.

    Parameters
    ----------
    base_out_dir: path to primary outputs/ directory containing finals/
    out_dir: outputs_extensions_v1/ directory where extension outputs go
    selection: parsed selection.json dict
    """
    import matplotlib
    matplotlib.use("Agg")

    conf_dir = out_dir / "confidence"
    conf_dir.mkdir(parents=True, exist_ok=True)

    final_dir = base_out_dir / "final"
    runs = _discover_runs(final_dir)
    if not runs:
        raise FileNotFoundError(f"No completed final runs with test_predictions.csv found in {final_dir}")

    architectures = selection.get("architectures", [])
    seeds = selection.get("seeds", [])
    representative_seed = int(selection.get("representative_seed", 42))
    reference_arch = selection.get("reference_architecture", architectures[0] if architectures else "")

    # Per-run records
    run_rows: list[dict[str, Any]] = []
    # Collected for per-architecture aggregation
    arch_ece_map: dict[str, list[float]] = {a: [] for a in architectures}
    all_hce: list[dict[str, Any]] = []
    all_bin_rows: list[dict[str, Any]] = []

    issues_log: list[str] = []

    for arch, seed, pred_path in sorted(runs, key=lambda x: (x[0], x[1])):
        source_label = f"{arch}/seed{seed}"
        try:
            raw = _load_predictions_csv(pred_path)
            records = _validate_predictions(raw, source_label)
        except (OSError, ValueError) as e:
            issues_log.append(str(e))
            run_rows.append({
                "architecture": arch, "seed": seed,
                "status": "invalid", "error": str(e),
            })
            continue

        bin_rows = _bin_metrics(records)
        ece = _ece(bin_rows, len(records))
        stats = _group_stats(records)
        hce = _high_confidence_errors(records, threshold=0.9)
        for h in hce:
            h.update({"architecture": arch, "seed": seed})
        all_hce.extend(hce)

        # Store bin rows for CSV
        for b in bin_rows:
            b_row = {"architecture": arch, "seed": seed, **b}
            all_bin_rows.append(b_row)

        # Per-seed reliability figure (representative seed → main figure)
        fig_path = conf_dir / f"reliability_{arch}_seed{seed}.png"
        seed_title = f"Reliability — {arch} seed {seed}\n(n={len(records)} test images)"
        _plot_reliability(bin_rows, seed_title, fig_path)

        # Per-seed confidence histogram
        correct_confs = [r["confidence"] for r in records if r["correct"] == 1]
        wrong_confs = [r["confidence"] for r in records if r["correct"] == 0]
        hist_path = conf_dir / f"confidence_hist_{arch}_seed{seed}.png"
        _plot_confidence_hist(correct_confs, wrong_confs,
                               f"Confidence distribution — {arch} seed {seed}", hist_path)

        if ece is not None:
            arch_ece_map.setdefault(arch, []).append(ece)

        run_rows.append({
            "architecture": arch, "seed": seed, "status": "ok",
            "n_total": stats["n_total"], "n_correct": stats["n_correct"], "n_wrong": stats["n_wrong"],
            "ece": ece,
            "mean_confidence_correct": stats["mean_confidence_correct"],
            "mean_confidence_wrong": stats["mean_confidence_wrong"],
            "n_high_confidence_errors": len(hce),
        })

    # Architecture summary: per-arch ECE mean ± SD
    arch_summary_rows: list[dict[str, Any]] = []
    for arch in architectures:
        ece_values = arch_ece_map.get(arch, [])
        completed = [r for r in run_rows if r["architecture"] == arch and r.get("status") == "ok"]
        n_correct = sum(r.get("n_correct", 0) for r in completed)
        n_wrong = sum(r.get("n_wrong", 0) for r in completed)
        arch_summary_rows.append({
            "architecture": arch,
            "completed_seeds": len(completed),
            "n_correct_total": n_correct,
            "n_wrong_total": n_wrong,
            "ece_mean": float(np.mean(ece_values)) if ece_values else None,
            "ece_sd": float(np.std(ece_values, ddof=1)) if len(ece_values) >= 2 else None,
            "ece_note": "bin-sensitive on small test set (n=85)",
        })

    # Pooled reliability diagram per architecture (labeled correctly)
    for arch in architectures:
        arch_records_all: list[dict[str, Any]] = []
        arch_seeds = [r for r in run_rows if r["architecture"] == arch and r.get("status") == "ok"]
        for arch_seed_row in arch_seeds:
            # Re-load to get individual records  (needed for pooled plot only)
            seed = arch_seed_row["seed"]
            pred_path_list = [(a2, s2, p) for a2, s2, p in runs if a2 == arch and s2 == seed]
            if not pred_path_list:
                continue
            _, _, pred_path = pred_path_list[0]
            try:
                raw = _load_predictions_csv(pred_path)
                recs = _validate_predictions(raw, f"{arch}/seed{seed}")
                arch_records_all.extend(recs)
            except (OSError, ValueError):
                pass

        if arch_records_all:
            pooled_bins = _bin_metrics(arch_records_all)
            n_seeds = len(arch_seeds)
            pool_fig = conf_dir / f"reliability_{arch}_pooled.png"
            _plot_reliability_pooled(pooled_bins, arch, n_seeds, pool_fig)

            # Representative seed main figure (labeled as single seed)
            rep_seed_rows = [r for r in run_rows if r["architecture"] == arch and r["seed"] == representative_seed and r.get("status") == "ok"]
            if rep_seed_rows:
                rep_pred = [(a2, s2, p) for a2, s2, p in runs if a2 == arch and s2 == representative_seed]
                if rep_pred:
                    _, _, rp = rep_pred[0]
                    try:
                        rraw = _load_predictions_csv(rp)
                        rrecs = _validate_predictions(rraw, f"{arch}/seed{representative_seed}")
                        rep_bins = _bin_metrics(rrecs)
                        main_fig = conf_dir / f"reliability_{arch}.png"
                        _plot_reliability(rep_bins,
                                          f"Reliability — {arch} (representative seed {representative_seed})",
                                          main_fig)
                        rc = [r["confidence"] for r in rrecs if r["correct"] == 1]
                        rw = [r["confidence"] for r in rrecs if r["correct"] == 0]
                        main_hist = conf_dir / f"confidence_hist_{arch}.png"
                        _plot_confidence_hist(rc, rw,
                                              f"Confidence distribution — {arch} seed {representative_seed}",
                                              main_hist)
                    except (OSError, ValueError):
                        pass

    # Save outputs
    save_csv(conf_dir / "run_metrics.csv", run_rows)
    save_csv(conf_dir / "architecture_summary.csv", arch_summary_rows)
    save_csv(conf_dir / "bin_metrics.csv", all_bin_rows)
    save_csv(conf_dir / "high_confidence_errors.csv", all_hce if all_hce else [
        {"note": "No high-confidence errors found (threshold=0.9)"}])

    config_out = {
        "stage": "confidence_analysis",
        "description": "Top-label reliability, ECE and confidence histograms from canonical final predictions",
        "bin_edges": BIN_EDGES.tolist(),
        "high_confidence_threshold": 0.9,
        "multiclass_correctness": "argmax matches true label",
        "ece_formula": "sum_b (n_b/N) * abs(accuracy_b - mean_confidence_b)",
        "pooled_note": ("Pooled plots pool repeated predictions of the same 85 test images "
                        "across seeds; they are NOT from independent images."),
        "architectures": architectures,
        "seeds": seeds,
        "representative_seed": representative_seed,
        "reference_architecture": reference_arch,
        "issues": issues_log,
    }
    save_json(conf_dir / "analysis_config.json", config_out)

    return {
        "run_rows": run_rows,
        "arch_summary": arch_summary_rows,
        "issues": issues_log,
        "out_dir": str(conf_dir),
    }


In [ ]:
%%project_source shrine_classifier_ext/diagnostics_analysis.py
"""Item 2 — Training dynamics and tuning analysis.

Reads canonical history.csv files and tuning trial CSVs to produce:
  - Per-run training dynamics table (best/stop epoch, gaps, etc.)
  - Architecture-level summaries
  - Trial inventory reconciled from all_trials.csv (canonical) vs sweep_summary.csv
  - Checkpoint validation against saved metadata
  - Figures: best-vs-stop, gap, and LR/WD scatter vs validation Macro F1

No model loading. Does not compute test metrics.
"""

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np

from shrine_classifier.data import CLASSES
from shrine_classifier.analysis import save_csv, save_json


# ---------------------------------------------------------------------------
# History loading
# ---------------------------------------------------------------------------

def _load_history(path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Load history.csv; separate epoch rows from summary/test-metric rows.

    Returns (epoch_rows, summary_rows).
    """
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        all_rows = list(csv.DictReader(f))

    epoch_rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []

    for row in all_rows:
        raw_epoch = row.get("epoch", "")
        raw_train_loss = row.get("train_loss", "")
        if raw_train_loss and raw_train_loss.strip():
            # This row has a train_loss value → it's an epoch measurement
            try:
                ep = int(float(row["epoch"]))
                train_loss = float(row["train_loss"])
            except (ValueError, TypeError, KeyError):
                summary_rows.append(row)
                continue
            epoch_rows.append({
                "epoch": ep,
                "train_loss": train_loss,
                "train_acc": _safe_float(row.get("train_acc")),
                "train_macro_f1": _safe_float(row.get("train_macro_f1")),
                "val_loss": _safe_float(row.get("val_loss")),
                "val_acc": _safe_float(row.get("val_acc")),
                "val_macro_f1": _safe_float(row.get("val_macro_f1")),
                "learning_rate": _safe_float(row.get("learning_rate")),
            })
        else:
            summary_rows.append(row)

    return epoch_rows, summary_rows


def _safe_float(val: Any) -> float | None:
    if val is None or val == "":
        return None
    try:
        v = float(val)
        return v if np.isfinite(v) else None
    except (ValueError, TypeError):
        return None


def _validate_epoch_rows(epoch_rows: list[dict[str, Any]], source: str) -> list[str]:
    """Return a list of issue descriptions."""
    issues = []
    epochs_seen = [r["epoch"] for r in epoch_rows]
    if sorted(set(epochs_seen)) != list(range(1, len(epochs_seen) + 1)):
        issues.append(f"{source}: epochs not unique/contiguous: {epochs_seen}")
    required_metrics = ["train_loss", "train_acc", "train_macro_f1",
                        "val_loss", "val_acc", "val_macro_f1"]
    for r in epoch_rows:
        missing = [m for m in required_metrics if r.get(m) is None]
        if missing:
            issues.append(f"{source} epoch {r['epoch']}: missing {missing}")
    return issues


def _best_epoch(epoch_rows: list[dict[str, Any]]) -> int:
    """Return best epoch index by val_macro_f1 → val_loss → earliest epoch."""
    def key(r):
        vf1 = r.get("val_macro_f1") or -999
        vl = r.get("val_loss") or 999
        ep = r.get("epoch", 999)
        return (-vf1, vl, ep)
    return min(epoch_rows, key=key)["epoch"]


# ---------------------------------------------------------------------------
# Dynamics extraction
# ---------------------------------------------------------------------------

def _extract_dynamics(arch: str, seed: int, run_dir: Path) -> dict[str, Any]:
    history_path = run_dir / "history.csv"
    config_path = run_dir / "config.json"
    checkpoint_meta_path = run_dir / "checkpoint_metadata.json"

    result: dict[str, Any] = {"architecture": arch, "seed": seed, "run_dir": str(run_dir)}

    if not history_path.exists():
        result["status"] = "missing_history"
        return result

    epoch_rows, summary_rows = _load_history(history_path)
    if not epoch_rows:
        result["status"] = "empty_history"
        return result

    issues = _validate_epoch_rows(epoch_rows, f"{arch}/seed{seed}")

    # Infer best epoch from history
    best_ep = _best_epoch(epoch_rows)
    stop_ep = epoch_rows[-1]["epoch"]
    epochs_after_best = stop_ep - best_ep

    best_row = next(r for r in epoch_rows if r["epoch"] == best_ep)
    last_row = epoch_rows[-1]

    # Gaps at best epoch
    acc_gap_at_best = (
        (best_row["train_acc"] - best_row["val_acc"])
        if best_row.get("train_acc") is not None and best_row.get("val_acc") is not None
        else None
    )
    f1_gap_at_best = (
        (best_row["train_macro_f1"] - best_row["val_macro_f1"])
        if best_row.get("train_macro_f1") is not None and best_row.get("val_macro_f1") is not None
        else None
    )
    loss_gap_at_best = (
        (best_row["val_loss"] - best_row["train_loss"])
        if best_row.get("val_loss") is not None and best_row.get("train_loss") is not None
        else None
    )

    # Validate against checkpoint metadata
    meta_issues: list[str] = []
    if checkpoint_meta_path.exists():
        try:
            meta = json.loads(checkpoint_meta_path.read_text(encoding="utf-8"))
            saved_best = meta.get("best_epoch")
            if saved_best is not None and int(saved_best) != best_ep:
                meta_issues.append(
                    f"History-derived best_epoch={best_ep} differs from "
                    f"checkpoint_metadata best_epoch={saved_best}"
                )
        except (OSError, json.JSONDecodeError, ValueError) as e:
            meta_issues.append(f"Could not read checkpoint_metadata.json: {e}")

    result.update({
        "status": "ok",
        "n_epochs": stop_ep,
        "best_epoch": best_ep,
        "stop_epoch": stop_ep,
        "epochs_after_best": epochs_after_best,
        "val_acc_at_best": best_row.get("val_acc"),
        "val_macro_f1_at_best": best_row.get("val_macro_f1"),
        "val_loss_at_best": best_row.get("val_loss"),
        "train_acc_at_best": best_row.get("train_acc"),
        "train_macro_f1_at_best": best_row.get("train_macro_f1"),
        "train_loss_at_best": best_row.get("train_loss"),
        "val_acc_at_stop": last_row.get("val_acc"),
        "val_macro_f1_at_stop": last_row.get("val_macro_f1"),
        "val_loss_at_stop": last_row.get("val_loss"),
        "acc_gap_at_best": acc_gap_at_best,
        "f1_gap_at_best": f1_gap_at_best,
        "loss_gap_at_best": loss_gap_at_best,
        "history_issues": issues,
        "meta_issues": meta_issues,
    })
    return result


# ---------------------------------------------------------------------------
# Trial loading
# ---------------------------------------------------------------------------

def _load_trials(tuning_dir: Path) -> list[dict[str, Any]]:
    """Load canonical all_trials.csv; reconcile counts against sweep_summary.csv."""
    canonical_path = tuning_dir / "all_trials.csv"
    sweep_path = tuning_dir / "sweep_summary.csv"
    issues: list[str] = []

    if not canonical_path.exists():
        raise FileNotFoundError(
            f"Canonical trial registry not found: {canonical_path}\n"
            "Run: python pipeline.py --stage tune"
        )

    with canonical_path.open("r", encoding="utf-8-sig", newline="") as f:
        canonical_rows = list(csv.DictReader(f))

    # Check for duplicate trial IDs (trial_id field if present)
    if "trial_id" in (canonical_rows[0] if canonical_rows else {}):
        ids = [r["trial_id"] for r in canonical_rows if r.get("trial_id")]
        duplicates = [tid for tid in set(ids) if ids.count(tid) > 1]
        if duplicates:
            issues.append(f"Duplicate trial_ids in canonical trials: {duplicates}")

    # Count per architecture
    arch_counts_canonical: dict[str, int] = {}
    for r in canonical_rows:
        a = r.get("architecture", "")
        arch_counts_canonical[a] = arch_counts_canonical.get(a, 0) + 1

    # Reconcile with sweep_summary.csv (may have additional attempt entries)
    if sweep_path.exists():
        with sweep_path.open("r", encoding="utf-8-sig", newline="") as f:
            sweep_rows = list(csv.DictReader(f))
        arch_counts_sweep: dict[str, int] = {}
        for r in sweep_rows:
            a = r.get("architecture", "")
            arch_counts_sweep[a] = arch_counts_sweep.get(a, 0) + 1
        for arch in set(arch_counts_canonical) | set(arch_counts_sweep):
            c = arch_counts_canonical.get(arch, 0)
            s = arch_counts_sweep.get(arch, 0)
            if s > c:
                issues.append(
                    f"{arch}: sweep_summary has {s} rows vs canonical {c}; "
                    "extra rows may be additional attempts not in canonical registry"
                )
    if issues:
        # Not fatal but report
        print(f"[diagnostics] Trial reconciliation notes: {issues}")

    for r in canonical_rows:
        r["_source"] = "canonical"
    return canonical_rows


# ---------------------------------------------------------------------------
# Figures
# ---------------------------------------------------------------------------

def _plot_best_vs_stop(dynamics: list[dict[str, Any]], out_dir: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    archs = list(dict.fromkeys(r["architecture"] for r in dynamics if r.get("status") == "ok"))
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(archs)))
    arch_color = dict(zip(archs, colors))

    for ax, metric, ylabel in [
        (axes[0], "best_epoch", "Best epoch"),
        (axes[1], "stop_epoch", "Stop epoch"),
    ]:
        for arch in archs:
            arch_rows = [r for r in dynamics if r["architecture"] == arch and r.get(metric) is not None]
            seeds = [r["seed"] for r in arch_rows]
            vals = [r[metric] for r in arch_rows]
            ax.scatter(seeds, vals, color=arch_color[arch], label=arch, s=60, zorder=3)
        ax.set_xlabel("Seed")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel + " by seed")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.25)

    fig.suptitle("Best and stop epoch per run", y=1.01)
    fig.tight_layout()
    path = out_dir / "best_vs_stop_epoch.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def _plot_gaps(dynamics: list[dict[str, Any]], out_dir: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    gap_keys = ["acc_gap_at_best", "f1_gap_at_best", "loss_gap_at_best"]
    gap_labels = [
        "Accuracy gap (train−val)",
        "Macro F1 gap (train−val)",
        "Loss gap (val−train)",
    ]

    archs = list(dict.fromkeys(r["architecture"] for r in dynamics if r.get("status") == "ok"))
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(archs)))
    arch_color = dict(zip(archs, colors))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, key, label in zip(axes, gap_keys, gap_labels):
        for arch in archs:
            arch_rows = [r for r in dynamics if r["architecture"] == arch
                         and r.get(key) is not None]
            seeds = [r["seed"] for r in arch_rows]
            vals = [r[key] for r in arch_rows]
            ax.scatter(seeds, vals, color=arch_color[arch], label=arch, s=60, zorder=3)
        ax.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
        ax.set_xlabel("Seed")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend(fontsize=6)
        ax.grid(alpha=0.25)

    note = ("Gaps use best-epoch train vs val. Train metrics include augmentation/param updates; "
            "validation uses eval mode. Gap is diagnostic, not a threshold for overfitting.")
    fig.text(0.5, -0.04, note, ha="center", fontsize=7, style="italic")
    fig.tight_layout()
    path = out_dir / "training_gaps.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def _plot_tuning_scatter(trial_rows: list[dict[str, Any]], out_dir: Path) -> None:
    """LR and weight decay scatter vs validation Macro F1, colour by optimizer."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    completed = [r for r in trial_rows if r.get("status") == "completed"
                 and _safe_float(r.get("best_val_macro_f1")) is not None
                 and _safe_float(r.get("learning_rate")) is not None]
    if not completed:
        return

    opt_colors = {"adamw": "#1565C0", "adam": "#2E7D32", "sgd": "#B71C1C"}
    archs = sorted(set(r.get("architecture", "?") for r in completed))

    fig, axes = plt.subplots(len(archs), 2, figsize=(10, 3.5 * len(archs)))
    if len(archs) == 1:
        axes = [axes]

    for i, arch in enumerate(archs):
        arch_trials = [r for r in completed if r.get("architecture") == arch]
        lrs = [float(r["learning_rate"]) for r in arch_trials]
        wds = [float(r["weight_decay"]) for r in arch_trials]
        f1s = [float(r["best_val_macro_f1"]) for r in arch_trials]
        opts = [str(r.get("optimizer", "")).lower() for r in arch_trials]
        colors = [opt_colors.get(o, "#666") for o in opts]

        ax_lr, ax_wd = axes[i][0], axes[i][1]
        for opt in ["adamw", "adam", "sgd"]:
            mask = [j for j, o in enumerate(opts) if o == opt]
            if mask:
                ax_lr.scatter([lrs[j] for j in mask], [f1s[j] for j in mask],
                               color=opt_colors[opt], label=f"{opt} (n={len(mask)})", s=50, alpha=0.75)
                ax_wd.scatter([wds[j] for j in mask], [f1s[j] for j in mask],
                               color=opt_colors[opt], label=f"{opt} (n={len(mask)})", s=50, alpha=0.75)
        ax_lr.set_xscale("log"); ax_wd.set_xscale("log")
        ax_lr.set_xlabel("Learning rate (log)")
        ax_lr.set_ylabel("Val Macro F1")
        ax_lr.set_title(f"{arch} — LR vs Val Macro F1")
        ax_lr.legend(fontsize=7); ax_lr.grid(alpha=0.25)
        ax_wd.set_xlabel("Weight decay (log)")
        ax_wd.set_ylabel("Val Macro F1")
        ax_wd.set_title(f"{arch} — WD vs Val Macro F1")
        ax_wd.legend(fontsize=7); ax_wd.grid(alpha=0.25)

    note = ("No causality implied: random budget differs per optimizer group; "
            "parameters are jointly sampled.")
    fig.text(0.5, -0.01, note, ha="center", fontsize=7, style="italic")
    fig.tight_layout()
    path = out_dir / "tuning_scatter.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# Main entry
# ---------------------------------------------------------------------------

def run_diagnostics_analysis(base_out_dir: Path, out_dir: Path,
                              selection: dict[str, Any]) -> dict[str, Any]:
    """Compute training dynamics and tuning analysis from cached artifacts."""
    diag_dir = out_dir / "diagnostics"
    diag_dir.mkdir(parents=True, exist_ok=True)

    final_dir = base_out_dir / "final"
    tuning_dir = base_out_dir / "tuning"
    architectures = selection.get("architectures", [])
    seeds = [int(s) for s in selection.get("seeds", [])]

    # --- Training dynamics ---
    dynamics: list[dict[str, Any]] = []
    checkpoint_validation_rows: list[dict[str, Any]] = []
    issues: list[str] = []

    for config_path in sorted(final_dir.glob("*/config.json")):
        try:
            cfg = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if cfg.get("stage") != "final":
            continue
        arch = cfg.get("architecture", "")
        seed = cfg.get("seed", None)
        if not arch or seed is None:
            continue
        run_dir = config_path.parent
        dyn = _extract_dynamics(arch, int(seed), run_dir)
        dynamics.append(dyn)

        if dyn.get("history_issues"):
            issues.extend(dyn["history_issues"])
        if dyn.get("meta_issues"):
            issues.extend(dyn["meta_issues"])

        # Checkpoint validation row
        checkpoint_validation_rows.append({
            "architecture": arch, "seed": seed,
            "history_derived_best_epoch": dyn.get("best_epoch"),
            "history_n_epochs": dyn.get("n_epochs"),
            "checkpoint_meta_match": not bool(dyn.get("meta_issues")),
            "issues": "; ".join(dyn.get("meta_issues", [])),
        })

    # Architecture dynamics summary
    arch_dyn_summary: list[dict[str, Any]] = []
    for arch in architectures:
        subset = [d for d in dynamics if d["architecture"] == arch and d.get("status") == "ok"]
        summary: dict[str, Any] = {"architecture": arch, "completed_runs": len(subset)}
        for key in ["best_epoch", "stop_epoch", "epochs_after_best",
                    "acc_gap_at_best", "f1_gap_at_best", "loss_gap_at_best"]:
            vals = [d[key] for d in subset if d.get(key) is not None]
            summary[key + "_mean"] = float(np.mean(vals)) if vals else None
            summary[key + "_sd"] = float(np.std(vals, ddof=1)) if len(vals) >= 2 else None
        arch_dyn_summary.append(summary)

    # --- Tuning trials ---
    trial_rows: list[dict[str, Any]] = []
    try:
        trial_rows = _load_trials(tuning_dir)
    except FileNotFoundError as e:
        issues.append(str(e))

    # Separate failed trials
    completed_trials = [r for r in trial_rows if r.get("status") == "completed"]
    failed_trials = [r for r in trial_rows if r.get("status") != "completed"]

    trial_inventory_rows: list[dict[str, Any]] = []
    for arch in architectures:
        arch_trials = [r for r in trial_rows if r.get("architecture") == arch]
        arch_complete = [r for r in arch_trials if r.get("status") == "completed"]
        trial_inventory_rows.append({
            "architecture": arch,
            "n_canonical_trials": len(arch_trials),
            "n_completed": len(arch_complete),
            "n_failed": len(arch_trials) - len(arch_complete),
        })

    # Figures
    if dynamics:
        _plot_best_vs_stop(dynamics, diag_dir)
        _plot_gaps(dynamics, diag_dir)
    if trial_rows:
        _plot_tuning_scatter(trial_rows, diag_dir)

    # Save outputs
    save_csv(diag_dir / "run_dynamics.csv", dynamics)
    save_csv(diag_dir / "architecture_dynamics_summary.csv", arch_dyn_summary)
    save_csv(diag_dir / "trial_inventory.csv", trial_inventory_rows)
    save_csv(diag_dir / "checkpoint_validation.csv", checkpoint_validation_rows)
    save_json(diag_dir / "issues.json", {"issues": issues})

    return {
        "dynamics": dynamics,
        "arch_dyn_summary": arch_dyn_summary,
        "trial_rows": trial_rows,
        "issues": issues,
        "out_dir": str(diag_dir),
    }


In [ ]:
%%project_source shrine_classifier_ext/extension_experiments.py
"""Extension training runners for items 3 and 4.

Provides fit_extension_run() which wraps the primary fit_run() API with:
  - finetuning mode support (full/frozen/partial)
  - augmentation/class-weight ablation
  - per-epoch mode enforcement (frozen BN invariants)
  - run identification that includes condition+mode
  - no changes to the primary fit_run() code

The shared fresh full baseline is a fresh training run with the locked
recipe (not a reuse of historical final runs). It is shared between
ablation and finetune stages via --resume logic.
"""

from __future__ import annotations

import hashlib
import json
import random
import time
import uuid
from pathlib import Path
from typing import Any

import numpy as np

from shrine_classifier.analysis import save_csv, save_json, classification_metrics
from shrine_classifier.data import CLASSES
from .finetuning_modes import apply_finetuning_mode, enforce_mode_per_epoch, log_mode_stats

EXTENSION_SEEDS = [42, 123, 456, 789, 1024]
CONDITIONS = ("full", "no_augmentation", "no_class_weights", "frozen", "partial")


# ---------------------------------------------------------------------------
# Internal fit function with mode support
# ---------------------------------------------------------------------------

def _fit_extension_run(
    *,
    root: Path,
    split_csv: Path,
    out: Path,
    architecture: str,
    seed: int,
    config: dict[str, Any],
    condition: str,
    mode: str,
    wandb_project: str,
    wandb_entity: str | None = None,
    wandb_mode: str = "online",
    device: str = "auto",
    workers: int = 0,
) -> dict[str, Any]:
    """Train one extension run with mode/condition and evaluate test in one pass."""
    import torch
    import torch.nn as nn
    from shrine_classifier.training import (
        seed_everything, load_split, make_loaders,
        _atomic_checkpoint, _save_test_predictions, _save_curves,
    )
    from shrine_classifier.models import build_model
    from shrine_classifier.tracking import RunLogger

    seed_everything(seed)
    rows = load_split(split_csv)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device_obj = torch.device(device)

    augmentation = bool(config.get("augmentation", True))
    use_class_weights = bool(config.get("use_class_weights", True))
    batch_size = int(config.get("batch_size", 16))

    loaders, weights, parts = make_loaders(
        root, rows, size=int(config.get("input_size", 224)),
        batch_size=batch_size, seed=seed, workers=workers,
        include_test=True,
        augmentation=augmentation,
        class_weighted=use_class_weights,
    )

    model, model_meta = build_model(architecture, pretrained=True)
    # Apply fine-tuning mode BEFORE creating optimizer
    mode_stats = apply_finetuning_mode(model, architecture, mode)
    model.to(device_obj)

    if use_class_weights:
        weight_tensor = torch.tensor(weights, dtype=torch.float32, device=device_obj)
    else:
        weight_tensor = torch.ones(len(CLASSES), dtype=torch.float32, device=device_obj)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor, reduction="none")

    optimizer_name = str(config.get("optimizer", "adamw")).lower()
    lr = float(config.get("learning_rate", 1e-4))
    decay = float(config.get("weight_decay", 1e-4))
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError(f"No trainable parameters for mode={mode}")

    if optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=decay)
    elif optimizer_name == "adam":
        optimizer = torch.optim.Adam(trainable_params, lr=lr, weight_decay=decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, nesterov=True, weight_decay=decay)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    split_sha256 = hashlib.sha256(split_csv.read_bytes()).hexdigest()

    run_id = f"ext_{condition}_{architecture}_seed{seed}_{int(time.time())}_{uuid.uuid4().hex[:6]}"
    run_dir = out / "experiments" / condition / run_id
    run_dir.mkdir(parents=True, exist_ok=False)

    cfg = {
        **config,
        **model_meta,
        "stage": f"extension_{condition}",
        "condition": condition,
        "mode": mode,
        "seed": seed,
        "architecture": architecture,
        "optimizer": optimizer_name,
        "learning_rate": lr,
        "weight_decay": decay,
        "batch_size": batch_size,
        "epochs": int(config.get("epochs", 30)),
        "early_stopping_patience": int(config.get("patience", 7)),
        "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=3)",
        "device": str(device_obj),
        "class_weights": weights.tolist(),
        "split_csv": str(split_csv.resolve()),
        "split_sha256": split_sha256,
        "augmentation": augmentation,
        "use_class_weights": use_class_weights,
        "augmentation_policy": "mild train-only" if augmentation else "disabled",
        "class_weight_policy": "inverse train frequency" if use_class_weights else "uniform loss",
        "padding_policy": "neutral ImageNet mean; cleaned source",
        "mode_stats": mode_stats,
    }
    save_json(run_dir / "config.json", cfg)

    logger = RunLogger(
        run_dir / "history.csv", mode=wandb_mode, project=wandb_project,
        group="ablation", name=run_id, config=cfg, entity=wandb_entity,
    )

    history: list[dict[str, Any]] = []
    best_score, best_loss, best_epoch, stale = -1.0, float("inf"), 0, 0
    max_epochs = int(config.get("epochs", 30))
    patience = int(config.get("patience", 7))
    checkpoint = run_dir / f"best_{architecture}_seed{seed}.pt"

    try:
        for epoch in range(1, max_epochs + 1):
            # Training phase with mode enforcement
            model.train()
            enforce_mode_per_epoch(model, architecture, mode)
            train_results = _phase_ext(model, loaders["train"], criterion, optimizer, device_obj, training=True)

            # Validation phase
            model.eval()
            val_results = _phase_ext(model, loaders["validation"], criterion, optimizer, device_obj, training=False)

            scheduler.step(val_results["macro_f1"])
            lr_now = float(optimizer.param_groups[0]["lr"])
            row = {
                "epoch": epoch,
                "train_loss": train_results["loss"], "train_acc": train_results["acc"],
                "train_macro_f1": train_results["macro_f1"],
                "val_loss": val_results["loss"], "val_acc": val_results["acc"],
                "val_macro_f1": val_results["macro_f1"],
                "learning_rate": lr_now,
            }
            history.append(row)
            logger.log(row)

            improvement = val_results["macro_f1"] > best_score + float(config.get("min_delta", 1e-4))
            checkpoint_better = (
                val_results["macro_f1"] > best_score
                or (val_results["macro_f1"] == best_score and val_results["loss"] < best_loss)
            )
            if checkpoint_better:
                best_score = val_results["macro_f1"]
                best_loss = val_results["loss"]
                best_epoch = epoch
                _atomic_checkpoint(checkpoint, {
                    "model_state": model.state_dict(),
                    "architecture": architecture, "seed": seed,
                    "epoch": epoch, "best_val_macro_f1": best_score,
                    "best_val_loss": best_loss, "config": cfg,
                })
            if improvement:
                stale = 0
            else:
                stale += 1
            if stale >= patience:
                break

        if best_epoch == 0:
            raise RuntimeError("No finite validation checkpoint was selected")

        # Load best checkpoint and evaluate test in one pass
        state = torch.load(checkpoint, map_location=device_obj, weights_only=False)
        model.load_state_dict(state["model_state"])
        model.eval()

        test_results = _phase_ext(model, loaders["test"], criterion, optimizer, device_obj, training=False)
        metrics = classification_metrics(test_results["y_true"], test_results["y_pred"])

        best_val_acc = next(r["val_acc"] for r in history if r["epoch"] == best_epoch)
        result: dict[str, Any] = {
            "architecture": architecture, "seed": seed, "condition": condition, "mode": mode,
            "status": "completed",
            "best_epoch": best_epoch, "best_val_acc": best_val_acc,
            "best_val_macro_f1": best_score, "best_val_loss": best_loss,
            "epochs_completed": len(history),
            "checkpoint": str(checkpoint), "run_dir": str(run_dir),
            "test_acc": metrics["accuracy"],
            "test_macro_precision": metrics["macro_precision"],
            "test_macro_recall": metrics["macro_recall"],
            "test_macro_f1": metrics["macro_f1"],
            "per_class": metrics["per_class"],
            "tracking_mode": logger.mode,
            "tracking_error": logger.tracking_error,
            "mode_stats": mode_stats,
        }
        save_json(run_dir / "test_metrics.json", metrics)
        _save_test_predictions(run_dir / "test_predictions.csv", parts["test"], model, loaders["test"], device_obj)

        from shrine_classifier.analysis import save_confusion_matrix
        save_confusion_matrix(run_dir / "confusion_matrix.png", metrics["confusion_matrix"])
        save_json(run_dir / "run_metrics.json", result)
        _save_curves(run_dir / "curves.png", history)

        logger.log({"epoch": len(history), "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc, "best_val_macro_f1": best_score})
        logger.log({"epoch": len(history), "test_acc": metrics["accuracy"],
                    "test_macro_f1": metrics["macro_f1"]})

        return result

    except Exception:
        logger.finish(exit_code=1)
        raise
    finally:
        logger.finish()


def _phase_ext(model, loader, criterion, optimizer, device, *, training: bool) -> dict[str, Any]:
    """Training/eval phase for extension runs."""
    import torch
    from sklearn.metrics import accuracy_score
    from shrine_classifier.analysis import classification_metrics

    loss_sum, weight_sum = 0.0, 0.0
    y_true, y_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels, _ids, _bbox in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            per_item = criterion(logits, labels)
            if training:
                per_item.mean().backward()
                optimizer.step()
            loss_sum += float(per_item.detach().sum().cpu())
            weight_sum += len(labels)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.detach().argmax(1).cpu().tolist())
    return {
        "loss": loss_sum / max(weight_sum, 1),
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": classification_metrics(y_true, y_pred)["macro_f1"],
        "y_true": y_true, "y_pred": y_pred,
    }


# ---------------------------------------------------------------------------
# Resume logic
# ---------------------------------------------------------------------------

def _find_resumable_extension_run(
    out_dir: Path,
    architecture: str,
    seed: int,
    condition: str,
    mode: str,
    expected_config: dict[str, Any],
    split_sha256: str,
) -> dict[str, Any] | None:
    """Return a completed extension run if protocol/config/split match."""
    required_keys = {"optimizer", "learning_rate", "weight_decay", "batch_size",
                     "epochs", "input_size", "augmentation", "use_class_weights"}
    for config_path in sorted((out_dir / "experiments" / condition).glob("*/config.json")):
        try:
            cfg = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if (cfg.get("architecture") != architecture
                or int(cfg.get("seed", -1)) != seed
                or cfg.get("condition") != condition
                or cfg.get("mode") != mode
                or cfg.get("split_sha256") != split_sha256):
            continue
        if any(cfg.get(k) != expected_config.get(k) for k in required_keys):
            continue
        run_dir = config_path.parent
        metrics_p = run_dir / "run_metrics.json"
        test_p = run_dir / "test_metrics.json"
        ckpt_p = run_dir / f"best_{architecture}_seed{seed}.pt"
        if not (metrics_p.is_file() and test_p.is_file() and ckpt_p.is_file()):
            continue
        try:
            result = json.loads(metrics_p.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if result.get("status") != "completed":
            continue
        result["run_dir"] = str(run_dir)
        result["checkpoint"] = str(ckpt_p)
        result["resumed"] = True
        return result
    return None


# ---------------------------------------------------------------------------
# High-level stage runners
# ---------------------------------------------------------------------------

def _condition_to_mode(condition: str) -> str:
    """Map condition name to fine-tuning mode."""
    if condition in ("full", "no_augmentation", "no_class_weights"):
        return "full"
    return condition  # "frozen" or "partial"


def run_extension_ablation(
    *,
    out_dir: Path,
    base_out_dir: Path,
    data_root: Path,
    split_csv: Path,
    selection: dict[str, Any],
    params: dict[str, Any],
    seeds: list[int],
    wandb_project: str,
    wandb_entity: str | None,
    wandb_mode: str,
    device: str,
    workers: int,
    resume: bool,
    dry_run: bool,
) -> dict[str, Any]:
    """Run shared full baseline + ablation conditions for Item 3.

    Conditions: full (shared baseline), no_augmentation, no_class_weights
    Each for all five seeds.
    """
    architecture = selection["reference_architecture"]
    split_sha256 = hashlib.sha256(split_csv.read_bytes()).hexdigest()
    best = params[architecture]
    base_cfg = {
        "optimizer": best["optimizer"],
        "learning_rate": best["learning_rate"],
        "weight_decay": best["weight_decay"],
        "batch_size": best["batch_size"],
        "epochs": 30,
        "patience": 7,
        "input_size": 224,
        "black_threshold": 10,
        "augmentation": True,
        "use_class_weights": True,
    }

    conditions = [
        ("full",              {**base_cfg}),
        ("no_augmentation",  {**base_cfg, "augmentation": False}),
        ("no_class_weights", {**base_cfg, "use_class_weights": False}),
    ]

    registry: list[dict[str, Any]] = _load_or_init_registry(out_dir)
    results_by_condition: dict[str, list[dict[str, Any]]] = {}

    for condition, cfg in conditions:
        mode = "full"
        cond_results: list[dict[str, Any]] = []
        for seed in seeds:
            seed_i = seed
            # Check resume
            existing = None
            if resume:
                existing = _find_resumable_extension_run(
                    out_dir, architecture, seed_i, condition, mode,
                    cfg, split_sha256
                )

            if dry_run:
                if existing:
                    status = "reusable"
                    cond_results.append({"architecture": architecture, "seed": seed_i,
                                         "condition": condition, "mode": mode,
                                         "status": "dry_run_reusable"})
                else:
                    cond_results.append({"architecture": architecture, "seed": seed_i,
                                         "condition": condition, "mode": mode,
                                         "status": "dry_run_new"})
                continue

            if existing:
                cond_results.append(existing)
                registry.append({"architecture": architecture, "seed": seed_i,
                                   "condition": condition, "mode": mode,
                                   "action": "reused", "run_dir": existing.get("run_dir")})
                continue

            try:
                r = _fit_extension_run(
                    root=data_root, split_csv=split_csv, out=out_dir,
                    architecture=architecture, seed=seed_i,
                    config=cfg, condition=condition, mode=mode,
                    wandb_project=wandb_project, wandb_entity=wandb_entity,
                    wandb_mode=wandb_mode, device=device, workers=workers,
                )
            except Exception as exc:
                r = {
                    "architecture": architecture, "seed": seed_i,
                    "condition": condition, "mode": mode,
                    "status": "failed", "error": f"{type(exc).__name__}: {exc}",
                }
            cond_results.append(r)
            registry.append({
                "architecture": architecture, "seed": seed_i,
                "condition": condition, "mode": mode,
                "action": "trained" if r.get("status") == "completed" else "failed",
                "run_dir": r.get("run_dir"), "error": r.get("error"),
            })

        results_by_condition[condition] = cond_results

    _save_registry(out_dir, registry)
    if not dry_run:
        _save_comparisons(out_dir, results_by_condition, seeds, architecture, selection)
    return results_by_condition


def run_extension_finetune(
    *,
    out_dir: Path,
    base_out_dir: Path,
    data_root: Path,
    split_csv: Path,
    selection: dict[str, Any],
    params: dict[str, Any],
    seeds: list[int],
    wandb_project: str,
    wandb_entity: str | None,
    wandb_mode: str,
    device: str,
    workers: int,
    resume: bool,
    dry_run: bool,
) -> dict[str, Any]:
    """Reuse shared full baseline + run frozen/partial for Item 4."""
    architecture = selection["reference_architecture"]
    split_sha256 = hashlib.sha256(split_csv.read_bytes()).hexdigest()
    best = params[architecture]
    base_cfg = {
        "optimizer": best["optimizer"],
        "learning_rate": best["learning_rate"],
        "weight_decay": best["weight_decay"],
        "batch_size": best["batch_size"],
        "epochs": 30,
        "patience": 7,
        "input_size": 224,
        "black_threshold": 10,
        "augmentation": True,
        "use_class_weights": True,
    }

    conditions = [
        ("full",    "full",    {**base_cfg}),
        ("frozen",  "frozen",  {**base_cfg}),
        ("partial", "partial", {**base_cfg}),
    ]

    registry = _load_or_init_registry(out_dir)
    results_by_condition: dict[str, list[dict[str, Any]]] = {}

    for condition, mode, cfg in conditions:
        cond_results: list[dict[str, Any]] = []
        for seed in seeds:
            existing = None
            if resume:
                existing = _find_resumable_extension_run(
                    out_dir, architecture, seed, condition, mode,
                    cfg, split_sha256
                )

            if dry_run:
                status = "dry_run_reusable" if existing else "dry_run_new"
                cond_results.append({"architecture": architecture, "seed": seed,
                                      "condition": condition, "mode": mode, "status": status})
                continue

            if existing:
                cond_results.append(existing)
                registry.append({"architecture": architecture, "seed": seed,
                                   "condition": condition, "mode": mode,
                                   "action": "reused", "run_dir": existing.get("run_dir")})
                continue

            try:
                r = _fit_extension_run(
                    root=data_root, split_csv=split_csv, out=out_dir,
                    architecture=architecture, seed=seed,
                    config=cfg, condition=condition, mode=mode,
                    wandb_project=wandb_project, wandb_entity=wandb_entity,
                    wandb_mode=wandb_mode, device=device, workers=workers,
                )
            except Exception as exc:
                r = {
                    "architecture": architecture, "seed": seed,
                    "condition": condition, "mode": mode,
                    "status": "failed", "error": f"{type(exc).__name__}: {exc}",
                }
            cond_results.append(r)
            registry.append({
                "architecture": architecture, "seed": seed,
                "condition": condition, "mode": mode,
                "action": "trained" if r.get("status") == "completed" else "failed",
                "run_dir": r.get("run_dir"), "error": r.get("error"),
            })

        results_by_condition[condition] = cond_results

    _save_registry(out_dir, registry)
    if not dry_run:
        _save_comparisons(out_dir, results_by_condition, seeds, architecture, selection)
    return results_by_condition


# ---------------------------------------------------------------------------
# Registry helpers
# ---------------------------------------------------------------------------

def _load_or_init_registry(out_dir: Path) -> list[dict[str, Any]]:
    path = out_dir / "run_registry.json"
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return []
    return []


def _save_registry(out_dir: Path, registry: list[dict[str, Any]]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    save_json(out_dir / "run_registry.json", registry)


# ---------------------------------------------------------------------------
# Comparison statistics (paired Holm, mean ± SD)
# ---------------------------------------------------------------------------

def _save_comparisons(
    out_dir: Path,
    results_by_condition: dict[str, list[dict[str, Any]]],
    seeds: list[int],
    architecture: str,
    selection: dict[str, Any],
) -> None:
    """Compute and save paired tests, summaries, and delta tables."""
    from scipy.stats import ttest_rel

    comp_dir = out_dir / "comparisons"
    comp_dir.mkdir(parents=True, exist_ok=True)

    # Merge all completed conditions from out_dir / "experiments" so all conditions are included
    merged_results: dict[str, list[dict[str, Any]]] = dict(results_by_condition)
    exp_dir = out_dir / "experiments"
    if exp_dir.exists():
        for cond_folder in sorted(exp_dir.iterdir()):
            if cond_folder.is_dir():
                cond_runs = []
                for run_file in sorted(cond_folder.glob("*/run_metrics.json")):
                    try:
                        data = json.loads(run_file.read_text(encoding="utf-8"))
                        if data.get("status") == "completed":
                            cond_runs.append(data)
                    except Exception:
                        pass
                if cond_runs:
                    merged_results[cond_folder.name] = cond_runs

    # Order conditions in canonical sequence
    order = ["full", "no_augmentation", "no_class_weights", "frozen", "partial"]
    results_by_condition = {c: merged_results[c] for c in order if c in merged_results}
    for c, runs in merged_results.items():
        if c not in results_by_condition:
            results_by_condition[c] = runs

    # Build seed-indexed tables: condition → seed → metrics
    seed_metrics: dict[str, dict[int, dict[str, float]]] = {}
    for condition, runs in results_by_condition.items():
        seed_metrics[condition] = {}
        for r in runs:
            if r.get("status") == "completed":
                s = int(r["seed"])
                seed_metrics[condition][s] = {
                    "test_acc": float(r["test_acc"]),
                    "test_macro_f1": float(r["test_macro_f1"]),
                }

    # Summary table: mean ± SD per condition
    summary_rows: list[dict[str, Any]] = []
    for condition, runs in results_by_condition.items():
        completed = [r for r in runs if r.get("status") == "completed"]
        accs = [float(r["test_acc"]) for r in completed]
        f1s = [float(r["test_macro_f1"]) for r in completed]
        summary_rows.append({
            "architecture": architecture, "condition": condition,
            "n_completed": len(completed),
            "test_acc_mean": float(np.mean(accs)) if accs else None,
            "test_acc_sd": float(np.std(accs, ddof=1)) if len(accs) >= 2 else None,
            "test_macro_f1_mean": float(np.mean(f1s)) if f1s else None,
            "test_macro_f1_sd": float(np.std(f1s, ddof=1)) if len(f1s) >= 2 else None,
        })
    save_csv(comp_dir / "condition_summary.csv", summary_rows)

    # Planned contrasts vs "full" baseline — 8 comparisons (4 variants × 2 metrics)
    # Lock family: no_augmentation-full, no_class_weights-full, frozen-full, partial-full
    # × accuracy, macro_f1
    if "full" not in seed_metrics:
        save_json(comp_dir / "statistical_tests.json", {"status": "no_full_baseline"})
        return

    planned_variants = [c for c in ("no_augmentation", "no_class_weights", "frozen", "partial")
                        if c in seed_metrics]
    metrics_list = ["test_acc", "test_macro_f1"]
    family_size = len(planned_variants) * len(metrics_list)

    test_records: list[dict[str, Any]] = []
    for variant in planned_variants:
        for metric in metrics_list:
            full_map = seed_metrics.get("full", {})
            var_map = seed_metrics.get(variant, {})
            shared_seeds = sorted(set(full_map) & set(var_map))
            if len(shared_seeds) < 2:
                test_records.append({
                    "reference": "full", "variant": variant, "metric": metric,
                    "paired_n": len(shared_seeds),
                    "mean_delta": None, "t_statistic": None, "p_value": None,
                    "p_holm": None, "significant_holm": None,
                    "status": "incomplete_pairs",
                    "note": "delta = variant minus full",
                    "alpha": 0.05,
                })
                continue
            full_vals = [full_map[s][metric] for s in shared_seeds]
            var_vals = [var_map[s][metric] for s in shared_seeds]
            diffs = np.array(var_vals) - np.array(full_vals)  # variant − full
            if not np.all(np.isfinite(diffs)) or np.std(diffs, ddof=1) == 0:
                test_records.append({
                    "reference": "full", "variant": variant, "metric": metric,
                    "paired_n": len(shared_seeds),
                    "mean_delta": float(np.mean(diffs)) if np.all(np.isfinite(diffs)) else None,
                    "t_statistic": None, "p_value": None,
                    "p_holm": None, "significant_holm": None,
                    "status": "zero_variance_or_nonfinite",
                    "note": "delta = variant minus full",
                    "alpha": 0.05,
                })
                continue
            result_t = ttest_rel(var_vals, full_vals, alternative="two-sided")
            test_records.append({
                "reference": "full", "variant": variant, "metric": metric,
                "paired_n": len(shared_seeds),
                "mean_delta": float(np.mean(diffs)),
                "t_statistic": float(result_t.statistic),
                "p_value": float(result_t.pvalue),
                "status": "ok",
                "note": "delta = variant minus full",
                "alpha": 0.05,
            })

    # Holm correction over the 8-comparison family
    valid = [r for r in test_records if r.get("p_value") is not None]
    valid.sort(key=lambda r: r["p_value"])
    remaining = family_size
    running_max = 0.0
    for row in valid:
        running_max = max(running_max, row["p_value"] * remaining)
        row["p_holm"] = min(1.0, running_max)
        row["significant_holm"] = bool(row["p_holm"] < 0.05)
        remaining -= 1
    for row in test_records:
        row.setdefault("p_holm", None)
        row.setdefault("significant_holm", None)

    save_csv(comp_dir / "statistical_tests.csv", test_records)
    save_json(comp_dir / "statistical_tests.json", {
        "records": test_records,
        "family_size": family_size,
        "family_description": "4 variants × 2 metrics; separate from primary 8-comparison family",
        "alpha": 0.05,
        "n_seeds": len(seeds),
        "seed_variation_note": (
            "n=5 reflects seed variation on one fixed split. "
            "This is exploratory analysis on a test split already inspected during final evaluation."
        ),
    })

    # Per-class deltas
    delta_rows: list[dict[str, Any]] = []
    for variant in planned_variants:
        var_runs = {int(r["seed"]): r for r in results_by_condition.get(variant, [])
                    if r.get("status") == "completed"}
        full_runs = {int(r["seed"]): r for r in results_by_condition.get("full", [])
                     if r.get("status") == "completed"}
        shared = sorted(set(var_runs) & set(full_runs))
        for cls in CLASSES:
            for metric in ("precision", "recall", "f1"):
                deltas = []
                for s in shared:
                    vr = var_runs[s].get("per_class", {}).get(cls, {})
                    fr = full_runs[s].get("per_class", {}).get(cls, {})
                    if metric in vr and metric in fr:
                        deltas.append(float(vr[metric]) - float(fr[metric]))
                delta_rows.append({
                    "variant": variant, "class": cls, "metric": metric,
                    "n_pairs": len(deltas),
                    "mean_delta": float(np.mean(deltas)) if deltas else None,
                    "sd_delta": float(np.std(deltas, ddof=1)) if len(deltas) >= 2 else None,
                })
    save_csv(comp_dir / "per_class_deltas.csv", delta_rows)


In [ ]:
%%project_source shrine_classifier_ext/extension_report.py
"""Extension report generator.

Summarises whatever artifacts actually exist in outputs_extensions_v1/.
Never starts training. Reports missing/incomplete outputs explicitly.
Generates slide-ready figures for comparison results.
"""

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np

from shrine_classifier.analysis import save_json, save_csv
from shrine_classifier.data import CLASSES


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _read_csv(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def _read_json(path: Path) -> Any | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


# ---------------------------------------------------------------------------
# Figure helpers
# ---------------------------------------------------------------------------

def _plot_condition_summary(summary_rows: list[dict[str, Any]], out_dir: Path) -> None:
    """Bar chart of mean test Macro F1 per condition with seed dots and SD error bars."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return

    if not summary_rows:
        return

    conditions = [r["condition"] for r in summary_rows if r.get("condition")]
    means = []
    sds = []
    for r in summary_rows:
        m = r.get("test_macro_f1_mean")
        s = r.get("test_macro_f1_sd")
        means.append(float(m) if m is not None else 0.0)
        sds.append(float(s) if s is not None else 0.0)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(conditions))
    bars = ax.bar(x, means, yerr=sds, capsize=4, color="#1565C0", alpha=0.75,
                  error_kw={"elinewidth": 1.5, "ecolor": "#333"})

    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=20, ha="right")
    ax.set_ylabel("Mean Test Macro F1 (± sample SD, n=5)")
    ax.set_title("Extension conditions — Test Macro F1\n"
                 "[EXPLORATORY: test set already inspected during final evaluation]")
    ax.set_ylim(max(0, min(means) - 0.1) if means else 0, 1.0)
    ax.grid(alpha=0.25, axis="y")

    note = ("Exploratory comparison on a test split inspected during the primary final evaluation. "
            "n=5 reflects seed variation on one fixed split only.")
    fig.text(0.5, -0.06, note, ha="center", fontsize=7, style="italic", wrap=True)
    fig.tight_layout()
    path = out_dir / "condition_comparison_macro_f1.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def _plot_condition_accuracy(summary_rows: list[dict[str, Any]], out_dir: Path) -> None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return

    if not summary_rows:
        return

    conditions = [r["condition"] for r in summary_rows if r.get("condition")]
    means = [float(r["test_acc_mean"]) if r.get("test_acc_mean") is not None else 0.0 for r in summary_rows]
    sds = [float(r["test_acc_sd"]) if r.get("test_acc_sd") is not None else 0.0 for r in summary_rows]

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(conditions))
    ax.bar(x, means, yerr=sds, capsize=4, color="#2E7D32", alpha=0.75,
           error_kw={"elinewidth": 1.5, "ecolor": "#333"})
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=20, ha="right")
    ax.set_ylabel("Mean Test Accuracy (± sample SD, n=5)")
    ax.set_title("Extension conditions — Test Accuracy\n[EXPLORATORY]")
    ax.set_ylim(max(0, min(means) - 0.1) if means else 0, 1.0)
    ax.grid(alpha=0.25, axis="y")
    fig.tight_layout()
    path = out_dir / "condition_comparison_accuracy.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# Main report
# ---------------------------------------------------------------------------

def run_report(out_dir: Path, base_out_dir: Path) -> dict[str, Any]:
    """Summarise all extension artifacts and generate slide-ready figures."""
    report_dir = out_dir / "report_assets"
    report_dir.mkdir(parents=True, exist_ok=True)

    status: dict[str, str] = {}

    # --- Confidence analysis ---
    conf_run_metrics = _read_csv(out_dir / "confidence" / "run_metrics.csv")
    conf_arch_summary = _read_csv(out_dir / "confidence" / "architecture_summary.csv")
    conf_bin_metrics = _read_csv(out_dir / "confidence" / "bin_metrics.csv")
    conf_hce = _read_csv(out_dir / "confidence" / "high_confidence_errors.csv")
    status["confidence_run_metrics"] = "present" if conf_run_metrics else "missing"
    status["confidence_arch_summary"] = "present" if conf_arch_summary else "missing"

    # --- Diagnostics analysis ---
    diag_dynamics = _read_csv(out_dir / "diagnostics" / "run_dynamics.csv")
    diag_arch_summary = _read_csv(out_dir / "diagnostics" / "architecture_dynamics_summary.csv")
    diag_trials = _read_csv(out_dir / "diagnostics" / "trial_inventory.csv")
    status["diagnostics_run_dynamics"] = "present" if diag_dynamics else "missing"
    status["diagnostics_arch_summary"] = "present" if diag_arch_summary else "missing"

    # --- Comparisons (extension experiments) ---
    comp_summary = _read_csv(out_dir / "comparisons" / "condition_summary.csv")
    comp_tests = _read_csv(out_dir / "comparisons" / "statistical_tests.csv")
    comp_per_class = _read_csv(out_dir / "comparisons" / "per_class_deltas.csv")
    status["comparisons_summary"] = "present" if comp_summary else "missing"
    status["comparisons_statistical_tests"] = "present" if comp_tests else "missing"

    # --- Protocol ---
    protocol = _read_json(out_dir / "protocol.json")
    status["protocol"] = "present" if protocol else "missing"

    # --- Figures for presentation ---
    if comp_summary:
        _plot_condition_summary(comp_summary, report_dir)
        _plot_condition_accuracy(comp_summary, report_dir)
        status["figure_condition_macro_f1"] = "generated"
        status["figure_condition_accuracy"] = "generated"
    else:
        status["figure_condition_macro_f1"] = "skipped_no_data"
        status["figure_condition_accuracy"] = "skipped_no_data"

    # --- README ---
    readme_lines = [
        "# Extension Report Assets",
        "",
        "Generated by `pipeline_extensions.py --stage report`.",
        "This is supplementary to the primary report; it does not overwrite primary outputs.",
        "",
        "## Artifact status",
        "",
    ]
    for key, val in sorted(status.items()):
        icon = "✅" if val in ("present", "generated") else "❌"
        readme_lines.append(f"- {icon} `{key}`: {val}")
    readme_lines += [
        "",
        "## Key outputs",
        "",
        "| File | Description |",
        "|---|---|",
        "| `confidence/run_metrics.csv` | Per-run ECE, correct/wrong confidence stats |",
        "| `confidence/architecture_summary.csv` | Per-arch ECE mean ± SD |",
        "| `confidence/reliability_<arch>.png` | Reliability diagram (representative seed) |",
        "| `confidence/confidence_hist_<arch>.png` | Confidence histogram |",
        "| `diagnostics/run_dynamics.csv` | Per-run best/stop epoch, gaps |",
        "| `diagnostics/training_gaps.png` | Acc/F1/loss gap figures |",
        "| `diagnostics/tuning_scatter.png` | LR/WD vs val Macro F1 |",
        "| `comparisons/condition_summary.csv` | Mean ± SD per condition |",
        "| `comparisons/statistical_tests.csv` | Paired Holm tests |",
        "| `report_assets/condition_comparison_*.png` | Slide-ready comparison bar charts |",
        "",
        "## Caveats",
        "",
        "- Extension experiments (ablation / fine-tuning modes) use a test set that was "
          "already inspected during the primary final evaluation. These are **exploratory** results.",
        "- n=5 seeds on one fixed split reflects seed variation only, not dataset uncertainty.",
        "- Holm family for extension: 4 variants × 2 metrics = 8 comparisons. "
          "Separate from the primary 8-comparison family.",
    ]
    (report_dir / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")

    # Summary JSON
    summary = {
        "status": status,
        "n_confidence_runs": len(conf_run_metrics),
        "n_diagnostic_dynamics": len(diag_dynamics),
        "n_comp_conditions": len(comp_summary),
        "n_stat_tests": len(comp_tests),
    }
    save_json(out_dir / "extension_summary.json", summary)

    # Also write Markdown extension summary
    md_lines = [
        "# Extension summary",
        "",
        "## Artifact status",
        "",
    ]
    for key, val in sorted(status.items()):
        md_lines.append(f"- `{key}`: **{val}**")
    md_lines += [
        "",
        "## Confidence analysis (Item 1)",
        f"- Runs analysed: {len(conf_run_metrics)}",
        f"- Architectures with ECE summary: {len(conf_arch_summary)}",
        "",
        "## Training dynamics (Item 2)",
        f"- Final runs analysed: {len(diag_dynamics)}",
        f"- Trial inventory entries: {len(diag_trials)}",
        "",
        "## Extension experiments (Items 3–4)",
        f"- Conditions with results: {len(comp_summary)}",
        f"- Paired tests recorded: {len(comp_tests)}",
        "",
        "> This summary was generated from existing artifacts only. "
        "Missing entries above indicate stages not yet completed.",
    ]
    (out_dir / "extension_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

    return {"status": status, "out_dir": str(report_dir)}


In [ ]:
%%project_source shrine_classifier_ext/finetuning_modes.py
"""Fine-tuning mode policies for items 3 & 4.

Provides:
  - apply_finetuning_mode(model, architecture, mode) → sets parameters trainable/frozen
  - enforce_mode_each_epoch(model, architecture, mode) → called after model.train() per epoch
  - log_mode_stats(model, architecture, mode) → returns dict with trainable/frozen counts

Modes:
  full     — all parameters trainable (default primary recipe; backbone in train mode)
  frozen   — backbone frozen+eval, head trainable+train
  partial  — backbone final stage + head trainable+train; rest frozen+eval
"""

from __future__ import annotations

from typing import Any

MODES = ("full", "frozen", "partial")


# ---------------------------------------------------------------------------
# Architecture-specific module resolution
# ---------------------------------------------------------------------------

def _get_head_modules(model, architecture: str):
    """Return a list of head modules for this architecture."""
    if architecture == "resnet50":
        return [model.fc]
    elif architecture in ("efficientnet_b0", "mobilenet_v3_large", "convnext_tiny"):
        return [model.classifier]
    elif architecture == "densenet121":
        return [model.classifier]
    raise ValueError(f"Unknown architecture: {architecture}")


def _get_partial_backbone_modules(model, architecture: str):
    """Return a list of modules that form the 'final backbone stage' for partial mode."""
    import torch.nn as nn

    if architecture == "resnet50":
        # Final stage: layer4
        return [model.layer4]

    elif architecture == "densenet121":
        # Final dense block + norm: features.denseblock4 and features.norm5
        # Do NOT train only features[-1] — must explicitly open both
        parts = []
        features = model.features
        if hasattr(features, "denseblock4"):
            parts.append(features.denseblock4)
        else:
            raise AttributeError("densenet121.features has no 'denseblock4' — check torchvision version")
        if hasattr(features, "norm5"):
            parts.append(features.norm5)
        else:
            raise AttributeError("densenet121.features has no 'norm5' — check torchvision version")
        return parts

    elif architecture == "convnext_tiny":
        # features is a Sequential of stages; the final ConvNeXt stage is features[-1]
        # Detect actual structure
        features = model.features
        if len(features) < 1:
            raise AttributeError("convnext_tiny.features appears empty; check torchvision version")
        # Final stage
        return [features[-1]]

    elif architecture == "efficientnet_b0":
        # features is a Sequential of MBConv stages.
        # Final MBConv stage = features[-2] (features[-1] is the projection Conv2dNormActivation)
        features = model.features
        n = len(features)
        if n < 2:
            raise AttributeError(f"efficientnet_b0.features too short (len={n})")
        # Final stage group + final projection
        parts = [features[-2], features[-1]]
        return parts

    elif architecture == "mobilenet_v3_large":
        # The torchvision MobileNetV3-Large tail has a semantic repeated stage
        # (three 160-channel inverted residual blocks in 0.28) followed by the
        # 1x1 feature projection. Resolve its boundary from module metadata.
        features = model.features
        n = len(features)
        if n < 2:
            raise AttributeError(f"mobilenet_v3_large.features too short (len={n})")
        last_inverted = next((i for i in range(n-1, -1, -1)
                              if type(features[i]).__name__ == "InvertedResidual"), None)
        if last_inverted is None or not hasattr(features[last_inverted], "out_channels"):
            raise AttributeError("Cannot resolve MobileNetV3-Large final InvertedResidual stage; inspect torchvision module names")
        stage_channels = features[last_inverted].out_channels
        start = last_inverted
        while start > 0 and type(features[start-1]).__name__ == "InvertedResidual" \
                and getattr(features[start-1], "out_channels", None) == stage_channels:
            start -= 1
        return [*features[start:last_inverted+1], *features[last_inverted+1:]]

    raise ValueError(f"Unknown architecture: {architecture}")


def _get_all_backbone_modules(model, architecture: str):
    """Return all backbone modules (everything except the final classification head)."""
    if architecture == "resnet50":
        return [model.conv1, model.bn1, model.relu, model.maxpool,
                model.layer1, model.layer2, model.layer3, model.layer4,
                model.avgpool]
    elif architecture == "efficientnet_b0":
        return [model.features, model.avgpool]
    elif architecture == "mobilenet_v3_large":
        return [model.features, model.avgpool]
    elif architecture == "densenet121":
        return [model.features]
    elif architecture == "convnext_tiny":
        return [model.features, model.avgpool]
    raise ValueError(f"Unknown architecture: {architecture}")


# ---------------------------------------------------------------------------
# Module helpers
# ---------------------------------------------------------------------------

def _set_requires_grad(module, requires_grad: bool) -> None:
    for param in module.parameters():
        param.requires_grad = requires_grad


def _set_eval_recursive(module) -> None:
    """Set a module and all submodules to eval mode (disables BN updates, dropout etc.)."""
    module.eval()


def _set_train_recursive(module) -> None:
    module.train()


def _collect_named_modules(module_list) -> list[str]:
    """Collect all fully-qualified module names from a list of module objects."""
    return [type(m).__name__ for m in module_list]


# ---------------------------------------------------------------------------
# Mode application
# ---------------------------------------------------------------------------

def apply_finetuning_mode(model, architecture: str, mode: str) -> dict[str, Any]:
    """Apply the trainability policy.

    Must be called after model is constructed (pretrained + head replaced) and
    before the optimizer is created (optimizer gets model.parameters() filtered here).

    Returns a stats dict for logging.
    """
    if mode not in MODES:
        raise ValueError(f"mode must be one of {MODES}, got {mode!r}")

    if mode == "full":
        # All parameters trainable; model.train() will be called before each epoch normally
        for param in model.parameters():
            param.requires_grad = True
        return _log_mode_stats(model, architecture, mode)

    elif mode == "frozen":
        # Backbone frozen+eval; only head is trained
        for m in _get_all_backbone_modules(model, architecture):
            _set_requires_grad(m, False)
        # Ensure head is trainable
        for m in _get_head_modules(model, architecture):
            _set_requires_grad(m, True)
        return _log_mode_stats(model, architecture, mode)

    elif mode == "partial":
        # First freeze everything, then unfreeze partial backbone + head
        for param in model.parameters():
            param.requires_grad = False
        for m in _get_partial_backbone_modules(model, architecture):
            _set_requires_grad(m, True)
        for m in _get_head_modules(model, architecture):
            _set_requires_grad(m, True)
        return _log_mode_stats(model, architecture, mode)

    raise ValueError(f"Unknown mode: {mode}")


def enforce_mode_per_epoch(model, architecture: str, mode: str) -> None:
    """Re-apply eval/train modes after model.train().

    Must be called AFTER model.train() at the start of each training epoch
    so that frozen BatchNorm buffers and dropout are not accidentally activated.
    """
    if mode == "full":
        return  # Nothing to override; all modules train normally

    elif mode == "frozen":
        # Backbone stays in eval; head stays in train
        for m in _get_all_backbone_modules(model, architecture):
            _set_eval_recursive(m)
        for m in _get_head_modules(model, architecture):
            _set_train_recursive(m)

    elif mode == "partial":
        # Frozen backbone modules stay in eval; partial + head in train
        all_backbone = _get_all_backbone_modules(model, architecture)
        trainable_partial = _get_partial_backbone_modules(model, architecture)
        trainable_ids = {id(m) for m in trainable_partial}
        for m in all_backbone:
            if id(m) in trainable_ids:
                _set_train_recursive(m)
            else:
                # Check if any submodule of m is in trainable set
                has_trainable_child = any(id(child) in trainable_ids
                                          for child in m.modules())
                if has_trainable_child:
                    # Mixed: individually set children
                    for child_name, child in m.named_children():
                        if id(child) in trainable_ids:
                            _set_train_recursive(child)
                        else:
                            _set_eval_recursive(child)
                else:
                    _set_eval_recursive(m)
        for m in _get_head_modules(model, architecture):
            _set_train_recursive(m)


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

def _log_mode_stats(model, architecture: str, mode: str) -> dict[str, Any]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    # Record exact qualified policy modules (e.g. features.13, features.14,
    # features.15 and features.16 for the current MobileNet semantic tail).
    if mode == "full":
        trainable_modules = [name for name, module in model.named_children()
                             if any(p.requires_grad for p in module.parameters())]
    else:
        selected = _get_head_modules(model, architecture)
        if mode == "partial":
            selected = _get_partial_backbone_modules(model, architecture) + selected
        selected_ids = {id(module) for module in selected}
        trainable_modules = [name for name, module in model.named_modules() if id(module) in selected_ids]

    return {
        "architecture": architecture,
        "mode": mode,
        "total_parameters": total,
        "trainable_parameters": trainable,
        "frozen_parameters": frozen,
        "trainable_module_names": trainable_modules,
        "bn_buffer_policy": (
            "frozen backbone BN buffers do not update (module in eval mode during training)"
            if mode in ("frozen", "partial")
            else "all BN buffers update normally (full fine-tuning)"
        ),
    }


def log_mode_stats(model, architecture: str, mode: str) -> dict[str, Any]:
    """Public API to log mode stats at any point (read-only)."""
    return _log_mode_stats(model, architecture, mode)


# ---------------------------------------------------------------------------
# Gradient/weight invariant tests (for test suite)
# ---------------------------------------------------------------------------

def test_gradient_invariants(model, architecture: str, mode: str, device) -> dict[str, Any]:
    """Run a forward+backward pass and verify gradient/weight invariants.

    Returns a dict with pass/fail status and details.
    """
    import torch
    import torch.nn as nn
    import copy

    model = model.to(device)
    model.train()
    enforce_mode_per_epoch(model, architecture, mode)

    # Save a copy of all parameters
    before = {name: p.detach().clone() for name, p in model.named_parameters()}

    # Dummy forward pass
    dummy = torch.zeros(1, 3, 224, 224, device=device)
    out = model(dummy)
    loss = out.sum()
    loss.backward()

    failures: list[str] = []

    # Check that frozen parameters have no gradient or zero gradient
    for name, param in model.named_parameters():
        if not param.requires_grad:
            if param.grad is not None and param.grad.abs().sum() > 0:
                failures.append(f"Frozen param {name} has non-zero gradient")

    # Check that trainable parameters have non-zero gradient (for this dummy input)
    for name, param in model.named_parameters():
        if param.requires_grad:
            if param.grad is None:
                failures.append(f"Trainable param {name} has no gradient after backward")
            # Note: may be zero for certain activations with dummy all-zero input; don't hard-fail on value

    # Simulate an optimizer step and verify frozen params unchanged
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3
    )
    optimizer.step()
    optimizer.zero_grad()

    after = {name: p.detach().clone() for name, p in model.named_parameters()}
    for name in before:
        param = dict(model.named_parameters())[name]
        if not param.requires_grad:
            if not torch.allclose(before[name], after[name], atol=1e-12):
                failures.append(f"Frozen param {name} changed after optimizer step")

    return {
        "architecture": architecture,
        "mode": mode,
        "passed": not failures,
        "failures": failures,
    }


## 4. ตัวรันและเครื่องมือ — source code

pipeline_modes.py เป็นตัวรันหลัก 15 setups; resnet_followup.py ทำ ResNet CAM/ablation; plot_learning_curves.py อ่าน saved histories เท่านั้น ส่วน pipeline.py และ pipeline_extensions.py เป็น entry points เดิมที่ยังต้องเก็บสำหรับ imports/tests ห้ามใช้ protocol เก่าแทนการทดลองปัจจุบัน


In [ ]:
%%project_source pipeline.py
from __future__ import annotations

import argparse
import csv
import json
import platform
import sys
from pathlib import Path

from shrine_classifier.data import CLASSES, audit_dataset, create_split


def parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Thai shrine image classification experiment pipeline")
    p.add_argument("--stage", required=True, choices=["audit", "prep", "baseline", "screen", "tune", "final", "gradcam", "eyeball", "report", "ablation", "smoke", "all"])
    p.add_argument("--data-root", type=Path, default=Path("dataset_cleaned"))
    p.add_argument("--padded-root", type=Path, default=Path("dataset_padding"))
    p.add_argument("--out-dir", type=Path, default=Path("outputs"))
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--runs", type=int)
    p.add_argument("--trials", type=int, default=12, help="Number of trials per architecture")
    p.add_argument("--input-size", type=int, default=224)
    p.add_argument("--wandb-project", default="tordave1150-thai-shrine-v1")
    p.add_argument("--wandb-entity")
    p.add_argument("--wandb-mode", choices=["online", "offline", "disabled"], default="online")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--split-seed", type=int, default=42)
    p.add_argument("--seeds", help="Comma-separated explicit final seed list")
    p.add_argument("--architectures", nargs="+", choices=["resnet50", "efficientnet_b0", "mobilenet_v3_large", "densenet121", "convnext_tiny"], default=["resnet50", "efficientnet_b0", "mobilenet_v3_large", "densenet121", "convnext_tiny"])
    p.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")
    p.add_argument("--num-workers", type=int, default=0)
    p.add_argument("--black-threshold", type=int, default=10)
    p.add_argument("--resume", action="store_true")
    p.add_argument("--dry-run", action="store_true")
    p.add_argument("--near-threshold", type=int, default=6)
    p.add_argument("--duplicate-review", type=Path, help="CSV with id_a,id_b,review_status columns")
    return p


def require_torchvision():
    try:
        import torch
        import torchvision
        return torch, torchvision
    except Exception as exc:
        raise RuntimeError("Model/training stages need compatible PyTorch and torchvision. Install the runtime-specific official PyTorch build; no random-weight fallback is used.") from exc


def main(argv: list[str] | None = None) -> int:
    args = parser().parse_args(argv)
    out = args.out_dir.resolve()
    stage = args.stage
    if stage == "smoke" and args.out_dir == Path("outputs"):
        out = (Path.cwd() / "outputs_smoke").resolve()
    if args.dry_run:
        count = len(args.architectures)
        runs = args.runs if args.runs is not None else 5
        print(json.dumps({"stage": stage, "data_root": str(args.data_root.resolve()), "out_dir": str(out),
                          "architectures": args.architectures,
                          "estimated_training_runs": count * (1 + args.trials + runs) if stage == "all" else None,
                          "max_epochs": count * (1 + args.trials + runs) * args.epochs if stage == "all" else None,
                          "test_metrics_access": stage in {"final", "gradcam", "eyeball", "report", "all"}}, indent=2))
        return 0
    if stage in {"audit", "all"}:
        summary = audit_dataset(args.data_root, out / "dataset_audit", near_threshold=args.near_threshold)
        padded_summary = audit_dataset(args.padded_root, out / "dataset_audit_padding", near_threshold=args.near_threshold, require_metadata=False)
        cleaned_ids = {(Path(r["path"]).parent.name, Path(r["path"]).stem) for r in _read_properties(out / "dataset_audit" / "image_properties.csv")}
        padded_ids = {(Path(r["path"]).parent.name, Path(r["path"]).stem) for r in _read_properties(out / "dataset_audit_padding" / "image_properties.csv")}
        pair_report = {"cleaned_count": len(cleaned_ids), "padding_count": len(padded_ids),
                       "paired_by_class_and_stem": len(cleaned_ids & padded_ids),
                       "cleaned_only": sorted("/".join(k) for k in cleaned_ids - padded_ids),
                       "padding_only": sorted("/".join(k) for k in padded_ids - cleaned_ids)}
        (out / "dataset_audit" / "version_pairing.json").write_text(json.dumps(pair_report, ensure_ascii=False, indent=2), encoding="utf-8")
        _write_environment(out / "environment.json")
        print(json.dumps({"stage": "audit", "num_images": summary["num_images"], "class_counts": summary["class_counts"],
                          "duplicate_groups": summary["exact_decoded_pixel_duplicate_groups"],
                          "near_duplicate_candidates": summary["num_near_duplicate_candidates"],
                          "padding_images": padded_summary["num_images"], "version_pairs": pair_report["paired_by_class_and_stem"],
                          "errors": len(summary["errors"]) + len(padded_summary["errors"]), "outputs": str(out / "dataset_audit")}, ensure_ascii=False, indent=2))
        if summary["errors"] or padded_summary["errors"] or pair_report["cleaned_only"] or pair_report["padding_only"]:
            return 2
        if stage == "audit":
            return 0
    if stage == "prep":
        audit = out / "dataset_audit"
        if not (audit / "image_properties.csv").exists():
            raise RuntimeError("Missing audit artifacts. Run: python pipeline.py --stage audit")
        existing = out / "splits.csv"
        if existing.exists():
            _check_existing_split(out, args.data_root)
            metadata = json.loads((out / "split_metadata.json").read_text(encoding="utf-8"))
            if metadata.get("split_seed") != args.split_seed:
                raise RuntimeError("Existing canonical split uses a different seed; use a new --out-dir")
            print(json.dumps({"stage": "prep", "status": "reused_existing_split", "split": metadata}, indent=2))
            return 0
        saved_review = audit / "near_duplicate_review_decisions.csv"
        review = args.duplicate_review or (saved_review if saved_review.exists() else None)
        split = create_split(args.data_root, audit, existing, split_seed=args.split_seed, review_csv=review)
        print(json.dumps({"stage": "prep", "split": split}, ensure_ascii=False, indent=2))
        return 0
    if stage == "smoke":
        return _run_smoke(args, out)
    if stage == "all" and not (out / "splits.csv").exists():
        review = out / "dataset_audit" / "near_duplicate_review_decisions.csv"
        create_split(args.data_root, out / "dataset_audit", out / "splits.csv",
                     split_seed=args.split_seed, review_csv=review if review.exists() else args.duplicate_review)
    if stage == "all" and (out / "splits.csv").exists():
        _check_existing_split(out, args.data_root)
    if stage in {"baseline", "screen", "tune", "final", "gradcam", "eyeball", "report", "ablation", "all"}:
        if stage == "report":
            results = _make_report(out)
            print(json.dumps({"stage": stage, "outputs": str(out), "results": results}, ensure_ascii=False, indent=2))
            return 0
        if stage != "report":
            require_torchvision()
        from shrine_classifier.experiments import run_ablation, run_final, run_screen, run_tuning
        results = {}
        stages = ["baseline", "screen", "tune", "final", "gradcam", "eyeball", "report"] if stage == "all" else [stage]
        for item in stages:
            if item in {"baseline", "screen", "tune", "final", "ablation"}:
                if item != "baseline" and not (out / "splits.csv").exists():
                    raise FileNotFoundError("Missing canonical split. Run: python pipeline.py --stage audit, then --stage prep")
            if item == "baseline":
                from shrine_classifier.baseline import run_baseline
                results[item] = run_baseline(args, out)
            elif item == "screen":
                results[item] = run_screen(args, out)
            elif item == "tune":
                results[item] = run_tuning(args, out)
            elif item == "final":
                results[item] = run_final(args, out)
            elif item == "ablation":
                results[item] = run_ablation(args, out)
            elif item in {"gradcam", "eyeball"}:
                from shrine_classifier.visualization import make_analysis_grids
                results[item] = make_analysis_grids(args, out, mode=item)
            elif item == "report":
                results[item] = _make_report(out)
        print(json.dumps({"stage": stage, "outputs": str(out), "results": results}, ensure_ascii=False, indent=2, default=str))
        return 0
    return 0


def _read_properties(path: Path) -> list[dict[str, str]]:
    import csv
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def _check_existing_split(out: Path, data_root: Path) -> None:
    with (out / "splits.csv").open("r", encoding="utf-8-sig", newline="") as stream:
        split = list(csv.DictReader(stream))
    if not split:
        raise RuntimeError("Existing splits.csv is empty")
    required = {"image_id", "path", "label", "label_index", "group_id", "split"}
    if not required.issubset(split[0]):
        raise RuntimeError("Existing split schema is invalid; use a new output directory")
    group_splits = {}
    for row in split:
        if row["label"] not in CLASSES or int(row["label_index"]) != CLASSES.index(row["label"]):
            raise RuntimeError("Existing split class order/index mismatch")
        previous = group_splits.setdefault(row["group_id"], row["split"])
        if previous != row["split"]:
            raise RuntimeError("Existing duplicate group crosses splits")
    meta_path = out / "split_metadata.json"
    if not meta_path.exists():
        raise RuntimeError("Existing splits.csv lacks split_metadata.json; use a new output directory")
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    digest = __import__("hashlib").sha256("\n".join(f"{r['image_id']}:{r['split']}" for r in sorted(split, key=lambda x: x["image_id"])).encode()).hexdigest()
    if digest != meta.get("split_sha256"):
        raise RuntimeError("Existing split fingerprint mismatch; refusing to continue or regenerate")
    props_path = out / "dataset_audit" / "image_properties.csv"
    from shrine_classifier.data import groups_from_audit
    with props_path.open("r", encoding="utf-8-sig", newline="") as stream:
        props = list(csv.DictReader(stream))
    manifest = __import__("hashlib").sha256("\n".join(f"{r['image_id']}:{r['pixel_sha256']}" for r in props).encode()).hexdigest()
    review = out / "dataset_audit" / "near_duplicate_review_decisions.csv"
    groups = groups_from_audit(props, review if review.exists() else None)
    group_digest = __import__("hashlib").sha256("\n".join(f"{k}:{groups[k]}" for k in sorted(groups)).encode()).hexdigest()
    if manifest != meta.get("manifest_sha256") or group_digest != meta.get("group_sha256"):
        raise RuntimeError("Image manifest or duplicate review changed after splitting; refusing to continue. Use a new --out-dir")
    for row in props:
        current = data_root / row["path"]
        if not current.is_file() or __import__("hashlib").sha256(current.read_bytes()).hexdigest() != row["byte_sha256"]:
            raise RuntimeError(f"Image changed after the audit: {row['path']}; re-audit and use a new --out-dir")
    audit_summary = json.loads((out / "dataset_audit" / "dataset_summary.json").read_text(encoding="utf-8"))
    metadata_file = data_root / "metadata_cleaned.csv"
    if not metadata_file.exists() or __import__("hashlib").sha256(metadata_file.read_bytes()).hexdigest() != audit_summary.get("metadata", {}).get("sha256"):
        raise RuntimeError("Source metadata changed or was not audited; re-audit and use a new --out-dir")
    if review.exists() and meta.get("duplicate_review_sha256"):
        if __import__("hashlib").sha256(review.read_bytes()).hexdigest() != meta["duplicate_review_sha256"]:
            raise RuntimeError("Near-duplicate review changed after splitting; refusing to continue")


def _run_smoke(args, out: Path) -> int:
    from importlib.util import find_spec
    out.mkdir(parents=True, exist_ok=True)
    missing = [name for name in ("torch", "torchvision") if find_spec(name) is None]
    record = {"stage": "smoke", "python": platform.python_version(), "python_executable": sys.executable,
              "torch_installed": "torch" not in missing, "torchvision_installed": "torchvision" not in missing,
              "wandb_installed": find_spec("wandb") is not None,
              "pretrained_weights_tested": False, "synthetic_backward_smoke": "not_run",
              "test_split_accessed": False, "missing_dependencies": missing}
    record["gpu"] = _gpu_inventory()
    try:
        import numpy
        record["numpy"] = numpy.__version__
        import PIL
        record["pillow"] = PIL.__version__
    except Exception as exc:
        record["environment_probe_error"] = f"{type(exc).__name__}: {exc}"
    if missing:
        record["status"] = "blocked_before_model_smoke"
        (out / "smoke_environment.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        raise RuntimeError(f"Smoke could not exercise the five CNNs; missing {', '.join(missing)}. GPU availability is not a substitute for the PyTorch runtime. See {out / 'smoke_environment.json'}")
    import torch
    import torchvision
    record.update({"torch": torch.__version__, "torchvision": torchvision.__version__,
                   "cuda_available": torch.cuda.is_available(),
                   "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})
    record["status"] = "model_smoke_not_yet_run"
    from shrine_classifier.models import ARCHITECTURES, build_model
    from shrine_classifier.data import IMAGENET_MEAN, IMAGENET_STD
    import numpy as np
    torch.manual_seed(args.seed)
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)
    dummy = torch.randn(2, 3, args.input_size, args.input_size, device=device)
    results = []
    for arch in ARCHITECTURES:
        entry = {"architecture": arch, "weights": "synthetic random initialization; not a result"}
        try:
            model, metadata = build_model(arch, pretrained=False)
            model.to(device).train()
            logits = model(dummy)
            loss = torch.nn.functional.cross_entropy(logits, torch.tensor([0, 1], device=device))
            loss.backward()
            entry.update({"status": "forward_backward_passed", "logit_shape": list(logits.shape), "loss": float(loss.detach().cpu()), **metadata})
            if arch == "resnet50":
                model.eval()
                cam = __import__("shrine_classifier.analysis", fromlist=["gradcam"]).gradcam(model, arch, dummy[:1].detach(), 0)
                entry["gradcam_shape"] = list(cam.shape)
                entry["gradcam_finite"] = bool(torch.isfinite(cam).all().item())
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as exc:
            entry.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}"})
        results.append(entry)
    record["models"] = results
    record["synthetic_backward_smoke"] = "passed" if all(r["status"] == "forward_backward_passed" for r in results) else "failed"
    record["status"] = "completed" if record["synthetic_backward_smoke"] == "passed" else "failed"
    (out / "smoke_environment.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(json.dumps(record, indent=2))
    return 0 if record["status"] == "completed" else 1


def _make_report(out: Path) -> dict:
    assets = out / "report_assets"
    assets.mkdir(parents=True, exist_ok=True)
    rows = []
    missing = []
    for rel in ("dataset_audit/class_distribution.png", "dataset_audit/sample_grid.png",
                "dataset_audit/preprocessing_examples.png", "baseline/pretrained_examples.png",
                "final/architecture_summary.csv", "final/statistical_tests.csv", "gradcam/gradcam_grid.png",
                "eyeball/eyeball_grid.png"):
        (missing if not (out / rel).exists() else rows).append(rel)
    body = "# Thai shrine classifier presentation assets\n\n"
    body += "Available assets:\n" + ("\n".join(f"- `{x}`" for x in rows) if rows else "- None yet") + "\n\n"
    body += "Not yet available:\n" + ("\n".join(f"- `{x}`" for x in missing) if missing else "- None") + "\n"
    (assets / "README.md").write_text(body, encoding="utf-8")
    return {"available": rows, "missing": missing, "readme": str(assets / "README.md")}


def _gpu_inventory() -> dict:
    import subprocess
    try:
        result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                capture_output=True, text=True, timeout=5, check=False)
        return {"detected": result.returncode == 0, "inventory": result.stdout.strip(), "error": result.stderr.strip() if result.returncode else None}
    except Exception as exc:
        return {"detected": False, "error": f"{type(exc).__name__}: {exc}"}


def _write_environment(path: Path) -> None:
    import importlib.metadata as metadata
    from importlib.util import find_spec
    packages = {}
    for name in ("numpy", "Pillow", "scipy", "scikit-learn", "matplotlib", "torch", "torchvision", "wandb"):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
    record = {"timestamp_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
              "python": platform.python_version(), "python_executable": sys.executable,
              "packages": packages, "gpu": _gpu_inventory(),
              "torch_importable": bool(find_spec("torch")), "wandb_importable": bool(find_spec("wandb"))}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2), encoding="utf-8")


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except Exception as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        raise SystemExit(1)


In [ ]:
%%project_source pipeline_extensions.py
"""Historical extension diagnostics and bounded mode smoke.

Current objective (2026-10-13): independently tune every architecture-mode pair.
The fixed-Full-recipe prepare/ablation/finetune/report CLI stages are retired
and fail before creating output or starting training, even with --dry-run.

Available: analyze cached historical Full-only predictions/histories, or run
bounded smoke of reusable mode APIs. --stage is required.

The new architecture-mode tuning runner is not implemented yet. Read
AGENTS.md and docs/MODE_TUNING_PROTOCOL.md for scope, budget and next work.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

STAGES = ("analyze", "prepare", "smoke", "ablation", "finetune", "report")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Historical confidence/dynamics analysis. Fixed-Full-recipe measured stages are retired; see docs/MODE_TUNING_PROTOCOL.md.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    p.add_argument("--stage", required=True, choices=STAGES,
                   help="Stage to run (required; no default that starts training).")
    p.add_argument("--base-out-dir", type=Path, default=Path("outputs"),
                   help="Primary outputs/ directory with canonical splits, final runs, tuning (default: outputs).")
    p.add_argument("--out-dir", type=Path, default=Path("outputs_extensions_v1"),
                   help="Extension output directory (default: outputs_extensions_v1).")
    p.add_argument("--data-root", type=Path, default=Path("dataset_cleaned"),
                   help="Dataset root (default: dataset_cleaned).")
    p.add_argument("--device", default="auto",
                   help="Device: auto | cpu | cuda (default: auto).")
    p.add_argument("--num-workers", type=int, default=0,
                   help="DataLoader workers (default: 0).")
    p.add_argument("--wandb-project", default="thai-shrine-classifier",
                   help="W&B project name.")
    p.add_argument("--wandb-entity", default=None,
                   help="W&B entity (team/user); uses default if not set.")
    p.add_argument("--wandb-mode", default="online", choices=("online", "offline", "disabled"),
                   help="W&B tracking mode (default: online).")
    p.add_argument("--resume", action="store_true",
                   help="Reuse completed extension runs whose protocol/config/split match.")
    p.add_argument("--dry-run", action="store_true",
                   help="Show what would be run without training; report new/reusable/failed runs.")
    return p.parse_args(argv)


# ---------------------------------------------------------------------------
# Prerequisites helpers
# ---------------------------------------------------------------------------

def _require_file(path: Path, hint: str) -> None:
    if not path.exists():
        sys.exit(f"ERROR: Required file not found: {path}\nRun: {hint}")


def _load_selection(base_out_dir: Path) -> dict:
    sel_path = base_out_dir / "selection.json"
    _require_file(sel_path, "python pipeline.py --stage tune --base-out-dir ...")
    return json.loads(sel_path.read_text(encoding="utf-8"))


def _load_params(base_out_dir: Path) -> dict:
    params_path = base_out_dir / "tuning" / "best_params_by_arch.json"
    _require_file(params_path, "python pipeline.py --stage tune --base-out-dir ...")
    return json.loads(params_path.read_text(encoding="utf-8"))


def _load_split_csv(base_out_dir: Path) -> Path:
    split_path = base_out_dir / "splits.csv"
    _require_file(split_path, "python pipeline.py --stage prep --base-out-dir ...")
    return split_path


# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------

def _collect_environment(device: str) -> dict:
    env: dict = {
        "python_version": sys.version,
        "platform": platform.platform(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "device_requested": device,
    }
    try:
        import torch
        env["torch_version"] = torch.__version__
        env["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            env["cuda_device_name"] = torch.cuda.get_device_name(0)
            env["cuda_device_memory_mib"] = torch.cuda.get_device_properties(0).total_memory // (1024 * 1024)
    except ImportError:
        env["torch_version"] = "not_installed"
        env["cuda_available"] = False
    try:
        import torchvision
        env["torchvision_version"] = torchvision.__version__
    except ImportError:
        env["torchvision_version"] = "not_installed"
    try:
        import wandb
        env["wandb_version"] = getattr(wandb, "__version__", "installed_no_version")
    except ImportError:
        env["wandb_version"] = "not_installed"
    try:
        import numpy as np
        env["numpy_version"] = np.__version__
    except ImportError:
        pass
    try:
        import PIL
        env["pillow_version"] = PIL.__version__
    except ImportError:
        pass
    try:
        import sklearn
        env["sklearn_version"] = sklearn.__version__
    except ImportError:
        pass
    return env


# ---------------------------------------------------------------------------
# Stage: analyze
# ---------------------------------------------------------------------------

def stage_analyze(args) -> None:
    print("[analyze] Running confidence/reliability analysis and training dynamics from cached artifacts...")
    base = args.base_out_dir
    out = args.out_dir
    out.mkdir(parents=True, exist_ok=True)

    # Verify prerequisites
    _require_file(base / "final" / "run_metrics.json",
                  "python pipeline.py --stage final ...")
    selection = _load_selection(base)

    from shrine_classifier_ext.confidence_analysis import run_confidence_analysis
    from shrine_classifier_ext.diagnostics_analysis import run_diagnostics_analysis

    print("[analyze] Item 1 — Confidence/reliability analysis")
    conf_result = run_confidence_analysis(base, out, selection)
    n_ok = sum(1 for r in conf_result["run_rows"] if r.get("status") == "ok")
    n_issues = len(conf_result["issues"])
    print(f"  -> {n_ok} runs processed; {n_issues} issues")
    if conf_result["issues"]:
        for iss in conf_result["issues"][:5]:
            print(f"     [ISSUE] {iss}")

    print("[analyze] Item 2 — Training dynamics and tuning analysis")
    diag_result = run_diagnostics_analysis(base, out, selection)
    print(f"  -> {len(diag_result['dynamics'])} run dynamics records; "
          f"{len(diag_result['trial_rows'])} tuning trials loaded")

    print(f"[analyze] Outputs written to {out}")
    _update_execution_status(
        "analyze",
        "completed",
        f"Confidence: {n_ok} runs OK, {n_issues} issues. "
        f"Diagnostics: {len(diag_result['dynamics'])} runs, "
        f"{len(diag_result['trial_rows'])} trials.",
        out,
    )


# ---------------------------------------------------------------------------
# Stage: prepare
# ---------------------------------------------------------------------------

def stage_prepare(args) -> None:
    print("[prepare] Writing protocol, preflight, and environment files...")
    base = args.base_out_dir
    out = args.out_dir
    out.mkdir(parents=True, exist_ok=True)

    selection = _load_selection(base)
    params = _load_params(base)
    split_csv = _load_split_csv(base)

    split_sha256_csv = hashlib.sha256(split_csv.read_bytes()).hexdigest()

    # Check if protocol.json already exists → protect it
    protocol_path = out / "protocol.json"
    if protocol_path.exists():
        print(f"  [WARNING] protocol.json already exists at {protocol_path}; not overwriting.")
        print("  Create a new --out-dir for an intentional protocol revision.")
    else:
        # Build image-ID split fingerprint
        import csv as _csv
        with split_csv.open("r", encoding="utf-8-sig", newline="") as f:
            split_rows = list(_csv.DictReader(f))
        id_split_pairs = sorted(f"{r['image_id']}:{r['split']}" for r in split_rows)
        id_split_hash = hashlib.sha256("|".join(id_split_pairs).encode()).hexdigest()

        arch = selection.get("reference_architecture", "")
        best = params.get(arch, {})

        protocol = {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "status": "extension_exploratory",
            "exploratory_note": (
                "Items 3 and 4 use the test split that was already inspected during "
                "the primary final evaluation. Protocol lock prevents further cherry-picking but "
                "does not restore unseen-test status."
            ),
            "reference_architecture": arch,
            "seeds": selection.get("seeds", []),
            "representative_seed": selection.get("representative_seed", 42),
            "base_out_dir": str(base.resolve()),
            "out_dir": str(out.resolve()),
            "source_selection_path": str((base / "selection.json").resolve()),
            "source_params_path": str((base / "tuning" / "best_params_by_arch.json").resolve()),
            "split_csv_path": str(split_csv.resolve()),
            "split_sha256_csv_bytes": split_sha256_csv,
            "split_sha256_id_split_pairs": id_split_hash,
            "class_order": ["san_phraphrom", "san_phraphum", "san_tayay", "tee_ju_eia"],
            "conditions": {
                "full": "Fresh full fine-tuning baseline (shared between Items 3 and 4)",
                "no_augmentation": "Disable train augmentation only; all else identical to full",
                "no_class_weights": "Uniform cross-entropy only; all else identical to full",
                "frozen": "Frozen backbone (eval), train head only",
                "partial": "Frozen backbone except final stage, train final stage + head",
            },
            "locked_recipe": {
                "architecture": arch,
                "optimizer": best.get("optimizer"),
                "learning_rate": best.get("learning_rate"),
                "weight_decay": best.get("weight_decay"),
                "batch_size": best.get("batch_size"),
                "epochs": 30,
                "patience": 7,
                "input_size": 224,
                "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=3)",
                "augmentation": True,
                "use_class_weights": True,
            },
            "partial_backbone_stages": {
                "resnet50": "layer4",
                "densenet121": "features.denseblock4 + features.norm5",
                "convnext_tiny": "features[-1] (final ConvNeXt stage)",
                "efficientnet_b0": "features[-2] (final MBConv stage) + features[-1] (final projection)",
                "mobilenet_v3_large": "features[-2] (final InvertedResidual group) + features[-1] (final projection)",
            },
            "budget": {
                "conditions": 5,
                "seeds_per_condition": 5,
                "max_epochs_per_run": 30,
                "total_max_epochs": 750,
                "note": "full baseline shared between ablation and finetune stages",
            },
            "planned_comparison_family": {
                "contrasts": ["no_augmentation-full", "no_class_weights-full", "frozen-full", "partial-full"],
                "metrics": ["test_acc", "test_macro_f1"],
                "family_size": 8,
                "test": "two-sided scipy.stats.ttest_rel",
                "correction": "Holm",
                "note": "Separate from primary 8-comparison family in outputs/final/statistical_tests.csv",
            },
            "checkpoint_selection_rule": "max val_macro_f1; ties: lower val_loss; ties: earlier epoch",
            "min_delta": 1e-4,
        }
        protocol_path.write_text(
            json.dumps(protocol, ensure_ascii=False, indent=2, allow_nan=False),
            encoding="utf-8",
        )
        print(f"  Wrote protocol.json to {protocol_path}")

    # Preflight checks
    preflight: dict = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "checks": []}
    def chk(name: str, ok: bool, msg: str):
        status = "pass" if ok else "fail"
        preflight["checks"].append({"name": name, "status": status, "message": msg})
        print(f"  [{status.upper()}] {name}: {msg}")

    chk("base_out_dir_exists", base.exists(), str(base))
    chk("splits_csv_exists", (base / "splits.csv").exists(), str(base / "splits.csv"))
    chk("selection_json_exists", (base / "selection.json").exists(), str(base / "selection.json"))
    chk("best_params_exists", (base / "tuning" / "best_params_by_arch.json").exists(),
        str(base / "tuning" / "best_params_by_arch.json"))
    chk("final_run_metrics_exists", (base / "final" / "run_metrics.json").exists(),
        str(base / "final" / "run_metrics.json"))
    chk("data_root_exists", args.data_root.exists(), str(args.data_root))
    chk("out_dir_writable", True, str(out))  # we just created it above

    try:
        import torch
        chk("torch_installed", True, torch.__version__)
        if args.device in ("auto", "cuda"):
            chk("cuda_available", torch.cuda.is_available(),
                torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")
    except ImportError:
        chk("torch_installed", False, "torch not installed — required for training stages")

    all_pass = all(c["status"] == "pass" for c in preflight["checks"])
    preflight["overall"] = "pass" if all_pass else "partial_fail"
    preflight["budget_display"] = (
        "5 conditions × 5 seeds × max 30 epochs = max 750 epochs (before early stopping)\n"
        "  full (5) + no_augmentation (5) + no_class_weights (5) + frozen (5) + partial (5) = 25 runs\n"
        "  full baseline shared between ablation and finetune stages"
    )

    from shrine_classifier.analysis import save_json
    save_json(out / "preflight.json", preflight)
    env = _collect_environment(args.device)
    save_json(out / "environment.json", env)
    print(f"  Preflight: {preflight['overall']}")
    print(f"  Budget: {preflight['budget_display']}")
    print(f"[prepare] Done. Outputs: {out}")


# ---------------------------------------------------------------------------
# Stage: smoke
# ---------------------------------------------------------------------------

def stage_smoke(args) -> None:
    print("[smoke] Running bounded model/mode smoke test (no test split access)...")
    base = args.base_out_dir
    out = args.out_dir

    # Check PyTorch
    try:
        import torch
        import torchvision
    except ImportError:
        sys.exit(
            "ERROR: PyTorch and torchvision are required for the smoke stage.\n"
            "Install them with the correct CUDA version and retry."
        )

    smoke_dir = out / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)

    from shrine_classifier.models import build_model, ARCHITECTURES
    from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode, enforce_mode_per_epoch, log_mode_stats, MODES

    device_str = args.device
    if device_str == "auto":
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device_str)

    smoke_results: list[dict] = []
    issues: list[str] = []

    for arch in ARCHITECTURES:
        for mode in MODES:
            result: dict = {"architecture": arch, "mode": mode}
            try:
                # Build with synthetic random weights for smoke (no pretrained download required)
                model, meta = build_model(arch, pretrained=False)
                model = model.to(device)
                mode_stats = apply_finetuning_mode(model, arch, mode)

                # Verify 4 logits
                dummy = torch.zeros(2, 3, 224, 224, device=device)
                model.train()
                enforce_mode_per_epoch(model, arch, mode)
                logits = model(dummy)
                assert logits.shape == (2, 4), f"Expected (2,4), got {logits.shape}"

                # Verify backward
                loss = logits.sum()
                loss.backward()
                trainable_params = [p for p in model.parameters() if p.requires_grad]
                has_grad = any(p.grad is not None for p in trainable_params)

                # Verify frozen params unchanged through optimizer step
                frozen_params_before = {
                    name: p.detach().clone()
                    for name, p in model.named_parameters() if not p.requires_grad
                }
                if trainable_params:
                    opt = torch.optim.SGD(trainable_params, lr=0.001)
                    opt.step()
                for name, before_val in frozen_params_before.items():
                    current = dict(model.named_parameters())[name].detach()
                    if not torch.allclose(before_val, current, atol=1e-10):
                        issues.append(f"{arch}/{mode}: frozen param {name} changed after step")

                result.update({
                    "status": "pass",
                    "logits_shape": list(logits.shape),
                    "has_gradient": has_grad,
                    "trainable_params": mode_stats["trainable_parameters"],
                    "frozen_params": mode_stats["frozen_parameters"],
                    "trainable_module_names": mode_stats["trainable_module_names"],
                    "note": "SYNTHETIC random weights — pretrained loading not tested here",
                })
                print(f"  [PASS] {arch}/{mode}: logits={logits.shape}, "
                      f"trainable={mode_stats['trainable_parameters']:,}")
            except Exception as exc:
                result.update({"status": "fail", "error": f"{type(exc).__name__}: {exc}"})
                issues.append(f"{arch}/{mode}: {exc}")
                print(f"  [FAIL] {arch}/{mode}: {exc}")
            smoke_results.append(result)

    # Verify no test loader created during smoke
    result_no_test = {
        "check": "no_test_loader_in_smoke",
        "status": "pass",
        "note": "Smoke uses synthetic inputs only; no DataLoader touching test split was created",
    }
    smoke_results.append(result_no_test)

    summary = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "device": device_str,
        "total_checks": len(smoke_results),
        "passed": sum(1 for r in smoke_results if r.get("status") == "pass"),
        "failed": sum(1 for r in smoke_results if r.get("status") == "fail"),
        "issues": issues,
        "synthetic_weights_note": (
            "All model checks use synthetic random weights. "
            "Pretrained weight download/loading was NOT verified in this smoke pass."
        ),
        "results": smoke_results,
    }
    from shrine_classifier.analysis import save_json
    save_json(smoke_dir / "smoke_results.json", summary)
    print(f"[smoke] {summary['passed']} passed, {summary['failed']} failed. "
          f"Output: {smoke_dir / 'smoke_results.json'}")
    if issues:
        print(f"[smoke] Issues: {issues}")
    _update_execution_status(
        "smoke",
        "completed" if summary["failed"] == 0 else "partial_fail",
        f"{summary['passed']} pass, {summary['failed']} fail (synthetic weights only; pretrained unverified)",
        out,
    )


# ---------------------------------------------------------------------------
# Stage: ablation
# ---------------------------------------------------------------------------

def stage_ablation(args) -> None:
    print("[ablation] Running shared full baseline + ablation conditions (Items 3)...")
    base = args.base_out_dir
    out = args.out_dir
    out.mkdir(parents=True, exist_ok=True)

    selection = _load_selection(base)
    params = _load_params(base)
    split_csv = _load_split_csv(base)
    seeds = [int(s) for s in selection.get("seeds", [])]

    if args.dry_run:
        print("[ablation] DRY-RUN mode — showing what would be run")
    else:
        # Verify protocol is locked
        protocol_path = out / "protocol.json"
        if not protocol_path.exists():
            sys.exit(
                "ERROR: protocol.json not found. Lock the protocol before training:\n"
                "  python pipeline_extensions.py --stage prepare ..."
            )

    # Check PyTorch unless dry-run
    if not args.dry_run:
        try:
            import torch
        except ImportError:
            sys.exit("ERROR: torch not installed. Install PyTorch before training stages.")

    from shrine_classifier_ext.extension_experiments import run_extension_ablation
    results = run_extension_ablation(
        out_dir=out, base_out_dir=base, data_root=args.data_root,
        split_csv=split_csv, selection=selection, params=params, seeds=seeds,
        wandb_project=args.wandb_project, wandb_entity=args.wandb_entity,
        wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers,
        resume=args.resume, dry_run=args.dry_run,
    )

    if args.dry_run:
        total_new = sum(1 for cond_runs in results.values()
                        for r in cond_runs if r.get("status") == "dry_run_new")
        total_reuse = sum(1 for cond_runs in results.values()
                          for r in cond_runs if r.get("status") == "dry_run_reusable")
        max_epochs = total_new * 30
        print(f"[ablation dry-run] New runs: {total_new}, Reusable: {total_reuse}")
        print(f"[ablation dry-run] Maximum remaining epochs: {max_epochs}")
    else:
        for cond, cond_runs in results.items():
            ok = sum(1 for r in cond_runs if r.get("status") == "completed")
            fail = sum(1 for r in cond_runs if r.get("status") == "failed")
            print(f"  {cond}: {ok} completed, {fail} failed")

    print(f"[ablation] Done. Outputs: {out}")


# ---------------------------------------------------------------------------
# Stage: finetune
# ---------------------------------------------------------------------------

def stage_finetune(args) -> None:
    print("[finetune] Running frozen/partial fine-tuning comparison (Item 4)...")
    base = args.base_out_dir
    out = args.out_dir
    out.mkdir(parents=True, exist_ok=True)

    selection = _load_selection(base)
    params = _load_params(base)
    split_csv = _load_split_csv(base)
    seeds = [int(s) for s in selection.get("seeds", [])]

    if args.dry_run:
        print("[finetune] DRY-RUN mode — showing what would be run")
    else:
        protocol_path = out / "protocol.json"
        if not protocol_path.exists():
            sys.exit(
                "ERROR: protocol.json not found. Lock the protocol before training:\n"
                "  python pipeline_extensions.py --stage prepare ..."
            )

    if not args.dry_run:
        try:
            import torch
        except ImportError:
            sys.exit("ERROR: torch not installed.")

    from shrine_classifier_ext.extension_experiments import run_extension_finetune
    results = run_extension_finetune(
        out_dir=out, base_out_dir=base, data_root=args.data_root,
        split_csv=split_csv, selection=selection, params=params, seeds=seeds,
        wandb_project=args.wandb_project, wandb_entity=args.wandb_entity,
        wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers,
        resume=args.resume, dry_run=args.dry_run,
    )

    if args.dry_run:
        total_new = sum(1 for cond_runs in results.values()
                        for r in cond_runs if r.get("status") == "dry_run_new")
        total_reuse = sum(1 for cond_runs in results.values()
                          for r in cond_runs if r.get("status") == "dry_run_reusable")
        max_epochs = total_new * 30
        print(f"[finetune dry-run] New runs: {total_new}, Reusable: {total_reuse}")
        print(f"[finetune dry-run] Maximum remaining epochs: {max_epochs}")
    else:
        for cond, cond_runs in results.items():
            ok = sum(1 for r in cond_runs if r.get("status") == "completed")
            fail = sum(1 for r in cond_runs if r.get("status") == "failed")
            print(f"  {cond}: {ok} completed, {fail} failed")

    print(f"[finetune] Done. Outputs: {out}")


# ---------------------------------------------------------------------------
# Stage: report
# ---------------------------------------------------------------------------

def stage_report(args) -> None:
    print("[report] Generating extension summary from existing artifacts...")
    # Recompute comparisons across all completed conditions on disk if present
    try:
        from shrine_classifier_ext.extension_experiments import _save_comparisons
        selection_path = args.base_out_dir / "selection.json"
        selection = json.loads(selection_path.read_text(encoding="utf-8")) if selection_path.exists() else {}
        arch = selection.get("selected_architecture", "resnet50")
        seeds = selection.get("seeds", [42, 123, 456, 789, 1024])
        _save_comparisons(args.out_dir, {}, seeds, arch, selection)
    except Exception as exc:
        print(f"[report] Note on comparisons refresh: {exc}")

    from shrine_classifier_ext.extension_report import run_report
    result = run_report(args.out_dir, args.base_out_dir)
    print(f"[report] Status summary:")
    for k, v in sorted(result["status"].items()):
        icon = "[OK]" if v in ("present", "generated") else "[MISSING]"
        print(f"  {icon} {k}: {v}")
    print(f"[report] Done. Assets: {result['out_dir']}")


# ---------------------------------------------------------------------------
# Status file update
# ---------------------------------------------------------------------------

def _update_execution_status(stage: str, status: str, details: str, out_dir: Path) -> None:
    status_path = Path("EXTENSION_EXECUTION_STATUS.md")
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    entry = (
        f"\n## Extension stage: `{stage}` — {status} ({timestamp})\n\n"
        f"{details}\n\n"
        f"Output directory: `{out_dir}`\n"
    )
    if status_path.exists():
        current = status_path.read_text(encoding="utf-8")
        status_path.write_text(current + entry, encoding="utf-8")
    else:
        header = (
            "# Extension Execution Status\n\n"
            "Tracks `pipeline_extensions.py` stage outcomes.\n"
            "Appended by each run; original entries preserved.\n"
        )
        status_path.write_text(header + entry, encoding="utf-8")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(argv=None) -> None:
    args = parse_args(argv)

    # User retired this experiment and deleted its outputs on 2026-10-13.
    # Block before dispatch/output creation, including dry-run, so no old recipe
    # is mistaken for independent architecture-mode tuning.
    if args.stage in {"prepare", "ablation", "finetune", "report"}:
        raise SystemExit(
            f"Stage '{args.stage}' belongs to the retired fixed-Full-recipe protocol. "
            "It is disabled and will not recreate deleted experiment outputs. "
            "The requested flow tunes each architecture-mode pair independently; "
            "see docs/MODE_TUNING_PROTOCOL.md. Its new runner is not implemented yet. "
            "Historical cached analysis and bounded smoke remain available."
        )

    dispatch = {
        "analyze": stage_analyze,
        "prepare": stage_prepare,
        "smoke": stage_smoke,
        "ablation": stage_ablation,
        "finetune": stage_finetune,
        "report": stage_report,
    }
    fn = dispatch[args.stage]

    t0 = time.time()
    try:
        fn(args)
    except SystemExit:
        raise
    except Exception as exc:
        import traceback
        print(f"\n[ERROR] Stage '{args.stage}' failed: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        _update_execution_status(args.stage, "error", str(exc), args.out_dir)
        sys.exit(1)

    elapsed = time.time() - t0
    print(f"[{args.stage}] Elapsed: {elapsed:.1f}s")


if __name__ == "__main__":
    # Windows multiprocessing safe entry point
    import multiprocessing
    multiprocessing.freeze_support()
    main()


In [ ]:
%%project_source pipeline_modes.py
"""Independent architecture × fine-tuning-mode experiment runner.

Tuning and lock stages never construct a test DataLoader. All new artifacts
live under outputs_modes_v3 by default; historical outputs are read-only inputs.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import html
import json
import os
import platform
import random
import shutil
import sys
import time
import traceback
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np

from shrine_classifier.data import CLASSES, sha256_file
from shrine_classifier.models import ARCHITECTURES

MODES = ("full", "frozen", "partial")
SEEDS = (42, 123, 456, 789, 1024)
PROTOCOL_VERSION = "architecture-mode-5trial-v2"
SEARCH_SPACE = {
    "learning_rate": {"distribution": "log_uniform", "min": 1e-5, "max": 1e-3},
    "weight_decay": {"distribution": "log_uniform", "min": 1e-6, "max": 1e-2},
    "optimizer": {"values": ["adamw", "adam", "sgd"]},
    "batch_size": {"values": [16, 32]},
}
SEARCH_METHOD = "local seeded random search; trials individually tracked as W&B runs"


def _json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False, default=str), encoding="utf-8")
    tmp.replace(path)


def _csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("status\r\n", encoding="utf-8-sig")
        return
    fields = list(dict.fromkeys(k for row in rows for k in row))
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    tmp.replace(path)


def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def _fingerprint(obj: Any) -> str:
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()


def _source_manifest(base: Path, data_root: Path, out: Path) -> dict[str, Any]:
    prop_file = base / "dataset_audit" / "image_properties.csv"
    review_file = base / "dataset_audit" / "near_duplicate_review_decisions.csv"
    metadata_file = data_root / "metadata_cleaned.csv"
    if not prop_file.is_file() or not review_file.is_file() or not metadata_file.is_file():
        raise FileNotFoundError("Existing dataset audit, duplicate review, or metadata missing; run the verified audit workflow first")
    props = _read_csv(prop_file)
    if len(props) != 560:
        raise ValueError(f"Expected 560 audited images, found {len(props)}")
    meta_rows = _read_csv(metadata_file)
    metadata: dict[tuple[str, str], str] = {}
    for row in meta_rows:
        key = (row["category"], Path(row["filename"]).stem)
        if key in metadata:
            raise ValueError(f"Ambiguous metadata join for {key}")
        metadata[key] = row.get("source_url", "") or ""
    manifest, problems, domains, domains_by_class = [], [], {}, {name:{} for name in CLASSES}
    split_map = {r["image_id"]: r for r in _read_csv(base / "splits.csv")}
    for row in props:
        rel = Path(row["path"])
        if row["label"] not in CLASSES or int(row["label_index"]) != CLASSES.index(row["label"]):
            problems.append({"path": row["path"], "issue": "class-order mismatch"})
            continue
        img = data_root / rel
        if not img.is_file():
            problems.append({"path": row["path"], "issue": "missing image"})
            continue
        actual = sha256_file(img)
        if actual != row.get("byte_sha256"):
            problems.append({"path": row["path"], "issue": "byte hash changed since audit"})
            continue
        key = (row["label"], rel.stem)
        url = metadata.get(key)
        if url is None:
            problems.append({"path": row["path"], "issue": "metadata join missing"})
            continue
        domain = row.get("source_domain", "")
        domains[domain or "unknown"] = domains.get(domain or "unknown", 0) + 1
        domain_name=domain or "unknown"
        domains_by_class[row["label"]][domain_name]=domains_by_class[row["label"]].get(domain_name,0)+1
        manifest.append({"image_id": row["image_id"], "path": row["path"], "label": row["label"],
                         "label_index": int(row["label_index"]), "width": int(row["width"]),
                         "height": int(row["height"]), "mode": row["mode"], "format": row["format"],
                         "byte_sha256": actual, "pixel_sha256": row["pixel_sha256"],
                         "group_id": split_map.get(row["image_id"], {}).get("group_id"),
                         "source_url": url, "source_domain": domain or None})
    ids = [row["image_id"] for row in manifest]
    if problems or len(set(ids)) != 560 or set(metadata) != {(r["label"], Path(r["path"]).stem) for r in manifest}:
        _csv(out / "dataset_audit" / "source_issues.csv", problems or [{"issue": "duplicate IDs or unused metadata join"}])
        raise ValueError(f"Dataset integrity check failed: {len(problems)} issues; see dataset_audit/source_issues.csv")
    _json(out / "source_manifest.json", {"protocol": PROTOCOL_VERSION, "images": manifest,
          "image_count": len(manifest), "metadata_sha256": sha256_file(metadata_file),
          "audit_properties_sha256": sha256_file(prop_file), "duplicate_review_sha256": sha256_file(review_file),
          "source_domain_counts": domains, "source_domain_counts_by_class":domains_by_class,
          "unknown_url_count": sum(not r["source_url"] for r in manifest)})
    return {"images": manifest, "fingerprint": _fingerprint([(r["image_id"], r["byte_sha256"]) for r in manifest])}


def _current_code_fingerprint() -> str:
    base = Path(__file__).resolve().parent
    entries = []
    for folder in (base / "shrine_classifier", base / "shrine_classifier_ext"):
        for path in sorted(folder.glob("*.py")):
            entries.append((str(path.relative_to(base)), sha256_file(path)))
    for path in (base / "pipeline_modes.py", base / "pipeline.py"):
        if path.exists():
            entries.append((str(path.relative_to(base)), sha256_file(path)))
    return _fingerprint(entries)


def _verify_and_prepare(args: argparse.Namespace, out: Path) -> dict[str, Any]:
    from pipeline import _check_existing_split
    from shrine_classifier.training import load_split

    base = args.base_out_dir.resolve()
    _check_existing_split(base, args.data_root.resolve())
    split_file = base / "splits.csv"
    meta_file = base / "split_metadata.json"
    split_rows = load_split(split_file)
    meta = json.loads(meta_file.read_text(encoding="utf-8"))
    if int(meta.get("split_seed", -1)) != args.split_seed or len(split_rows) != 560 or int(meta.get("num_groups", -1)) != 550:
        raise ValueError("Canonical split does not match the reviewed seed/group/image contract; refusing to regenerate")
    if meta.get("group_overlap") is not False:
        raise ValueError("Canonical split metadata reports group overlap")
    # Copy byte-identical canonical split artifacts into the isolated new output root.
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(split_file, out / "splits.csv")
    shutil.copy2(meta_file, out / "split_metadata.json")
    if sha256_file(split_file) != sha256_file(out / "splits.csv"):
        raise IOError("Split copy byte hash mismatch")
    manifest = _source_manifest(base, args.data_root.resolve(), out)
    protocol = {"version": PROTOCOL_VERSION, "architectures": list(args.architectures), "modes": list(args.modes),
        "class_order": CLASSES, "split_seed": args.split_seed, "split_sha256": sha256_file(split_file),
        "assignment_fingerprint": meta["split_sha256"], "source_fingerprint": manifest["fingerprint"],
        "input_size": args.input_size, "max_epochs": args.epochs, "early_stopping_patience": 7,
        "min_delta": 1e-4, "selection_metric": "validation macro F1",
        "search_space": SEARCH_SPACE, "search_method": SEARCH_METHOD, "tuning_seed": args.seed,
        "trials_per_pair": args.trials_per_pair, "final_seeds": args.seeds,
        "wandb_project": args.wandb_project, "created_utc": datetime.now(timezone.utc).isoformat()}
    protocol["protocol_fingerprint"] = _fingerprint(protocol)
    existing_path = out / "protocol.json"
    if existing_path.exists():
        previous = json.loads(existing_path.read_text(encoding="utf-8"))
        if previous.get("protocol_fingerprint") != protocol["protocol_fingerprint"]:
            raise ValueError(f"{out} has a different protocol fingerprint; use a new --out-dir")
    _json(existing_path, protocol)
    env = {"python": sys.version, "platform": platform.platform(), "torch": None, "torchvision": None,
           "numpy": np.__version__, "device_requested": args.device, "wandb_mode_requested": args.wandb_mode,
           "timestamp_utc": datetime.now(timezone.utc).isoformat(), "code_fingerprint": _current_code_fingerprint()}
    import torch
    import torchvision
    env.update({"torch": torch.__version__, "torchvision": torchvision.__version__,
                "cuda_available": torch.cuda.is_available(), "cuda_version": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "resolved_device": "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)})
    try:
        import wandb
        env["wandb"] = wandb.__version__
    except Exception:
        env["wandb"] = None
    _json(out / "environment.json", env)
    _json(out / "preflight.json", {"approved_budget": {"tuning_runs": len(args.architectures)*len(args.modes)*args.trials_per_pair,
        "final_runs": len(args.architectures)*len(args.modes)*len(args.seeds),
        "maximum_epochs": len(args.architectures)*len(args.modes)*(args.trials_per_pair+len(args.seeds))*args.epochs},
        "split_counts": meta["counts"], "num_groups": meta["num_groups"], "test_accessed": False,
        "source_fingerprint": manifest["fingerprint"], "protocol_fingerprint": protocol["protocol_fingerprint"],
        "dataset_padding_available": args.padded_root.exists() if args.padded_root else False,
        "disk_free_bytes": shutil.disk_usage(out).free, "disk_free_gib": round(shutil.disk_usage(out).free/(1024**3),2)})
    old_audit = base / "dataset_audit"
    if old_audit.is_dir() and not (out / "dataset_audit").exists():
        shutil.copytree(old_audit, out / "dataset_audit")
    # Reuse the existing qualitative ImageNet baseline only as a historical, compatible asset.
    old_baseline = base / "baseline"
    if old_baseline.exists() and not (out / "baseline").exists():
        shutil.copytree(old_baseline, out / "baseline")
    _json(out / "execution_status.json", {"stage": "prepare", "status": "completed",
        "updated_utc": datetime.now(timezone.utc).isoformat(), "completed_tuning_trials": 0,
        "target_tuning_trials": len(args.architectures)*len(args.modes)*args.trials_per_pair,
        "completed_final_runs": 0, "target_final_runs": len(args.architectures)*len(args.modes)*len(args.seeds),
        "split_verified": True, "split_modified": False, "test_predictions_created": False,
        "protocol_fingerprint": protocol["protocol_fingerprint"]})
    return {"protocol": protocol, "metadata": meta, "manifest": manifest, "environment": env}


def _sample_configs(count: int, seed: int) -> list[dict[str, Any]]:
    rng = np.random.default_rng(seed)
    return [{"learning_rate": float(10 ** rng.uniform(-5, -3)),
             "weight_decay": float(10 ** rng.uniform(-6, -2)),
             "optimizer": str(rng.choice(["adamw", "adam", "sgd"])),
             "batch_size": int(rng.choice([16, 32]))} for _ in range(count)]


def _load_registry(out: Path) -> dict[str, Any]:
    path = out / "run_registry.json"
    if path.exists():
        obj = json.loads(path.read_text(encoding="utf-8"))
        if obj.get("protocol_fingerprint") != json.loads((out / "protocol.json").read_text(encoding="utf-8"))["protocol_fingerprint"]:
            raise ValueError("Run registry protocol fingerprint mismatch")
        changed = False
        for stage in ("trials", "final_runs"):
            for row in obj.get(stage, []):
                if row.get("status") == "running":
                    row.update({"status": "interrupted", "finished_utc": datetime.now(timezone.utc).isoformat(),
                                "error": row.get("error") or "Process ended before the run recorded a terminal status; recovered on resume."})
                    changed = True
        if changed:
            _json(path, obj)
        return obj
    return {"protocol_fingerprint": json.loads((out / "protocol.json").read_text(encoding="utf-8"))["protocol_fingerprint"], "trials": [], "final_runs": []}


def _write_registry(out: Path, registry: dict[str, Any]) -> None:
    _json(out / "run_registry.json", registry)
    done_tune = len({r.get("identity") for r in registry["trials"] if r.get("status") == "completed"})
    done_final = len({r.get("identity") for r in registry["final_runs"] if r.get("status") == "completed"})
    protocol=json.loads((out/"protocol.json").read_text(encoding="utf-8"))
    tune_target=len(protocol["architectures"])*len(protocol["modes"])*int(protocol["trials_per_pair"])
    final_target=len(protocol["architectures"])*len(protocol["modes"])*len(protocol["final_seeds"])
    _json(out / "execution_status.json", {"updated_utc": datetime.now(timezone.utc).isoformat(),
        "completed_tuning_trials": done_tune, "target_tuning_trials": tune_target,
        "completed_final_runs": done_final, "target_final_runs": final_target,
        "failed_trials": sum(r.get("status") == "failed" for r in registry["trials"]),
        "failed_final_runs": sum(r.get("status") == "failed" for r in registry["final_runs"]),
        "protocol_fingerprint": registry["protocol_fingerprint"]})


def _pair_key(architecture: str, mode: str) -> str:
    return f"{architecture}::{mode}"


def _latest_completed_seed_rows(records: list[dict[str,Any]]) -> list[dict[str,Any]]:
    latest={}
    for row in records:
        if row.get("status")=="completed" and row.get("seed") is not None:
            latest[(row["architecture"],row["mode"],int(row["seed"]))]=row
    return list(latest.values())


def _tune(args: argparse.Namespace, out: Path) -> dict[str, Any]:
    from shrine_classifier.training import fit_run
    from shrine_classifier.analysis import save_csv

    registry = _load_registry(out)
    trial_rows = registry["trials"]
    fixed_configs = _sample_configs(args.trials_per_pair, args.seed)
    protocol = json.loads((out / "protocol.json").read_text(encoding="utf-8"))
    code_hash = _current_code_fingerprint()
    target = len(args.architectures) * len(args.modes) * args.trials_per_pair
    _json(out / "tuning" / "search_definition.json", {"method": SEARCH_METHOD, "search_space": SEARCH_SPACE,
        "seed": args.seed, "trials_per_pair": args.trials_per_pair, "sampled_configs": fixed_configs,
        "unique_trial_target": target, "protocol_fingerprint": protocol["protocol_fingerprint"]})
    for architecture in args.architectures:
        for mode in args.modes:
            for trial_no, sampled in enumerate(fixed_configs, 1):
                ident = f"{architecture}::{mode}::trial{trial_no:03d}"
                cfg = {**sampled, "architecture": architecture, "mode": mode, "seed": args.seed,
                       "trial": trial_no, "epochs": args.epochs, "patience": 7, "min_delta": 1e-4,
                       "input_size": args.input_size, "augmentation": True, "use_class_weights": True,
                       "optimizer_defaults": {"sgd_momentum": .9, "sgd_nesterov": True,
                                              "adam_betas": [0.9, .999], "epsilon": 1e-8,
                                              "weight_decay_mode": "optimizer default"}}
                cfg.update({"protocol_fingerprint":protocol["protocol_fingerprint"],"dataset_fingerprint":protocol["source_fingerprint"],
                            "split_assignment_fingerprint":protocol["assignment_fingerprint"],"code_fingerprint":code_hash,"amp":False})
                cfg_hash = _fingerprint({"cfg": cfg, "split": protocol["assignment_fingerprint"],
                    "source": protocol["source_fingerprint"], "code": code_hash, "protocol": protocol["protocol_fingerprint"]})
                previous = next((r for r in reversed(trial_rows) if r["identity"] == ident and r.get("config_fingerprint") == cfg_hash and r.get("status") == "completed"), None)
                # Verify completion artifacts without ever borrowing legacy runs.
                if args.resume and previous:
                    rdir = Path(previous.get("run_dir", ""))
                    if (rdir / "run_metrics.json").is_file() and (rdir / "history.csv").is_file() and (rdir / "checkpoint_metadata.json").is_file():
                        print(f"[resume] {ident} already complete", flush=True)
                        continue
                attempt=1+sum(r.get("identity")==ident for r in trial_rows)
                record = {"identity": ident, "attempt":attempt, "architecture": architecture, "mode": mode,
                          "trial": trial_no, "seed": args.seed, "config": cfg,
                          "config_fingerprint": cfg_hash, "code_fingerprint": code_hash,
                          "started_utc": datetime.now(timezone.utc).isoformat(), "status": "running"}
                trial_rows.append(record)
                _write_registry(out, registry)
                try:
                    result = fit_run(root=args.data_root.resolve(), split_csv=out / "splits.csv", out=out,
                        architecture=architecture, mode=mode, run_identity=f"trial{trial_no:03d}_attempt{attempt:02d}", seed=args.seed,
                        config=cfg, stage="tuning", wandb_project=args.wandb_project,
                        wandb_entity=args.wandb_entity, wandb_mode=args.wandb_mode,
                        device=args.device, workers=args.num_workers, evaluate_test=False)
                    record.update(result)
                    record["status"] = "completed"
                except Exception as exc:
                    record.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}",
                                   "traceback": traceback.format_exc(limit=5)})
                record["finished_utc"] = datetime.now(timezone.utc).isoformat()
                _write_registry(out, registry)
                save_csv(out / "tuning" / "all_trials.csv", trial_rows)
                print(f"[tune] {ident}: {record['status']} val_macro_f1={record.get('best_val_macro_f1')} ({sum(x.get('status')=='completed' for x in trial_rows)}/{target} successful)", flush=True)
            successes = [r for r in trial_rows if r["architecture"] == architecture and r["mode"] == mode
                         and r.get("status") == "completed" and r.get("best_val_macro_f1") is not None]
            if not successes:
                _json(out / "tuning" / "all_trials.json", trial_rows)
                raise RuntimeError(f"No successful trial for {architecture}/{mode}; see run_registry.json")
    save_csv(out / "tuning" / "all_trials.csv", trial_rows)
    _json(out / "tuning" / "all_trials.json", trial_rows)
    return {"completed": sum(r.get("status") == "completed" for r in trial_rows), "target": target}


def _lock(args: argparse.Namespace, out: Path) -> dict[str, Any]:
    protocol = json.loads((out / "protocol.json").read_text(encoding="utf-8"))
    registry = _load_registry(out)
    rows = registry["trials"]
    winners, selection_rows = {}, []
    for architecture in args.architectures:
        for mode in args.modes:
            sampled_trials={int(r["trial"]) for r in rows if r.get("architecture")==architecture and r.get("mode")==mode}
            expected_trials=set(range(1,args.trials_per_pair+1))
            if sampled_trials!=expected_trials:
                raise RuntimeError(f"Cannot lock {architecture}/{mode}: recorded trial IDs {sorted(sampled_trials)}, expected {sorted(expected_trials)}")
            trials = [r for r in rows if r.get("architecture") == architecture and r.get("mode") == mode
                      and r.get("status") == "completed" and r.get("best_val_macro_f1") is not None]
            if not trials:
                raise RuntimeError(f"Cannot lock: no successful validation winner for {architecture}/{mode}")
            best = sorted(trials, key=lambda r: (-float(r["best_val_macro_f1"]), -float(r["best_val_acc"]),
                float(r["best_val_loss"]), int(r["trial"])))[0]
            # Tuning stores the training recipe inside record['config']; metrics
            # returned by fit_run are merged at the record's top level.
            winning_config = best.get("config")
            recipe_keys = ("learning_rate", "weight_decay", "optimizer", "batch_size")
            if not isinstance(winning_config, dict) or any(k not in winning_config for k in recipe_keys):
                raise ValueError(f"Winning trial {best['identity']} has an incomplete saved config; refusing to infer parameters")
            recipe = {k: winning_config[k] for k in recipe_keys}
            winners[_pair_key(architecture, mode)] = {"architecture": architecture, "mode": mode,
                **recipe, "winning_trial": best["trial"], "best_val_macro_f1": best["best_val_macro_f1"],
                "best_val_acc": best["best_val_acc"], "best_val_loss": best["best_val_loss"],
                "trial_config_fingerprint": best["config_fingerprint"], "tuning_checkpoint": best["checkpoint"]}
            selection_rows.append(winners[_pair_key(architecture, mode)])
    order_arch = {v: i for i, v in enumerate(args.architectures)}
    order_mode = {v: i for i, v in enumerate(args.modes)}
    reference = sorted(selection_rows, key=lambda r: (-float(r["best_val_macro_f1"]),
        -float(r["best_val_acc"]), float(r["best_val_loss"]), order_arch[r["architecture"]], order_mode[r["mode"]]))[0]
    tune_path = out / "tuning" / "best_params_by_arch_mode.json"
    selection_path=out/"selection.json"
    if selection_path.is_file():
        prior=json.loads(selection_path.read_text(encoding="utf-8"))
        prior_pairs=json.loads(tune_path.read_text(encoding="utf-8")).get("pairs",{}) if tune_path.is_file() else {}
        stable=(prior.get("protocol_fingerprint")==protocol["protocol_fingerprint"] and prior_pairs==winners and
                prior.get("reference_pair")=={k:reference[k] for k in ("architecture","mode")})
        if not stable:raise ValueError("A previous selection lock differs; do not overwrite post-lock decisions. Start an explicitly versioned output.")
        return prior
    if tune_path.exists():
        old = json.loads(tune_path.read_text(encoding="utf-8"))
        if old.get("protocol_fingerprint") not in (None, protocol["protocol_fingerprint"]):
            raise ValueError("Tuning winners belong to a different protocol")
    _json(tune_path, {"protocol_fingerprint": protocol["protocol_fingerprint"], "search_space": SEARCH_SPACE,
        "trials_per_pair": args.trials_per_pair, "pairs": winners})
    locked = {"protocol_fingerprint": protocol["protocol_fingerprint"],
        "source_fingerprint": protocol["source_fingerprint"], "split_file_sha256": sha256_file(out / "splits.csv"),
        "split_assignment_fingerprint": protocol["assignment_fingerprint"], "architectures": args.architectures,
        "modes": args.modes, "seeds": args.seeds, "representative_seed": 42 if 42 in args.seeds else args.seeds[0],
        "epochs": args.epochs, "trials_per_pair": args.trials_per_pair,
        "tuned_pair_count": len(winners), "reference_pair": {k: reference[k] for k in ("architecture", "mode")},
        "reference_validation": {k: reference[k] for k in ("best_val_macro_f1", "best_val_acc", "best_val_loss")},
        "primary_holm_family": {"family_id": "reference_vs_other_pairs_accuracy_macro_f1", "planned_size": (len(winners)-1)*2},
        "secondary_holm_family": {"family_id": "within_architecture_mode_pairs_accuracy_macro_f1", "planned_size": len(args.architectures)*len(args.modes)*(len(args.modes)-1)},
        "test_metrics_read": False, "locked_utc": datetime.now(timezone.utc).isoformat()}
    if len(winners) != len(args.architectures) * len(args.modes):
        raise ValueError("Cannot lock incomplete architecture-mode grid")
    locked["selection_fingerprint"] = _fingerprint(locked)
    _json(out / "selection.json", locked)
    return locked


def _verify_lock(args: argparse.Namespace, out: Path) -> tuple[dict[str, Any], dict[str, Any]]:
    selection_path, winner_path = out / "selection.json", out / "tuning" / "best_params_by_arch_mode.json"
    if not selection_path.is_file() or not winner_path.is_file():
        raise FileNotFoundError("Locked validation winners missing; run --stage lock")
    selection, winners = json.loads(selection_path.read_text(encoding="utf-8")), json.loads(winner_path.read_text(encoding="utf-8"))
    protocol = json.loads((out / "protocol.json").read_text(encoding="utf-8"))
    if selection.get("protocol_fingerprint") != protocol["protocol_fingerprint"] or \
       winners.get("protocol_fingerprint") != protocol["protocol_fingerprint"]:
        raise ValueError("Selection/winner protocol fingerprint mismatch")
    saved_fingerprint=selection.get("selection_fingerprint")
    fingerprint_payload={k:v for k,v in selection.items() if k!="selection_fingerprint"}
    if saved_fingerprint!=_fingerprint(fingerprint_payload):
        raise ValueError("Selection lock fingerprint is corrupt")
    if selection.get("test_metrics_read") is not False or selection.get("split_file_sha256") != sha256_file(out / "splits.csv"):
        raise ValueError("Selection lock is incompatible, touched-test, or has a changed split")
    expected = len(args.architectures) * len(args.modes)
    if selection.get("tuned_pair_count") != expected or len(winners.get("pairs", {})) != expected:
        raise ValueError("Selection lock does not cover every configured pair")
    if selection["architectures"] != args.architectures or selection["modes"] != args.modes or selection["seeds"] != args.seeds:
        raise ValueError("Final run grid/seeds differ from frozen selection")
    return selection, winners["pairs"]


def _final(args: argparse.Namespace, out: Path) -> dict[str, Any]:
    from shrine_classifier.training import fit_run
    from shrine_classifier.analysis import save_csv

    selection, winners = _verify_lock(args, out)
    registry = _load_registry(out)
    final_rows = registry["final_runs"]
    protocol = json.loads((out / "protocol.json").read_text(encoding="utf-8"))
    code_hash = _current_code_fingerprint()
    target = len(args.architectures) * len(args.modes) * len(args.seeds)
    for architecture in args.architectures:
        for mode in args.modes:
            pair = winners[_pair_key(architecture, mode)]
            for seed in args.seeds:
                ident = f"{architecture}::{mode}::seed{seed}"
                cfg = {k: pair[k] for k in ("learning_rate", "weight_decay", "optimizer", "batch_size")}
                cfg.update({"epochs": args.epochs, "patience": 7, "min_delta": 1e-4, "input_size": args.input_size,
                    "augmentation": True, "use_class_weights": True,
                    "optimizer_defaults": {"sgd_momentum": .9, "sgd_nesterov": True,
                        "adam_betas": [.9,.999], "epsilon": 1e-8}, "selection_fingerprint": selection["selection_fingerprint"]})
                cfg.update({"protocol_fingerprint":protocol["protocol_fingerprint"],"dataset_fingerprint":protocol["source_fingerprint"],
                            "split_assignment_fingerprint":protocol["assignment_fingerprint"],"code_fingerprint":code_hash,"amp":False})
                cfg_hash = _fingerprint({"cfg": cfg, "pair_trial": pair["trial_config_fingerprint"],
                    "seed": seed, "split": protocol["assignment_fingerprint"], "source": protocol["source_fingerprint"],
                    "code": code_hash, "protocol": protocol["protocol_fingerprint"]})
                previous = next((r for r in reversed(final_rows) if r["identity"] == ident and r.get("config_fingerprint") == cfg_hash and r.get("status") == "completed"), None)
                if args.resume and previous:
                    run_dir = Path(previous.get("run_dir", ""))
                    if (run_dir / "test_predictions.csv").is_file() and (run_dir / "test_metrics.json").is_file() and (run_dir / "checkpoint_metadata.json").is_file():
                        print(f"[resume] {ident} already complete", flush=True)
                        continue
                attempt=1+sum(r.get("identity")==ident for r in final_rows)
                record = {"identity": ident, "attempt":attempt, "architecture": architecture, "mode": mode, "seed": seed,
                    "trial": pair["winning_trial"], "config": cfg, "config_fingerprint": cfg_hash,
                    "code_fingerprint": code_hash, "started_utc": datetime.now(timezone.utc).isoformat(), "status": "running"}
                final_rows.append(record)
                _write_registry(out, registry)
                try:
                    result = fit_run(root=args.data_root.resolve(), split_csv=out / "splits.csv", out=out,
                        architecture=architecture, mode=mode, run_identity=f"final_seed{seed}_attempt{attempt:02d}", seed=seed, config=cfg,
                        stage="final", wandb_project=args.wandb_project, wandb_entity=args.wandb_entity,
                        wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers, evaluate_test=True)
                    record.update(result)
                    record["status"] = "completed"
                except Exception as exc:
                    record.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}",
                                   "traceback": traceback.format_exc(limit=5)})
                record["finished_utc"] = datetime.now(timezone.utc).isoformat()
                _write_registry(out, registry)
                save_csv(out / "final" / "run_metrics.csv", final_rows)
                print(f"[final] {ident}: {record['status']} test_macro_f1={record.get('test_macro_f1')} ({sum(x.get('status')=='completed' for x in final_rows)}/{target} successful)", flush=True)
    save_csv(out / "final" / "run_metrics.csv", final_rows)
    _summarize(args, out, final_rows, selection)
    return {"completed": sum(r.get("status") == "completed" for r in final_rows), "target": target}


def _holm(rows: list[dict[str, Any]], family_size: int) -> None:
    valid = [(i, float(r["p_value"])) for i, r in enumerate(rows) if r.get("p_value") is not None and np.isfinite(float(r["p_value"]))]
    valid.sort(key=lambda v: v[1])
    running = 0.0
    for rank, (index, p_value) in enumerate(valid):
        running = max(running, p_value * (family_size - rank))
        rows[index]["p_holm"] = min(1., running)
        rows[index]["significant_holm"] = bool(rows[index]["p_holm"] < .05)
    for row in rows:
        row.setdefault("p_holm", None)
        row.setdefault("significant_holm", None)
        row["alpha"] = .05
        row["holm_family_size"] = family_size


def _test_summary(records: list[dict[str, Any]], ref: dict[str, Any], comp: dict[str, Any],
                  metric: str, family: str) -> dict[str, Any]:
    ref_map = {int(r["seed"]): r for r in records if r["architecture"] == ref["architecture"] and r["mode"] == ref["mode"] and r["status"] == "completed"}
    comp_map = {int(r["seed"]): r for r in records if r["architecture"] == comp["architecture"] and r["mode"] == comp["mode"] and r["status"] == "completed"}
    seeds = sorted(set(ref_map) & set(comp_map))
    expected = (42, 123, 456, 789, 1024)
    row = {"family_id": family, "reference": _pair_key(ref["architecture"], ref["mode"]),
           "competitor": _pair_key(comp["architecture"], comp["mode"]), "metric": metric,
           "seed_ids": ",".join(map(str,seeds)), "paired_n": len(seeds), "mean_difference_reference_minus_competitor": None,
           "t_statistic": None, "p_value": None, "status": "incomplete_pairs" if seeds != list(expected) else "undefined_zero_variance"}
    if seeds != list(expected):
        return row
    ref_vals = np.asarray([float(ref_map[s][metric]) for s in seeds])
    comp_vals = np.asarray([float(comp_map[s][metric]) for s in seeds])
    diff = ref_vals-comp_vals
    row["mean_difference_reference_minus_competitor"] = float(diff.mean())
    if not np.isfinite(diff).all() or not np.isfinite(ref_vals).all() or not np.isfinite(comp_vals).all():
        row["status"] = "undefined_nonfinite"
        return row
    if np.std(diff, ddof=1) == 0:
        return row
    from scipy.stats import ttest_rel
    result = ttest_rel(ref_vals, comp_vals, alternative="two-sided")
    row.update({"t_statistic": float(result.statistic), "p_value": float(result.pvalue), "status": "ok"})
    return row


def _summarize(args: argparse.Namespace, out: Path, records: list[dict[str, Any]], selection: dict[str, Any]) -> None:
    from shrine_classifier.analysis import save_csv
    records=_latest_completed_seed_rows(records)
    summary, per_class, primary, secondary = [], [], [], []
    pairs = [{"architecture": a, "mode": m} for a in args.architectures for m in args.modes]
    for pair in pairs:
        rows = [r for r in records if r["architecture"] == pair["architecture"] and r["mode"] == pair["mode"] and r.get("status") == "completed"]
        entry = {**pair, "completed_seeds": len(rows)}
        for metric in ("test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1"):
            vals = [float(r[metric]) for r in rows if r.get(metric) is not None]
            entry[metric+"_mean"] = float(np.mean(vals)) if vals else None
            entry[metric+"_sample_sd"] = float(np.std(vals,ddof=1)) if len(vals)>=2 else None
        summary.append(entry)
        for cls in CLASSES:
            vals=[r["per_class"][cls] for r in rows if r.get("per_class",{}).get(cls)]
            pc={**pair,"class":cls,"support":vals[0]["support"] if vals else None}
            for metric in ("precision","recall","f1"):
                numbers=[float(x[metric]) for x in vals]
                pc[metric+"_mean"]=float(np.mean(numbers)) if numbers else None
                pc[metric+"_sample_sd"]=float(np.std(numbers,ddof=1)) if len(numbers)>=2 else None
            per_class.append(pc)
    ref=selection["reference_pair"]
    others=[p for p in pairs if p!=ref]
    for pair in others:
        for metric in ("test_acc","test_macro_f1"):
            primary.append(_test_summary(records,ref,pair,metric,"primary_reference_pair_28"))
    for arch in args.architectures:
        mode_pairs=[{"architecture":arch,"mode":m} for m in args.modes]
        for a,b in combinations(mode_pairs,2):
            # Fixed direction follows user-specified mode order.
            for metric in ("test_acc","test_macro_f1"):
                secondary.append(_test_summary(records,a,b,metric,"secondary_within_arch_mode_30"))
    _holm(primary, max(0, (len(args.architectures)*len(args.modes)-1)*2))
    _holm(secondary, len(args.architectures)*len(args.modes)*(len(args.modes)-1))
    save_csv(out/"final"/"pair_summary.csv",summary)
    save_csv(out/"final"/"per_class_summary.csv",per_class)
    save_csv(out/"final"/"statistical_tests.csv",primary+secondary)
    _json(out/"final"/"architecture_mode_summary.json",summary)
    _plot_summary(summary,out)


def _plot_summary(summary: list[dict[str, Any]], out: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    outdir=out/"report_assets"; outdir.mkdir(parents=True,exist_ok=True)
    labels=[f"{r['architecture']}\n{r['mode']}" for r in summary]
    for metric,title,name in (("test_macro_f1","Test Macro F1","macro_f1"),("test_acc","Test accuracy","accuracy")):
        vals=[r[metric+"_mean"] for r in summary]; sd=[r[metric+"_sample_sd"] or 0 for r in summary]
        fig,ax=plt.subplots(figsize=(13,5)); x=np.arange(len(labels))
        ax.bar(x,[v or 0 for v in vals],yerr=sd,capsize=3,color=["#167d9a" if r["mode"]=="full" else "#e69542" if r["mode"]=="frozen" else "#7c70b5" for r in summary])
        ax.set_xticks(x,labels,rotation=50,ha="right",fontsize=8); ax.set_ylim(0,1); ax.set_ylabel(title); ax.set_title(f"{title}: mean ± sample SD across matched seeds"); ax.grid(axis="y",alpha=.2); fig.tight_layout(); fig.savefig(outdir/f"pair_{name}.png",dpi=170); plt.close(fig)


def _confidence(args: argparse.Namespace, out: Path, records: list[dict[str,Any]]) -> dict[str,Any]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    confidence_dir=out/"confidence"; confidence_dir.mkdir(parents=True,exist_ok=True)
    records=_latest_completed_seed_rows(records)
    run_rows=[]; bin_rows=[]; rep_seed=42 if 42 in args.seeds else args.seeds[0]
    for record in records:
        if record.get("status")!="completed": continue
        path=Path(record["run_dir"])/"test_predictions.csv"
        if not path.is_file(): continue
        predictions=_read_csv(path)
        conf=np.asarray([float(r["confidence"]) for r in predictions]); correct=np.asarray([r["true_label"]==r["predicted_label"] for r in predictions],dtype=float)
        edges=np.linspace(0.,1.,11); bins=[]; ece=0.
        for b in range(10):
            mask=(conf>=edges[b]) & ((conf<edges[b+1]) if b<9 else (conf<=edges[b+1]))
            n=int(mask.sum()); avg=float(conf[mask].mean()) if n else None; acc=float(correct[mask].mean()) if n else None
            if n: ece+=(n/len(conf))*abs(acc-avg)
            item={"architecture":record["architecture"],"mode":record["mode"],"seed":record["seed"],"bin":b,"lower":edges[b],"upper":edges[b+1],"count":n,"mean_confidence":avg,"accuracy":acc}
            bins.append(item); bin_rows.append(item)
        row={"architecture":record["architecture"],"mode":record["mode"],"seed":record["seed"],"run_identity":record.get("identity"),
             "test_ece_10bin":float(ece),"correct_mean_confidence":float(conf[correct.astype(bool)].mean()) if correct.any() else None,
             "incorrect_mean_confidence":float(conf[~correct.astype(bool)].mean()) if (~correct.astype(bool)).any() else None,"test_n":len(predictions),"status":"completed"}
        run_rows.append(row)
        if int(record["seed"])==rep_seed:
            points=[x for x in bins if x["count"]]
            fig,ax=plt.subplots(figsize=(5,5));ax.plot([0,1],[0,1],"--",color="gray",label="ideal")
            if points:ax.plot([x["mean_confidence"] for x in points],[x["accuracy"] for x in points],"o-",label=f"ECE={ece:.3f}")
            ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean confidence",ylabel="Accuracy",title=f"{record['architecture']} · {record['mode']} · seed {rep_seed}")
            ax.grid(alpha=.2);ax.legend();fig.tight_layout();dest=confidence_dir/f"reliability_{record['architecture']}_{record['mode']}_seed{rep_seed}.png";fig.savefig(dest,dpi=150);plt.close(fig)
            fig,ax=plt.subplots(figsize=(6,4))
            if correct.any():ax.hist(conf[correct.astype(bool)],bins=np.linspace(0,1,11),alpha=.65,label="Correct")
            if (~correct.astype(bool)).any():ax.hist(conf[~correct.astype(bool)],bins=np.linspace(0,1,11),alpha=.65,label="Incorrect")
            ax.set(xlim=(0,1),xlabel="Top-class confidence",ylabel="Test image count",title=f"Confidence · {record['architecture']} · {record['mode']} · seed {rep_seed}")
            ax.grid(axis="y",alpha=.2);ax.legend();fig.tight_layout();fig.savefig(confidence_dir/f"confidence_hist_{record['architecture']}_{record['mode']}_seed{rep_seed}.png",dpi=150);plt.close(fig)
    _csv(confidence_dir/"run_metrics.csv",run_rows);_csv(confidence_dir/"bin_metrics.csv",bin_rows)
    summary=[]
    for arch in args.architectures:
        for mode in args.modes:
            vals=[r["test_ece_10bin"] for r in run_rows if r["architecture"]==arch and r["mode"]==mode]
            summary.append({"architecture":arch,"mode":mode,"completed_seeds":len(vals),"test_ece_10bin_mean":float(np.mean(vals)) if vals else None,
                            "test_ece_10bin_sample_sd":float(np.std(vals,ddof=1)) if len(vals)>=2 else None})
    _csv(confidence_dir/"pair_summary.csv",summary)
    return {"runs":len(run_rows),"bins":10}


def _analyze(args: argparse.Namespace, out: Path) -> dict[str, Any]:
    registry=_load_registry(out); selection=json.loads((out/"selection.json").read_text(encoding="utf-8"))
    records=registry["final_runs"]
    _summarize(args,out,records,selection)
    confidence=_confidence(args,out,records)
    _logged_curves(args,out,records)
    dynamics=_training_dynamics(args,out,records)
    _tuning_plot(out,registry["trials"])
    return {"run_attempts":len(records),"completed_unique_seeds":len(_latest_completed_seed_rows(records)),"confidence":confidence,"dynamics":dynamics}


def _logged_curves(args: argparse.Namespace, out: Path, records: list[dict[str,Any]]) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for record in _latest_completed_seed_rows(records):
        if record.get("status")!="completed" or int(record["seed"]) != (42 if 42 in args.seeds else args.seeds[0]): continue
        path=Path(record["run_dir"])/"history.csv"
        if not path.exists(): continue
        hist=[r for r in _read_csv(path) if r.get("train_loss") not in (None,"") and r.get("val_loss") not in (None,"")]
        if not hist: continue
        fig,axes=plt.subplots(1,3,figsize=(11,3.3))
        for ax,keys,title in zip(axes,(('train_loss','val_loss'),('train_acc','val_acc'),('train_macro_f1','val_macro_f1')),('Loss','Accuracy','Macro F1')):
            for key in keys: ax.plot([int(h['epoch']) for h in hist],[float(h[key]) for h in hist],label=key)
            ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=.2); ax.legend(fontsize=7)
        fig.suptitle(f"{record['architecture']} · {record['mode']} · seed {record['seed']}"); fig.tight_layout()
        p=out/'final'/'curves'/record['architecture']/record['mode']/f"seed{record['seed']}.png"; p.parent.mkdir(parents=True,exist_ok=True); fig.savefig(p,dpi=150); plt.close(fig)


def _tuning_plot(out: Path, rows: list[dict[str,Any]]) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    values=[r for r in rows if r.get('status')=='completed' and r.get('best_val_macro_f1') is not None]
    if not values:return
    fig,ax=plt.subplots(figsize=(8,5))
    for mode,marker in zip(MODES,('o','s','^')):
        subset=[r for r in values if r['mode']==mode]
        ax.scatter([r['config']['learning_rate'] for r in subset],[r['best_val_macro_f1'] for r in subset],label=mode,marker=marker,alpha=.7)
    ax.set_xscale('log');ax.set_xlabel('Learning rate');ax.set_ylabel('Best validation Macro F1');ax.set_title('Logged search trials (train/validation only)');ax.grid(alpha=.2);ax.legend()
    p=out/'tuning'/'tuning_validation_scatter.png';p.parent.mkdir(parents=True,exist_ok=True);fig.tight_layout();fig.savefig(p,dpi=160);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,5))
    for mode,marker in zip(MODES,('o','s','^')):
        subset=[r for r in values if r['mode']==mode]
        ax.scatter([r['config']['weight_decay'] for r in subset],[r['best_val_macro_f1'] for r in subset],label=mode,marker=marker,alpha=.7)
    ax.set_xscale('log');ax.set_xlabel('Weight decay');ax.set_ylabel('Best validation Macro F1');ax.set_title('Logged trial weight decay vs validation score');ax.grid(alpha=.2);ax.legend()
    fig.tight_layout();fig.savefig(out/'tuning'/'weight_decay_validation_scatter.png',dpi=160);plt.close(fig)


def _training_dynamics(args: argparse.Namespace, out: Path, records: list[dict[str,Any]]) -> dict[str,Any]:
    rows=[]
    for record in _latest_completed_seed_rows(records):
        path=Path(record["run_dir"])/"history.csv"
        if not path.is_file():continue
        history=[r for r in _read_csv(path) if r.get("train_loss") not in (None,"") and r.get("val_loss") not in (None,"")]
        best=next((r for r in history if int(r["epoch"])==int(record["best_epoch"])),None)
        if best is None:continue
        rows.append({"architecture":record["architecture"],"mode":record["mode"],"seed":record["seed"],
            "run_identity":record.get("identity"),"best_epoch":record["best_epoch"],"stop_epoch":record.get("epochs_completed"),
            "train_val_loss_gap_at_best":float(best["val_loss"])-float(best["train_loss"]),
            "train_val_accuracy_gap_at_best":float(best["train_acc"])-float(best["val_acc"]),
            "train_val_macro_f1_gap_at_best":float(best["train_macro_f1"])-float(best["val_macro_f1"]),"status":"completed"})
    dest=out/"diagnostics";dest.mkdir(parents=True,exist_ok=True);_csv(dest/"run_dynamics.csv",rows)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if rows:
        labels=[];loss_gaps=[];f1_gaps=[]
        for arch in args.architectures:
            for mode in args.modes:
                group=[r for r in rows if r["architecture"]==arch and r["mode"]==mode]
                labels.append(f"{arch}\n{mode}");loss_gaps.append(float(np.mean([r["train_val_loss_gap_at_best"] for r in group])) if group else 0)
                f1_gaps.append(float(np.mean([r["train_val_macro_f1_gap_at_best"] for r in group])) if group else 0)
        fig,axes=plt.subplots(1,2,figsize=(14,5));x=np.arange(len(labels))
        axes[0].bar(x,loss_gaps);axes[0].set_title("Validation − train loss at selected checkpoint");axes[0].set_ylabel("Loss gap")
        axes[1].bar(x,f1_gaps);axes[1].set_title("Train − validation Macro F1 at selected checkpoint");axes[1].set_ylabel("Macro F1 gap")
        for ax in axes:ax.set_xticks(x,labels,rotation=55,ha="right",fontsize=7);ax.grid(axis="y",alpha=.2)
        fig.tight_layout();fig.savefig(dest/"training_gaps.png",dpi=160);plt.close(fig)
    return {"run_rows":len(rows),"gap_figure":str(dest/"training_gaps.png") if rows else None}


def _eyeball_and_cam(args: argparse.Namespace, out: Path) -> dict[str,Any]:
    import torch
    from PIL import Image
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from torchvision import transforms, models
    from shrine_classifier.data import open_rgb, letterbox, IMAGENET_MEAN, IMAGENET_STD
    from shrine_classifier.models import build_model
    from shrine_classifier.analysis import gradcam
    selection,winners=_verify_lock(args,out); ref=selection['reference_pair']; seed=selection['representative_seed']
    row=next(r for r in _latest_completed_seed_rows(_load_registry(out)['final_runs']) if r.get('architecture')==ref['architecture'] and r.get('mode')==ref['mode'] and int(r.get('seed',-1))==seed)
    preds=_read_csv(Path(row['run_dir'])/'test_predictions.csv')
    chosen=[]
    for cls in CLASSES:
        subset=[p for p in preds if p['true_label']==cls]
        for right in (True,False):
            found=[p for p in subset if (p['predicted_label']==cls)==right]
            if found: chosen.append(sorted(found,key=lambda p:(float(p['confidence']),p['image_id']) if not right else (-float(p['confidence']),p['image_id']))[0])
            elif subset:
                candidate=min(subset,key=lambda p:(float(p['confidence']),p['image_id']))
                if candidate not in chosen: chosen.append(candidate)
    if not chosen: raise RuntimeError('No test predictions available for CAM/eyeball')
    device='cuda' if args.device=='auto' and torch.cuda.is_available() else ('cpu' if args.device=='auto' else args.device)
    # The ImageNet output space remains 1,000 classes; save its generic top label
    # for the exact same images, and never calculate shrine-class metrics from it.
    enum_name={'resnet50':'ResNet50_Weights','efficientnet_b0':'EfficientNet_B0_Weights','mobilenet_v3_large':'MobileNet_V3_Large_Weights','densenet121':'DenseNet121_Weights','convnext_tiny':'ConvNeXt_Tiny_Weights'}[ref['architecture']]
    weights=getattr(models,enum_name).DEFAULT; imagenet_model=getattr(models,ref['architecture'])(weights=weights).to(device).eval(); image_transform=weights.transforms()
    split_lookup={r['image_id']:r for r in _read_csv(out/'splits.csv')}
    imagenet_top1={}
    with torch.no_grad():
        for item in chosen:
            image=open_rgb(args.data_root/split_lookup[item['image_id']]['path'])
            logits=imagenet_model(image_transform(image).unsqueeze(0).to(device)); probs=torch.softmax(logits,dim=1)[0]
            score,index=probs.max(0);imagenet_top1[item['image_id']]={'imagenet_label':weights.meta['categories'][int(index)],'imagenet_confidence':float(score)}
    del imagenet_model
    if torch.cuda.is_available():torch.cuda.empty_cache()
    model,meta=build_model(ref['architecture'],pretrained=False); state=torch.load(row['checkpoint'],map_location=device,weights_only=False); model.load_state_dict(state['model_state']);model.to(device).eval()
    transforms_tensor=transforms.Compose([transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    cam_dir=out/'gradcam';cam_dir.mkdir(parents=True,exist_ok=True)
    fig,axes=plt.subplots(len(chosen),3,figsize=(10,3.1*len(chosen)),squeeze=False)
    summary=[]
    for i,item in enumerate(chosen):
        rec=next(r for r in _read_csv(out/'splits.csv') if r['image_id']==item['image_id'])
        image=open_rgb(args.data_root/rec['path']);canvas,bbox=letterbox(image,args.input_size);tensor=transforms_tensor(canvas).unsqueeze(0).to(device)
        idx=CLASSES.index(item['predicted_label'])
        cam=gradcam(model,ref['architecture'],tensor,idx)[0].detach().cpu().numpy()
        x1,y1,x2,y2=bbox;cam=Image.fromarray(np.uint8(np.clip(cam[y1:y2,x1:x2],0,1)*255)).resize(image.size,Image.Resampling.BILINEAR)
        heat=np.asarray(cam,dtype=np.float32)/255
        base=np.asarray(image,dtype=np.float32); colored=plt.get_cmap('jet')(heat)[...,:3]*255
        overlay=np.uint8(np.clip(base*.55+colored*.45,0,255))
        axes[i,0].imshow(image);axes[i,0].set_title(f"Original · true {item['true_label']}")
        axes[i,1].imshow(heat,cmap='jet');axes[i,1].set_title(f"Grad-CAM target: {item['predicted_label']}")
        axes[i,2].imshow(overlay);axes[i,2].set_title(f"Pred {item['predicted_label']} · conf {float(item['confidence']):.2f}")
        for ax in axes[i]:ax.axis('off')
        sample_dir=cam_dir/'samples'/item['image_id'].replace('/','__');sample_dir.mkdir(parents=True,exist_ok=True)
        image.save(sample_dir/'original.png');cam.save(sample_dir/'heatmap.png');Image.fromarray(overlay).save(sample_dir/'overlay.png')
        summary.append({**item,**imagenet_top1[item['image_id']],'reason':'correct case' if item['true_label']==item['predicted_label'] else 'misclassified case', 'explained_target':item['predicted_label'],'checkpoint':row['checkpoint'],'cam_files':str(sample_dir.relative_to(out))})
    fig.tight_layout();fig.savefig(cam_dir/'reference_pair_cam_grid.png',dpi=150);plt.close(fig)
    _csv(cam_dir/'selected_cases.csv',summary);_csv(out/'eyeball'/'eyeball_cases.csv',summary)
    fig,axes=plt.subplots(len(chosen),2,figsize=(8,3*len(chosen)),squeeze=False)
    for i,item in enumerate(chosen):
        rec=next(r for r in _read_csv(out/'splits.csv') if r['image_id']==item['image_id']);image=open_rgb(args.data_root/rec['path'])
        axes[i,0].imshow(image);axes[i,0].set_title(f"True: {item['true_label']} · {item['image_id']}")
        axes[i,1].text(0,.75,f"ImageNet top-1: {summary[i]['imagenet_label']} ({summary[i]['imagenet_confidence']:.2f})\nFine-tuned: {item['predicted_label']} ({float(item['confidence']):.2f})\nCorrect: {item['true_label']==item['predicted_label']}\nReason: {summary[i]['reason']}",fontsize=11);axes[i,1].axis('off');axes[i,0].axis('off')
    fig.tight_layout();(out/'eyeball').mkdir(parents=True,exist_ok=True);fig.savefig(out/'eyeball'/'eyeball_grid.png',dpi=150);plt.close(fig)
    return {'reference_pair':ref,'seed':seed,'samples':len(chosen)}


def _report(args: argparse.Namespace, out: Path) -> dict[str,Any]:
    summary=_read_csv(out/'final'/'pair_summary.csv')
    tests=_read_csv(out/'final'/'statistical_tests.csv')
    selection=json.loads((out/'selection.json').read_text(encoding='utf-8'))
    all_registry=_load_registry(out)
    protocol=json.loads((out/'protocol.json').read_text(encoding='utf-8'))
    tuning_target=len(protocol['architectures'])*len(protocol['modes'])*int(protocol['trials_per_pair'])
    final_target=len(protocol['architectures'])*len(protocol['modes'])*len(protocol['final_seeds'])
    seeds_per_pair=len(protocol['final_seeds'])
    completed=len(_latest_completed_seed_rows(all_registry['final_runs']))
    completed_trials=len({r.get('identity') for r in all_registry['trials'] if r.get('status')=='completed'})
    winners=json.loads((out/'tuning'/'best_params_by_arch_mode.json').read_text(encoding='utf-8'))['pairs']
    recipe_rows=''.join(f"<tr><td>{html.escape(v['architecture'])}</td><td>{html.escape(v['mode'])}</td><td>{float(v['learning_rate']):.3g}</td><td>{float(v['weight_decay']):.3g}</td><td>{html.escape(v['optimizer'])}</td><td>{v['batch_size']}</td><td>{v['winning_trial']}</td></tr>" for v in winners.values())
    source=json.loads((out/'source_manifest.json').read_text(encoding='utf-8'))
    sections=[]
    for row in summary:
        f1=row.get('test_macro_f1_mean');sd=row.get('test_macro_f1_sample_sd')
        value = f"{float(f1):.3f} ± {float(sd):.3f}" if f1 not in (None, "", "nan") and sd not in (None, "", "nan") else "incomplete"
        sections.append(f"<tr><td>{html.escape(str(row['architecture']))}</td><td>{html.escape(str(row['mode']))}</td><td>{int(row['completed_seeds'])}/{seeds_per_pair}</td><td>{value}</td></tr>")
    figure_sources=[(p,p.name) for p in sorted((out/'report_assets').glob('*.png'))]
    for rel in ('dataset_audit/class_distribution.png','dataset_audit/sample_grid.png','dataset_audit/preprocessing_examples.png',
                'dataset_audit/near_duplicate_review.png','baseline/pretrained_examples.png',
                'gradcam/reference_pair_cam_grid.png','eyeball/eyeball_grid.png'):
        p=out/rel
        if p.is_file():figure_sources.append((p,'../'+rel.replace('\\','/')))
    reference=selection['reference_pair'];representative=selection['representative_seed']
    for rel in (f"confidence/reliability_{reference['architecture']}_{reference['mode']}_seed{representative}.png",
                f"confidence/confidence_hist_{reference['architecture']}_{reference['mode']}_seed{representative}.png"):
        p=out/rel
        if p.is_file():figure_sources.append((p,'../'+rel))
    for rel in ('tuning/tuning_validation_scatter.png','tuning/weight_decay_validation_scatter.png','diagnostics/training_gaps.png',
                f"final/curves/{reference['architecture']}/{reference['mode']}/seed{representative}.png"):
        p=out/rel
        if p.is_file():figure_sources.append((p,'../'+rel))
    figures=''.join(f"<figure><img src='{src}'><figcaption>{html.escape(p.name)}</figcaption></figure>" for p,src in figure_sources)
    counts={}
    for row in tests:
        counts[row.get('status','unknown')]=counts.get(row.get('status','unknown'),0)+1
    body=f"""<!doctype html><html lang='th'><meta charset='utf-8'><title>สรุปการทดลอง Thai Shrine</title><style>body{{font-family:system-ui,'Noto Sans Thai',sans-serif;max-width:1100px;margin:2rem auto;line-height:1.6}}table{{border-collapse:collapse;width:100%}}th,td{{border:1px solid #aaa;padding:.45rem;text-align:left}}th{{background:#17324d;color:white}}img{{max-width:100%}}figure{{margin:2rem 0}}code{{background:#eee;padding:.15rem}}</style><h1>อธิบายผลการทดลอง Thai Shrine Classifier</h1>
<p>เอกสารนี้สร้างจาก logs ภายใต้ <code>{html.escape(out.name)}</code> เท่านั้น งานที่บันทึกเสร็จ: tuning สำเร็จ {completed_trials}/{tuning_target} unique trials; final {completed}/{final_target} unique seeds.</p>
<h2>ข้อมูลและ preprocessing</h2><p>ใช้ภาพ {source['image_count']} ภาพใน 4 classes และ canonical grouped split เดิม train 390 / validation 85 / test 85 จาก 550 groups. URL ไม่ทราบ {source['unknown_url_count']} ภาพ. แปลงภาพเป็น RGB, EXIF orientation, resize โดยรักษาสัดส่วนลง canvas 224 × 224 สี neutral RGB (124,116,104), แล้ว normalize แบบ ImageNet. Augmentation ใช้ train เท่านั้น.</p>
<h2>คำถามที่ทดลอง</h2><p>เปรียบเทียบ CNN 5 แบบและนโยบายการฝึก 3 แบบ โดยให้แต่ละคู่สุ่มค้น learning rate, weight decay, optimizer และ batch size จาก validation ของตัวเอง ดังนั้น Full, Frozen และ Partial มี hyperparameters ชนะของตัวเอง. ใช้ ImageNet weights, class-weighted cross entropy, AdamW/Adam/SGD และ ReduceLROnPlateau. Frozen ฝึก classifier; Partial เปิด semantic final stage กับ classifier; Full ฝึก backbone ทั้งหมด.</p>
<h2>ขั้นตอนและการอ่านผล</h2><ol><li>ตรวจไฟล์/provenance และใช้ grouped split เดิม 70/15/15 เพื่อกัน duplicate group ข้าม split</li><li>สุ่ม {int(protocol['trials_per_pair'])} configurations ต่อ architecture × mode ด้วย seed {int(protocol['tuning_seed'])}; trials ใช้ train/validation เท่านั้น คะแนนในกราฟ tuning คือผล validation ไม่ใช่ test</li><li>เลือกสูตรชนะต่อคู่ตาม best validation Macro F1, Accuracy, lower loss, trial order แล้วล็อก reference pair จาก {len(protocol['architectures'])*len(protocol['modes'])} คู่ด้วย validation</li><li>ฝึกจาก ImageNet weights ใหม่ {seeds_per_pair} matched seeds ต่อคู่ เลือก checkpoint ด้วย validation แล้วประเมิน test ภายหลัง</li><li>สรุป mean ± sample SD จาก seeds; paired tests join ด้วย seed ID และปรับ Holm แยก primary family 28 และ secondary family 30</li></ol>
<h2>Validation-selected reference</h2><p><b>{html.escape(selection['reference_pair']['architecture'])} / {html.escape(selection['reference_pair']['mode'])}</b>; selection ไม่ได้ใช้ test scores. test split เคยเปิดดูในงานก่อนหน้า การทดลองใหม่นี้จึงเป็น exploratory บน split เดิม.</p>
<h2>ผลแยกตามคู่</h2><table><thead><tr><th>Architecture</th><th>Mode</th><th>Seeds</th><th>Test Macro F1 mean ± sample SD</th></tr></thead><tbody>{''.join(sections)}</tbody></table>
<h2>สูตรที่ชนะจาก validation ต่อคู่</h2><table><thead><tr><th>Architecture</th><th>Mode</th><th>Learning rate</th><th>Weight decay</th><th>Optimizer</th><th>Batch</th><th>Trial</th></tr></thead><tbody>{recipe_rows}</tbody></table>
<p>ผล class-specific อยู่ใน <code>final/per_class_summary.csv</code>. Confusion matrix มีทั้งจำนวนภาพและสัดส่วนต่อ true class ต่อ run.</p><p>Calibration ใช้ Expected Calibration Error 10 bins จาก confidence ของ predictions หลัง final; ตาราง/รูปแยก architecture × mode × seed อยู่ใน <code>confidence/</code>.</p>
<p>สถานะ paired tests: {html.escape(json.dumps(counts,ensure_ascii=False))}. อ่านค่า effect size, p และ adjusted p ใน <code>final/statistical_tests.csv</code>; ค่า undefined/incomplete ไม่ใช่ผลไม่มีนัยสำคัญ.</p>
<h2>Figures จาก log</h2>{figures or '<p>ยังไม่มีกราฟจากผล final ที่บันทึก</p>'}
<h2>ข้อจำกัด</h2><ul><li>ผลต่างระหว่าง seeds สะท้อนความแปรปรวนบน split เดียว ไม่แทนชุดข้อมูลใหม่</li><li>การทำ CAM เป็นเครื่องมือวินิจฉัย ไม่ยืนยันเหตุและผล</li><li>อ่านค่า W&amp;B ตามสถานะจริงของแต่ละ run; การมีไฟล์ local ไม่ยืนยันว่า sync ขึ้น cloud</li><li>แนวทางผู้สอนและ PDF อ้างอิงไม่มีในโฟลเดอร์นี้ จึงยังยืนยันความสอดคล้องกับผู้สอนไม่ได้</li></ul>
<h2>แหล่งข้อมูล</h2><p>metrics: <code>final/run_metrics.csv</code>; configs/history: ไดเรกทอรี run ใน <code>tuning/</code> และ <code>final/</code>; definitions: <code>protocol.json</code>, <code>selection.json</code>; pair summaries, class metrics, statistical tests และรูปภาพอยู่ใต้ out-dir เดียวกัน.</p></html>"""
    p=out/'report_assets'/'understanding_th.html';p.parent.mkdir(parents=True,exist_ok=True);p.write_text(body,encoding='utf-8')
    return {'path':str(p),'final_runs':completed,'tuning_trials':len(_load_registry(out)['trials'])}


def _smoke(args: argparse.Namespace, out: Path) -> dict[str,Any]:
    import torch
    from shrine_classifier.models import build_model
    from shrine_classifier_ext.finetuning_modes import (apply_finetuning_mode,enforce_mode_per_epoch,
        _get_head_modules,_get_partial_backbone_modules)
    device='cuda' if args.device=='auto' and torch.cuda.is_available() else ('cpu' if args.device=='auto' else args.device)
    results=[]
    for arch in args.architectures:
        for mode in args.modes:
            model,meta=build_model(arch,pretrained=False);model.to(device);stats=apply_finetuning_mode(model,arch,mode)
            parameters=[p for p in model.parameters() if p.requires_grad]
            all_parameters=list(model.parameters())
            expected_ids={id(p) for p in all_parameters} if mode=='full' else {id(p) for m in _get_head_modules(model,arch) for p in m.parameters()}
            if mode=='partial':expected_ids.update(id(p) for m in _get_partial_backbone_modules(model,arch) for p in m.parameters())
            actual_ids={id(p) for p in all_parameters if p.requires_grad}
            optimizer=torch.optim.SGD(parameters,lr=1e-4)
            model.train();enforce_mode_per_epoch(model,arch,mode)
            x=torch.randn(2,3,args.input_size,args.input_size,device=device);y=torch.tensor([0,1],device=device)
            before={n:p.detach().clone() for n,p in model.named_parameters() if not p.requires_grad}
            before_trainable={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
            frozen_bn_buffers={}
            for module_name,module in model.named_modules():
                if isinstance(module,torch.nn.modules.batchnorm._BatchNorm) and not module.training:
                    for buffer_name,value in module.named_buffers(recurse=False):
                        frozen_bn_buffers[f'{module_name}.{buffer_name}']=value.detach().clone()
            loss=torch.nn.functional.cross_entropy(model(x),y);loss.backward();optimizer.step()
            unchanged=all(torch.equal(v,dict(model.named_parameters())[n]) for n,v in before.items())
            current_params=dict(model.named_parameters())
            updated=any(not torch.equal(value,current_params[name]) for name,value in before_trainable.items())
            current_buffers=dict(model.named_buffers())
            bn_stable=all(torch.equal(value,current_buffers[name]) for name,value in frozen_bn_buffers.items())
            no_frozen_grad=all(current_params[name].grad is None for name in before)
            passed=(actual_ids==expected_ids and unchanged and updated and bn_stable and no_frozen_grad and loss.isfinite().item())
            row={'architecture':arch,'mode':mode,'logits':[2,4],'trainable_parameters':stats['trainable_parameters'],'frozen_parameters':stats['frozen_parameters'],
                 'trainable_membership_verified':actual_ids==expected_ids,'frozen_unchanged':unchanged,'trainable_parameter_updated':updated,
                 'frozen_bn_buffers_stable':bn_stable,'frozen_gradient_absent':no_frozen_grad,'pretrained':False,'test_split_accessed':False,'status':'passed' if passed else 'failed'}
            results.append(row);del model,optimizer,x,loss
            if torch.cuda.is_available():torch.cuda.empty_cache()
            if row['status']!='passed':raise RuntimeError(f"Mode smoke failed: {row}")
    _json(out/'smoke'/'mode_smoke.json',{'models':results,'status':'passed','test_split_accessed':False,'measured_training':False,'weights':'synthetic random; not pretrained'})
    return {'models':len(results),'status':'passed','test_split_accessed':False}


def _parser() -> argparse.ArgumentParser:
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--stage',required=True,choices=('prepare','smoke','tune','lock','final','analyze','gradcam','eyeball','report','all'))
    p.add_argument('--data-root',type=Path,default=Path('dataset_cleaned'));p.add_argument('--padded-root',type=Path,default=Path('dataset_padding'))
    p.add_argument('--base-out-dir',type=Path,default=Path('outputs'));p.add_argument('--out-dir',type=Path,default=Path('outputs_modes_v3'))
    p.add_argument('--architectures',nargs='+',choices=ARCHITECTURES,default=list(ARCHITECTURES));p.add_argument('--modes',nargs='+',choices=MODES,default=list(MODES))
    p.add_argument('--trials-per-pair','--trials',dest='trials_per_pair',type=int,default=5);p.add_argument('--runs',type=int,default=5);p.add_argument('--seeds',type=int,nargs='+',default=list(SEEDS))
    p.add_argument('--epochs',type=int,default=30);p.add_argument('--input-size',type=int,default=224);p.add_argument('--seed',type=int,default=42);p.add_argument('--split-seed',type=int,default=42)
    p.add_argument('--device',choices=('auto','cpu','cuda'),default='auto');p.add_argument('--num-workers',type=int,default=0)
    p.add_argument('--wandb-project',default='thai-shrine-classifier');p.add_argument('--wandb-entity',default=None);p.add_argument('--wandb-mode',choices=('online','offline','disabled'),default='online')
    p.add_argument('--resume',action='store_true');p.add_argument('--dry-run',action='store_true')
    return p


def main(argv: list[str] | None = None) -> int:
    args=_parser().parse_args(argv)
    args.out_dir=args.out_dir.resolve();args.data_root=args.data_root.resolve();args.base_out_dir=args.base_out_dir.resolve();args.padded_root=args.padded_root.resolve()
    if args.runs!=len(args.seeds) or len(set(args.seeds))!=len(args.seeds):raise ValueError('--runs must equal count of unique --seeds')
    if args.trials_per_pair<1 or args.epochs<1 or not args.architectures or not args.modes:raise ValueError('architecture/mode/trial/epoch selections must be non-empty and positive')
    if args.dry_run:
        print(json.dumps({'stage':args.stage,'out_dir':str(args.out_dir),'architectures':args.architectures,'modes':args.modes,'trials_per_pair':args.trials_per_pair,'tuning_runs':len(args.architectures)*len(args.modes)*args.trials_per_pair,'final_runs':len(args.architectures)*len(args.modes)*args.runs,'max_epochs':len(args.architectures)*len(args.modes)*(args.trials_per_pair+args.runs)*args.epochs,'test_access_during_tune_or_lock':False,'measured_run_started':False},ensure_ascii=False,indent=2));return 0
    args.out_dir.mkdir(parents=True,exist_ok=True);(args.out_dir/'wandb').mkdir(exist_ok=True)
    os.environ['WANDB_DIR']=str(args.out_dir/'wandb');os.environ.setdefault('WANDB_INIT_TIMEOUT','15');os.environ.setdefault('WANDB_START_METHOD','thread')
    stages=('prepare','smoke','tune','lock','final','analyze','gradcam','report') if args.stage=='all' else (args.stage,)
    for stage in stages:
        start=time.time()
        if stage=='prepare': result=_verify_and_prepare(args,args.out_dir)
        elif stage=='smoke':result=_smoke(args,args.out_dir)
        elif stage=='tune':result=_tune(args,args.out_dir)
        elif stage=='lock':result=_lock(args,args.out_dir)
        elif stage=='final':result=_final(args,args.out_dir)
        elif stage=='analyze':result=_analyze(args,args.out_dir)
        elif stage in ('gradcam','eyeball'):result=_eyeball_and_cam(args,args.out_dir)
        elif stage=='report':result=_report(args,args.out_dir)
        print(f"[{stage}] {json.dumps(result,ensure_ascii=False,default=str)[:1200]} elapsed={time.time()-start:.1f}s",flush=True)
    return 0


if __name__=='__main__':
    import multiprocessing
    multiprocessing.freeze_support()
    raise SystemExit(main())


In [ ]:
%%project_source plot_learning_curves.py
"""
plot_learning_curves.py
=======================
Plot mean ± 1 SD learning curves (loss and accuracy) for every
architecture × mode pair in outputs_modes_v3_td.

Each pair's 5 seeds are aggregated epoch-by-epoch:
  - Solid/dashed line  = mean across seeds
  - Shaded band        = ±1 SD across seeds
  - Dotted vertical    = mean best epoch (from run_metrics.json)

Outputs
-------
  outputs_modes_v3_td/final/curves/<arch>/<mode>/mean_sd_curves.png
      one figure per pair (loss + acc subplots)

  outputs_modes_v3_td/final/curves/<arch>/mean_sd_grid.png
      3-mode side-by-side grid per architecture (mirrors reference Fig 7.2)

  outputs_modes_v3_td/final/curves/all_pairs_grid.png
      single large overview of all 15 pairs

Usage
-----
  python plot_learning_curves.py
  python plot_learning_curves.py --out-dir outputs_modes_v3_td --metric loss
  python plot_learning_curves.py --arch efficientnet_b0 --mode full frozen
  python plot_learning_curves.py --no-per-pair --no-overview
"""

import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # headless — no display required
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# ── canonical orders (AGENTS.md) ────────────────────────────────────────────
ARCHITECTURES = [
    "resnet50", "efficientnet_b0", "mobilenet_v3_large",
    "densenet121", "convnext_tiny",
]
MODES = ["full", "frozen", "partial"]
SEEDS = [42, 123, 456, 789, 1024]

ARCH_LABELS = {
    "resnet50":           "ResNet-50",
    "efficientnet_b0":    "EfficientNet-B0",
    "mobilenet_v3_large": "MobileNetV3-Large",
    "densenet121":        "DenseNet-121",
    "convnext_tiny":      "ConvNeXt-Tiny",
}
MODE_LABELS = {
    "full":    "Full fine-tuning",
    "frozen":  "Frozen backbone",
    "partial": "Partial (last stage)",
}

# ── visual style ─────────────────────────────────────────────────────────────
TRAIN_COLOR = "#4878CF"
VAL_COLOR   = "#E07B39"
BEST_COLOR  = "#666666"
BAND_ALPHA  = 0.20

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})


# ── data helpers ─────────────────────────────────────────────────────────────

def find_run_dirs(base: Path, arch: str, mode: str) -> list:
    pair_dir = base / "final" / arch / mode
    if not pair_dir.exists():
        return []
    return sorted(d for d in pair_dir.iterdir() if d.is_dir())


def load_history(run_dir: Path):
    csv = run_dir / "history.csv"
    if not csv.exists():
        return None
    try:
        df = pd.read_csv(csv)
        required = {"epoch", "train_loss", "val_loss", "train_acc", "val_acc"}
        if not required.issubset(df.columns):
            print(f"    [warn] missing columns in {csv}")
            return None
        # Drop summary/test rows appended at the end (they have NaN train_loss)
        df = df[df["train_loss"].notna()].copy()
        # Deduplicate by epoch, keeping the first (actual training) row
        df = df.drop_duplicates(subset="epoch", keep="first")
        return df.set_index("epoch")
    except Exception as exc:
        print(f"    [warn] cannot read {csv}: {exc}")
        return None


def load_best_epoch(run_dir: Path):
    p = run_dir / "run_metrics.json"
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())["best_epoch"]
    except Exception:
        return None


def aggregate(dfs: list, col: str):
    """
    Align per-seed histories by epoch (NaN-pad shorter runs).
    Returns (epochs_1d, mean_1d, sd_1d).
    """
    if not dfs:
        return np.array([]), np.array([]), np.array([])

    max_ep = max(df.index.max() for df in dfs)
    epochs = np.arange(1, int(max_ep) + 1)
    arr = np.full((len(dfs), len(epochs)), np.nan)
    for i, df in enumerate(dfs):
        vals = df[col].reindex(epochs).values
        arr[i, :len(vals)] = vals

    mean = np.nanmean(arr, axis=0)
    sd   = np.nanstd(arr, axis=0, ddof=1)
    sd   = np.where(np.isnan(sd), 0.0, sd)   # single-seed → 0 not NaN
    return epochs, mean, sd


# ── low-level drawing ────────────────────────────────────────────────────────

def _draw(
    ax, epochs,
    tr_mean, tr_sd,
    vl_mean, vl_sd,
    mean_best_epoch,
    ylabel, title,
    show_legend=True,
    show_xlabel=True,
):
    h1, = ax.plot(epochs, tr_mean, color=TRAIN_COLOR, linestyle="--",
                  linewidth=1.6, label="Train (mean ± 1 SD)")
    ax.fill_between(epochs, tr_mean - tr_sd, tr_mean + tr_sd,
                    color=TRAIN_COLOR, alpha=BAND_ALPHA)

    h2, = ax.plot(epochs, vl_mean, color=VAL_COLOR, linestyle="-",
                  linewidth=1.6, label="Validation (mean ± 1 SD)")
    ax.fill_between(epochs, vl_mean - vl_sd, vl_mean + vl_sd,
                    color=VAL_COLOR, alpha=BAND_ALPHA)

    h3 = None
    if mean_best_epoch is not None:
        h3 = ax.axvline(mean_best_epoch, color=BEST_COLOR, linestyle=":",
                        linewidth=1.2, label="Mean best epoch")

    ax.set_title(title, fontsize=9, pad=4)
    ax.set_ylabel(ylabel, fontsize=8)
    if show_xlabel:
        ax.set_xlabel("Epoch", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))
    ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.55)
    ax.set_xlim(left=1)

    if show_legend:
        handles = [h1, h2] + ([h3] if h3 else [])
        ax.legend(handles=handles, fontsize=6.5, loc="best", framealpha=0.85)

    return [h1, h2] + ([h3] if h3 else [])


def _collect(base, arch, mode):
    """Return (dfs, mean_best_epoch_or_None)."""
    run_dirs    = find_run_dirs(base, arch, mode)
    dfs         = [df for rd in run_dirs if (df := load_history(rd)) is not None]
    best_epochs = [be for rd in run_dirs if (be := load_best_epoch(rd)) is not None]
    mean_be     = float(np.mean(best_epochs)) if best_epochs else None
    return dfs, mean_be


# ── figure 1: per-pair ───────────────────────────────────────────────────────

def plot_pair(base: Path, arch: str, mode: str, save_dir: Path, metric: str = "all"):
    """Loss / accuracy (/ F1) subplots for one arch × mode pair."""
    dfs, mean_be = _collect(base, arch, mode)
    if not dfs:
        print(f"    [skip] {arch}/{mode} — no valid history.csv")
        return None

    n = len(dfs)
    has_f1 = "train_macro_f1" in dfs[0].columns

    rows_to_plot = []
    if metric in ("all", "loss"):
        rows_to_plot.append("loss")
    if metric in ("all", "acc"):
        rows_to_plot.append("acc")
    if metric in ("all", "f1") and has_f1:
        rows_to_plot.append("f1")

    fig, axes = plt.subplots(len(rows_to_plot), 1,
                             figsize=(6, 3 * len(rows_to_plot)), squeeze=False)
    fig.suptitle(
        f"Learning curves (mean ± SD, n={n} seeds)\n"
        f"{ARCH_LABELS.get(arch, arch)} — {MODE_LABELS.get(mode, mode)}",
        fontsize=10, fontweight="bold", y=1.01,
    )

    for r, m in enumerate(rows_to_plot):
        ax = axes[r, 0]
        is_last = (r == len(rows_to_plot) - 1)
        label = chr(97 + r)
        if m == "loss":
            ep, tr_m, tr_s = aggregate(dfs, "train_loss")
            _,  vl_m, vl_s = aggregate(dfs, "val_loss")
            _draw(ax, ep, tr_m, tr_s, vl_m, vl_s, mean_be,
                  ylabel="Weighted CE loss", title=f"({label}) Loss",
                  show_legend=(r == 0), show_xlabel=is_last)
        elif m == "acc":
            ep, tr_m, tr_s = aggregate(dfs, "train_acc")
            _,  vl_m, vl_s = aggregate(dfs, "val_acc")
            _draw(ax, ep, tr_m, tr_s, vl_m, vl_s, mean_be,
                  ylabel="Accuracy", title=f"({label}) Accuracy",
                  show_legend=(r == 0), show_xlabel=is_last)
            ax.set_ylim(0, 1.05)
        elif m == "f1":
            ep, tr_m, tr_s = aggregate(dfs, "train_macro_f1")
            _,  vl_m, vl_s = aggregate(dfs, "val_macro_f1")
            _draw(ax, ep, tr_m, tr_s, vl_m, vl_s, mean_be,
                  ylabel="Macro F1", title=f"({label}) Macro F1",
                  show_legend=(r == 0), show_xlabel=is_last)
            ax.set_ylim(0, 1.05)

    plt.tight_layout()
    save_dir.mkdir(parents=True, exist_ok=True)
    out = save_dir / "mean_sd_curves.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    [saved] {out}")
    return out


# ── figure 2: per-architecture 3-mode grid (mirrors reference Fig 7.2) ──────

def plot_per_arch_grid(base: Path, arch: str, save_path: Path):
    """
    2 rows × 3 cols
      rows:    loss (top), accuracy (bottom)
      columns: full, frozen, partial
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), squeeze=False)
    arch_label = ARCH_LABELS.get(arch, arch)

    fig.suptitle(
        f"Learning curves (mean ± SD over seeds) — {arch_label}",
        fontsize=12, fontweight="bold",
    )

    legend_handles = None

    for m_idx, mode in enumerate(MODES):
        dfs, mean_be = _collect(base, arch, mode)
        n = len(dfs)
        ml = MODE_LABELS.get(mode, mode)

        for row, metric in enumerate(["loss", "acc"]):
            ax = axes[row, m_idx]
            # subplot label: (a)…(f) column-major order to match reference
            col_letter = chr(97 + row * 3 + m_idx)
            sub_title = f"({col_letter}) {ml} — {metric}"
            if n:
                sub_title += f"  [n={n}]"

            if not dfs:
                ax.set_title(sub_title + "\n(no data)", fontsize=8.5)
                continue

            col_tr = "train_loss" if metric == "loss" else "train_acc"
            col_vl = "val_loss"   if metric == "loss" else "val_acc"
            ylabel = "Weighted CE loss" if metric == "loss" else "Accuracy"

            ep, tr_m, tr_s = aggregate(dfs, col_tr)
            _,  vl_m, vl_s = aggregate(dfs, col_vl)

            handles = _draw(
                ax, ep, tr_m, tr_s, vl_m, vl_s, mean_be,
                ylabel=ylabel, title=sub_title,
                show_legend=False,
                show_xlabel=(row == 1),
            )
            if legend_handles is None:
                legend_handles = handles

            if metric == "acc":
                ax.set_ylim(0, 1.05)

    # shared legend below title
    if legend_handles:
        labels = [h.get_label() for h in legend_handles]
        fig.legend(legend_handles, labels,
                   loc="upper center", ncol=len(legend_handles),
                   fontsize=8, bbox_to_anchor=(0.5, 0.975), framealpha=0.9)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    [saved] {save_path}")


# ── figure 3: large overview (all architectures × all modes) ────────────────

def plot_overview_grid(base: Path, save_path: Path):
    """
    Rows = architecture × metric (loss then acc), Columns = mode.
    5 architectures × 2 metrics = 10 rows; 3 mode columns.
    """
    n_arch  = len(ARCHITECTURES)
    n_rows  = n_arch * 2    # loss + acc per arch
    n_cols  = len(MODES)

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 3.2 * n_rows),
                             squeeze=False)
    fig.suptitle(
        "Learning curves — mean ± 1 SD over seeds  (all architectures × modes)",
        fontsize=14, fontweight="bold", y=1.003,
    )

    legend_handles = None

    for a_idx, arch in enumerate(ARCHITECTURES):
        arch_label = ARCH_LABELS.get(arch, arch)
        for m_idx, mode in enumerate(MODES):
            dfs, mean_be = _collect(base, arch, mode)
            ml = MODE_LABELS.get(mode, mode)

            for row_offset, metric in enumerate(["loss", "acc"]):
                row = a_idx * 2 + row_offset
                ax  = axes[row, m_idx]

                col_tr = "train_loss" if metric == "loss" else "train_acc"
                col_vl = "val_loss"   if metric == "loss" else "val_acc"
                ylabel = "CE loss" if metric == "loss" else "Accuracy"
                title  = f"{arch_label}\n{ml} — {metric}"

                if not dfs:
                    ax.set_title(title + "\n(no data)", fontsize=7)
                    continue

                ep, tr_m, tr_s = aggregate(dfs, col_tr)
                _,  vl_m, vl_s = aggregate(dfs, col_vl)

                handles = _draw(
                    ax, ep, tr_m, tr_s, vl_m, vl_s, mean_be,
                    ylabel=ylabel, title=title,
                    show_legend=False,
                    show_xlabel=(row_offset == 1),
                )
                if legend_handles is None:
                    legend_handles = handles

                if metric == "acc":
                    ax.set_ylim(0, 1.05)

    if legend_handles:
        labels = [h.get_label() for h in legend_handles]
        fig.legend(legend_handles, labels,
                   loc="upper center", ncol=len(legend_handles),
                   fontsize=9, bbox_to_anchor=(0.5, 1.005), framealpha=0.9)

    plt.tight_layout(rect=[0, 0, 1, 0.998])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  [saved overview] {save_path}")


# ── CLI ──────────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(
        description="Plot mean ± SD learning curves from history.csv logs.")
    p.add_argument("--out-dir", default="outputs_modes_v3_td",
                   help="Root output dir  (default: outputs_modes_v3_td)")
    p.add_argument("--metric", choices=["loss", "acc", "f1", "all"], default="all",
                   help="Metric(s) for per-pair figures  (default: all)")
    p.add_argument("--arch", nargs="+", default=None,
                   help="Limit to specific architecture(s)  (default: all 5)")
    p.add_argument("--mode", nargs="+", default=None,
                   help="Limit to specific mode(s)  (default: all 3)")
    p.add_argument("--no-per-pair", action="store_true",
                   help="Skip per-pair figures")
    p.add_argument("--no-per-arch", action="store_true",
                   help="Skip per-architecture 3-mode grid figures")
    p.add_argument("--no-overview", action="store_true",
                   help="Skip the large all-pairs overview grid")
    return p.parse_args()


def main():
    args  = parse_args()
    base  = Path(args.out_dir)
    archs = args.arch or ARCHITECTURES
    modes = args.mode or MODES
    curves_root = base / "final" / "curves"

    print(f"Output root  : {base.resolve()}")
    print(f"Architectures: {archs}")
    print(f"Modes        : {modes}")
    print()

    # 1. Per-pair
    if not args.no_per_pair:
        print("=== Per-pair figures ===")
        for arch in archs:
            for mode in modes:
                print(f"  {arch} / {mode}")
                plot_pair(base, arch, mode,
                          save_dir=curves_root / arch / mode,
                          metric=args.metric)
        print()

    # 2. Per-architecture 3-mode grids
    if not args.no_per_arch:
        print("=== Per-architecture 3-mode grids ===")
        for arch in archs:
            out = curves_root / arch / "mean_sd_grid.png"
            print(f"  {arch}")
            plot_per_arch_grid(base, arch, save_path=out)
        print()

    # 3. Overview grid
    if not args.no_overview:
        print("=== Overview grid (all pairs) ===")
        plot_overview_grid(base, save_path=curves_root / "all_pairs_grid.png")

    print("\nDone.")


if __name__ == "__main__":
    main()


In [ ]:
%%project_source resnet_followup.py
"""ResNet50/Full follow-up on the completed outputs_modes_v3_td experiment.

Explicit stages only: gradcam, lock, ablation, analyze, report.
No tuning or baseline training. Ablation adds two conditions x five matched
seeds (10 fresh ImageNet runs, at most 300 epochs). The user runs this CLI.
The original selection.json, registry, final results and statistical families
are never rewritten. This study is exploratory after test-informed selection.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import platform
import shutil
import sys
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from pipeline_modes import _csv, _fingerprint, _holm, _json, _read_csv
from shrine_classifier.analysis import classification_metrics
from shrine_classifier.data import CLASSES, sha256_file

SEEDS = [42, 123, 456, 789, 1024]
PAIR = {"architecture": "resnet50", "mode": "full"}
VARIANTS = {"no_augmentation": {"augmentation": False},
            "no_class_weights": {"use_class_weights": False}}
CONFIG_KEYS = ("learning_rate", "weight_decay", "optimizer", "batch_size", "epochs",
               "patience", "min_delta", "input_size", "augmentation", "use_class_weights", "optimizer_defaults")
METRICS = ("test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1")
FAMILY = "resnet50_full_ablation_variant_minus_baseline_accuracy_macro_f1_4"


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def utc():
    return datetime.now(timezone.utc).isoformat()


def code_hash():
    base = Path(__file__).resolve().parent
    files = [Path(__file__).resolve(), base / "pipeline_modes.py", base / "pipeline.py"]
    for folder in ("shrine_classifier", "shrine_classifier_ext"):
        files.extend(sorted((base / folder).glob("*.py")))
    return _fingerprint([(str(p.relative_to(base)).replace("\\", "/"), sha256_file(p)) for p in files])


def locate(root, saved):
    """Resolve a saved path inside this output root, including a moved checkout."""
    root = root.resolve()
    value = str(saved).replace("\\", "/")
    candidate = Path(value)
    if not candidate.is_absolute():
        candidate = root / candidate
    if candidate.exists() and candidate.resolve().is_relative_to(root):
        return candidate.resolve()
    # Absolute paths in the registry may refer to the original Windows checkout.
    token = "/" + root.name + "/"
    if token in value:
        candidate = root / value.split(token, 1)[1]
        if candidate.exists() and candidate.resolve().is_relative_to(root):
            return candidate.resolve()
    raise FileNotFoundError(f"Artifact is missing or outside {root}: {saved}")


def rel(root, path):
    return str(Path(path).resolve().relative_to(root.resolve())).replace("\\", "/")


def validate_run(run, root, test_rows, expected_config=None):
    """Validate identities, saved files and metrics before baseline reuse/resume."""
    if run.get("status") != "completed" or any(run.get(k) != v for k, v in PAIR.items()):
        raise ValueError("Expected a completed ResNet50/Full run")
    directory = locate(root, run["run_dir"])
    checkpoint = locate(root, run["checkpoint"])
    if checkpoint.parent != directory:
        raise ValueError("Checkpoint/run-directory identity mismatch")
    cfg = read_json(directory / "config.json")
    metadata = read_json(directory / "checkpoint_metadata.json")
    metrics = read_json(directory / "run_metrics.json")
    if metadata.get("class_order") != CLASSES:
        raise ValueError(f"Class order mismatch: {directory}")
    for item in (cfg, metadata, metrics):
        if any(item.get(k) != v for k, v in PAIR.items()) or int(item.get("seed", -1)) != int(run["seed"]):
            raise ValueError(f"Saved run identity mismatch: {directory}")
    if int(metadata["best_epoch"]) != int(run["best_epoch"]):
        raise ValueError("Selected checkpoint epoch mismatch")
    if metadata.get("split_sha256") != sha256_file(root / "splits.csv"):
        raise ValueError(f"Run split hash mismatch: {directory}")
    if expected_config is not None:
        for key, value in expected_config.items():
            if cfg.get(key) != value:
                raise ValueError(f"Config mismatch for {key}: {directory}")
    predictions = _read_csv(directory / "test_predictions.csv")
    if [r["image_id"] for r in predictions] != [r["image_id"] for r in test_rows]:
        raise ValueError(f"Canonical test ID order mismatch: {directory}")
    for pred, truth in zip(predictions, test_rows):
        if pred["true_label"] != truth["label"] or pred["predicted_label"] not in CLASSES:
            raise ValueError("Prediction label mismatch")
        probs = np.array([float(pred[f"prob_{c}"]) for c in CLASSES])
        if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1).any() or not np.isclose(probs.sum(), 1, atol=1e-5):
            raise ValueError("Invalid saved probability vector")
        if CLASSES[int(probs.argmax())] != pred["predicted_label"] or not math.isclose(float(pred["confidence"]), float(probs.max()), abs_tol=1e-6):
            raise ValueError("Saved confidence/prediction mismatch")
    measured = classification_metrics([CLASSES.index(p["true_label"]) for p in predictions],
                                      [CLASSES.index(p["predicted_label"]) for p in predictions])
    for metric in METRICS:
        key = "accuracy" if metric == "test_acc" else metric.removeprefix("test_")
        if not math.isclose(float(run[metric]), measured[key], abs_tol=1e-9):
            raise ValueError(f"Prediction-derived {metric} mismatch: {directory}")
    artifacts = {rel(root, p): sha256_file(p) for p in
                 (checkpoint, directory / "config.json", directory / "checkpoint_metadata.json",
                  directory / "run_metrics.json", directory / "test_metrics.json", directory / "test_predictions.csv")}
    return {**run, "run_dir": str(directory), "checkpoint": str(checkpoint),
            "per_class": measured["per_class"]}, cfg, predictions, artifacts


def context(args):
    root = args.out_dir.resolve()
    from pipeline import _check_existing_split
    _check_existing_split(root, args.data_root.resolve())
    selection = read_json(root / "selection.json")
    protocol = read_json(root / "protocol.json")
    payload = {k: v for k, v in selection.items() if k != "selection_fingerprint"}
    if _fingerprint(payload) != selection["selection_fingerprint"]:
        raise ValueError("Original selection lock fingerprint is corrupt")
    if selection["protocol_fingerprint"] != protocol["protocol_fingerprint"]:
        raise ValueError("Original protocol/selection mismatch")
    if selection["split_file_sha256"] != sha256_file(root / "splits.csv"):
        raise ValueError("Canonical split CSV byte hash mismatch")
    if read_json(root / "split_metadata.json")["split_sha256"] != selection["split_assignment_fingerprint"]:
        raise ValueError("Canonical split assignment fingerprint mismatch")
    split = _read_csv(root / "splits.csv")
    if len(split) != 560 or len({r["image_id"] for r in split}) != 560:
        raise ValueError("Expected 560 unique canonical image IDs")
    # Verify the actual image files; never re-audit into or rewrite old outputs.
    source = read_json(root / "source_manifest.json")
    if _fingerprint([(r["image_id"], r["byte_sha256"]) for r in source["images"]]) != selection["source_fingerprint"]:
        raise ValueError("Source manifest fingerprint differs from the original lock")
    by_id = {r["image_id"]: r for r in split}
    if len(source["images"]) != 560:
        raise ValueError("Incomplete source manifest")
    for image in source["images"]:
        row = by_id[image["image_id"]]
        file = (args.data_root.resolve() / image["path"]).resolve()
        if not file.is_relative_to(args.data_root.resolve()) or sha256_file(file) != image["byte_sha256"]:
            raise ValueError(f"Source image hash/path mismatch: {image['image_id']}")
        if row["label"] != image["label"] or row["path"] != image["path"] or CLASSES[int(row["label_index"])] != row["label"]:
            raise ValueError("Source/split class or path mismatch")
    test_rows = [r for r in split if r["split"] == "test"]
    registry = read_json(root / "run_registry.json")
    winners = read_json(root / "tuning" / "best_params_by_arch_mode.json")["pairs"]["resnet50::full"]
    latest = {}
    for run in registry["final_runs"]:
        if run.get("status") == "completed" and all(run.get(k) == v for k, v in PAIR.items()):
            latest[int(run["seed"])] = run
    if sorted(latest) != sorted(SEEDS) or selection["seeds"] != SEEDS:
        raise ValueError("Expected all five original matched final seeds")
    baseline, configs, artifacts = [], [], {}
    for seed in SEEDS:
        row, cfg, _, files = validate_run(latest[seed], root, test_rows)
        for key in ("learning_rate", "weight_decay", "optimizer", "batch_size"):
            if cfg[key] != winners[key]:
                raise ValueError(f"Baseline is not the independently tuned winner: {key}")
        if cfg.get("selection_fingerprint") != selection["selection_fingerprint"] or not cfg.get("augmentation") or not cfg.get("use_class_weights"):
            raise ValueError("Incompatible baseline selection or factors")
        if (cfg.get("protocol_fingerprint") != protocol["protocol_fingerprint"]
                or cfg.get("split_assignment_fingerprint") != selection["split_assignment_fingerprint"]
                or cfg.get("dataset_fingerprint") != selection["source_fingerprint"]
                or cfg.get("weights") != "ResNet50_Weights.DEFAULT" or not cfg.get("pretrained")):
            raise ValueError("Baseline provenance/pretrained weights mismatch")
        baseline.append(row); configs.append({k: cfg[k] for k in CONFIG_KEYS}); artifacts.update(files)
    if any(c != configs[0] for c in configs[1:]):
        raise ValueError("Baseline configs differ across seeds")
    if configs[0]["epochs"] != 30 or configs[0]["input_size"] != 224 or configs[0]["patience"] != 7 or configs[0]["min_delta"] != 1e-4:
        raise ValueError("Baseline training policy differs from the approved follow-up")
    return {"root": root, "selection": selection, "protocol": protocol, "baseline": baseline,
            "config": configs[0], "test_rows": test_rows, "artifacts": artifacts,
            "code_fingerprint": code_hash()}


def study_dir(args):
    if not args.study_name or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in args.study_name):
        raise ValueError("Use a lowercase study name with letters, digits, '_' or '-'")
    return args.out_dir.resolve() / "ablation" / args.study_name


def lock_study(args, ctx):
    directory = study_dir(args)
    payload = {"version": 1, **PAIR, "seeds": SEEDS, "representative_seed": 42,
        "baseline_config": ctx["config"], "variants": VARIANTS,
        "selection_context": "ResNet50/Full selected after viewing test results; exploratory follow-up on an already examined split.",
        "original_statistical_reference": ctx["selection"]["reference_pair"],
        "original_selection_fingerprint": ctx["selection"]["selection_fingerprint"],
        "original_protocol_fingerprint": ctx["selection"]["protocol_fingerprint"],
        "split_sha256": sha256_file(ctx["root"] / "splits.csv"),
        "baseline_artifact_sha256": ctx["artifacts"], "code_fingerprint": ctx["code_fingerprint"],
        "baseline_environment_sha256": sha256_file(ctx["root"] / "environment.json"),
        "baseline_reused_runs": 5, "new_runs": 10, "max_new_epochs": 10 * int(ctx["config"]["epochs"]),
        "statistics": {"family_id": FAMILY, "family_size": 4, "metrics": ["test_acc", "test_macro_f1"],
                       "direction": "variant minus baseline", "alternative": "two-sided", "alpha": .05},
        "baseline_sources": [{"seed": r["seed"], "run_dir": rel(ctx["root"], r["run_dir"]),
                              "checkpoint": rel(ctx["root"], r["checkpoint"])} for r in ctx["baseline"]]}
    fingerprint = _fingerprint(payload)
    target = directory / "ablation_lock.json"
    if target.exists():
        previous = read_json(target)
        if previous.get("study_fingerprint") != fingerprint or _fingerprint({k: v for k, v in previous.items() if k not in ("study_fingerprint", "locked_utc")}) != fingerprint:
            raise ValueError("Study lock differs or is corrupt; use a new --study-name")
        return previous
    result = {**payload, "study_fingerprint": fingerprint, "locked_utc": utc()}
    if not args.dry_run:
        _json(target, result)
    return result


def require_lock(args, ctx):
    if not (study_dir(args) / "ablation_lock.json").is_file():
        raise FileNotFoundError("Run --stage lock before training/analyzing this ablation")
    return lock_study(args, ctx)  # Validate all hashes/configs against the immutable lock.


def ablation(args, ctx):
    lock = require_lock(args, ctx)
    directory = study_dir(args)
    registry_path = directory / "run_registry.json"
    registry = read_json(registry_path) if registry_path.exists() else {"study_fingerprint": lock["study_fingerprint"], "attempts": []}
    if registry["study_fingerprint"] != lock["study_fingerprint"]:
        raise ValueError("Ablation registry belongs to a different lock")
    if args.dry_run:
        return {"new_training_started": False, "new_runs": 10, "baseline_reused": 5,
                "max_new_epochs": lock["max_new_epochs"]}
    import torch
    import torchvision
    from shrine_classifier.training import fit_run
    for variable, suffix in (("WANDB_DIR", "runs"), ("WANDB_CACHE_DIR", "cache"), ("WANDB_DATA_DIR", "data")):
        tracking_dir = directory / "wandb" / suffix
        tracking_dir.mkdir(parents=True, exist_ok=True)
        os.environ[variable] = str(tracking_dir)
    os.environ.setdefault("WANDB_INIT_TIMEOUT", "45")
    split_copy = directory / "splits.csv"
    if not split_copy.exists():
        shutil.copy2(ctx["root"] / "splits.csv", split_copy)
    if sha256_file(split_copy) != lock["split_sha256"]:
        raise ValueError("Ablation split copy hash mismatch")
    _json(directory / "environment.json", {"created_utc": utc(), "python": sys.version,
        "platform": platform.platform(), "torch": torch.__version__, "torchvision": torchvision.__version__,
        "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "workers": args.num_workers, "amp": False, "disk_free_bytes": shutil.disk_usage(directory).free,
        "deterministic_algorithms": True, "deterministic_warn_only": True, "cudnn_benchmark": False,
        "requested_tracking_mode": args.wandb_mode, "code_fingerprint": ctx["code_fingerprint"],
        "baseline_environment": read_json(ctx["root"] / "environment.json")})
    print(f"Reusing 5 baseline runs; at most 10 new runs / {lock['max_new_epochs']} epochs; free disk {shutil.disk_usage(directory).free / 2**30:.1f} GiB", flush=True)
    for row in registry["attempts"]:
        if row.get("status") == "running":
            row.update(status="interrupted", error="Previous process ended without completion; restarting as a new attempt")
    for condition, overrides in VARIANTS.items():
        for seed in SEEDS:
            ident = f"{condition}::seed{seed}"
            cfg = {**ctx["config"], **overrides, "selection_fingerprint": ctx["selection"]["selection_fingerprint"],
                "protocol_fingerprint": ctx["protocol"]["protocol_fingerprint"],
                "dataset_fingerprint": ctx["protocol"]["source_fingerprint"],
                "split_assignment_fingerprint": ctx["selection"]["split_assignment_fingerprint"],
                "code_fingerprint": ctx["code_fingerprint"], "amp": False,
                "ablation_condition": condition, "ablation_study_fingerprint": lock["study_fingerprint"]}
            cfg_hash = _fingerprint({"config": cfg, "seed": seed})
            done = next((r for r in reversed(registry["attempts"]) if r["identity"] == ident and r.get("status") == "completed"), None)
            if done:
                if not args.resume:
                    raise ValueError(f"{ident} is complete; use --resume to reuse it")
                if done.get("config_fingerprint") != cfg_hash:
                    raise ValueError(f"Incompatible completed ablation config: {ident}")
                validate_run(done, directory, ctx["test_rows"], cfg)
                for file, digest in done["artifact_sha256"].items():
                    if sha256_file(directory / file) != digest:
                        raise ValueError(f"Completed artifact changed: {file}")
                print(f"[resume] {ident}", flush=True)
                continue
            attempt = 1 + sum(r["identity"] == ident for r in registry["attempts"])
            row = {**PAIR, "identity": ident, "seed": seed, "condition": condition, "attempt": attempt,
                   "config": cfg, "config_fingerprint": cfg_hash, "status": "running", "started_utc": utc()}
            registry["attempts"].append(row)
            _json(registry_path, registry)
            try:
                result = fit_run(root=args.data_root.resolve(), split_csv=ctx["root"] / "splits.csv", out=directory,
                    architecture="resnet50", mode="full", seed=seed, config=cfg,
                    stage=f"ablation_{condition}", run_identity=f"{ident.replace('::', '_')}_attempt{attempt:02d}",
                    wandb_project=args.wandb_project, wandb_entity=args.wandb_entity,
                    wandb_mode=args.wandb_mode, device=args.device, workers=args.num_workers, evaluate_test=True)
                row.update(result)
                _, _, _, artifacts = validate_run(row, directory, ctx["test_rows"], cfg)
                row["artifact_sha256"] = artifacts
            except KeyboardInterrupt:
                row.update(status="interrupted", error="User/process interruption; no optimizer/RNG resume state")
                row["finished_utc"] = utc(); _json(registry_path, registry)
                raise
            except Exception as exc:
                row.update(status="failed", error=f"{type(exc).__name__}: {exc}", traceback=traceback.format_exc(limit=8))
            row["finished_utc"] = utc()
            _json(registry_path, registry)
            _csv(directory / "run_metrics.csv", registry["attempts"])
            print(f"[ablation] {ident} attempt {attempt}: {row['status']}", flush=True)
    complete = {r["identity"] for r in registry["attempts"] if r.get("status") == "completed"}
    _json(directory / "execution_status.json", {"completed": len(complete), "target": 10, "baseline_reused": 5,
        "next_stage": "analyze" if len(complete) == 10 else "ablation --resume"})
    if len(complete) != 10:
        raise RuntimeError(f"Only {len(complete)}/10 ablation runs completed; inspect registry then use --resume")
    return {"completed": 10, "baseline_reused": 5, "next_stage": "analyze"}


def analyze(args, ctx):
    lock = require_lock(args, ctx)
    directory = study_dir(args)
    registry = read_json(directory / "run_registry.json")
    if registry.get("study_fingerprint") != lock["study_fingerprint"]:
        raise ValueError("Ablation registry/lock mismatch")
    groups = {"baseline": ctx["baseline"]}
    for condition, overrides in VARIANTS.items():
        rows = {}
        for row in registry["attempts"]:
            if row.get("condition") == condition and row.get("status") == "completed":
                rows[int(row["seed"])] = row
        if sorted(rows) != sorted(SEEDS):
            raise ValueError(f"Incomplete ablation: {condition} requires all five seeds")
        group = []
        for seed in SEEDS:
            row = rows[seed]
            if row["config"].get("ablation_study_fingerprint") != lock["study_fingerprint"]:
                raise ValueError("Run study fingerprint mismatch")
            validated, _, _, artifacts = validate_run(row, directory, ctx["test_rows"],
                                                     {**ctx["config"], **overrides})
            if artifacts != row.get("artifact_sha256"):
                raise ValueError("Ablation artifact hashes changed")
            group.append(validated)
        groups[condition] = group
    summary, deltas, tests, per_class = [], [], [], []
    baseline = {int(r["seed"]): r for r in groups["baseline"]}
    for condition, rows in groups.items():
        entry = {**PAIR, "condition": condition, "completed_seeds": 5}
        for metric in METRICS:
            vals = np.array([float(r[metric]) for r in rows])
            entry[metric+"_mean"], entry[metric+"_sample_sd"] = float(vals.mean()), float(vals.std(ddof=1))
        summary.append(entry)
        for cls in CLASSES:
            item = {"condition": condition, "class": cls, "support": rows[0]["per_class"][cls]["support"]}
            for metric in ("precision", "recall", "f1"):
                vals = np.array([r["per_class"][cls][metric] for r in rows])
                item[metric+"_mean"], item[metric+"_sample_sd"] = float(vals.mean()), float(vals.std(ddof=1))
            per_class.append(item)
        if condition == "baseline":
            continue
        for row in rows:
            deltas.append({"condition": condition, "seed": row["seed"],
                **{metric+"_delta_pp": 100 * (float(row[metric]) - float(baseline[int(row["seed"])][metric])) for metric in METRICS}})
        from scipy.stats import ttest_rel
        for metric in ("test_acc", "test_macro_f1"):
            matched = {int(r["seed"]): r for r in rows}
            a = np.array([float(matched[s][metric]) for s in SEEDS])
            b = np.array([float(baseline[s][metric]) for s in SEEDS])
            diff = a - b
            item = {"family_id": FAMILY, "condition": condition, "direction": "variant minus baseline", "metric": metric,
                "seed_ids": ",".join(map(str, SEEDS)), "paired_n": 5, "mean_difference": float(diff.mean()),
                "mean_difference_pp": 100*float(diff.mean()), "t_statistic": None, "p_value": None,
                "status": "undefined_zero_variance"}
            if not np.isfinite(diff).all():
                item["status"] = "undefined_nonfinite"
            elif float(diff.std(ddof=1)) != 0:
                result = ttest_rel(a, b, alternative="two-sided")
                item.update(t_statistic=float(result.statistic), p_value=float(result.pvalue), status="ok")
            tests.append(item)
    _holm(tests, 4)
    _csv(directory / "condition_summary.csv", summary)
    _csv(directory / "per_class_summary.csv", per_class)
    _csv(directory / "matched_deltas.csv", deltas)
    _csv(directory / "statistical_tests.csv", tests)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    labels = ["Baseline", "No augmentation", "No class weights"]
    for ax, metric, title in zip(axes, ("test_acc", "test_macro_f1"), ("Accuracy (%)", "Macro F1 (%)")):
        for x, (condition, rows) in enumerate(groups.items()):
            vals = np.array([float(r[metric])*100 for r in rows])
            ax.scatter(x + np.linspace(-.06, .06, 5), vals, color="#3c9659", marker="D", alpha=.4)
            ax.errorbar(x, vals.mean(), yerr=vals.std(ddof=1), fmt="D", color="#26703d", capsize=7)
        ax.set_xticks(range(3), labels); ax.set_ylabel(title); ax.set_ylim(0, 101); ax.grid(axis="y", alpha=.2)
    fig.suptitle("ResNet50 / Full ablation: mean ± sample SD, 5 matched seeds", y=.99)
    fig.tight_layout(rect=(0, 0, 1, .94)); fig.savefig(directory / "ablation_summary.png", dpi=180); plt.close(fig)
    result = {"study_fingerprint": lock["study_fingerprint"], "completed_new_runs": 10, "baseline_reused": 5,
        "exploratory": True, "figures": ["ablation_summary.png"], "registry_sha256": sha256_file(directory / "run_registry.json"),
        "artifact_sha256": {p.name: sha256_file(p) for p in [directory / n for n in
            ("condition_summary.csv", "per_class_summary.csv", "matched_deltas.csv", "statistical_tests.csv", "ablation_summary.png")]},
        "note": "Seed variation on a previously examined split, not independent external confirmation. Weighted/unweighted loss values are not directly comparable."}
    _json(directory / "analysis.json", result)
    return result


def make_gradcam(args, ctx):
    if args.dry_run:
        return {"model_inference_started": False, **PAIR, "seed": 42, "checkpoint": ctx["baseline"][0]["checkpoint"]}
    import torch
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image
    from shrine_classifier.analysis import gradcam
    from shrine_classifier.data import open_rgb, letterbox, IMAGENET_MEAN, IMAGENET_STD
    from shrine_classifier.models import build_model
    from shrine_classifier.training import seed_everything
    from torchvision import transforms
    seed_everything(42)
    run = next(r for r in ctx["baseline"] if int(r["seed"]) == 42)
    checkpoint_digest = sha256_file(Path(run["checkpoint"]))
    predictions = _read_csv(Path(run["run_dir"]) / "test_predictions.csv")
    chosen = []
    for cls in CLASSES:
        subset = [r for r in predictions if r["true_label"] == cls]
        correct = sorted((r for r in subset if r["predicted_label"] == cls), key=lambda r: (-float(r["confidence"]), r["image_id"]))
        wrong = sorted((r for r in subset if r["predicted_label"] != cls), key=lambda r: (-float(r["confidence"]), r["image_id"]))
        if correct:
            chosen.append((correct[0], "highest_confidence_correct"))
        if wrong:
            chosen.append((wrong[0], "highest_confidence_misclassified"))
        elif len(correct) > 1:
            chosen.append((min(correct[1:], key=lambda r: (float(r["confidence"]), r["image_id"])), "lowest_confidence_correct_no_errors_in_class"))
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)
    model, _ = build_model("resnet50", pretrained=False)
    state = torch.load(run["checkpoint"], map_location="cpu", weights_only=False)
    if state.get("architecture") != "resnet50" or state.get("mode") != "full" or int(state.get("seed", -1)) != 42 or int(state.get("epoch", -1)) != int(run["best_epoch"]):
        raise ValueError("Checkpoint identity does not match the saved final run")
    model.load_state_dict(state["model_state"], strict=True); model.to(device).eval()
    tensor_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    directory = ctx["root"] / "gradcam" / "resnet50_full" / "seed42" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    directory.mkdir(parents=True, exist_ok=False)
    fig, axes = plt.subplots(len(chosen), 3, figsize=(14, 3.8*len(chosen)), squeeze=False)
    records = []
    lookup = {r["image_id"]: r for r in ctx["test_rows"]}
    try:
        for i, (pred, reason) in enumerate(chosen):
            row = lookup[pred["image_id"]]
            image = open_rgb(args.data_root / row["path"])
            canvas, bbox = letterbox(image, int(ctx["config"]["input_size"]))
            tensor = tensor_transform(canvas).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = model(tensor).softmax(1)[0].cpu().numpy()
            saved_probs = np.array([float(pred[f"prob_{c}"]) for c in CLASSES])
            abs_err = float(np.abs(probs - saved_probs).max())
            rel_err = float((np.abs(probs - saved_probs) / (np.abs(saved_probs) + 1e-8)).max())
            if probs.argmax() != saved_probs.argmax():
                raise ValueError(
                    f"Checkpoint/preprocessing prediction CHANGED class for {pred['image_id']}: "
                    f"saved={CLASSES[saved_probs.argmax()]}, recomputed={CLASSES[probs.argmax()]}, "
                    f"max_abs_err={abs_err:.2e}, max_rel_err={rel_err:.2e}"
                )
            if abs_err > 1e-3:
                print(f"[gradcam] WARNING: large probability discrepancy for {pred['image_id']}: "
                      f"max_abs_err={abs_err:.2e}, max_rel_err={rel_err:.2e} "
                      f"(batched vs single-image float32 rounding; class unchanged)")
            elif abs_err > 5e-5:
                print(f"[gradcam] NOTE: minor float32 rounding for {pred['image_id']}: "
                      f"max_abs_err={abs_err:.2e} (batched vs single-image inference; class unchanged)")
            target = CLASSES.index(pred["predicted_label"])
            with torch.enable_grad():
                cam = gradcam(model, "resnet50", tensor, target)[0].detach().cpu().numpy()
            x1, y1, x2, y2 = bbox
            crop = cam[y1:y2, x1:x2]
            if not np.isfinite(crop).all() or crop.size == 0 or not np.any(crop > 0):
                raise ValueError(f"CAM is empty/non-finite/all-zero for {pred['image_id']}; no valid figure published")
            heat = np.asarray(Image.fromarray(crop.astype(np.float32)).resize(image.size, Image.Resampling.BILINEAR))
            colored = np.uint8(plt.get_cmap("jet")(np.clip(heat, 0, 1))[..., :3]*255)
            overlay = Image.blend(image, Image.fromarray(colored), .42)
            sample = directory / pred["image_id"].replace("/", "__"); sample.mkdir()
            image.save(sample / "original.png"); Image.fromarray(colored).save(sample / "heatmap.png"); overlay.save(sample / "overlay.png")
            np.save(sample / "cam_original_alignment.npy", heat)
            for ax, picture, title in zip(axes[i], (image, Image.fromarray(colored), overlay), ("Original", "Grad-CAM", "Overlay")):
                ax.imshow(picture); ax.axis("off"); ax.set_title(title)
            axes[i, 0].set_title(f"{row['image_id']}\nTrue: {row['label']}", fontsize=10)
            axes[i, 2].set_title(f"Predicted / explained: {pred['predicted_label']}\nConfidence: {float(pred['confidence'])*100:.2f} · {reason}", fontsize=9)
            records.append({**pred, "selection_reason": reason, "explained_target": pred["predicted_label"],
                "content_bbox": list(bbox), "checkpoint": run["checkpoint"], "checkpoint_sha256": checkpoint_digest,
                "sample_directory": rel(ctx["root"], sample)})
        fig.suptitle("ResNet50 / Full · final seed 42 · Grad-CAM at layer4[-1]", fontsize=16)
        fig.tight_layout(rect=(0, 0, 1, .985)); fig.savefig(directory / "gradcam_grid.png", dpi=160)
        _csv(directory / "selected_cases.csv", records)
        result = {**PAIR, "seed": 42, "examples": len(records), "checkpoint_sha256": checkpoint_digest,
            "device": str(device), "torch": torch.__version__, "created_utc": utc(),
            "code_fingerprint": ctx["code_fingerprint"], "selection_context": "Post-hoc report-selected model; Grad-CAM is diagnostic, not causal evidence.",
            "directory": rel(ctx["root"], directory), "artifact_sha256": {p.name: sha256_file(p) for p in (directory / "gradcam_grid.png", directory / "selected_cases.csv")}}
        _json(directory / "manifest.json", result)
        _json(directory.parent / "latest.json", result)
        return result
    except Exception as exc:
        _json(directory / "failure.json", {"error": f"{type(exc).__name__}: {exc}", "completed_cases": len(records)})
        raise
    finally:
        plt.close(fig)


def export_report(args, ctx):
    """Insert only measured, verified follow-up figures/tables into local README."""
    import re
    repo = Path(__file__).resolve().parent
    readme = repo / "README.md"
    text = readme.read_text(encoding="utf-8")
    assets = repo / "report" / "report_assets" / "latest"
    assets.mkdir(parents=True, exist_ok=True)
    updated = []
    source_map = {}

    def replace_block(name, body):
        nonlocal text
        pattern = rf"(<!-- resnet-followup:{name}:start -->).*?(<!-- resnet-followup:{name}:end -->)"
        if not re.search(pattern, text, flags=re.S):
            raise ValueError(f"README follow-up markers missing: {name}")
        text = re.sub(pattern, lambda m: m[1] + "\n\n" + body + "\n\n" + m[2], text, flags=re.S)

    latest = ctx["root"] / "gradcam" / "resnet50_full" / "seed42" / "latest.json"
    if latest.is_file():
        manifest = read_json(latest)
        directory = locate(ctx["root"], manifest["directory"])
        if manifest["checkpoint_sha256"] != sha256_file(Path(next(r for r in ctx["baseline"] if int(r["seed"]) == 42)["checkpoint"])):
            raise ValueError("CAM checkpoint changed")
        for name, digest in manifest["artifact_sha256"].items():
            if sha256_file(directory / name) != digest:
                raise ValueError(f"CAM artifact hash mismatch: {name}")
        shutil.copy2(directory / "gradcam_grid.png", assets / "gradcam_resnet50_full_seed42.png")
        shutil.copy2(directory / "selected_cases.csv", repo / "report" / "resnet_gradcam_cases.csv")
        replace_block("gradcam", "![Grad-CAM ของ ResNet50 Full seed 42](report/report_assets/latest/gradcam_resnet50_full_seed42.png)\n\n"
            f"Grad-CAM ของ ResNet50/Full, final seed 42 จำนวน {manifest['examples']} ภาพ อธิบายคลาสที่โมเดลทำนาย เลือกกรณีถูก/ผิดแยกตาม true class; ถ้าไม่มีภาพผิดใช้กรณีถูกที่ confidence ต่ำ แผนที่จัดแนวกลับสู่ภาพต้นฉบับหลังตัด canvas padding ออก เป็นหลักฐานประกอบการวิเคราะห์ ไม่ได้พิสูจน์เหตุผลเชิงสาเหตุของโมเดล [ข้อมูลรายภาพ](report/resnet_gradcam_cases.csv)\n\n"
            "ภาพ MobileNet ด้านล่างเป็นหลักฐานประกอบของ reference เดิม แยกจากโมเดล ResNet ที่เลือกในรายงาน")
        source_map["gradcam"] = manifest; updated.append("gradcam")
    directory = study_dir(args)
    if (directory / "analysis.json").is_file():
        lock = require_lock(args, ctx)
        analysis = read_json(directory / "analysis.json")
        if analysis["study_fingerprint"] != lock["study_fingerprint"] or analysis["registry_sha256"] != sha256_file(directory / "run_registry.json"):
            raise ValueError("Analysis/lock mismatch")
        for name, digest in analysis["artifact_sha256"].items():
            if sha256_file(directory / name) != digest:
                raise ValueError(f"Analysis artifact hash mismatch: {name}")
        for name in ("condition_summary.csv", "matched_deltas.csv", "statistical_tests.csv", "per_class_summary.csv"):
            shutil.copy2(directory / name, repo / "report" / ("resnet_ablation_" + name))
        shutil.copy2(directory / "ablation_summary.png", assets / "resnet_ablation_summary.png")
        rows = _read_csv(directory / "condition_summary.csv")
        lines = ["| Condition | Accuracy (%) | Macro precision (%) | Macro recall (%) | Macro F1 (%) |", "| --- | ---: | ---: | ---: | ---: |"]
        for row in rows:
            cells = [row["condition"]] + [f"{float(row[m+'_mean'])*100:.2f} ± {float(row[m+'_sample_sd'])*100:.2f}" for m in METRICS]
            lines.append("| " + " | ".join(cells) + " |")
        replace_block("ablation", "ใช้ ResNet50/Full สูตรเดิมและ split เดิม เปลี่ยนทีละปัจจัย: ปิด augmentation หรือปิด class weights ฝึกใหม่จาก ImageNet variant ละ 5 matched seeds รวม 10 runs และใช้ baseline final เดิม 5 seeds โดยไม่ tune ใหม่ ค่าตารางเป็น mean (%) ± sample SD (จุดเปอร์เซ็นต์)\n\n"
            + "\n".join(lines) + "\n\n![ResNet50 Full ablation](report/report_assets/latest/resnet_ablation_summary.png)\n\n"
            "[ผลต่างจับคู่ราย seed](report/resnet_ablation_matched_deltas.csv) · [สถิติครบ 4 comparisons ใน Holm family แยก](report/resnet_ablation_statistical_tests.csv) · [ผลรายคลาส](report/resnet_ablation_per_class_summary.csv)\n\n"
            "ผลนี้เป็น exploratory follow-up หลังเลือก ResNet จาก test บน split ที่เคยตรวจดูแล้ว สถิติอธิบายความแปรปรวนข้าม seeds ไม่ใช่การยืนยันกับข้อมูลภายนอก และ loss ของเงื่อนไขใช้/ไม่ใช้ class weights ไม่อยู่บนสเกลวัตถุประสงค์เดียวกัน")
        source_map["ablation"] = analysis; updated.append("ablation")
    if not updated:
        raise FileNotFoundError("No completed ResNet follow-up artifacts to insert; run gradcam or ablation + analyze first")
    readme.write_text(text, encoding="utf-8")
    (repo / "report" / "report_source.md").write_text("<!-- Generated mirror of ../README.md; edit the canonical README narrative. -->\n" + text.replace("](report/", "]("), encoding="utf-8")
    _json(repo / "report" / "resnet_followup_sources.json", source_map)
    selected_source = repo / "report" / "selected_model_report_sources.json"
    if selected_source.exists():
        original = read_json(selected_source)
        if "gradcam" in source_map:
            original["gradcam_resnet50_full"] = source_map["gradcam"]
        if "ablation" in source_map:
            original["ablation_resnet50_full"] = source_map["ablation"]
        _json(selected_source, original)
    return {"local_readme_updated": updated, "github_push_performed": False}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stage", required=True, choices=("gradcam", "lock", "ablation", "analyze", "report"))
    parser.add_argument("--out-dir", type=Path, default=Path("outputs_modes_v3_td"))
    parser.add_argument("--data-root", type=Path, default=Path("dataset_cleaned"))
    parser.add_argument("--study-name", default="resnet50_full_v1")
    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda"))
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--wandb-project", default="thai-shrine-classifier-td")
    parser.add_argument("--wandb-entity", default=None)
    parser.add_argument("--wandb-mode", default="online", choices=("online", "offline", "disabled"))
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--dry-run", action="store_true", help="Read/validate only; never train or infer")
    args = parser.parse_args()
    ctx = context(args)
    if args.dry_run and args.stage in ("analyze", "report"):
        result = {"dry_run": True, "stage": args.stage, "writes_performed": False}
    else:
        result = {"gradcam": make_gradcam, "lock": lock_study, "ablation": ablation,
                  "analyze": analyze, "report": export_report}[args.stage](args, ctx)
    print(json.dumps(result, indent=2, ensure_ascii=False, allow_nan=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 5. Test source — รวมไว้แต่ไม่รันอัตโนมัติ

รวม test code เดิมเพื่อให้ notebook เก็บโครงการครบ ไม่มีเซลล์ใดเรียก tests หรือ smoke โดยอัตโนมัติ การจัดแพ็กครั้งนี้ตรวจแบบ static เท่านั้น ไม่ถือเป็นการทดสอบ Python ผ่านแล้ว


In [ ]:
%%project_source tests/test_analysis.py
import unittest

from shrine_classifier.analysis import classification_metrics, paired_holm_tests
from shrine_classifier.data import CLASSES


class AnalysisContractTests(unittest.TestCase):
    def test_four_class_macro_metrics_keep_fixed_labels(self):
        metrics = classification_metrics([0, 1, 2, 3], [0, 1, 2, 3])
        self.assertEqual(metrics["accuracy"], 1.0)
        self.assertEqual(metrics["macro_f1"], 1.0)
        self.assertEqual(list(metrics["per_class"]), CLASSES)
        self.assertEqual(metrics["confusion_matrix"], [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    def test_paired_results_join_by_seed_and_holm_adjusts_family(self):
        rows = []
        for seed, val in ((123, .7), (42, .8), (456, .9), (789, .75)):
            rows.append({"architecture": "ref", "seed": seed, "status": "completed", "test_acc": val, "test_macro_f1": val})
        for seed, val in ((456, .5), (789, .6), (42, .6), (123, .55)):
            rows.append({"architecture": "other", "seed": seed, "status": "completed", "test_acc": val, "test_macro_f1": val})
        result = paired_holm_tests(rows, "ref", architecture_order=["ref", "other"])
        self.assertEqual(len(result), 2)
        self.assertTrue(all(r["paired_n"] == 4 and r["status"] == "ok" for r in result))
        self.assertAlmostEqual(result[0]["mean_difference"], .225)
        self.assertGreaterEqual(result[0]["p_holm"], result[0]["p_value"])

    def test_missing_or_zero_variance_pairs_are_not_claimed_significant(self):
        rows = []
        for seed in (42, 123, 456):
            rows.extend([{ "architecture": "ref", "seed": seed, "status": "completed", "test_acc": .5, "test_macro_f1": .5},
                         { "architecture": "other", "seed": seed, "status": "completed", "test_acc": .5, "test_macro_f1": .5}])
        result = paired_holm_tests(rows, "ref", architecture_order=["ref", "other"])
        self.assertTrue(all(r["p_value"] is None and r["significant_holm"] is None for r in result))
        rows = [r for r in rows if not (r["architecture"] == "other" and r["seed"] == 123)]
        result = paired_holm_tests(rows, "ref", architecture_order=["ref", "other"])
        self.assertTrue(all(r["status"] == "undefined_zero_variance_or_nonfinite" for r in result))


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%project_source tests/test_data.py
import csv
import tempfile
import unittest
from pathlib import Path

from PIL import Image

from shrine_classifier.data import CLASSES, NEUTRAL, groups_from_audit, letterbox, open_rgb


class DataContractTests(unittest.TestCase):
    def test_letterbox_preserves_aspect_and_uses_neutral_padding(self):
        image = Image.new("RGB", (640, 320), (180, 20, 30))
        canvas, bbox = letterbox(image, 224)
        self.assertEqual(canvas.size, (224, 224))
        self.assertEqual(bbox, (0, 56, 224, 168))
        self.assertEqual(canvas.getpixel((0, 0)), NEUTRAL)
        self.assertEqual(canvas.getpixel((100, 100)), (180, 20, 30))

    def test_transparency_is_composited_on_neutral_color(self):
        with tempfile.TemporaryDirectory() as temp:
            path = Path(temp) / "alpha.png"
            image = Image.new("RGBA", (5, 5), (255, 0, 0, 0))
            image.putpixel((2, 2), (200, 10, 10, 255))
            image.save(path)
            converted = open_rgb(path)
            self.assertEqual(converted.mode, "RGB")
            self.assertEqual(converted.getpixel((0, 0)), NEUTRAL)
            self.assertEqual(converted.getpixel((2, 2)), (200, 10, 10))

    def test_confirmed_duplicate_groups_are_transitive_and_label_safe(self):
        rows = [
            {"image_id": "san_phraphum/a", "label": "san_phraphum", "pixel_sha256": "same", "source_url": "u1"},
            {"image_id": "san_phraphum/b", "label": "san_phraphum", "pixel_sha256": "b", "source_url": "u1"},
            {"image_id": "san_phraphum/c", "label": "san_phraphum", "pixel_sha256": "c", "source_url": "u2"},
        ]
        with tempfile.TemporaryDirectory() as temp:
            review = Path(temp) / "review.csv"
            with review.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=["id_a", "id_b", "review_status"])
                writer.writeheader(); writer.writerow({"id_a": rows[1]["image_id"], "id_b": rows[2]["image_id"], "review_status": "confirmed_same_source"})
            groups = groups_from_audit(rows, review)
            self.assertEqual(len(set(groups.values())), 1)

    def test_class_order_is_explicit(self):
        self.assertEqual(CLASSES, ["san_phraphrom", "san_phraphum", "san_tayay", "tee_ju_eia"])


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%project_source tests/test_extensions.py
"""Tests for extension pipeline components.

Tests: confidence analysis, diagnostics history parsing, fine-tuning mode invariants,
ablation config isolation, seed join/statistics, Holm family size,
and protocol protection guards.
"""

from __future__ import annotations

import csv
import json
import math
import unittest
from pathlib import Path
import tempfile


# ---------------------------------------------------------------------------
# Test helpers
# ---------------------------------------------------------------------------

def _make_predictions_csv(path: Path, records: list[dict]) -> None:
    if not records:
        return
    fields = list(records[0].keys())
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def _pred_row(image_id, true_label, predicted_label,
              probs=(0.7, 0.1, 0.1, 0.1), correct=True):
    """Helper: build a synthetic prediction row."""
    from shrine_classifier.data import CLASSES
    pred_idx = CLASSES.index(predicted_label)
    confidence = probs[pred_idx]
    row = {
        "image_id": image_id,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "confidence": str(confidence),
    }
    for i, cls in enumerate(CLASSES):
        row[f"prob_{cls}"] = str(probs[i])
    return row


# ---------------------------------------------------------------------------
# Confidence analysis tests
# ---------------------------------------------------------------------------

class TestConfidenceValidation(unittest.TestCase):

    def test_valid_predictions_pass(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            _pred_row("san_phraphrom/img_0001", "san_phraphrom", "san_phraphrom",
                      probs=(0.80, 0.10, 0.05, 0.05)),
        ]
        result = _validate_predictions(rows, "test_source")
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["correct"], 1)

    def test_wrong_prediction_marked_incorrect(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            _pred_row("san_phraphum/img_0001", "san_phraphum", "san_phraphrom",
                      probs=(0.7, 0.1, 0.1, 0.1)),
        ]
        result = _validate_predictions(rows, "test")
        self.assertEqual(result[0]["correct"], 0)

    def test_invalid_probability_sum_fails(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            {
                "image_id": "san_phraphrom/img_0001",
                "true_label": "san_phraphrom",
                "predicted_label": "san_phraphrom",
                "confidence": "0.7",
                "prob_san_phraphrom": "0.7",
                "prob_san_phraphum": "0.2",
                "prob_san_tayay": "0.2",  # sum = 1.2
                "prob_tee_ju_eia": "0.1",
            }
        ]
        with self.assertRaises(ValueError):
            _validate_predictions(rows, "test")

    def test_out_of_range_probability_fails(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            {
                "image_id": "san_phraphrom/img_0001",
                "true_label": "san_phraphrom",
                "predicted_label": "san_phraphrom",
                "confidence": "1.1",
                "prob_san_phraphrom": "1.1",
                "prob_san_phraphum": "-0.1",
                "prob_san_tayay": "0.0",
                "prob_tee_ju_eia": "0.0",
            }
        ]
        with self.assertRaises(ValueError):
            _validate_predictions(rows, "test")

    def test_argmax_mismatch_fails(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            {
                "image_id": "san_phraphrom/img_0001",
                "true_label": "san_phraphrom",
                "predicted_label": "san_phraphum",  # wrong: probs say phraphrom
                "confidence": "0.7",
                "prob_san_phraphrom": "0.7",
                "prob_san_phraphum": "0.1",
                "prob_san_tayay": "0.1",
                "prob_tee_ju_eia": "0.1",
            }
        ]
        with self.assertRaises(ValueError):
            _validate_predictions(rows, "test")

    def test_duplicate_image_ids_fails(self):
        from shrine_classifier_ext.confidence_analysis import _validate_predictions
        rows = [
            _pred_row("san_phraphrom/img_0001", "san_phraphrom", "san_phraphrom",
                      probs=(0.8, 0.1, 0.05, 0.05)),
            _pred_row("san_phraphrom/img_0001", "san_phraphrom", "san_phraphrom",  # duplicate
                      probs=(0.8, 0.1, 0.05, 0.05)),
        ]
        with self.assertRaises(ValueError):
            _validate_predictions(rows, "test")

    def test_bin_metrics_empty_bin_has_none_accuracy(self):
        from shrine_classifier_ext.confidence_analysis import _bin_metrics, BIN_EDGES
        # Records all with confidence 0.95 → all in last bin only
        records = [
            {"confidence": 0.95, "correct": 1},
            {"confidence": 0.97, "correct": 0},
        ]
        bins = _bin_metrics(records)
        # All but last bin should have count 0 and accuracy None
        empty_bins = [b for b in bins if b["count"] == 0]
        for b in empty_bins:
            self.assertIsNone(b["accuracy"],
                              f"Empty bin at [{b['bin_lower']}, {b['bin_upper']}) should have accuracy=None")

    def test_ece_hand_calculation(self):
        from shrine_classifier_ext.confidence_analysis import _bin_metrics, _ece
        # 2 samples in bin [0.7, 0.8): confidence 0.75 mean, accuracy 1.0 → gap 0.25
        # 1 sample in bin [0.5, 0.6): confidence 0.55 mean, accuracy 0.0 → gap 0.55
        records = [
            {"confidence": 0.75, "correct": 1},
            {"confidence": 0.76, "correct": 1},
            {"confidence": 0.55, "correct": 0},
        ]
        bins = _bin_metrics(records)
        ece = _ece(bins, 3)
        # Expected: (2/3)*|1.0 - 0.755| + (1/3)*|0.0 - 0.55|
        mean_07 = (0.75 + 0.76) / 2
        expected = (2 / 3) * abs(1.0 - mean_07) + (1 / 3) * abs(0.0 - 0.55)
        self.assertAlmostEqual(ece, expected, places=6)

    def test_ece_confidence_zero_and_one_included(self):
        from shrine_classifier_ext.confidence_analysis import _bin_metrics, _ece
        # Confidence 0.0 → bin [0.0, 0.1); confidence 1.0 → bin [0.9, 1.0]
        records = [
            {"confidence": 0.0, "correct": 0},
            {"confidence": 1.0, "correct": 1},
        ]
        bins = _bin_metrics(records)
        counts_in_edge_bins = sum(b["count"] for b in bins if b["bin_lower"] in (0.0, 0.9))
        self.assertEqual(counts_in_edge_bins, 2)
        ece = _ece(bins, 2)
        self.assertIsNotNone(ece)
        self.assertGreaterEqual(ece, 0.0)

    def test_all_correct_group_stats(self):
        from shrine_classifier_ext.confidence_analysis import _group_stats
        records = [{"confidence": 0.9, "correct": 1}, {"confidence": 0.8, "correct": 1}]
        stats = _group_stats(records)
        self.assertEqual(stats["n_correct"], 2)
        self.assertEqual(stats["n_wrong"], 0)
        self.assertIsNone(stats["mean_confidence_wrong"])

    def test_all_wrong_group_stats(self):
        from shrine_classifier_ext.confidence_analysis import _group_stats
        records = [{"confidence": 0.9, "correct": 0}]
        stats = _group_stats(records)
        self.assertEqual(stats["n_wrong"], 1)
        self.assertIsNone(stats["mean_confidence_correct"])


# ---------------------------------------------------------------------------
# Diagnostics history tests
# ---------------------------------------------------------------------------

class TestDiagnosticsHistory(unittest.TestCase):

    def _write_history(self, tmp_dir: Path, rows: list[dict]) -> Path:
        path = tmp_dir / "history.csv"
        fields = ["epoch", "train_loss", "train_acc", "train_macro_f1",
                  "val_loss", "val_acc", "val_macro_f1", "learning_rate",
                  "best_epoch", "best_val_acc", "best_val_macro_f1",
                  "test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1",
                  "precision_san_phraphrom", "precision_san_phraphum", "precision_san_tayay",
                  "precision_tee_ju_eia", "recall_san_phraphrom", "recall_san_phraphum",
                  "recall_san_tayay", "recall_tee_ju_eia",
                  "f1_san_phraphrom", "f1_san_phraphum", "f1_san_tayay", "f1_tee_ju_eia",
                  "support_san_phraphrom", "support_san_phraphum", "support_san_tayay",
                  "support_tee_ju_eia"]
        with path.open("w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            for row in rows:
                writer.writerow({k: row.get(k, "") for k in fields})
        return path

    def test_valid_epoch_rows_extracted(self):
        from shrine_classifier_ext.diagnostics_analysis import _load_history
        with tempfile.TemporaryDirectory() as td:
            p = Path(td)
            epoch_data = [
                {"epoch": 1, "train_loss": 1.0, "train_acc": 0.5, "train_macro_f1": 0.48,
                 "val_loss": 0.9, "val_acc": 0.55, "val_macro_f1": 0.52, "learning_rate": 1e-4},
                {"epoch": 2, "train_loss": 0.8, "train_acc": 0.7, "train_macro_f1": 0.68,
                 "val_loss": 0.7, "val_acc": 0.72, "val_macro_f1": 0.70, "learning_rate": 1e-4},
            ]
            path = self._write_history(p, epoch_data)
            epoch_rows, summary_rows = _load_history(path)
            self.assertEqual(len(epoch_rows), 2)
            self.assertEqual(epoch_rows[0]["epoch"], 1)

    def test_summary_rows_not_counted_as_epochs(self):
        from shrine_classifier_ext.diagnostics_analysis import _load_history
        with tempfile.TemporaryDirectory() as td:
            p = Path(td)
            # Mix of epoch rows and summary rows (no train_loss in summary)
            data = [
                {"epoch": 1, "train_loss": 1.0, "train_acc": 0.5, "train_macro_f1": 0.48,
                 "val_loss": 0.9, "val_acc": 0.55, "val_macro_f1": 0.52, "learning_rate": 1e-4},
                {"epoch": 1, "best_epoch": 1, "best_val_acc": 0.55, "best_val_macro_f1": 0.52},
                {"epoch": 1, "test_acc": 0.9, "test_macro_f1": 0.88},
            ]
            path = self._write_history(p, data)
            epoch_rows, summary_rows = _load_history(path)
            self.assertEqual(len(epoch_rows), 1)
            self.assertGreater(len(summary_rows), 0)

    def test_best_epoch_tie_resolution(self):
        from shrine_classifier_ext.diagnostics_analysis import _best_epoch
        # Two epochs with same val_macro_f1 → pick lower val_loss
        rows = [
            {"epoch": 3, "val_macro_f1": 0.9, "val_loss": 0.3, "val_acc": 0.9},
            {"epoch": 5, "val_macro_f1": 0.9, "val_loss": 0.2, "val_acc": 0.9},
        ]
        self.assertEqual(_best_epoch(rows), 5)

    def test_best_epoch_second_tie_uses_earlier(self):
        from shrine_classifier_ext.diagnostics_analysis import _best_epoch
        # Same F1, same loss → earlier epoch wins
        rows = [
            {"epoch": 3, "val_macro_f1": 0.9, "val_loss": 0.2, "val_acc": 0.9},
            {"epoch": 7, "val_macro_f1": 0.9, "val_loss": 0.2, "val_acc": 0.9},
        ]
        self.assertEqual(_best_epoch(rows), 3)

    def test_unequal_histories_not_filled(self):
        from shrine_classifier_ext.diagnostics_analysis import _load_history
        with tempfile.TemporaryDirectory() as td:
            p = Path(td)
            data_5 = [
                {"epoch": i, "train_loss": 0.5, "train_acc": 0.8, "train_macro_f1": 0.78,
                 "val_loss": 0.4, "val_acc": 0.85, "val_macro_f1": 0.82, "learning_rate": 1e-4}
                for i in range(1, 6)
            ]
            path = self._write_history(p, data_5)
            epoch_rows, _ = _load_history(path)
            self.assertEqual(len(epoch_rows), 5)
            self.assertEqual(epoch_rows[-1]["epoch"], 5)


# ---------------------------------------------------------------------------
# Fine-tuning mode tests
# ---------------------------------------------------------------------------

@unittest.skipUnless(
    __import__("importlib").util.find_spec("torch") is not None,
    "PyTorch not installed"
)
class TestFinetuningModes(unittest.TestCase):

    def _build(self, arch: str, mode: str):
        from shrine_classifier.models import build_model
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode
        import torch
        model, _ = build_model(arch, pretrained=False)
        device = torch.device("cpu")
        model = model.to(device)
        apply_finetuning_mode(model, arch, mode)
        return model, device

    def _check_logits_shape(self, arch: str, mode: str):
        import torch
        model, device = self._build(arch, mode)
        dummy = torch.zeros(1, 3, 224, 224, device=device)
        model.eval()
        out = model(dummy)
        self.assertEqual(out.shape, (1, 4),
                         f"{arch}/{mode}: expected (1,4), got {out.shape}")

    def test_all_architectures_modes_logit_shape(self):
        from shrine_classifier.models import ARCHITECTURES
        from shrine_classifier_ext.finetuning_modes import MODES
        for arch in ARCHITECTURES:
            for mode in MODES:
                with self.subTest(arch=arch, mode=mode):
                    self._check_logits_shape(arch, mode)

    def test_full_mode_all_trainable(self):
        from shrine_classifier.models import ARCHITECTURES
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode
        from shrine_classifier.models import build_model
        import torch
        for arch in ARCHITECTURES:
            with self.subTest(arch=arch):
                model, _ = build_model(arch, pretrained=False)
                stats = apply_finetuning_mode(model, arch, "full")
                self.assertEqual(stats["frozen_parameters"], 0,
                                 f"{arch}/full: expected 0 frozen params, got {stats['frozen_parameters']}")

    def test_frozen_mode_head_trainable_backbone_frozen(self):
        from shrine_classifier.models import ARCHITECTURES
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode, _get_head_modules
        from shrine_classifier.models import build_model
        import torch
        for arch in ARCHITECTURES:
            with self.subTest(arch=arch):
                model, _ = build_model(arch, pretrained=False)
                apply_finetuning_mode(model, arch, "frozen")
                # Head must be trainable
                for m in _get_head_modules(model, arch):
                    for p in m.parameters():
                        self.assertTrue(p.requires_grad,
                                        f"{arch}/frozen: head param not trainable")
                # Some params must be frozen
                frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
                self.assertGreater(frozen, 0,
                                   f"{arch}/frozen: expected frozen params > 0")

    def test_partial_mode_densenet_opens_denseblock4(self):
        from shrine_classifier.models import build_model
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode
        import torch
        model, _ = build_model("densenet121", pretrained=False)
        apply_finetuning_mode(model, "densenet121", "partial")
        # denseblock4 params must be trainable
        trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
        db4_trainable = [n for n in trainable_names if "denseblock4" in n]
        self.assertGreater(len(db4_trainable), 0,
                           "densenet121/partial: features.denseblock4 should be trainable")

    def test_frozen_bn_buffers_not_updated(self):
        """Frozen BN should not update running_mean/var after a step."""
        import torch
        from shrine_classifier.models import build_model
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode, enforce_mode_per_epoch

        model, _ = build_model("resnet50", pretrained=False)
        device = torch.device("cpu")
        model = model.to(device)
        apply_finetuning_mode(model, "resnet50", "frozen")

        # Save BN running stats from backbone
        before_mean = model.layer1[0].bn1.running_mean.clone()
        before_var = model.layer1[0].bn1.running_var.clone()

        # Forward pass with mode enforcement
        model.train()
        enforce_mode_per_epoch(model, "resnet50", "frozen")
        dummy = torch.randn(2, 3, 224, 224, device=device)
        out = model(dummy)
        out.sum().backward()

        after_mean = model.layer1[0].bn1.running_mean.clone()
        after_var = model.layer1[0].bn1.running_var.clone()

        # Frozen backbone BN should not have updated running stats
        self.assertTrue(torch.allclose(before_mean, after_mean),
                        "resnet50/frozen: BN running_mean changed (backbone should be in eval)")
        self.assertTrue(torch.allclose(before_var, after_var),
                        "resnet50/frozen: BN running_var changed")

    def test_trainable_params_get_gradient(self):
        import torch
        from shrine_classifier.models import build_model
        from shrine_classifier_ext.finetuning_modes import apply_finetuning_mode, enforce_mode_per_epoch

        model, _ = build_model("resnet50", pretrained=False)
        device = torch.device("cpu")
        model = model.to(device)
        apply_finetuning_mode(model, "resnet50", "frozen")

        model.train()
        enforce_mode_per_epoch(model, "resnet50", "frozen")
        dummy = torch.randn(1, 3, 224, 224, device=device)
        out = model(dummy)
        out.sum().backward()

        trainable_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
        self.assertGreater(len(trainable_params), 0)
        for name, param in trainable_params:
            self.assertIsNotNone(param.grad, f"Trainable param {name} has no gradient")


# ---------------------------------------------------------------------------
# Ablation config tests
# ---------------------------------------------------------------------------

class TestAblationConfigIsolation(unittest.TestCase):

    def test_no_augmentation_only_changes_augmentation(self):
        """no_augmentation config must differ only in augmentation flag."""
        base = {
            "optimizer": "adamw", "learning_rate": 1e-4, "weight_decay": 1e-4,
            "batch_size": 16, "epochs": 30, "patience": 7, "input_size": 224,
            "augmentation": True, "use_class_weights": True,
        }
        no_aug = {**base, "augmentation": False}
        diffs = {k for k in base if base[k] != no_aug.get(k)}
        self.assertEqual(diffs, {"augmentation"})

    def test_no_class_weights_only_changes_class_weights(self):
        base = {
            "optimizer": "adamw", "learning_rate": 1e-4, "weight_decay": 1e-4,
            "batch_size": 16, "epochs": 30, "patience": 7, "input_size": 224,
            "augmentation": True, "use_class_weights": True,
        }
        no_cw = {**base, "use_class_weights": False}
        diffs = {k for k in base if base[k] != no_cw.get(k)}
        self.assertEqual(diffs, {"use_class_weights"})


# ---------------------------------------------------------------------------
# Statistics: seed join and Holm family
# ---------------------------------------------------------------------------

class TestStatistics(unittest.TestCase):

    def test_seed_join_tolerates_reversed_order(self):
        """Pairing by seed ID not row position."""
        import numpy as np
        full_by_seed = {42: 0.90, 123: 0.88, 456: 0.91, 789: 0.87, 1024: 0.92}
        var_by_seed = {1024: 0.88, 789: 0.85, 456: 0.89, 123: 0.86, 42: 0.88}  # reversed
        shared = sorted(set(full_by_seed) & set(var_by_seed))
        diffs = [var_by_seed[s] - full_by_seed[s] for s in shared]
        # Mean diff should be negative (variant generally lower)
        # Values: 42→-0.02, 123→-0.02, 456→-0.02, 789→-0.02, 1024→-0.04 → mean=-0.024
        self.assertLess(float(np.mean(diffs)), 0.0,
                        "variant should be on average below full")

    def test_holm_family_size_eight(self):
        """Holm family must be exactly 8 for 4 variants × 2 metrics."""
        variants = ["no_augmentation", "no_class_weights", "frozen", "partial"]
        metrics = ["test_acc", "test_macro_f1"]
        family = len(variants) * len(metrics)
        self.assertEqual(family, 8)

    def test_zero_variance_difference_status(self):
        """Zero-variance diffs must produce explicit status, not a p-value."""
        import numpy as np
        diffs = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
        if np.std(diffs, ddof=1) == 0:
            status = "zero_variance_or_nonfinite"
        else:
            status = "ok"
        self.assertEqual(status, "zero_variance_or_nonfinite")

    def test_constant_diff_is_zero_variance(self):
        import numpy as np
        full_vals = [0.9, 0.91, 0.92, 0.89, 0.93]
        var_vals = [0.8, 0.81, 0.82, 0.79, 0.83]  # constant gap of 0.1
        diffs = np.array(var_vals) - np.array(full_vals)
        # sd should be essentially 0 (constant delta)
        sd = np.std(diffs, ddof=1)
        self.assertAlmostEqual(sd, 0.0, places=10)

    def test_sample_sd_ddof1(self):
        import numpy as np
        values = [0.90, 0.91, 0.89, 0.92, 0.88]
        sd = float(np.std(values, ddof=1))
        # Check ddof=1 vs ddof=0
        sd0 = float(np.std(values, ddof=0))
        self.assertGreater(sd, sd0)


# ---------------------------------------------------------------------------
# Protocol protection
# ---------------------------------------------------------------------------

class TestProtocolProtection(unittest.TestCase):

    def test_prepare_does_not_overwrite_existing_protocol(self):
        """A second prepare run on an existing protocol.json must not overwrite."""
        with tempfile.TemporaryDirectory() as td:
            out = Path(td) / "out"
            out.mkdir()
            proto = out / "protocol.json"
            original = {"test": "immutable_content"}
            proto.write_text(json.dumps(original))
            # Simulate prepare: it checks and skips if exists
            if proto.exists():
                existing = json.loads(proto.read_text())
                self.assertEqual(existing["test"], "immutable_content")
                # Do NOT overwrite
                should_overwrite = False
            else:
                should_overwrite = True
            self.assertFalse(should_overwrite)
            # File should be unchanged
            self.assertEqual(json.loads(proto.read_text())["test"], "immutable_content")


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%project_source tests/test_mode_protocol_v2.py
import contextlib
import io
import json
import tempfile
import unittest
from pathlib import Path
from types import SimpleNamespace

import pipeline_modes as modes


class ModeProtocolTests(unittest.TestCase):
    def test_search_configs_repeat_from_fixed_seed(self):
        first = modes._sample_configs(12, 42)
        self.assertEqual(first, modes._sample_configs(12, 42))
        self.assertEqual(len(first), 12)
        self.assertTrue(all(1e-5 <= r["learning_rate"] <= 1e-3 for r in first))
        self.assertTrue(all(1e-6 <= r["weight_decay"] <= 1e-2 for r in first))

    def test_mobilenet_partial_mode_opens_whole_tail_stage_and_projection(self):
        from shrine_classifier.models import build_model
        from shrine_classifier_ext.finetuning_modes import _get_partial_backbone_modules

        model, _ = build_model("mobilenet_v3_large", pretrained=False)
        stage = _get_partial_backbone_modules(model, "mobilenet_v3_large")
        names = [name for name, module in model.named_modules() if any(module is item for item in stage)]
        self.assertEqual(names, ["features.13", "features.14", "features.15", "features.16"])

    def test_lock_selects_validation_and_keeps_architecture_mode_pairs(self):
        with tempfile.TemporaryDirectory() as temp:
            out = Path(temp)
            (out / "protocol.json").write_text(json.dumps({"protocol_fingerprint": "fixed-protocol", "source_fingerprint": "source", "assignment_fingerprint": "split"}), encoding="utf-8")
            (out / "tuning").mkdir()
            archs, selected_modes = ["resnet50", "densenet121"], ["full", "frozen"]
            rows = []
            for ai, arch in enumerate(archs):
                for mi, mode in enumerate(selected_modes):
                    for trial in (1, 2):
                        score = .75 + .01 * ai + .02 * mi + .001 * trial
                        rows.append({"identity": f"{arch}::{mode}::trial{trial:03d}", "architecture": arch,
                            "mode": mode, "trial": trial, "status": "completed",
                            "config": {"learning_rate": trial * 1e-4,
                                "weight_decay": 1e-4, "optimizer": "adamw", "batch_size": 16},
                            "best_val_macro_f1": score, "best_val_acc": score, "best_val_loss": 1-score,
                            "test_macro_f1": .01 if mode == "full" else .99,
                            "config_fingerprint": f"fp-{arch}-{mode}-{trial}", "checkpoint": f"{arch}-{mode}-{trial}.pt"})
            (out / "run_registry.json").write_text(json.dumps({"protocol_fingerprint": "fixed-protocol", "trials": rows, "final_runs": []}), encoding="utf-8")
            (out / "splits.csv").write_text("image_id,split\n", encoding="utf-8")
            args = SimpleNamespace(architectures=archs, modes=selected_modes, trials_per_pair=2,
                                   seeds=[42,123,456,789,1024], epochs=30)
            locked = modes._lock(args, out)
            self.assertEqual(locked["reference_pair"], {"architecture": "densenet121", "mode": "frozen"})
            self.assertFalse(locked["test_metrics_read"])
            winners = json.loads((out / "tuning" / "best_params_by_arch_mode.json").read_text(encoding="utf-8"))
            self.assertEqual(set(winners["pairs"]), {f"{a}::{m}" for a in archs for m in selected_modes})
            self.assertEqual(winners["pairs"]["resnet50::full"]["winning_trial"], 2)
            self.assertEqual(winners["pairs"]["resnet50::full"]["learning_rate"], 2e-4)
            self.assertEqual(modes._lock(args, out), locked)

    def test_tuning_plots_read_nested_logged_config(self):
        with tempfile.TemporaryDirectory() as temp:
            out = Path(temp)
            rows = [{"architecture": "resnet50", "mode": mode, "status": "completed",
                     "best_val_macro_f1": .8,
                     "config": {"learning_rate": 1e-4, "weight_decay": 1e-5}}
                    for mode in modes.MODES]
            modes._tuning_plot(out, rows)
            for filename in ("tuning_validation_scatter.png", "weight_decay_validation_scatter.png"):
                self.assertGreater((out / "tuning" / filename).stat().st_size, 0)

    def test_test_rows_join_by_seed_and_fixed_family_size_survives_undefined(self):
        ref, other = {"architecture": "resnet50", "mode": "full"}, {"architecture": "resnet50", "mode": "frozen"}
        rows = []
        for seed in [456, 42, 1024, 123, 789]:
            rows.extend([{"architecture": "resnet50", "mode": "full", "seed": seed, "status": "completed", "test_macro_f1": .8},
                         {"architecture": "resnet50", "mode": "frozen", "seed": seed, "status": "completed", "test_macro_f1": .8}])
        result = modes._test_summary(rows, ref, other, "test_macro_f1", "primary")
        self.assertEqual(result["seed_ids"], "42,123,456,789,1024")
        self.assertEqual(result["status"], "undefined_zero_variance")
        self.assertIsNone(result["p_value"])
        record = {"p_value": .01}
        modes._holm([record], 28)
        self.assertEqual(record["holm_family_size"], 28)
        self.assertEqual(record["p_holm"], .28)

    def test_cli_dry_run_reports_revised_five_trial_grid_without_starting(self):
        with tempfile.TemporaryDirectory() as temp:
            destination = Path(temp) / "new"
            output = io.StringIO()
            with contextlib.redirect_stdout(output):
                code = modes.main(["--stage", "all", "--dry-run", "--out-dir", str(destination)])
            plan = json.loads(output.getvalue())
            self.assertEqual(code, 0)
            self.assertEqual(plan["tuning_runs"], 75)
            self.assertEqual(plan["final_runs"], 75)
            self.assertEqual(plan["max_epochs"], 4500)
            self.assertFalse(plan["measured_run_started"])
            self.assertFalse(destination.exists())


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%project_source tests/test_pipeline_smoke.py
import importlib.util
import unittest

from shrine_classifier.models import ARCHITECTURES


@unittest.skipUnless(importlib.util.find_spec("torch") and importlib.util.find_spec("torchvision"),
                     "PyTorch/torchvision absent; run pipeline.py --stage smoke on the model runtime")
class ArchitectureBackwardSmokeTests(unittest.TestCase):
    def test_five_models_emit_four_logits_and_backpropagate(self):
        import torch
        from shrine_classifier.models import build_model
        device = "cuda" if torch.cuda.is_available() else "cpu"
        x = torch.randn(2, 3, 64, 64, device=device)
        for name in ARCHITECTURES:
            model, _ = build_model(name, pretrained=False)
            model.to(device).train()
            output = model(x)
            self.assertEqual(tuple(output.shape), (2, 4), name)
            torch.nn.functional.cross_entropy(output, torch.tensor([0, 1], device=device)).backward()
            self.assertTrue(any(p.grad is not None for p in model.parameters()), name)
            del model


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%project_source tests/test_retired_protocol.py
"""Ensure cleanup cannot be undone by accidentally invoking the retired CLI."""
import unittest
from unittest.mock import patch
import pipeline_extensions


class RetiredProtocolTests(unittest.TestCase):
    def test_retired_stages_fail_before_dispatch(self):
        for stage in ("prepare", "ablation", "finetune", "report"):
            with self.subTest(stage=stage), patch.object(pipeline_extensions, "stage_" + stage) as dispatch:
                with self.assertRaises(SystemExit) as error:
                    pipeline_extensions.main(["--stage", stage])
                self.assertIn("retired fixed-Full-recipe", str(error.exception))
                self.assertIn("not implemented yet", str(error.exception))
                dispatch.assert_not_called()

    def test_dry_run_does_not_restore_retired_stages(self):
        for stage in ("prepare", "ablation", "finetune", "report"):
            with self.subTest(stage=stage), patch.object(pipeline_extensions, "stage_" + stage) as dispatch:
                with self.assertRaises(SystemExit):
                    pipeline_extensions.main(["--stage", stage, "--dry-run"])
                dispatch.assert_not_called()

    def test_historical_analysis_still_dispatches(self):
        with patch.object(pipeline_extensions, "stage_analyze") as dispatch, patch("builtins.print"):
            pipeline_extensions.main(["--stage", "analyze"])
            dispatch.assert_called_once()


if __name__ == "__main__":
    unittest.main()


## 6. Report utilities และ dependency list — source code

รวม JavaScript/PowerShell และ matched-CAM Python ที่ใช้ประกอบรายงานไว้ด้วย การ Run All เพียงเตรียมไฟล์ ไม่เรียก report tools และไม่แก้ README แพ็กเกจ/เครื่องมือ Node, Sharp หรือ PowerShell ของ report utilities เป็น dependencies แยกจาก Python


In [ ]:
%%project_source report/build_report.ps1
# Rebuild report evidence from saved artifacts only. No Python or training.
$ErrorActionPreference='Stop'
$root=Split-Path $PSScriptRoot -Parent
$out=Join-Path $root 'outputs_modes_v3_td'
$assets=Join-Path $PSScriptRoot 'report_assets'
New-Item -ItemType Directory -Path $assets -Force | Out-Null
$culture=[Globalization.CultureInfo]::InvariantCulture
function F($x,$fmt='F4'){ ([double]$x).ToString($fmt,$culture) }
function Mean($v){ ($v | Measure-Object -Average).Average }
function SD($v){ $m=Mean $v; [math]::Sqrt((($v | ForEach-Object {([double]$_-$m)*([double]$_-$m)} | Measure-Object -Sum).Sum)/($v.Count-1)) }
function Put($path,$content){ [IO.File]::WriteAllText($path,($content -join "`n"),[Text.UTF8Encoding]::new($false)) }
function Table($headers,$rows){
 $s=[Collections.Generic.List[string]]::new();$s.Add('| '+($headers -join ' | ')+' |');$s.Add('| '+(($headers|ForEach-Object{'---'}) -join ' | ')+' |')
 foreach($row in $rows){
  $cells=@($row)
  if($cells.Count-ne$headers.Count){throw "Table row has $($cells.Count) cells; expected $($headers.Count): $row"}
  $s.Add('| '+(($cells|ForEach-Object{([string]$_).Replace('|','\|')}) -join ' | ')+' |')
 };$s -join "`n"
}
$reg=Get-Content (Join-Path $out 'run_registry.json') -Raw|ConvertFrom-Json
$selection=Get-Content (Join-Path $out 'selection.json') -Raw|ConvertFrom-Json
$classes=@('san_phraphrom','san_phraphum','san_tayay','tee_ju_eia')
$archs=@('resnet50','efficientnet_b0','mobilenet_v3_large','densenet121','convnext_tiny')
$modes=@('full','frozen','partial')
$display=@{resnet50='ResNet50';efficientnet_b0='EfficientNet-B0';mobilenet_v3_large='MobileNetV3-Large';densenet121='DenseNet121';convnext_tiny='ConvNeXt-Tiny'}
$split=Import-Csv (Join-Path $out 'splits.csv')
$testIDs=@($split|Where-Object split -eq 'test'|ForEach-Object image_id)
$groupOverlap=@($split|Group-Object group_id|Where-Object{@($_.Group.split|Select-Object -Unique).Count -gt 1}).Count
$runs=@($reg.final_runs|Where-Object status -eq 'completed')
if($runs.Count-ne 75 -or @($runs.identity|Select-Object -Unique).Count-ne 75){throw 'Expected 75 unique completed final runs'}
$raw=[Collections.Generic.List[object]]::new();$perclass=[Collections.Generic.List[object]]::new()
$histories=@{};$configs=@{};$maxRawError=0.0;$maxSummaryError=0.0;$maxClassError=0.0;$maxStatError=0.0
foreach($r in $runs){
 $pred=@(Import-Csv (Join-Path $r.run_dir 'test_predictions.csv'))
 if($pred.Count-ne 85 -or (($pred.image_id -join '|')-ne($testIDs -join '|'))){throw "Test order mismatch $($r.identity)"}
 $cfg=Get-Content (Join-Path $r.run_dir 'config.json') -Raw|ConvertFrom-Json
 $configs[$r.identity]=$cfg
 $histories[$r.identity]=@(Import-Csv (Join-Path $r.run_dir 'history.csv')|Where-Object{$_.train_loss-ne ''})
 $report=Get-Content (Join-Path $r.run_dir 'run_metrics.json') -Raw|ConvertFrom-Json
 $ms=@();foreach($c in $classes){
  $tp=@($pred|Where-Object{$_.true_label-eq$c -and $_.predicted_label-eq$c}).Count
  $support=@($pred|Where-Object true_label -eq $c).Count;$positive=@($pred|Where-Object predicted_label -eq $c).Count
  $p=if($positive){$tp/$positive}else{0};$rec=$tp/$support;$f1=if($p+$rec){2*$p*$rec/($p+$rec)}else{0}
  $item=[pscustomobject]@{architecture=$r.architecture;mode=$r.mode;seed=[int]$r.seed;class=$c;support=$support;precision=$p;recall=$rec;f1=$f1}
  $perclass.Add($item);$ms+=,$item
  foreach($k in @('precision','recall','f1','support')){$maxRawError=[math]::Max($maxRawError,[math]::Abs([double]$item.$k-[double]$report.per_class.$c.$k))}
 }
 $acc=@($pred|Where-Object{$_.true_label-eq$_.predicted_label}).Count/85
 $row=[pscustomobject]@{identity=$r.identity;architecture=$r.architecture;mode=$r.mode;seed=[int]$r.seed;test_acc=$acc;test_macro_precision=(Mean $ms.precision);test_macro_recall=(Mean $ms.recall);test_macro_f1=(Mean $ms.f1)}
 foreach($k in @('test_acc','test_macro_precision','test_macro_recall','test_macro_f1')){$maxRawError=[math]::Max($maxRawError,[math]::Abs([double]$row.$k-[double]$report.$k))}
 $raw.Add($row)
}
$summary=[Collections.Generic.List[object]]::new();$classSummary=[Collections.Generic.List[object]]::new()
foreach($a in $archs){foreach($m in $modes){
 $rr=@($raw|Where-Object{$_.architecture-eq$a -and $_.mode-eq$m})
 if(($rr.seed|Sort-Object)-join ',' -ne '42,123,456,789,1024'){throw "Seed mismatch $a $m"}
 $record=[ordered]@{architecture=$a;mode=$m;completed_seeds=5}
 foreach($k in @('test_acc','test_macro_precision','test_macro_recall','test_macro_f1')){$record[$k+'_mean']=Mean $rr.$k;$record[$k+'_sample_sd']=SD $rr.$k}
 $summary.Add([pscustomobject]$record)
 foreach($c in $classes){$cc=@($perclass|Where-Object{$_.architecture-eq$a -and $_.mode-eq$m -and $_.class-eq$c});$rec=[ordered]@{architecture=$a;mode=$m;class=$c;support=$cc[0].support};foreach($k in @('precision','recall','f1')){$rec[$k+'_mean']=Mean $cc.$k;$rec[$k+'_sample_sd']=SD $cc.$k};$classSummary.Add([pscustomobject]$rec)}
}}
foreach($old in (Import-Csv (Join-Path $out 'final/pair_summary.csv'))){$new=$summary|Where-Object{$_.architecture-eq$old.architecture -and $_.mode-eq$old.mode};foreach($k in $new.PSObject.Properties.Name|Where-Object{$_ -match '_(mean|sample_sd)$'}){$maxSummaryError=[math]::Max($maxSummaryError,[math]::Abs([double]$new.$k-[double]$old.$k))}}
foreach($old in (Import-Csv (Join-Path $out 'final/per_class_summary.csv'))){$new=$classSummary|Where-Object{$_.architecture-eq$old.architecture -and $_.mode-eq$old.mode -and $_.class-eq$old.class};foreach($k in @('precision_mean','precision_sample_sd','recall_mean','recall_sample_sd','f1_mean','f1_sample_sd','support')){$maxClassError=[math]::Max($maxClassError,[math]::Abs([double]$new.$k-[double]$old.$k))}}
$stats=@(Import-Csv (Join-Path $out 'final/statistical_tests.csv'))
if(@($stats|Group-Object family_id,reference,competitor,metric|Where-Object Count -gt 1).Count){throw 'Duplicate hypotheses'}
$checked=[Collections.Generic.List[object]]::new()
foreach($s in $stats){
 $ar=$s.reference.Split('::',[StringSplitOptions]::RemoveEmptyEntries);$ac=$s.competitor.Split('::',[StringSplitOptions]::RemoveEmptyEntries)
 $r1=@($raw|Where-Object{$_.architecture-eq$ar[0] -and $_.mode-eq$ar[1]}|Sort-Object seed);$r2=@($raw|Where-Object{$_.architecture-eq$ac[0] -and $_.mode-eq$ac[1]}|Sort-Object seed)
 $d=@(0..4|ForEach-Object{[double]$r1[$_].($s.metric)-[double]$r2[$_].($s.metric)})
 $mean=Mean $d;$sd=SD $d;$t=$mean/($sd/[math]::Sqrt(5));$u=[math]::Abs($t)/[math]::Sqrt($t*$t+4)
 # Exact two-sided Student t tail at df=4: .5*(1-u)^2*(u+2).
 $p=.5*(1-$u)*(1-$u)*($u+2)
 $checked.Add([pscustomobject]@{family_id=$s.family_id;reference=$s.reference;competitor=$s.competitor;metric=$s.metric;paired_n=5;mean_difference=$mean;t=$t;p_raw=$p;p_holm=0.0;cohen_dz=($mean/$sd);ci95_lower=($mean-2.776445105*$sd/[math]::Sqrt(5));ci95_upper=($mean+2.776445105*$sd/[math]::Sqrt(5));significant=$false})
 $maxStatError=[math]::Max($maxStatError,[math]::Abs($p-[double]$s.p_value));$maxStatError=[math]::Max($maxStatError,[math]::Abs($t-[double]$s.t_statistic))
}
foreach($fam in ($checked|Group-Object family_id)){$ordered=@($fam.Group|Sort-Object p_raw);$prev=0.0;for($i=0;$i-lt$ordered.Count;$i++){$v=[math]::Min(1.0,[math]::Max($prev,($ordered.Count-$i)*$ordered[$i].p_raw));$ordered[$i].p_holm=$v;$ordered[$i].significant=$v-lt.05;$prev=$v}}
foreach($s in $stats){$r=$checked|Where-Object{$_.family_id-eq$s.family_id -and $_.reference-eq$s.reference -and $_.competitor-eq$s.competitor -and $_.metric-eq$s.metric};$maxStatError=[math]::Max($maxStatError,[math]::Abs($r.p_holm-[double]$s.p_holm))}
if([math]::Max([math]::Max($maxRawError,$maxSummaryError),[math]::Max($maxClassError,$maxStatError))-gt 1e-9){throw "Metric discrepancy exceeds 1e-9: raw=$maxRawError summary=$maxSummaryError perclass=$maxClassError stats=$maxStatError"}
$raw|Export-Csv (Join-Path $PSScriptRoot 'verified_run_metrics.csv') -NoTypeInformation -Encoding utf8
$summary|Export-Csv (Join-Path $PSScriptRoot 'verified_pair_summary.csv') -NoTypeInformation -Encoding utf8
$classSummary|Export-Csv (Join-Path $PSScriptRoot 'verified_per_class_summary.csv') -NoTypeInformation -Encoding utf8
$checked|Export-Csv (Join-Path $PSScriptRoot 'verified_statistics.csv') -NoTypeInformation -Encoding utf8
$validation=[ordered]@{output_root='outputs_modes_v3_td';final_unique_runs=$raw.Count;test_predictions=($raw.Count*85);canonical_test_order_verified=$true;group_overlap=$groupOverlap;max_prediction_vs_run_metric_error=$maxRawError;max_summary_error=$maxSummaryError;max_perclass_summary_error=$maxClassError;max_statistical_error=$maxStatError;holm_families=@(28,30);statistical_rows=$checked.Count;significant_rows=@($checked|Where-Object significant).Count;test_loss_recomputed=$false;python_executed=$false;verification_method='PowerShell; metrics from predictions; paired n=5, exact Student t df=4 tail; Holm within each predeclared family'}
Put (Join-Path $PSScriptRoot 'verification.json') ($validation|ConvertTo-Json -Depth 5)
# Full project inventory, excluding installed runtime and bytecode caches.
$allFiles=@(Get-ChildItem $root -File -Force)+@(Get-ChildItem $root -Directory -Force|Where-Object Name -NotIn @('.venv','__pycache__','.git','report')|ForEach-Object{Get-ChildItem $_.FullName -Recurse -File -Force|Where-Object FullName -NotMatch '\\__pycache__\\'})
$inventory=foreach($f in $allFiles){$relative=$f.FullName.Substring($root.Length+1).Replace('\','/');$scope=if($relative.StartsWith('outputs_modes_v3_td/')){'latest'}elseif($relative.StartsWith('dataset_cleaned/')){'dataset'}elseif($relative-match '^outputs|^wandb/'){'excluded_old_or_smoke'}else{'source_or_instruction'};$a=($archs|Where-Object{$relative.Contains($_)}|Select-Object -First 1);$seed=if($relative-match 'seed(\d+)'){$Matches[1]}else{''};[pscustomobject]@{path=$relative;bytes=$f.Length;type=$f.Extension;scope=$scope;architecture=$a;seed=$seed;evidence_kind=if($f.Extension-eq'.csv'-or$f.Extension-eq'.json'){'structured'}elseif($f.Extension-eq'.pt'){'checkpoint'}elseif($f.Extension-eq'.png'){'figure'}else{'other'}}}
$inventory|Export-Csv (Join-Path $PSScriptRoot 'evidence_inventory.csv') -NoTypeInformation -Encoding utf8
# Copy selected existing figures with GitHub-safe relative names, without modification.
$copy=@{'dataset_audit/class_distribution.png'='class_distribution.png';'dataset_audit/sample_grid.png'='sample_grid.png';'dataset_audit/preprocessing_examples.png'='preprocessing.png';'baseline/pretrained_examples.png'='imagenet_baseline.png';'tuning/tuning_validation_scatter.png'='tuning_lr.png';'tuning/weight_decay_validation_scatter.png'='tuning_weight_decay.png';'report_assets/pair_accuracy.png'='pair_accuracy.png';'report_assets/pair_macro_f1.png'='pair_macro_f1.png';'gradcam/reference_pair_cam_grid.png'='gradcam_reference.png';'eyeball/eyeball_grid.png'='eyeball_reference.png';'diagnostics/training_gaps.png'='training_gaps.png';'confidence/reliability_mobilenet_v3_large_partial_seed42.png'='reference_reliability.png'}
foreach($src in $copy.Keys){Copy-Item -LiteralPath (Join-Path $out $src) -Destination (Join-Path $assets $copy[$src])}
foreach($a in $archs){$r=$runs|Where-Object{$_.architecture-eq$a -and $_.mode-eq'full' -and $_.seed-eq42};Copy-Item (Join-Path $r.run_dir 'confusion_matrix_normalized.png') (Join-Path $assets "confusion_${a}_full_seed42.png")}
$ref=$runs|Where-Object{$_.architecture-eq'mobilenet_v3_large' -and $_.mode-eq'partial' -and $_.seed-eq42}
Copy-Item (Join-Path $ref.run_dir 'confusion_matrix_normalized.png') (Join-Path $assets 'confusion_reference_seed42.png')
# Use the user's existing mean ± SD plots; do not regenerate training curves.
& (Join-Path $PSScriptRoot 'copy_learning_curves.ps1')
# Audit EDA distribution SVG (actual properties, not inferred brightness).
$props=@(Import-Csv (Join-Path $out 'dataset_audit/image_properties.csv'))
$svg=[Collections.Generic.List[string]]::new();$svg.Add('<svg xmlns="http://www.w3.org/2000/svg" width="1100" height="640" viewBox="0 0 1100 640"><rect width="1100" height="640" fill="white"/><text x="30" y="30" font-family="sans-serif" font-size="20">Image properties | n=560 | audit of canonical dataset</text>')
$fields=@('width','height','aspect_ratio','file_bytes');$maxima=@(4500,4500,4,2000000)
for($i=0;$i-lt4;$i++){$vals=@($props|ForEach-Object{if($i-eq2){[double]$_.width/[double]$_.height}else{[double]$_.($fields[$i])}});$bins=New-Object int[] 10;foreach($v in $vals){$idx=[math]::Min(9,[math]::Floor($v/$maxima[$i]*10));$bins[$idx]++};$mx=($bins|Measure-Object -Maximum).Maximum;$x=60+($i%2)*540;$y=75+[math]::Floor($i/2)*285;$svg.Add("<text x='$x' y='$y' font-family='sans-serif' font-size='16'>$($fields[$i]) | last bin includes overflow</text>");for($b=0;$b-lt10;$b++){$height=$bins[$b]/$mx*170;$xx=$x+$b*45;$yy=$y+195-$height;$label=if($i-eq3){F ($b*$maxima[$i]/10/1000) 'F0'}else{F ($b*$maxima[$i]/10) 'G3'};$svg.Add("<rect x='$xx' y='$(F $yy)' width='38' height='$(F $height)' fill='#24496b'/><text x='$xx' y='$(F ($yy-3))' font-size='10'>$($bins[$b])</text><text x='$xx' y='$($y+215)' font-size='10'>$label</text>")};if($i-eq3){$svg.Add("<text x='$x' y='$($y+235)' font-size='12'>x-axis: file size (kB)</text>")}}
$svg.Add('</svg>');Put (Join-Path $assets 'image_properties.svg') $svg
"Verified $($raw.Count) runs, 6375 predictions; max metric error=$maxRawError; stat error=$maxStatError; inventory=$($inventory.Count) files"


In [ ]:
%%project_source report/copy_learning_curves.ps1
# Copy existing measured figures and record their sources. Does not run Python.
$ErrorActionPreference = 'Stop'
$curveWorkspace = Split-Path $PSScriptRoot -Parent
$curveOutput = Join-Path $curveWorkspace 'outputs_modes_v3_td'
$curveRegistry = Get-Content -LiteralPath (Join-Path $curveOutput 'run_registry.json') -Raw | ConvertFrom-Json
$curveArchitectures = @('resnet50','efficientnet_b0','mobilenet_v3_large','densenet121','convnext_tiny')
$curveModes = @('full','frozen','partial')
$curveSeeds = @(42,123,456,789,1024)
$curveCounts = [Collections.Generic.List[object]]::new()
$curvePairs = [Collections.Generic.List[object]]::new()
$curveFiles = [Collections.Generic.List[object]]::new()
$curveRelativeFiles = [Collections.Generic.List[string]]::new()
$curveRelativeFiles.Add('all_pairs_grid.png')

foreach ($curveArchitecture in $curveArchitectures) {
    $curveRelativeFiles.Add("$curveArchitecture/mean_sd_grid.png")
    foreach ($curveMode in $curveModes) {
        $curveRuns = @($curveRegistry.final_runs | Where-Object {
            $_.architecture -eq $curveArchitecture -and $_.mode -eq $curveMode
        })
        if ($curveRuns.Count -ne 5 -or @($curveRuns | Where-Object status -ne 'completed').Count) {
            throw "Expected five completed runs: $curveArchitecture/$curveMode"
        }
        if ((@($curveRuns.seed | Sort-Object) -join ',') -ne ($curveSeeds -join ',')) {
            throw "Unexpected seed set: $curveArchitecture/$curveMode"
        }
        # The plotting script scans every directory, rather than the registry.
        $curveRunDirectories = @(Get-ChildItem -LiteralPath (Join-Path $curveOutput "final/$curveArchitecture/$curveMode") -Directory)
        if ($curveRunDirectories.Count -ne 5 -or @(Compare-Object @($curveRuns.run_dir | Sort-Object) @($curveRunDirectories.FullName | Sort-Object)).Count) {
            throw "Plot input directories differ from registered runs: $curveArchitecture/$curveMode"
        }
        $curveEnds = @{}
        $curveHistorySources = foreach ($curveRun in $curveRuns) {
            $curveHistoryPath = Join-Path $curveRun.run_dir 'history.csv'
            $curveHistory = @(Import-Csv -LiteralPath $curveHistoryPath | Where-Object { $_.train_loss -ne '' } | Sort-Object { [int]$_.epoch } -Unique)
            $curveEpochs = @($curveHistory | ForEach-Object { [int]$_.epoch })
            $curveEnd = ($curveEpochs | Measure-Object -Maximum).Maximum
            if (($curveEpochs -join ',') -ne ((1..$curveEnd) -join ',')) { throw "Non-contiguous history: $curveHistoryPath" }
            $curveEnds[[int]$curveRun.seed] = [int]$curveEnd
            [ordered]@{
                seed = [int]$curveRun.seed
                history = $curveHistoryPath.Substring($curveWorkspace.Length+1).Replace('\','/')
                history_sha256 = (Get-FileHash -LiteralPath $curveHistoryPath -Algorithm SHA256).Hash.ToLowerInvariant()
                stop_epoch = [int]$curveEnd
                selected_epoch = [int]$curveRun.best_epoch
            }
        }
        $curveMinEnd = ($curveEnds.Values | Measure-Object -Minimum).Minimum
        $curveMaxEnd = ($curveEnds.Values | Measure-Object -Maximum).Maximum
        foreach ($curveEpoch in 1..$curveMaxEnd) {
            $curveAvailable = @($curveSeeds | Where-Object { $curveEnds[$_] -ge $curveEpoch })
            $curveCounts.Add([pscustomobject]@{
                architecture=$curveArchitecture; mode=$curveMode; epoch=$curveEpoch
                available_runs=$curveAvailable.Count; available_seeds=($curveAvailable -join ';')
                sample_sd_defined=($curveAvailable.Count -ge 2)
            })
        }
        $curvePairs.Add([ordered]@{
            architecture=$curveArchitecture; mode=$curveMode; initial_n=5
            all_five_available_through_epoch=[int]$curveMinEnd
            last_epoch=[int]$curveMaxEnd; histories=@($curveHistorySources)
        })
    }
}
$curveRelativeFiles.Add('mobilenet_v3_large/partial/mean_sd_curves.png')
foreach ($curveRelative in $curveRelativeFiles) {
    $curveSource = Join-Path $curveOutput "final/curves/$curveRelative"
    $curveTarget = Join-Path $PSScriptRoot "report_assets/learning_curves/$curveRelative"
    New-Item -ItemType Directory -Path (Split-Path $curveTarget -Parent) -Force | Out-Null
    Copy-Item -LiteralPath $curveSource -Destination $curveTarget
    $curveSourceHash = (Get-FileHash -LiteralPath $curveSource -Algorithm SHA256).Hash.ToLowerInvariant()
    $curveTargetHash = (Get-FileHash -LiteralPath $curveTarget -Algorithm SHA256).Hash.ToLowerInvariant()
    if ($curveSourceHash -ne $curveTargetHash) { throw "Copy hash mismatch: $curveRelative" }
    $curveFiles.Add([ordered]@{
        source="outputs_modes_v3_td/final/curves/$curveRelative"
        report_asset="report/report_assets/learning_curves/$curveRelative"
        sha256=$curveSourceHash; copied_unchanged=$true
    })
}
$curveCounts | Export-Csv -LiteralPath (Join-Path $PSScriptRoot 'learning_curve_epoch_counts.csv') -NoTypeInformation -Encoding utf8
$curveMap = [ordered]@{
    output_root='outputs_modes_v3_td'; protocol_fingerprint=$curveRegistry.protocol_fingerprint
    plot_script='plot_learning_curves.py'
    plot_script_sha256=(Get-FileHash -LiteralPath (Join-Path $curveWorkspace 'plot_learning_curves.py') -Algorithm SHA256).Hash.ToLowerInvariant()
    script_executed_for_report=$false
    aggregation='Per-epoch available-run mean and sample SD (ddof=1); shorter histories NaN-padded; no forward fill. When n=1 the plotting script substitutes SD=0, although sample SD is undefined.'
    checkpoint_marker='Mean of the five run-specific selected epochs, using validation Macro F1.'
    files=@($curveFiles); pairs=@($curvePairs)
}
[IO.File]::WriteAllText((Join-Path $PSScriptRoot 'learning_curve_sources.json'), ($curveMap | ConvertTo-Json -Depth 10), [Text.UTF8Encoding]::new($false))
"Copied $($curveFiles.Count) existing figures; verified 15 pairs with five registered seeds each."


In [ ]:
%%project_source report/figure_theme.js
// Presentation only. Numerical results remain in their original saved units.
const modeColors = { frozen: '#0072B2', partial: '#D55E00', full: '#009E73' };
const symbols = { frozen: 'circle', partial: 'square', full: 'diamond' };
// Remaining Okabe–Ito colours; class and fine-tuning mappings are separate.
const classColors = ['#56B4E9', '#E69F00', '#CC79A7', '#000000'];
const fontFamily = 'Arial,"Noto Sans Thai","Leelawadee UI",Tahoma,sans-serif';
const ink = '#1F2937', muted = '#6B7280', grid = '#E5E7EB';
const esc = value => String(value).replaceAll('&', '&amp;').replaceAll('<', '&lt;').replaceAll('>', '&gt;').replaceAll('"', '&quot;');
const styles = 'text{font-family:' + fontFamily + ';fill:' + ink + '}.title{font-size:23px;font-weight:700}.sub{font-size:14px;font-weight:600}.label{font-size:13px}.small{font-size:11px}.axis{stroke-width:1}.grid{stroke:' + grid + ';stroke-width:1}';
function heatColor(percent) {
  const t = Math.max(0, Math.min(100, percent)) / 100;
  const low = [244, 247, 248], high = [14, 74, 92];
  return 'rgb(' + low.map((v, i) => Math.round(v + t * (high[i] - v))).join(',') + ')';
}
module.exports = { modeColors, symbols, classColors, fontFamily, ink, muted, grid, styles, heatColor, esc };


In [ ]:
%%project_source report/generate_context_figures.js
// Percentage labels for compatible inherited baseline/reference evidence.
// Reads saved predictions and images only; never runs an ImageNet or fine-tuned model.
const fs = require('fs');
const path = require('path');
const {root, readCsv, hash, sharpRuntime} = require('./report_utils');
const theme = require('./figure_theme');
const sharp = sharpRuntime();
const assets = 'report/report_assets/latest/';
const inputPaths = new Set(['report/figure_theme.js','outputs_modes_v3_td/baseline/pretrained_predictions.csv','outputs_modes_v3_td/eyeball/eyeball_cases.csv','outputs_modes_v3_td/splits.csv']);
const labels = {san_phraphrom:'พระพรหม',san_phraphum:'พระภูมิ',san_tayay:'ตายาย',tee_ju_eia:'ตี่จู้เอี๊ยะ'};
const text = (x,y,value,size=27) => '<text x="'+x+'" y="'+y+'" text-anchor="middle" font-size="'+size+'">'+theme.esc(value)+'</text>';
const pct = n => (Number(n)*100).toFixed(2);
function header(w,h,title,subtitle) { return '<svg xmlns="http://www.w3.org/2000/svg" width="'+w+'" height="'+h+'"><style>text{font-family:'+theme.fontFamily+';fill:'+theme.ink+'}</style>'+text(w/2,48,title,38)+text(w/2,88,subtitle,27); }
async function thumb(relative,width,height) {
  inputPaths.add(relative);
  return sharp(path.join(root,relative)).resize(width,height,{fit:'contain',background:'white'}).png().toBuffer();
}
async function save(relative,width,height,svg,composites) {
  composites.push({input:Buffer.from(svg+'</svg>'),left:0,top:0});
  await sharp({create:{width,height,channels:3,background:'white'}}).composite(composites).png().toFile(path.join(root,relative));
  return relative;
}
async function run() {
  const outputs = [];
  const splits = new Map(readCsv('outputs_modes_v3_td/splits.csv').map(r=>[r.image_id,r]));
  const allBaseline = readCsv('outputs_modes_v3_td/baseline/pretrained_predictions.csv');
  if(allBaseline.length!==390)throw Error('Expected inherited ImageNet predictions for all 390 train images');
  const baseline = Object.keys(labels).flatMap(label => allBaseline.filter(r=>r.true_custom_class===label).slice(0,3));
  if(baseline.length!==12)throw Error('Missing three examples per class');
  const w=1800,h=1900,composites=[];
  let svg=header(w,h,'ResNet50 / ImageNet: original label space','Fixed train examples · ImageNet top-1 confidence (%)');
  for(let i=0;i<baseline.length;i++){
    const r=baseline[i],split=splits.get(r.image_id),left=20+(i%3)*600,top=120+Math.floor(i/3)*440;
    if(!split||split.split!=='train'||split.label!==r.true_custom_class)throw Error('Incompatible inherited baseline image');
    composites.push({input:await thumb('dataset_cleaned/'+split.path,550,300),left,top});
    svg+=text(left+275,top+330,r.image_id.split('/')[1],23)+text(left+275,top+368,'จริง: '+labels[r.true_custom_class],29);
    svg+=text(left+275,top+408,'ImageNet: '+r.imagenet_top1+' | '+pct(r.imagenet_confidence),28);
  }
  outputs.push(await save(assets+'imagenet_baseline_percent.png',w,h,svg,composites));

  const cases=readCsv('outputs_modes_v3_td/eyeball/eyeball_cases.csv');
  if(cases.length!==8)throw Error('Expected eight reference cases');
  const cw=2160,ch=2060,cam=[],ew=1800,eh=2100,eye=[];
  let cs=header(cw,ch,'MobileNetV3-Large / Partial · seed 42 · supplementary Grad-CAM','Saved reference examples · confidence (%)');
  let es=header(ew,eh,'MobileNetV3-Large / Partial · seed 42 · supplementary eyeball','ImageNet and fine-tuned predictions on the same image · confidence (%)');
  for(let i=0;i<cases.length;i++){
    const c=cases[i],dir='outputs_modes_v3_td/'+c.cam_files.replaceAll('\\','/'),left=35+(i%2)*1080,top=132+Math.floor(i/2)*475;
    cs+=text(left+510,top+25,c.image_id.split('/')[1],27);
    for(let j=0;j<3;j++){
      const file=['original.png','heatmap.png','overlay.png'][j];
      cam.push({input:await thumb(dir+'/'+file,320,320),left:left+j*342,top:top+78});
      cs+=text(left+j*342+160,top+61,['Original','Grad-CAM','Overlay'][j],25);
    }
    cs+=text(left+510,top+435,'จริง: '+labels[c.true_label]+' → ทำนาย: '+labels[c.predicted_label]+' | '+pct(c.confidence)+' '+(c.true_label===c.predicted_label?'✓':'✗'),29);
    const ex=30+(i%2)*900,ey=130+Math.floor(i/2)*480;
    eye.push({input:await thumb(dir+'/original.png',330,350),left:ex,top:ey+40});
    es+=text(ex+420,ey+12,c.image_id.split('/')[1],25);
    es+=text(ex+600,ey+135,'จริง: '+labels[c.true_label],30);
    es+=text(ex+600,ey+190,'ImageNet: '+c.imagenet_label,27)+text(ex+600,ey+233,pct(c.imagenet_confidence),29);
    es+=text(ex+600,ey+300,'Fine-tuned: '+labels[c.predicted_label],27)+text(ex+600,ey+343,pct(c.confidence)+' '+(c.true_label===c.predicted_label?'✓':'✗'),29);
  }
  outputs.push(await save(assets+'gradcam_reference_percent.png',cw,ch,cs,cam));
  outputs.push(await save(assets+'eyeball_reference_percent.png',ew,eh,es,eye));
  fs.writeFileSync(path.join(__dirname,'context_figure_sources.json'),JSON.stringify({generated_utc:new Date().toISOString(),
    source_sha256:Object.fromEntries([...inputPaths].map(p=>[p,hash(p)])),output_sha256:Object.fromEntries(outputs.map(p=>[p,hash(p)])),
    baseline_example_ids:baseline.map(r=>r.image_id),
    note:'Compatible inherited ImageNet baseline and latest reference diagnostics. First three logged train examples per class, fixed canonical class order. Re-layout and percentage conversion only. No inference/training. Original images and saved grids preserved.'},null,2)+'\n');
  console.log('Reformatted inherited baseline and reference examples with confidence (%).');
}
run().catch(error=>{console.error(error);process.exitCode=1;});


In [ ]:
%%project_source report/generate_followup_figures.js
// Re-layout saved CAM images and plot saved ablation metrics. No inference or training.
const fs = require('fs');
const path = require('path');
const { root, readCsv, hash, sharpRuntime, mean, sd } = require('./report_utils');
const theme = require('./figure_theme');
const assets = path.join(__dirname, 'report_assets/latest');
const sources = ['report/figure_theme.js', 'report/resnet_gradcam_cases.csv', 'report/resnet_followup_sources.json',
  'outputs_modes_v3_td/final/run_metrics.csv', 'outputs_modes_v3_td/ablation/resnet50_full_v1/run_metrics.csv',
  'outputs_modes_v3_td/ablation/resnet50_full_v1/condition_summary.csv'];
const originalHash = hash('report/report_assets/latest/gradcam_resnet50_full_seed42.png');
const conditions = ['baseline', 'no_augmentation', 'no_class_weights'];
const labels = ['Baseline', 'No augmentation', 'No class weights'];
const base = readCsv(sources[3]).filter(r => r.status === 'completed' && r.architecture === 'resnet50' && r.mode === 'full').map(r => ({ ...r, condition: 'baseline' }));
const runs = [...base, ...readCsv(sources[4]).filter(r => r.status === 'completed')];
const summary = readCsv(sources[5]);
const text = (x, y, value, size = 14, extra = '') => '<text x="' + x + '" y="' + y + '" text-anchor="middle" font-size="' + size + '" ' + extra + '>' + theme.esc(value) + '</text>';
const line = (x1, y1, x2, y2, color, width = 1) => '<line x1="' + x1 + '" y1="' + y1 + '" x2="' + x2 + '" y2="' + y2 + '" stroke="' + color + '" stroke-width="' + width + '"/>';
function diamond(x, y, r, opacity) { return '<path d="M ' + x + ' ' + (y-r) + ' l ' + r + ' ' + r + ' l ' + (-r) + ' ' + r + ' l ' + (-r) + ' ' + (-r) + ' Z" fill="' + theme.modeColors.full + '" opacity="' + opacity + '"/>'; }
let svg = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1100 550" role="img" aria-label="ResNet50 Full ablation: five matched seeds"><rect width="1100" height="550" fill="white"/><style>' + theme.styles + '</style>';
svg += text(550, 34, 'ResNet50 / Full: ablation on test', 23, 'font-weight="700"');
svg += text(550, 65, '5 matched seeds · mean ± sample SD · diamonds = individual seeds', 14);
['test_acc', 'test_macro_f1'].forEach((metric, panel) => {
  const left = 75 + panel * 540, top = 128, width = 430, height = 280;
  const fy = value => top + height - (value - 80) / 20 * height;
  svg += text(left + width/2, 104, panel ? 'Macro F1 (%)' : 'Accuracy (%)', 17, 'font-weight="700"');
  svg += text(left-55, top+height/2, panel ? 'Macro F1 (%)' : 'Accuracy (%)', 14, 'transform="rotate(-90 '+(left-55)+' '+(top+height/2)+')"');
  for (const tick of [80, 85, 90, 95, 100]) {
    svg += line(left, fy(tick), left + width, fy(tick), theme.grid) + text(left - 24, fy(tick) + 5, tick, 13);
  }
  svg += line(left, top, left, top + height, theme.muted) + line(left, top + height, left + width, top + height, theme.muted);
  conditions.forEach((condition, i) => {
    const subset = runs.filter(r => r.condition === condition).sort((a, b) => Number(a.seed) - Number(b.seed));
    if (subset.length !== 5 || new Set(subset.map(r => r.seed)).size !== 5) throw Error('Incomplete matched seeds: ' + condition);
    const values = subset.map(r => Number(r[metric]) * 100), m = mean(values), s = sd(values);
    const saved = summary.find(r => r.condition === condition);
    if (Math.abs(m - Number(saved[metric + '_mean']) * 100) > 1e-9 || Math.abs(s - Number(saved[metric + '_sample_sd']) * 100) > 1e-9) throw Error('Ablation summary mismatch');
    if (Math.min(...values, m-s) < 80 || Math.max(...values, m+s) > 100) throw Error('Ablation axis would clip data');
    const cx = left + 65 + i * 150;
    svg += line(cx, fy(m-s), cx, fy(m+s), theme.modeColors.full, 2);
    for (const v of [m-s, m+s]) svg += line(cx-8, fy(v), cx+8, fy(v), theme.modeColors.full, 2);
    values.forEach((v, j) => { svg += diamond(cx + (j-2)*7, fy(v), 4, 0.4); });
    svg += diamond(cx, fy(m), 7, 1) + text(cx, 445, labels[i], 13);
    svg += text(cx, 471, m.toFixed(2) + ' ± ' + s.toFixed(2), 13);
  });
});
svg += text(550, 520, 'Display axes: 80–100 · all conditions use Full mode · values below plots: mean (%) ± SD (pp)', 12) + '</svg>\n';
const ablationFigure = 'report/report_assets/latest/resnet_ablation_summary.svg';
fs.writeFileSync(path.join(root, ablationFigure), svg);

async function camGrid() {
  const sharp = sharpRuntime();
  const cases = readCsv('report/resnet_gradcam_cases.csv');
  if (cases.length !== 8) throw Error('Expected eight saved CAM cases');
  const width = 2160, height = 2060, composites = [];
  const short = { san_phraphrom: 'พระพรหม', san_phraphum: 'พระภูมิ', san_tayay: 'ตายาย', tee_ju_eia: 'ตี่จู้เอี๊ยะ' };
  let captions = '<svg xmlns="http://www.w3.org/2000/svg" width="2160" height="2060"><style>text{font-family:' + theme.fontFamily + ';fill:' + theme.ink + '}</style>';
  captions += text(1080, 49, 'ResNet50 / Full · seed 42 · Grad-CAM at layer4[-1]', 38, 'font-weight="700"');
  captions += text(1080, 92, 'Selected examples · confidence (%) · 6 correct / 2 incorrect', 28);
  for (let i = 0; i < cases.length; i++) {
    const c = cases[i], left = 35 + (i%2)*1080, top = 132 + Math.floor(i/2)*475;
    captions += '<rect x="' + (left-12) + '" y="' + (top-8) + '" width="1055" height="457" rx="9" fill="none" stroke="' + theme.grid + '" stroke-width="2"/>';
    captions += text(left+510, top+25, c.image_id.split('/')[1], 27, 'font-weight="700"');
    for (let j = 0; j < 3; j++) {
      const filename = ['original.png', 'heatmap.png', 'overlay.png'][j];
      const relative = 'outputs_modes_v3_td/' + c.sample_directory + '/' + filename;
      sources.push(relative);
      const input = path.join(root, relative);
      const resized = await sharp(input).resize(320, 320, {fit:'contain', background:'white'}).png().toBuffer();
      composites.push({input: resized, left: left+j*342, top: top+78});
      captions += text(left+j*342+160, top+61, ['Original', 'Grad-CAM', 'Overlay'][j], 25);
    }
    const correct = c.true_label === c.predicted_label;
    const label = 'จริง: ' + short[c.true_label] + ' → ทำนาย: ' + short[c.predicted_label] + ' | ' + (Number(c.confidence)*100).toFixed(2) + ' ' + (correct?'✓':'✗');
    captions += text(left+510, top+435, label, 29);
  }
  captions += '</svg>';
  composites.push({input: Buffer.from(captions), left:0, top:0});
  const figure = 'report/report_assets/latest/gradcam_resnet50_full_seed42_readable.png';
  await sharp({create:{width,height,channels:3,background:'white'}}).composite(composites).png().toFile(path.join(root, figure));
  if (hash('report/report_assets/latest/gradcam_resnet50_full_seed42.png') !== originalHash) throw Error('Original CAM grid changed');
  const inputs = Object.fromEntries([...new Set(sources)].map(p => [p, hash(p)]));
  fs.writeFileSync(path.join(__dirname, 'followup_figure_sources.json'), JSON.stringify({
    generated_utc: new Date().toISOString(), source_sha256: inputs,
    figures: [ablationFigure, figure], output_sha256: {[ablationFigure]:hash(ablationFigure), [figure]:hash(figure)},
    note: 'Presentation only: plotted saved run metrics and resized existing original/heatmap/overlay PNGs. No heatmap recomputation, inference, retuning, or training. Original eight-row grid preserved.'
  }, null, 2) + '\n');
  console.log('Generated ablation SVG and re-laid out eight saved CAM cases.');
}
camGrid().catch(error => { console.error(error); process.exitCode = 1; });


In [ ]:
%%project_source report/generate_latest_figures.js
/** Rebuild README figures from the completed outputs_modes_v3_td experiment.
 * Run: node report/generate_latest_figures.js
 * Only reads existing audit, tuning, final, and confidence records; no model code runs.
 */
const fs = require('fs');
const path = require('path');
const crypto = require('crypto');
const theme = require('./figure_theme');
const root = path.resolve(__dirname, '..');
const out = path.join(root, 'outputs_modes_v3_td');
const assets = path.join(__dirname, 'report_assets', 'latest');
fs.mkdirSync(assets, {recursive: true});
const archs = ['resnet50','efficientnet_b0','mobilenet_v3_large','densenet121','convnext_tiny'];
const names = {resnet50:'ResNet50',efficientnet_b0:'EfficientNet-B0',mobilenet_v3_large:'MobileNetV3-Large',densenet121:'DenseNet121',convnext_tiny:'ConvNeXt-Tiny'};
const modes = ['frozen','partial','full'];
const colors = theme.modeColors;
const symbols = theme.symbols;
const classes = ['san_phraphrom','san_phraphum','san_tayay','tee_ju_eia'];
const classNames = ['San Phra Phrom','San Phra Phum','San Ta Yai','Tee Ju Eia'];
const shortClassNames = ['พระพรหม','พระภูมิ','ตายาย','ตี่จู้เอี๊ยะ'];
const classColors = theme.classColors;
const source = {};
read('report/figure_theme.js');
const winners = JSON.parse(read('outputs_modes_v3_td/tuning/best_params_by_arch_mode.json')).pairs;
function esc(v){return String(v).replaceAll('&','&amp;').replaceAll('<','&lt;').replaceAll('>','&gt;').replaceAll('"','&quot;');}
function csv(s){let rows=[],row=[],field='',q=false;for(let i=0;i<s.length;i++){const c=s[i];if(q){if(c==='"'&&s[i+1]==='"'){field+='"';i++;}else if(c==='"')q=false;else field+=c;}else if(c==='"')q=true;else if(c===','){row.push(field);field='';}else if(c==='\n'){row.push(field.replace(/\r$/,''));rows.push(row);row=[];field='';}else field+=c;}if(field||row.length){row.push(field.replace(/\r$/,''));rows.push(row);}const h=rows.shift();return rows.filter(r=>r.length===h.length).map(r=>Object.fromEntries(h.map((k,i)=>[k,r[i]])));}
function read(rel){const p=path.join(root,rel);const b=fs.readFileSync(p);source[rel]=crypto.createHash('sha256').update(b).digest('hex');return b.toString('utf8');}
function readCsv(rel){return csv(read(rel).replace(/^\uFEFF/,''));}
function save(name,svg){fs.writeFileSync(path.join(assets,name),svg+'\n');}
function fmt(n,d=1){return Number(n).toFixed(d);}
function mean(a){return a.reduce((x,y)=>x+y,0)/a.length;}
function sd(a){const m=mean(a);return Math.sqrt(a.reduce((x,y)=>x+(y-m)**2,0)/(a.length-1));}
function quant(a,p){const b=[...a].sort((x,y)=>x-y),i=(b.length-1)*p,j=Math.floor(i);return b[j]+(b[Math.ceil(i)]-b[j])*(i-j);}
function svgStart(w,h,title){return `<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 ${w} ${h}" role="img" aria-label="${esc(title)}"><rect width="${w}" height="${h}" fill="white"/><style>${theme.styles}</style><text x="${w/2}" y="34" class="title" text-anchor="middle">${esc(title)}</text>`;}
function line(x1,y1,x2,y2,cls='axis',extra=''){const stroke=extra.includes('stroke="')?'':`stroke="${theme.muted}"`;return `<line x1="${x1}" y1="${y1}" x2="${x2}" y2="${y2}" class="${cls}" ${stroke} ${extra}/>`;}
function txt(x,y,t,cls='label',anchor='middle',extra=''){return `<text x="${x}" y="${y}" class="${cls}" text-anchor="${anchor}" ${extra}>${esc(t)}</text>`;}
function rect(x,y,w,h,fill,extra=''){return `<rect x="${x}" y="${y}" width="${w}" height="${h}" fill="${fill}" ${extra}/>`;}
function marker(x,y,mode,r=4,opacity=.75){const c=colors[mode];if(symbols[mode]==='square')return rect(x-r,y-r,2*r,2*r,c,`opacity="${opacity}"`);if(symbols[mode]==='diamond')return `<path d="M ${x} ${y-r*1.2} L ${x+r*1.2} ${y} L ${x} ${y+r*1.2} L ${x-r*1.2} ${y} Z" fill="${c}" opacity="${opacity}"/>`;return `<circle cx="${x}" cy="${y}" r="${r}" fill="${c}" opacity="${opacity}"/>`;}
function legend(x,y){let s='';modes.forEach((m,i)=>{const px=x+i*130;s+=marker(px,y-4,m,5,1)+txt(px+12,y,m[0].toUpperCase()+m.slice(1),'label','start');});return s;}
function box(x,values,map,color,width=23,mode=null){const lo=Math.min(...values),hi=Math.max(...values),q1=quant(values,.25),q2=quant(values,.5),q3=quant(values,.75);let s=line(x,map(lo),x,map(hi),'axis',`stroke="${color}"`)+line(x-8,map(lo),x+8,map(lo),'axis',`stroke="${color}"`)+line(x-8,map(hi),x+8,map(hi),'axis',`stroke="${color}"`);s+=rect(x-width/2,map(q3),width,Math.max(1,map(q1)-map(q3)),color,'opacity="0.55" stroke="#35434a"')+line(x-width/2,map(q2),x+width/2,map(q2),'axis','stroke="#263238" stroke-width="2"');if(values.length<=10)values.forEach((v,i)=>{const px=x+(i-(values.length-1)/2)*3.4;s+=mode?marker(px,map(v),mode,2.6,1):`<circle cx="${px}" cy="${map(v)}" r="2.5" fill="${color}" stroke="white" stroke-width=".5"/>`;});return s;}
function axes(x,y,w,h,min,max,ticks,label){let s=line(x,y+h,x+w,y+h)+line(x,y,x,y+h);ticks.forEach(v=>{const py=y+h-(v-min)/(max-min)*h;s+=line(x,py,x+w,py,'grid')+txt(x-8,py+4,fmt(v,Number.isInteger(v)?0:1),'small','end');});s+=txt(x+w/2,y+h+39,label,'label');return s;}

const audit = readCsv('outputs_modes_v3_td/dataset_audit/image_properties.csv');
if(audit.length!==560)throw Error(`Expected 560 audited images, got ${audit.length}`);
const finals=readCsv('outputs_modes_v3_td/final/run_metrics.csv').filter(r=>r.status==='completed');
if(finals.length!==75)throw Error(`Expected 75 completed final runs, got ${finals.length}`);
const trials=readCsv('outputs_modes_v3_td/tuning/all_trials.csv').filter(r=>r.status==='completed');
if(trials.length!==75)throw Error(`Expected 75 completed tuning trials, got ${trials.length}`);
const pairs=new Map();for(const r of finals){const k=`${r.architecture}/${r.mode}`;if(!pairs.has(k))pairs.set(k,[]);pairs.get(k).push(r);}for(const [k,v] of pairs)if(v.length!==5||new Set(v.map(r=>r.seed)).size!==5)throw Error(`Incomplete seed set for ${k}`);

// Dataset charts use the audited images, never the example screenshot counts.
{
 const counts=classes.map(c=>audit.filter(r=>r.label===c).length);let s=svgStart(920,430,'Cleaned dataset: images per class')+axes(82,77,790,285,0,220,[0,50,100,150,200],'Class');
 counts.forEach((v,i)=>{const x=110+i*196;s+=rect(x,77+285-v/220*285,123,v/220*285,classColors[i])+txt(x+61.5,72+285-v/220*285,v,'sub')+txt(x+61.5,388,classNames[i],'label');});save('class_distribution.svg',s+'</svg>');
}
{
 let s=svgStart(1200,510,'Image dimensions and aspect ratio by class');const left={x:80,y:75,w:510,h:360},right={x:690,y:75,w:450,h:360};
 s+=txt(330,67,'Original width vs. height (px)','sub')+txt(915,67,'Width / height (count)','sub');s+=axes(left.x,left.y,left.w,left.h,0,4500,[0,1000,2000,3000,4000],'Width (px)');[0,1000,2000,3000,4000].forEach(v=>s+=txt(left.x+v/4500*left.w,left.y+left.h+17,v,'small'));s+=line(left.x,left.y+left.h,left.x+left.w,left.y,'grid','stroke-dasharray="6 5"');
 audit.forEach(r=>{const i=classes.indexOf(r.label),x=left.x+Number(r.width)/4500*left.w,y=left.y+left.h-Number(r.height)/4500*left.h;s+=`<circle cx="${x}" cy="${y}" r="2.7" fill="${classColors[i]}" opacity=".58"/>`;});
 s+=txt(23,260,'Height (px)','label','middle','transform="rotate(-90 23 260)"');s+=axes(right.x,right.y,right.w,right.h,0,100,[0,20,40,60,80,100],'Aspect ratio (width / height)');
 const bins=Array.from({length:18},()=>[0,0,0,0]);audit.forEach(r=>{const v=Number(r.width)/Number(r.height),i=Math.min(17,Math.max(0,Math.floor((v-.3)/.1)));bins[i][classes.indexOf(r.label)]++;});
 bins.forEach((b,i)=>b.forEach((n,c)=>{const bx=right.x+i*(right.w/18)+c*5.5;s+=rect(bx,right.y+right.h-n/100*right.h,5,n/100*right.h,classColors[c],`opacity=".8"`);}));
 s+=txt(right.x,453,'0.3','small')+txt(right.x+right.w*.39,453,'1.0','small')+txt(right.x+right.w,453,'2.1+','small');classes.forEach((_,i)=>{s+=rect(80+i*235,482,12,12,classColors[i])+txt(98+i*235,493,classNames[i],'small','start');});save('image_dimensions.svg',s+'</svg>');
}
{
 let s=svgStart(1180,430,'Decoded colour modes in the latest audit'),ms=['RGB','RGBA','P','CMYK'];const total=ms.map(m=>audit.filter(r=>r.mode===m).length);s+=txt(274,67,'Overall count','sub')+txt(825,67,'Count by class','sub');s+=axes(70,83,460,275,0,560,[0,100,200,300,400,500],'Colour mode');
 ms.forEach((m,i)=>{const h=total[i]/560*275,x=95+i*107;s+=rect(x,358-h,67,h,['#374151','#6B7280','#9CA3AF','#D1D5DB'][i])+txt(x+33,351-h,total[i],'small')+txt(x+33,382,m,'label');});
 s+=axes(640,83,490,275,0,210,[0,50,100,150,200],'Class');classes.forEach((c,i)=>{ms.forEach((m,j)=>{const n=audit.filter(r=>r.label===c&&r.mode===m).length,h=n/210*275;s+=rect(660+i*119+j*19,358-h,17,h,['#374151','#6B7280','#9CA3AF','#D1D5DB'][j]);});s+=txt(690+i*119,380,classNames[i].replace('San ','').replace('Tee Ju ','TJ '),'small');});
 ms.forEach((m,i)=>{s+=rect(720+i*92,411,11,11,['#374151','#6B7280','#9CA3AF','#D1D5DB'][i])+txt(736+i*92,421,m,'small','start');});save('colour_modes.svg',s+'</svg>');
}

// Search response: saved nested trial configs, with exactly the two tuned numeric variables.
{
 let s=svgStart(1220,545,'Tuning trials: validation Macro F1 (%)');const panels=[{key:'learning_rate',label:'Learning rate (log scale)',min:1e-5,max:1e-3,x:80},{key:'weight_decay',label:'Weight decay (log scale)',min:1e-6,max:1e-2,x:665}];
 for(const p of panels){s+=txt(p.x+245,78,p.label,'sub')+axes(p.x,95,500,325,40,100,[40,50,60,70,80,90,100],p.label);for(let power=Math.log10(p.min);power<=Math.log10(p.max);power++)s+=txt(p.x+(power-Math.log10(p.min))/(Math.log10(p.max)-Math.log10(p.min))*500,440,`10^${power}`,'small');for(const r of trials){const hit=r.config.match(new RegExp("'"+p.key+"': ([0-9.eE+-]+)"));if(!hit)throw Error(`Missing ${p.key} in ${r.identity}`);const v=Number(hit[1]),f=Number(r.best_val_macro_f1)*100;if(!Number.isFinite(v)||!Number.isFinite(f))throw Error('Non-finite trial');const x=p.x+(Math.log10(v)-Math.log10(p.min))/(Math.log10(p.max)-Math.log10(p.min))*500,y=420-(f-40)/60*325;s+=marker(x,y,r.mode,4.2,.62);if(Number(r.trial)===Number(winners[r.architecture+"::"+r.mode]?.winning_trial))s+="<circle cx=\""+x+"\" cy=\""+y+"\" r=\"7\" fill=\"none\" stroke=\""+theme.ink+"\" stroke-width=\"1.3\"/>";}}s+=legend(405,515);save('tuning_response.svg',s+'</svg>');
}

// Four aligned panels: each dot is one of five final seeds, bars are mean ± sample SD.
{
 const W=1300,H=950;let s=svgStart(W,H,'Independently tuned modes: five matched final seeds')+legend(450,66);const metrics=[['best_val_acc','Validation accuracy (%)'],['test_acc','Test accuracy (%)'],['best_val_macro_f1','Validation Macro F1 (%)'],['test_macro_f1','Test Macro F1 (%)']];
 metrics.forEach(([key,label],pi)=>{const col=pi%2,row=Math.floor(pi/2),x=68+col*630,y=125+row*425,w=550,h=292;s+=txt(x-46,y+h/2,label,'small','middle','transform="rotate(-90 '+(x-46)+' '+(y+h/2)+')"')+txt(x+w/2,y-20,label,'sub')+axes(x,y,w,h,75,100,[75,80,85,90,95,100],'Architecture');archs.forEach((a,ai)=>{modes.forEach((m,mi)=>{const v=pairs.get(`${a}/${m}`).map(r=>Number(r[key])*100),cx=x+55+ai*105+(mi-1)*22,mu=mean(v),sigma=sd(v),fy=t=>y+h-(t-75)/25*h;s+=line(cx,fy(mu-sigma),cx,fy(mu+sigma),'axis',`stroke="${colors[m]}" stroke-width="2"`)+line(cx-6,fy(mu-sigma),cx+6,fy(mu-sigma),'axis',`stroke="${colors[m]}"`)+line(cx-6,fy(mu+sigma),cx+6,fy(mu+sigma),'axis',`stroke="${colors[m]}"`);v.forEach((z,i)=>s+=marker(cx+(i-2)*1.8,fy(z),m,2.7,.42));s+=marker(cx,fy(mu),m,5.5,1);});s+=txt(x+55+ai*105,y+h+18,names[a].replace('EfficientNet-','EffNet-').replace('MobileNetV3-','MobV3-').replace('ConvNeXt-','ConvNeXt-'),'small');});});save('mode_comparison.svg',s+'</svg>');
}

// Checkpoint box plots replace crowded temporal aggregates in README.
for(const a of archs){let s=svgStart(970,480,`${names[a]}: selected-checkpoint distributions (n = 5 seeds)`)+legend(286,69);const panels=[{key:'best_val_macro_f1',title:'Validation Macro F1 (%)',min:75,max:100,ticks:[75,80,85,90,95,100],scale:100},{key:'best_epoch',title:'Best epoch',min:0,max:30,ticks:[0,5,10,15,20,25,30],scale:1}];panels.forEach((p,pi)=>{const x=75+pi*470,y=115,w=395,h=265;s+=txt(x-48,y+h/2,p.title,'small','middle','transform="rotate(-90 '+(x-48)+' '+(y+h/2)+')"')+txt(x+w/2,y-16,p.title,'sub')+axes(x,y,w,h,p.min,p.max,p.ticks,'Fine-tuning mode');modes.forEach((m,i)=>{const v=pairs.get(`${a}/${m}`).map(r=>Number(r[p.key])*p.scale),cx=x+92+i*108,fy=z=>y+h-(z-p.min)/(p.max-p.min)*h;s+=box(cx,v,fy,colors[m],34,m)+txt(cx,y+h+19,m[0].toUpperCase()+m.slice(1),'small');});});s+=txt(485,462,'Box: IQR and median · whiskers: observed min/max · dots: individual seeds','small');save(`checkpoint_boxes_${a}.svg`,s+'</svg>');}

// Per-seed row-normalized matrices: mean in %, SD in percentage points.
const confusionRows=[];
for(const a of archs)for(const m of modes){const runs=pairs.get(`${a}/${m}`),mats=[];for(const r of runs){const rel=path.relative(root,path.join(r.run_dir,'test_predictions.csv')).replaceAll('\\','/'),pred=readCsv(rel);if(pred.length!==85)throw Error(`Expected 85 predictions: ${rel}`);const mat=classes.map(()=>classes.map(()=>0)),support=classes.map(()=>0);for(const q of pred){const i=classes.indexOf(q.true_label),j=classes.indexOf(q.predicted_label);if(i<0||j<0)throw Error('Unknown class in predictions');mat[i][j]++;support[i]++;}mats.push(mat.map((row,i)=>row.map(n=>n/support[i]*100)));}
 let s=svgStart(880,640,`${names[a]} / ${m[0].toUpperCase()+m.slice(1)}: test confusion (%)`);s+=txt(440,72,'Cell values: mean (%) ± sample SD (pp), n = 5 seeds','sub');const x0=270,y0=140,cell=109;for(let i=0;i<4;i++){s+=txt(x0+i*cell+cell/2,126,shortClassNames[i],'label');s+=txt(x0-12,y0+i*cell+cell/2+4,shortClassNames[i],'label','end');for(let j=0;j<4;j++){const vals=mats.map(mat=>mat[i][j]),mu=mean(vals),std=sd(vals),fill=theme.heatColor(mu),xx=x0+j*cell,yy=y0+i*cell;confusionRows.push({architecture:a,mode:m,true_class:classes[i],predicted_class:classes[j],n_seeds:5,mean_pct:fmt(mu,4),sd_pp:fmt(std,4)});s+=rect(xx,yy,cell-3,cell-3,fill,'stroke="#e0e6ea"')+txt(xx+cell/2,yy+49,fmt(mu,1),'sub','middle',mu>55?'style="fill:white"':'')+txt(xx+cell/2,yy+69,`± ${fmt(std,1)}`,'small','middle',mu>55?'style="fill:white"':'');}}
 s+=txt(490,110,'Predicted class','label')+txt(70,355,'True class','label','middle','transform="rotate(-90 70 355)"');s+=txt(490,608,'Rows: true class · columns: predicted class · fixed 0–100% colour scale','small');save(`confusion_${a}_${m}.svg`,s+'</svg>');}
fs.writeFileSync(path.join(__dirname,'confusion_mean_sd.csv'),['architecture,mode,true_class,predicted_class,n_seeds,mean_pct,sd_pp',...confusionRows.map(r=>Object.values(r).join(','))].join('\n')+'\n');

// Reference calibration from the saved 10-bin diagnostics, not a refit.
{
 const bins=readCsv('outputs_modes_v3_td/confidence/bin_metrics.csv').filter(r=>r.architecture==='mobilenet_v3_large'&&r.mode==='partial'&&r.seed==='42');if(bins.length!==10)throw Error(`Missing reference calibration bins: found ${bins.length}`);let s=svgStart(800,570,'Reference calibration on test: seed 42 (%)');const x=90,y=90,w=580,h=370;s+=axes(x,y,w,h,0,100,[0,20,40,60,80,100],'Mean confidence (%)')+txt(28,275,'Observed accuracy (%)','label','middle','transform="rotate(-90 28 275)"')+line(x,y+h,x+w,y,'grid','stroke-dasharray="7 5"');[0,20,40,60,80,100].forEach(v=>s+=txt(x+v/100*w,y+h+18,v,'small'));bins.forEach(b=>{if(Number(b.count)===0)return;const xx=x+Number(b.mean_confidence)*w,yy=y+h-Number(b.accuracy)*h;s+=`<circle cx="${xx}" cy="${yy}" r="${Math.max(5,Math.sqrt(Number(b.count))*2.6)}" fill="${colors.partial}" opacity=".65" stroke="#35434a"/>`+txt(xx,yy-13,b.count,'small');});s+=txt(400,532,'Dot area scales with bin count; label = images in bin (total test n = 85)','small');save('reference_calibration.svg',s+'</svg>');
}

async function brightness(){const values=readCsv('report/derived_brightness.csv').map(r=>({image_id:r.image_id,label:r.label,luminance:Number(r.mean_luminance_0_255)}));if(values.length!==560||new Set(values.map(r=>r.image_id)).size!==560||values.some(r=>!Number.isFinite(r.luminance)||!audit.some(a=>a.image_id===r.image_id&&a.label===r.label)))throw Error('Invalid saved brightness values');let s=svgStart(1150,500,'Original-image mean luminance by class (0–255)');s+=txt(290,74,'Distribution (560 decoded originals)','sub')+txt(865,74,'Image count by luminance bin','sub');const x=72,y=98,w=480,h=310;s+=axes(x,y,w,h,0,255,[0,50,100,150,200,250],'Class');classes.forEach((c,i)=>{const v=values.filter(z=>z.label===c).map(z=>z.luminance),cx=x+62+i*116,fy=t=>y+h-t/255*h;s+=box(cx,v,fy,classColors[i],40)+txt(cx,y+h+18,classNames[i].replace('San ','').replace('Tee Ju ','TJ '),'small');});const bx=640,bw=460,bins=Array.from({length:20},()=>[0,0,0,0]);values.forEach(v=>{const b=Math.min(19,Math.floor(v.luminance/256*20));bins[b][classes.indexOf(v.label)]++;});s+=axes(bx,y,bw,310,0,65,[0,15,30,45,60],'Mean luminance (0–255)');bins.forEach((row,b)=>row.forEach((n,i)=>{const xx=bx+b*bw/20+i*5.4,h=n/65*310;s+=rect(xx,y+310-h,5.2,h,classColors[i],'opacity=".65"');}));s+=txt(bx,426,'0','small')+txt(bx+bw/2,426,'128','small')+txt(bx+bw,426,'255','small');save('brightness.svg',s+'</svg>');}

brightness().then(async()=>{await require('./generate_learning_report').generate();fs.writeFileSync(path.join(__dirname,'latest_figure_sources.json'),JSON.stringify({generated_utc:new Date().toISOString(),protocol:'outputs_modes_v3_td',source_sha256:source,notes:'All model charts use 75 latest tuning and 75 latest final records. Brightness uses the saved derived_brightness.csv values from 560 originals (EXIF-aware decode, Rec.709); class colours are separate from mode colours.'},null,2)+'\n');console.log('Generated latest figures from 560 audited images, 75 tuning trials, 75 final runs.');}).catch(e=>{console.error(e);process.exitCode=1;});


In [ ]:
%%project_source report/generate_learning_report.js
// Enhanced report figures, strictly from saved audit / history / configurations.
// Run directly, or automatically after generate_latest_figures.js.
const fs = require('fs');
const path = require('path');
const {root, parseCsv, hash, mean, sd} = require('./report_utils');
const T = require('./figure_theme');
const A = ['resnet50','efficientnet_b0','mobilenet_v3_large','densenet121','convnext_tiny'];
const N = ['ResNet50','EfficientNet-B0','MobileNetV3-Large','DenseNet121','ConvNeXt-Tiny'];
const M = ['full','frozen','partial'];
const C = ['san_phraphrom','san_phraphum','san_tayay','tee_ju_eia'];
const CN = ['San Phra Phrom','San Phra Phum','San Ta Yai','Tee Ju Eia'];
const dash = ['', '9 5', '2 5', '10 4 2 4'];
const destination = 'report/report_assets/latest/';
const sources = {}, figures = [], histograms = [];
function read(p) { sources[p]=hash(p); return fs.readFileSync(path.join(root,p),'utf8'); }
function csv(p) { return parseCsv(read(p)); }
function writeCsv(p,rows) {
  if(!rows.length)throw Error('No rows: '+p);
  const fields=Object.keys(rows[0]), cell=v=>'"'+String(v??'').replaceAll('"','""')+'"';
  fs.writeFileSync(path.join(root,p),[fields.map(cell).join(','),...rows.map(r=>fields.map(f=>cell(r[f])).join(','))].join('\n')+'\n');
}
const f=(v,d=1)=>Number(v).toFixed(d);
const text=(x,y,v,size=16,extra='')=>{const anchor=extra.match(/text-anchor="([^"]+)"/);return '<text x="'+x+'" y="'+y+'" text-anchor="'+(anchor?anchor[1]:'middle')+'" font-size="'+size+'" '+extra.replace(/text-anchor="[^"]+"/,'')+'>'+T.esc(v)+'</text>';};
const line=(x,y,xx,yy,color=T.muted,extra='')=>'<line x1="'+x+'" y1="'+y+'" x2="'+xx+'" y2="'+yy+'" stroke="'+color+'" '+extra+'/>';
function frame(w,h,title,subtitle='') { return '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 '+w+' '+h+'" role="img" aria-label="'+T.esc(title)+'"><rect width="'+w+'" height="'+h+'" fill="white"/><style>text{font-family:'+T.fontFamily+';fill:'+T.ink+'}</style>'+text(w/2,34,title,25,'font-weight="700"')+(subtitle?text(w/2,67,subtitle,16):''); }
function save(name,s) { const p=destination+name; fs.writeFileSync(path.join(root,p),s+'</svg>\n'); figures.push(p); }
function axes(b,xmin,xmax,ymin,ymax,xt,yt,xlabel,ylabel) {
  const x=v=>b.x+(v-xmin)/(xmax-xmin)*b.w, y=v=>b.y+b.h-(v-ymin)/(ymax-ymin)*b.h;
  let s='';for(const v of yt)s+=line(b.x,y(v),b.x+b.w,y(v),T.grid)+text(b.x-13,y(v)+5,Number.isInteger(v)?v:f(v),14,'text-anchor="end"');
  for(const v of xt)s+=text(x(v),b.y+b.h+24,Number.isInteger(v)?v:f(v),14);
  s+=line(b.x,b.y,b.x,b.y+b.h)+line(b.x,b.y+b.h,b.x+b.w,b.y+b.h);
  s+=text(b.x+b.w/2,b.y+b.h+55,xlabel,17)+text(b.x-56,b.y+b.h/2,ylabel,16,'transform="rotate(-90 '+(b.x-56)+' '+(b.y+b.h/2)+')"');
  return {s,x,y};
}
function marker(x,y,m,r=4,opacity=1) {
  const color=T.modeColors[m];
  if(m==='partial')return '<rect x="'+(x-r)+'" y="'+(y-r)+'" width="'+(2*r)+'" height="'+(2*r)+'" fill="'+color+'" opacity="'+opacity+'"/>';
  if(m==='full')return '<path d="M '+x+' '+(y-r)+' l '+r+' '+r+' l '+(-r)+' '+r+' l '+(-r)+' '+(-r)+' Z" fill="'+color+'" opacity="'+opacity+'"/>';
  return '<circle cx="'+x+'" cy="'+y+'" r="'+r+'" fill="'+color+'" opacity="'+opacity+'"/>';
}
function classLegend(y,w=1440) {let s='';C.forEach((_,i)=>{const x=135+i*(w-100)/4;s+=line(x-50,y-5,x-12,y-5,T.classColors[i],'stroke-width="4" stroke-dasharray="'+dash[i]+'"')+text(x,y,CN[i],15,'text-anchor="start"');});return s;}
function proportions(values,lo,hi,step,label) {
  const bins=Array.from({length:Math.round((hi-lo)/step)},()=>[0,0,0,0]);
  for(const r of values){if(r.value<lo||r.value>hi)throw Error('Histogram would omit observation');const i=Math.min(bins.length-1,Math.floor((r.value-lo)/step+1e-10));bins[i][C.indexOf(r.label)]++;}
  const totals=C.map(c=>values.filter(r=>r.label===c).length);
  bins.forEach((counts,i)=>counts.forEach((n,c)=>histograms.push({figure:label,class:C[c],bin_left:lo+i*step,bin_right:lo+(i+1)*step,count:n,class_total:totals[c],proportion_pct:n/totals[c]*100})));
  return {bins,totals};
}
function histogram(b,values,lo,hi,step,xlabel,label,meanLines=false) {
  const {bins,totals}=proportions(values,lo,hi,step,label);
  const max=Math.max(...bins.flatMap(row=>row.map((n,c)=>n/totals[c]*100))), ymax=Math.ceil(max/10)*10;
  const xt=[lo,...[0.5,1,1.5,2].filter(v=>v>lo&&v<hi),hi];
  const ticks=xlabel.includes('brightness')?[0,64,128,192,255]:xt;
  const p=axes(b,lo,hi,0,ymax,ticks,Array.from({length:ymax/10+1},(_,i)=>i*10),xlabel,'Proportion of images (%)');let s=p.s;
  for(let c=0;c<4;c++){
    const points=[];for(let i=0;i<bins.length;i++){const y=p.y(bins[i][c]/totals[c]*100);points.push([p.x(lo+i*step),y],[p.x(lo+(i+1)*step),y]);}
    const d='M '+p.x(lo)+' '+p.y(0)+' L '+points.map(v=>v.join(' ')).join(' L ')+' L '+p.x(hi)+' '+p.y(0)+' Z';
    s+='<path d="'+d+'" fill="'+T.classColors[c]+'" fill-opacity=".25" stroke="'+T.classColors[c]+'" stroke-width="2.5" stroke-dasharray="'+dash[c]+'"/>';
    if(meanLines){const mu=mean(values.filter(r=>r.label===C[c]).map(r=>r.value));s+=line(p.x(mu),b.y,p.x(mu),b.y+b.h,T.classColors[c],'stroke-width="2" stroke-dasharray="'+(dash[c]||'4 3')+'"');}
  }
  if(!meanLines)s+=line(p.x(1),b.y,p.x(1),b.y+b.h,T.ink,'stroke-width="2" stroke-dasharray="7 5"')+text(p.x(1)+8,b.y+20,'Square (1:1)',14,'text-anchor="start"');
  return s;
}
function datasetFigures(audit) {
  let s=frame(1100,500,'Class distribution','560 original images · fixed class order');
  const p=axes({x:100,y:100,w:940,h:300},0,4,0,220,[],[0,50,100,150,200],'Class','Number of images');s+=p.s;
  C.forEach((c,i)=>{const n=audit.filter(r=>r.label===c).length,x=p.x(i+.5);s+='<rect x="'+(x-65)+'" y="'+p.y(n)+'" width="130" height="'+(p.y(0)-p.y(n))+'" fill="'+T.classColors[i]+'"/>'+text(x,p.y(n)-12,n,19)+text(x,425,CN[i],16);});save('class_distribution.svg',s);
  s=frame(1440,650,'Image dimensions and aspect ratio','Original decoded dimensions · histogram uses common bins and within-class percentages');
  const a=axes({x:100,y:120,w:550,h:400},0,4500,0,4500,[0,1000,2000,3000,4000],[0,1000,2000,3000,4000],'Width (px)','Height (px)');s+=a.s+line(a.x(0),a.y(0),a.x(4500),a.y(4500),T.muted,'stroke-dasharray="7 5"');
  audit.forEach(r=>{const c=C.indexOf(r.label),xx=a.x(Number(r.width)),yy=a.y(Number(r.height));s+=['<circle cx="'+xx+'" cy="'+yy+'" r="3"/>','<rect x="'+(xx-3)+'" y="'+(yy-3)+'" width="6" height="6"/>','<path d="M '+xx+' '+(yy-4)+' l 4 7 h -8 Z"/>','<path d="M '+(xx-3)+' '+(yy-3)+' l 6 6 M '+(xx+3)+' '+(yy-3)+' l -6 6"/>'][c].replace('/>',' fill="'+T.classColors[c]+'" stroke="'+T.classColors[c]+'" opacity=".5"/>');});
  const vals=audit.map(r=>({label:r.label,value:Number(r.width)/Number(r.height)}));const lo=Math.floor(Math.min(...vals.map(r=>r.value))*10)/10,hi=Math.ceil(Math.max(...vals.map(r=>r.value))*10)/10;
  s+=histogram({x:810,y:120,w:570,h:400},vals,lo,hi,.1,'Aspect ratio (width / height)','aspect_ratio')+classLegend(620);save('image_dimensions.svg',s);
  s=frame(1330,620,'Decoded colour modes','Direct counts and percentages; non-RGB modes shown separately at an expanded scale');
  const modes=['RGB','RGBA','P','CMYK'], labels=['RGB','RGBA','Palette','CMYK'],patterns=['','////','....','xxxx'];
  s+='<defs>'+modes.map((m,i)=>'<pattern id="pattern'+i+'" width="8" height="8" patternUnits="userSpaceOnUse"><rect width="8" height="8" fill="'+['#374151','#D1D5DB','#F3F4F6','#9CA3AF'][i]+'"/>'+(['','<path d="M 0 8 L 8 0" stroke="#374151"/>','<circle cx="4" cy="4" r="1.5" fill="#374151"/>','<path d="M 0 0 L 8 8 M 8 0 L 0 8" stroke="#374151"/>'][i])+'</pattern>').join('')+'</defs>';
  for(let panel=0;panel<2;panel++){const b={x:130+panel*650,y:120,w:440,h:340},max=panel?10:560,series=panel?[1,2,3]:[0,1,2,3];s+=text(b.x+b.w/2,100,panel?'Non-RGB detail':'All images',19,'font-weight="700"');
    s+=line(b.x,b.y,b.x,b.y+b.h)+line(b.x,b.y+b.h,b.x+b.w,b.y+b.h)+text(b.x+b.w/2,520,'Number of images',17)+text(b.x-100,b.y+b.h/2,'Colour mode',17,'transform="rotate(-90 '+(b.x-100)+' '+(b.y+b.h/2)+')"');
    for(const t of (panel?[0,2,4,6,8,10]:[0,100,200,300,400,500]))s+=line(b.x+t/max*b.w,b.y,b.x+t/max*b.w,b.y+b.h,T.grid)+text(b.x+t/max*b.w,485,t,14);
    series.forEach((idx,k)=>{const count=audit.filter(r=>r.mode===modes[idx]).length,yy=b.y+30+k*78,ww=count/max*b.w;s+='<rect x="'+b.x+'" y="'+yy+'" width="'+ww+'" height="42" fill="url(#pattern'+idx+')" stroke="#374151"/>'+text(b.x-12,yy+27,labels[idx],16,'text-anchor="end"')+text(b.x+ww+10,yy+27,count+' ('+f(count/560*100,2)+')',16,'text-anchor="start"');});}
  s+=text(665,579,'Labels: image count (percentage of all 560 images); patterns distinguish colour modes without hue',16);save('colour_modes.svg',s);
  const v=csv('report/derived_brightness.csv').map(r=>({label:r.label,value:Number(r.mean_luminance_0_255)}));
  if(v.length!==560)throw Error('Brightness evidence incomplete');
  s=frame(1440,680,'Image brightness by class','One value per original image · approximate RGB-weighted brightness (0–255)');
  const bp=axes({x:110,y:120,w:540,h:410},0,4,0,255,[],[0,50,100,150,200,250],'Class','Mean image brightness (0–255)');s+=bp.s;
  const q=(a,p)=>{const b=[...a].sort((x,y)=>x-y),k=(b.length-1)*p,j=Math.floor(k);return b[j]+(b[Math.ceil(k)]-b[j])*(k-j);};
  C.forEach((c,i)=>{const vs=v.filter(r=>r.label===c).map(r=>r.value),cx=bp.x(i+.5),lo=Math.min(...vs),hi=Math.max(...vs),q1=q(vs,.25),q2=q(vs,.5),q3=q(vs,.75),mu=mean(vs);s+=line(cx,bp.y(lo),cx,bp.y(hi),T.classColors[i],'stroke-width="2"')+line(cx-12,bp.y(lo),cx+12,bp.y(lo))+line(cx-12,bp.y(hi),cx+12,bp.y(hi))+'<rect x="'+(cx-38)+'" y="'+bp.y(q3)+'" width="76" height="'+(bp.y(q1)-bp.y(q3))+'" fill="'+T.classColors[i]+'" fill-opacity=".35" stroke="'+T.ink+'"/>'+line(cx-38,bp.y(q2),cx+38,bp.y(q2),T.ink,'stroke-width="3"')+line(cx-7,bp.y(mu)-7,cx+7,bp.y(mu)+7,T.ink,'stroke-width="3"')+line(cx-7,bp.y(mu)+7,cx+7,bp.y(mu)-7,T.ink,'stroke-width="3"')+text(cx,556,CN[i].replace('San ',''),15);});
  const globalMean=mean(v.map(r=>r.value));s+=line(110,bp.y(globalMean),650,bp.y(globalMean),T.muted,'stroke-dasharray="8 5" stroke-width="2"');
  s+=histogram({x:810,y:120,w:570,h:410},v,0,255,12.75,'Mean image brightness (0–255)','brightness',true)+classLegend(623);
  s+=text(720,660,'Box: IQR · thick line: median · ×: class mean · horizontal dashed line: overall mean · histogram vertical lines: class means',15);save('brightness.svg',s);
  writeCsv('report/image_property_histograms.csv',histograms);
  const colorRows=C.flatMap(c=>modes.map(m=>({class:c,colour_mode:m,count:audit.filter(r=>r.label===c&&r.mode===m).length})));writeCsv('report/colour_mode_counts.csv',colorRows);
}
function loadHistories(registry) {
  const records=[];
  for(const stage of ['trials','final_runs'])for(const r of registry[stage].filter(r=>r.status==='completed')){
    const directory=path.relative(root,r.run_dir).replaceAll('\\','/');
    if(!directory.startsWith('outputs_modes_v3_td/'))throw Error('Historical result mixing');
    const historyFile=directory+'/history.csv';const rows=csv(historyFile).filter(h=>h.train_loss!=='').map(h=>Object.fromEntries(Object.entries(h).filter(([k])=>['epoch','train_loss','val_loss','train_acc','val_acc','val_macro_f1'].includes(k)).map(([k,v])=>[k,Number(v)])));
    if(rows.some((h,i)=>h.epoch!==i+1||Object.values(h).some(v=>!Number.isFinite(v))))throw Error('Invalid history '+historyFile);
    const selected=rows.find(h=>h.epoch===Number(r.best_epoch));if(!selected||Math.abs(selected.val_macro_f1-Number(r.best_val_macro_f1))>1e-10)throw Error('Selected epoch mismatch');
    records.push({...r,stage,rows,selected,historyFile,history_sha256:sources[historyFile]});
  }
  if(records.filter(r=>r.stage==='trials').length!==75||records.filter(r=>r.stage==='final_runs').length!==75)throw Error('Expected 75+75 completed records');
  return records;
}
function tuningFigures(records,winners) {
  const trials=records.filter(r=>r.stage==='trials');
  const metrics=[['learning_rate','Learning rate (log scale)',true],['weight_decay','Weight decay (log scale)',true],['train_loss','Training loss at selected epoch',false],['val_loss','Validation loss at selected epoch',false],['epoch','Selected epoch',false],['train_acc','Training accuracy (%)',false],['val_acc','Validation accuracy (%)',false]];
  const rows=trials.map(r=>({architecture:r.architecture,mode:r.mode,trial:r.trial,seed:r.seed,identity:r.identity,learning_rate:r.config.learning_rate,weight_decay:r.config.weight_decay,...r.selected,winner:Number(winners[r.architecture+'::'+r.mode].winning_trial)===Number(r.trial),history_file:r.historyFile,history_sha256:r.history_sha256}));
  writeCsv('report/tuning_checkpoint_observations.csv',rows);
  let s=frame(1440,1540,'Hyperparameter search and selected-epoch diagnostics','75 trials · tuning seed 42 · rings mark 15 within-setup winners · no test scores used');
  metrics.forEach(([key,label,log],i)=>{const b={x:95+(i%2)*720,y:135+Math.floor(i/2)*345,w:570,h:225};
    const vals=rows.map(r=>Number(r[key])*(key.endsWith('_acc')?100:1)),min=log?(key==='learning_rate'?1e-5:1e-6):0,max=log?(key==='learning_rate'?1e-3:1e-2):key.endsWith('_acc')?100:key==='epoch'?30:Math.ceil(Math.max(...vals)*10)/10;
    const xl=log?Math.log10(min):min,xh=log?Math.log10(max):max,xt=log?[]:[0,max/2,max],p=axes(b,xl,xh,40,100,xt,[40,60,80,100],label,'Validation Macro F1 (%)');s+=p.s;
    if(log)for(let v=Math.log10(min);v<=Math.log10(max);v++)s+=text(p.x(v),b.y+b.h+24,'10^'+v,14);
    rows.forEach((r,j)=>{const xx=p.x(log?Math.log10(vals[j]):vals[j]),yy=p.y(r.val_macro_f1*100);s+=marker(xx,yy,r.mode,4,.65);if(r.winner)s+='<circle cx="'+xx+'" cy="'+yy+'" r="7" fill="none" stroke="'+T.ink+'" stroke-width="1.5"/>';});
  });
  let lx=850;for(const m of ['frozen','partial','full']){s+=marker(lx,1290,m,7)+text(lx+18,1296,m[0].toUpperCase()+m.slice(1),18,'text-anchor="start"');lx+=170;}
  s+=text(1080,1350,'Loss / accuracy values share the selected epoch',17)+text(1080,1382,'These are associations, not causal effects',17)+text(1080,1414,'15 setups = 5 architectures × 3 strategies',17);save('tuning_response.svg',s);
  s=frame(1440,610,'Tuning trials within each predeclared setup','5 configurations per setup · ring = selected recipe · points are configurations, not repeated seeds');
  const p=axes({x:90,y:120,w:1290,h:310},0,15,40,100,[],[40,50,60,70,80,90,100],'','Validation Macro F1 (%)');s+=p.s;
  A.forEach((a,ai)=>M.forEach((m,mi)=>{const x=ai*3+mi+.5;rows.filter(r=>r.architecture===a&&r.mode===m).forEach((r,j)=>{const xx=p.x(x+(j-2)*.045),yy=p.y(r.val_macro_f1*100);s+=marker(xx,yy,m,5,.6);if(r.winner)s+='<circle cx="'+xx+'" cy="'+yy+'" r="9" fill="none" stroke="'+T.ink+'" stroke-width="2"/>';});s+=text(p.x(x),454,m,14);if(mi===1)s+=text(p.x(x),485,N[ai].replace('MobileNetV3-','MobV3-').replace('EfficientNet-','EffNet-'),16);}));
  s+=text(720,535,'Architecture × fine-tuning strategy',17)+text(720,580,'Each setup selects its own recipe using validation Macro F1; none is screened out by Full-only results',16);save('tuning_setup_summary.svg',s);
}
function learningFigures(records) {
  const finals=records.filter(r=>r.stage==='final_runs'), aggregates=[],checkpoints=[];
  const keys=['train_loss','val_loss','train_acc','val_acc','val_macro_f1'];
  for(const a of A)for(const m of M){const runs=finals.filter(r=>r.architecture===a&&r.mode===m).sort((x,y)=>Number(x.seed)-Number(y.seed));
    if(runs.length!==5||runs.map(r=>Number(r.seed)).join(',')!=='42,123,456,789,1024')throw Error('Missing matched seeds');
    for(const r of runs)checkpoints.push({architecture:a,mode:m,seed:r.seed,run_id:r.identity,selected_epoch:r.best_epoch,stopped_epoch:r.rows.length,validation_macro_f1:r.selected.val_macro_f1,validation_accuracy:r.selected.val_acc,checkpoint:path.relative(root,r.checkpoint).replaceAll('\\','/'),history_file:r.historyFile,history_sha256:r.history_sha256});
    for(let e=1;e<=Math.max(...runs.map(r=>r.rows.length));e++){const available=runs.filter(r=>r.rows.some(h=>h.epoch===e));for(const key of keys){const values=available.map(r=>r.rows.find(h=>h.epoch===e)[key]);aggregates.push({architecture:a,mode:m,epoch:e,metric:key,n:values.length,seeds:available.map(r=>r.seed).join(';'),mean:mean(values),sample_sd:values.length>1?sd(values):'',unit:key.includes('loss')?'weighted_cross_entropy':'fraction'});}}
  }
  writeCsv('report/learning_curve_summary.csv',aggregates);writeCsv('report/final_checkpoint_summary.csv',checkpoints);
  writeCsv('report/mean_selected_epochs.csv', A.flatMap(a=>M.map(m=>({architecture:a,mode:m,n:5,mean_selected_epoch:mean(checkpoints.filter(r=>r.architecture===a&&r.mode===m).map(r=>Number(r.selected_epoch))),seed_ids:'42;123;456;789;1024'}))));
  const losses=aggregates.filter(r=>r.metric.includes('loss'));const lossMin=Math.min(0,...losses.filter(r=>r.sample_sd!=='').map(r=>r.mean-r.sample_sd));const lossMax=Math.max(...losses.map(r=>r.mean+(r.sample_sd===''?0:r.sample_sd)));
  const lossLo=Math.floor(lossMin*2)/2,lossHi=Math.ceil(lossMax*2)/2;
  const fractions=aggregates.filter(r=>!r.metric.includes('loss'));
  const pctLo=Math.floor(Math.min(0,...fractions.map(r=>(r.mean-(r.sample_sd===''?0:r.sample_sd))*100))/5)*5;
  const pctHi=Math.ceil(Math.max(100,...fractions.map(r=>(r.mean+(r.sample_sd===''?0:r.sample_sd))*100))/5)*5;
  A.forEach((a,ai)=>{
    let s=frame(1560,1450,N[ai]+': learning curves across five matched seeds','Solid: training · dashed: validation · shading: ±1 sample SD · ×: each run’s selected validation checkpoint');
    M.forEach((m,mi)=>{s+=marker(200+mi*510,99,m,7)+text(225+mi*510,105,m[0].toUpperCase()+m.slice(1),19,'text-anchor="start"');
      const rows=[{keys:['train_loss','val_loss'],label:'Weighted cross-entropy loss',lo:lossLo,hi:lossHi,scale:1},{keys:['train_acc','val_acc'],label:'Accuracy (%)',lo:pctLo,hi:pctHi,scale:100},{keys:['val_macro_f1'],label:'Validation Macro F1 (%)',lo:pctLo,hi:pctHi,scale:100},{keys:[],label:'Available runs',lo:0,hi:5,scale:1}];
      rows.forEach((spec,ri)=>{const b={x:90+mi*510,y:140+ri*315,w:400,h:205},p=axes(b,1,30,spec.lo,spec.hi,[1,5,10,15,20,25,30],ri===0?[spec.lo,(spec.lo+spec.hi)/2,spec.hi]:ri===3?[0,1,2,3,4,5]:[0,25,50,75,100],'Epoch',spec.label);s+=p.s;
        const get=key=>aggregates.filter(r=>r.architecture===a&&r.mode===m&&r.metric===key);
        if(ri===3){const data=get('val_macro_f1');let d='M '+p.x(1)+' '+p.y(data[0].n);for(let i=1;i<data.length;i++)d+=' H '+p.x(data[i].epoch)+' V '+p.y(data[i].n);const last=data[data.length-1].epoch;if(last<30)d+=' H '+p.x(last+1)+' V '+p.y(0)+' H '+p.x(30);s+='<path d="'+d+'" fill="none" stroke="'+T.modeColors[m]+'" stroke-width="3"/>';return;}
        spec.keys.forEach((key,k)=>{const data=get(key),band=data.filter(r=>r.n>=2);if(band.length){const upper=band.map(r=>[p.x(r.epoch),p.y((r.mean+r.sample_sd)*spec.scale)]),lower=band.map(r=>[p.x(r.epoch),p.y((r.mean-r.sample_sd)*spec.scale)]).reverse();s+='<path d="M '+[...upper,...lower].map(v=>v.join(' ')).join(' L ')+' Z" fill="'+T.modeColors[m]+'" opacity="'+(k===0?.13:.23)+'"/>';}
          s+='<path d="M '+data.map(r=>p.x(r.epoch)+' '+p.y(r.mean*spec.scale)).join(' L ')+'" fill="none" stroke="'+T.modeColors[m]+'" stroke-width="3" stroke-dasharray="'+(key.startsWith('val')?'9 5':'')+'"/>';
        });
        if(ri===2)finals.filter(r=>r.architecture===a&&r.mode===m).forEach(r=>{const xx=p.x(r.best_epoch),yy=p.y(r.selected.val_macro_f1*100);s+=line(xx-4,yy-4,xx+4,yy+4,T.ink,'stroke-width="2"')+line(xx-4,yy+4,xx+4,yy-4,T.ink,'stroke-width="2"');});
        if(ri===2){const selectedMean=mean(finals.filter(r=>r.architecture===a&&r.mode===m).map(r=>Number(r.best_epoch)));s+=line(p.x(selectedMean),b.y,p.x(selectedMean),b.y+b.h,T.ink,'stroke-width="2" stroke-dasharray="3 5"')+text(b.x+b.w-8,b.y+b.h-16,'Mean selected epoch = '+selectedMean.toFixed(1),15,'text-anchor="end" font-weight="600"');}
      });
    });
    s+=text(780,1424,'Observed epochs only; no zero-fill or carry-forward · n = 1: mean only, no sample SD · common axes across architectures',16);save('learning_curves_'+a+'.svg',s);
  });
}
async function generate() {
  read('report/figure_theme.js');const audit=csv('outputs_modes_v3_td/dataset_audit/image_properties.csv');if(audit.length!==560)throw Error('Expected 560 audited originals');
  const registry=JSON.parse(read('outputs_modes_v3_td/run_registry.json'));const winners=JSON.parse(read('outputs_modes_v3_td/tuning/best_params_by_arch_mode.json')).pairs;
  const records=loadHistories(registry);datasetFigures(audit);tuningFigures(records,winners);learningFigures(records);
  for(const [p,h]of Object.entries(sources))if(hash(p)!==h)throw Error('Source changed during rendering: '+p);
  fs.writeFileSync(path.join(__dirname,'enhanced_figure_sources.json'),JSON.stringify({generated_utc:new Date().toISOString(),experiment:'outputs_modes_v3_td',source_sha256:sources,output_sha256:Object.fromEntries(figures.map(p=>[p,hash(p)])),tuning_trials:75,final_runs:75,final_pairs:15,sample_sd_ddof:1,histogram_unit:'within-class percentage per bin; not probability density',learning_curves:'Observed epochs only; sample SD omitted at n=1; train metrics logged during augmented training, validation after each epoch.',python_inference_training_performed:false},null,2)+'\n');
  console.log('Generated enhanced EDA, seven tuning diagnostics and learning curves for all 15 five-seed setups.');
}
module.exports={generate};
if(require.main===module)generate().catch(e=>{console.error(e);process.exitCode=1;});


In [ ]:
%%project_source report/generate_matched_mobilenet_cam.py
"""Generate only eight MobileNet CAMs matched to the saved ResNet cases.

Explicit user authorization: 22 October 2026. No training or tuning.
Run from the project root: .venv/Scripts/python.exe report/generate_matched_mobilenet_cam.py
"""
from pathlib import Path
import sys
ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import json
import uuid
from datetime import datetime, timezone
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
from matplotlib import colormaps
from PIL import Image
from torchvision import transforms
from pipeline import _check_existing_split
from pipeline_modes import _read_csv, _csv, _json
from resnet_followup import locate
from shrine_classifier.data import CLASSES, open_rgb, letterbox, sha256_file, IMAGENET_MEAN, IMAGENET_STD
from shrine_classifier.models import build_model
from shrine_classifier.analysis import gradcam
from shrine_classifier.training import seed_everything


def main():
    out = ROOT / "outputs_modes_v3_td"
    data = ROOT / "dataset_cleaned"
    _check_existing_split(out, data)
    registry = json.loads((out / "run_registry.json").read_text(encoding="utf-8"))
    selection = json.loads((out / "selection.json").read_text(encoding="utf-8"))
    assert selection["reference_pair"] == {"architecture": "mobilenet_v3_large", "mode": "partial"}
    matching = [r for r in registry["final_runs"] if r["status"] == "completed"
                and r["architecture"] == "mobilenet_v3_large" and r["mode"] == "partial" and int(r["seed"]) == 42]
    if len(matching) != 1:
        raise ValueError("Expected exactly one completed MobileNet/Partial seed-42 run")
    run = matching[0]
    checkpoint = locate(out, run["checkpoint"])
    run_dir = locate(out, run["run_dir"])
    cfg = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    cases_path = ROOT / "report/resnet_gradcam_cases.csv"
    cases = _read_csv(cases_path)
    if len(cases) != 8 or len({r["image_id"] for r in cases}) != 8:
        raise ValueError("Expected eight fixed ResNet case IDs")
    preds_path = run_dir / "test_predictions.csv"
    preds = {r["image_id"]: r for r in _read_csv(preds_path)}
    split = {r["image_id"]: r for r in _read_csv(out / "splits.csv") if r["split"] == "test"}
    if len(preds) != 85 or set(preds) != set(split):
        raise ValueError("Saved prediction IDs do not match canonical test split")
    parents = out / "gradcam/mobilenet_v3_large_partial_matched_resnet/seed42"
    parents.mkdir(parents=True, exist_ok=True)
    identity = {"checkpoint_sha256": sha256_file(checkpoint), "case_selection_sha256": sha256_file(cases_path),
                "prediction_sha256": sha256_file(preds_path), "selection_sha256": sha256_file(out / "selection.json")}
    latest = parents / "latest.json"
    if latest.exists():
        previous = json.loads(latest.read_text(encoding="utf-8"))
        if previous["inputs"] != identity:
            raise ValueError("Existing matched-CAM inputs differ; inspect before repeating")
        for relative, digest in previous["artifact_sha256"].items():
            if sha256_file(out / relative) != digest:
                raise ValueError("Existing matched-CAM artifact changed")
        print("Existing complete eight-case matched CAM retained.")
        return
    seed_everything(42)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, _ = build_model("mobilenet_v3_large", pretrained=False)
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    if (state["architecture"], state["mode"], int(state["seed"]), int(state["epoch"])) != ("mobilenet_v3_large", "partial", 42, int(run["best_epoch"])):
        raise ValueError("Checkpoint identity mismatch")
    model.load_state_dict(state["model_state"], strict=True)
    model.to(device).eval()
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    dest = parents / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    dest.mkdir(exist_ok=False)
    records = []
    for case in cases:
        image_id = case["image_id"]
        row, pred = split[image_id], preds[image_id]
        if row["label"] != case["true_label"] or pred["true_label"] != row["label"]:
            raise ValueError("Case label mismatch")
        original = open_rgb(data / row["path"])
        canvas, bbox = letterbox(original, int(cfg["input_size"]))
        tensor = transform(canvas).unsqueeze(0).to(device)
        with torch.no_grad():
            probabilities = model(tensor).softmax(1)[0].cpu().numpy()
        saved = np.asarray([float(pred["prob_" + c]) for c in CLASSES])
        error = float(np.abs(probabilities - saved).max())
        if probabilities.argmax() != saved.argmax():
            _json(dest / "failure.json", {"image_id": image_id, "completed_cases": len(records), "error": "predicted class changed", "max_probability_error": error})
            raise ValueError(f"Prediction changed for {image_id}: max probability error {error}")
        if error > 1e-3:
            print(f"[note] {image_id}: same predicted class; saved batched and current single-image probabilities differ by {error:.6f}. Both are retained in metadata.", flush=True)
        with torch.enable_grad():
            heat = gradcam(model, "mobilenet_v3_large", tensor, CLASSES.index(pred["predicted_label"]))[0].detach().cpu().numpy()
        x1, y1, x2, y2 = bbox
        crop = heat[y1:y2, x1:x2]
        if not np.isfinite(crop).all() or not crop.size or not np.any(crop > 0):
            raise ValueError("Empty/nonfinite CAM: " + image_id)
        aligned = np.asarray(Image.fromarray(crop.astype(np.float32)).resize(original.size, Image.Resampling.BILINEAR))
        color = Image.fromarray(np.uint8(colormaps["jet"](np.clip(aligned, 0, 1))[..., :3] * 255))
        sample = dest / image_id.replace("/", "__")
        sample.mkdir()
        original.save(sample / "original.png")
        color.save(sample / "heatmap.png")
        Image.blend(original, color, .42).save(sample / "overlay.png")
        np.save(sample / "cam_original_alignment.npy", aligned)
        records.append({**pred, "explained_target": pred["predicted_label"], "cam_layer": "features[-1]",
                        "selection_reason": "same image ID and order as the saved ResNet50/Full eight cases",
                        "content_bbox": json.dumps(list(bbox)), "checkpoint_sha256": identity["checkpoint_sha256"],
                        "probability_max_abs_error": error, "cam_inference_confidence": float(probabilities.max()),
                        "cam_inference_probabilities": json.dumps(probabilities.tolist()), "sample_directory": sample.relative_to(out).as_posix()})
        print(f"[{len(records)}/8] {image_id}: {pred['predicted_label']}, saved confidence={float(pred['confidence']):.6f}, error={error:.2e}", flush=True)
    _csv(dest / "selected_cases.csv", records)
    artifact_hashes = {p.relative_to(out).as_posix(): sha256_file(p) for p in sorted(dest.rglob("*")) if p.is_file()}
    manifest = {"architecture": "mobilenet_v3_large", "mode": "partial", "seed": 42, "examples": 8,
                "inputs": identity, "directory": dest.relative_to(out).as_posix(), "artifact_sha256": artifact_hashes,
                "device": device, "torch": torch.__version__, "training_runs_added": 0,
                "case_ids": [c["image_id"] for c in cases], "explained_target": "each model's own predicted class",
                "created_utc": datetime.now(timezone.utc).isoformat()}
    _json(dest / "manifest.json", manifest)
    _json(latest, manifest)
    print(json.dumps({"completed_cases": 8, "training_runs_added": 0, "latest": str(latest)}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%project_source report/generate_preprocessing_examples.js
/**
 * Visual previews of the latest neutral letterbox and train augmentation policy.
 * Run: node report/generate_preprocessing_examples.js
 * This does not invoke Python, train models, or use the attached reference images.
 * The augmentation preview implements the recorded ranges in JavaScript for
 * illustration; it is not a saved torchvision minibatch from measured runs.
 */
const fs = require('fs');
const path = require('path');
const crypto = require('crypto');
const root = path.resolve(__dirname, '..');
const assetDir = path.join(__dirname, 'report_assets', 'latest');
fs.mkdirSync(assetDir, {recursive: true});
let sharp;
try { sharp = require('sharp'); }
catch {
  const home = process.env.USERPROFILE || process.env.HOME || '';
  try { sharp = require(path.join(home, '.cache', 'codex-runtimes', 'codex-primary-runtime', 'dependencies', 'node', 'node_modules', 'sharp')); }
  catch { throw Error('Install the sharp Node package (npm install sharp) to generate these image previews.'); }
}
const N = 224;
const NEUTRAL = [124, 116, 104];
const classOrder = ['san_phraphrom', 'san_phraphum', 'san_tayay', 'tee_ju_eia'];
const classLabels = [['San Phra', 'Phrom'], ['San Phra', 'Phum'], ['San Ta', 'Yai'], ['Tee Ju', 'Eia']];
const classColors = require('./figure_theme').classColors;
function parseCsv(text) {
  const rows = []; let row = [], field = '', quoted = false;
  for (let i = 0; i < text.length; i++) {
    const c = text[i];
    if (quoted) { if (c === '"' && text[i+1] === '"') { field += '"'; i++; } else if (c === '"') quoted = false; else field += c; }
    else if (c === '"') quoted = true;
    else if (c === ',') { row.push(field); field = ''; }
    else if (c === '\n') { row.push(field.replace(/\r$/, '')); rows.push(row); row = []; field = ''; }
    else field += c;
  }
  if (field || row.length) { row.push(field.replace(/\r$/, '')); rows.push(row); }
  const header = rows.shift().map((v,i) => i === 0 ? v.replace(/^\uFEFF/, '') : v);
  return rows.filter(r => r.length === header.length).map(r => Object.fromEntries(header.map((h,i) => [h,r[i]])));
}
function hashFile(file) { return crypto.createHash('sha256').update(fs.readFileSync(file)).digest('hex'); }
function seedFor(id) { return crypto.createHash('sha256').update('readme-augmentation-preview-v1|' + id).digest().readUInt32LE(0); }
function rng(seed) { let x = seed >>> 0; return () => { x = (x + 0x6D2B79F5) >>> 0; let z = Math.imul(x ^ (x >>> 15), 1 | x); z ^= z + Math.imul(z ^ (z >>> 7), 61 | z); return ((z ^ (z >>> 14)) >>> 0) / 4294967296; }; }
function between(rand, lo, hi) { return lo + rand() * (hi - lo); }
function clamp(v) { return Math.max(0, Math.min(255, Math.round(v))); }
function emptyCanvas() { const a = Buffer.alloc(N*N*3); for(let i=0;i<a.length;i+=3){a[i]=NEUTRAL[0];a[i+1]=NEUTRAL[1];a[i+2]=NEUTRAL[2];} return a; }
async function openRgb(file) {
  const {data,info} = await sharp(file, {animated:false,page:0}).rotate().flatten({background:{r:NEUTRAL[0],g:NEUTRAL[1],b:NEUTRAL[2]}}).toColourspace('srgb').removeAlpha().raw().toBuffer({resolveWithObject:true});
  if(info.channels !== 3) throw Error(`Expected RGB for ${file}, got ${info.channels} channels`);
  return {data,width:info.width,height:info.height};
}
async function letterbox(source) {
  const scale = Math.min(N/source.width, N/source.height);
  const width = Math.max(1, Math.round(source.width*scale)), height = Math.max(1,Math.round(source.height*scale));
  const resized = await sharp(source.data,{raw:{width:source.width,height:source.height,channels:3}}).resize(width,height,{kernel:'lanczos3'}).raw().toBuffer();
  const canvas=emptyCanvas(), left=Math.floor((N-width)/2), top=Math.floor((N-height)/2);
  for(let y=0;y<height;y++) resized.copy(canvas,((top+y)*N+left)*3,y*width*3,(y+1)*width*3);
  return {data:canvas,bbox:[left,top,left+width,top+height]};
}
function warp(input, mapper) {
  const out=emptyCanvas();
  for(let y=0;y<N;y++) for(let x=0;x<N;x++) {
    const [sx,sy]=mapper(x,y),ix=Math.round(sx),iy=Math.round(sy);
    if(ix<0||ix>=N||iy<0||iy>=N) continue;
    const from=(iy*N+ix)*3,to=(y*N+x)*3;
    out[to]=input[from];out[to+1]=input[from+1];out[to+2]=input[from+2];
  }
  return out;
}
function homography(dest, src) {
  const a=[];
  for(let i=0;i<4;i++) {
    const [x,y]=dest[i], [u,v]=src[i];
    a.push([x,y,1,0,0,0,-u*x,-u*y,u]);
    a.push([0,0,0,x,y,1,-v*x,-v*y,v]);
  }
  for(let col=0;col<8;col++) {
    let pivot=col;for(let r=col+1;r<8;r++)if(Math.abs(a[r][col])>Math.abs(a[pivot][col]))pivot=r;
    [a[col],a[pivot]]=[a[pivot],a[col]];
    if(Math.abs(a[col][col])<1e-10)throw Error('Degenerate perspective transform');
    const div=a[col][col];for(let c=col;c<9;c++)a[col][c]/=div;
    for(let r=0;r<8;r++)if(r!==col){const f=a[r][col];for(let c=col;c<9;c++)a[r][c]-=f*a[col][c];}
  }
  const h=a.map(r=>r[8]);
  return (x,y)=>{const d=h[6]*x+h[7]*y+1;return [(h[0]*x+h[1]*y+h[2])/d,(h[3]*x+h[4]*y+h[5])/d];};
}
function rgbToHsv(r,g,b){r/=255;g/=255;b/=255;const max=Math.max(r,g,b),min=Math.min(r,g,b),d=max-min;let h=0;if(d){if(max===r)h=((g-b)/d)%6;else if(max===g)h=(b-r)/d+2;else h=(r-g)/d+4;h/=6;if(h<0)h+=1;}return [h,max===0?0:d/max,max];}
function hsvToRgb(h,s,v){const a=v*s,f=n=>{const k=(n+h*6)%6;return v-a*Math.max(0,Math.min(k,4-k,1));};return [f(5),f(3),f(1)].map(z=>clamp(z*255));}
function jitter(input, factors){
  const out=Buffer.from(input);let avg=0;for(let i=0;i<out.length;i+=3)avg+=(out[i]+out[i+1]+out[i+2])/3;avg/=N*N;
  for(let i=0;i<out.length;i+=3){let rgb=[out[i],out[i+1],out[i+2]].map(v=>clamp((v-avg)*factors.contrast+avg));const gray=.299*rgb[0]+.587*rgb[1]+.114*rgb[2];rgb=rgb.map(v=>clamp(gray+(v-gray)*factors.saturation));const hsv=rgbToHsv(...rgb);hsv[0]=(hsv[0]+factors.hue+1)%1;rgb=hsvToRgb(...hsv).map(v=>clamp(v*factors.brightness));out[i]=rgb[0];out[i+1]=rgb[1];out[i+2]=rgb[2];}
  return out;
}
function augment(canvas, rand) {
  const p={rotation_deg:between(rand,-12,12),translate_x_px:0,translate_y_px:0,scale:between(rand,.92,1.08),perspective:false,horizontal_flip:false,brightness:between(rand,.9,1.1),contrast:between(rand,.9,1.1),saturation:between(rand,.9,1.1),hue:between(rand,-.02,.02),erasing:false};
  let a=canvas, cx=(N-1)/2, cy=(N-1)/2, t=p.rotation_deg*Math.PI/180, co=Math.cos(t), si=Math.sin(t);
  a=warp(a,(x,y)=>[cx+co*(x-cx)+si*(y-cy),cy-si*(x-cx)+co*(y-cy)]);
  p.translate_x_px=Math.round(between(rand,-.05*N,.05*N));p.translate_y_px=Math.round(between(rand,-.05*N,.05*N));
  a=warp(a,(x,y)=>[cx+(x-cx-p.translate_x_px)/p.scale,cy+(y-cy-p.translate_y_px)/p.scale]);
  if(rand()<.20){const corners=[[0,0],[N-1,0],[N-1,N-1],[0,N-1]],d=corners.map(([x,y])=>[x+between(rand,-.05*N,.05*N),y+between(rand,-.05*N,.05*N)]);a=warp(a,homography(d,corners));p.perspective=true;}
  if(rand()<.5){a=warp(a,(x,y)=>[N-1-x,y]);p.horizontal_flip=true;}
  a=jitter(a,p);
  if(rand()<.05){const area=between(rand,.02,.08)*N*N,ratio=Math.exp(between(rand,Math.log(.3),Math.log(3.3))),w=Math.min(N,Math.max(1,Math.round(Math.sqrt(area*ratio)))),h=Math.min(N,Math.max(1,Math.round(Math.sqrt(area/ratio)))),x0=Math.floor(rand()*(N-w+1)),y0=Math.floor(rand()*(N-h+1));for(let y=y0;y<y0+h;y++)for(let x=x0;x<x0+w;x++){const i=(y*N+x)*3;a[i]=NEUTRAL[0];a[i+1]=NEUTRAL[1];a[i+2]=NEUTRAL[2];}p.erasing=true;p.erase_bbox=[x0,y0,w,h];}
  return {data:a,params:p};
}
function escapeXml(v){return String(v).replaceAll('&','&amp;').replaceAll('<','&lt;').replaceAll('>','&gt;').replaceAll('"','&quot;');}
async function png(raw,width=N,height=N){return sharp(raw,{raw:{width,height,channels:3}}).png().toBuffer();}
async function paddingPanel(row) {
  const sourceFile=path.join(root,'dataset_cleaned',row.path),source=await openRgb(sourceFile),letter=await letterbox(source);
  const original=await sharp(source.data,{raw:{width:source.width,height:source.height,channels:3}}).resize({width:720,height:560,fit:'inside',kernel:'lanczos3'}).png().toBuffer();
  const om=await sharp(original).metadata(),padded=await sharp(await png(letter.data)).resize(640,640,{kernel:'nearest'}).png().toBuffer();
  const W=1540,H=825,svg=Buffer.from(`<svg xmlns="http://www.w3.org/2000/svg" width="${W}" height="${H}"><style>text{font-family:Arial,sans-serif;fill:#263238}</style><rect width="100%" height="100%" fill="white"/><text x="770" y="42" text-anchor="middle" font-size="27" font-weight="bold">Aspect-ratio-preserving neutral letterbox</text><text x="405" y="92" text-anchor="middle" font-size="22">Original (${source.width} × ${source.height})</text><text x="1150" y="92" text-anchor="middle" font-size="22">Model input (224 × 224; enlarged view)</text><text x="770" y="805" text-anchor="middle" font-size="16">${escapeXml(row.image_id)} · train split · neutral fill RGB (124,116,104)</text></svg>`);
  const panel=await sharp({create:{width:W,height:H,channels:3,background:'#fff'}}).composite([{input:svg,left:0,top:0},{input:original,left:405-Math.floor(om.width/2),top:125+Math.floor((640-om.height)/2)},{input:padded,left:830,top:125}]).png().toBuffer();
  fs.writeFileSync(path.join(assetDir,'padding_large.png'),panel);
  return {image_id:row.image_id,path:row.path,original_width:source.width,original_height:source.height,model_input_size:N,letterbox_bbox:letter.bbox,source_sha256:hashFile(sourceFile)};
}
async function augmentationGrid(rows) {
  const chosen=[];for(const label of classOrder){const candidates=rows.filter(r=>r.split==='train'&&r.label===label);const distinct=new Map();for(const r of candidates){const key=r.group_id||r.image_id;if(!distinct.has(key))distinct.set(key,r);}const pick=[...distinct.values()].sort((a,b)=>seedFor(a.image_id)-seedFor(b.image_id)).slice(0,5);if(pick.length!==5)throw Error(`Expected five independent train groups: ${label}`);chosen.push(...pick);}
  const W=1390,H=1120,tile=224,x0=190,y0=100,dx=235,dy=250,composites=[],manifest=[];
  for(let i=0;i<chosen.length;i++){const r=chosen[i],file=path.join(root,'dataset_cleaned',r.path),source=await openRgb(file),base=await letterbox(source),a=augment(base.data,rng(seedFor(r.image_id)));composites.push({input:await png(a.data),left:x0+(i%5)*dx,top:y0+Math.floor(i/5)*dy});manifest.push({class:r.label,image_id:r.image_id,path:r.path,split:r.split,group_id:r.group_id,source_sha256:hashFile(file),seed:seedFor(r.image_id),...a.params});}
  let svg=`<svg xmlns="http://www.w3.org/2000/svg" width="${W}" height="${H}"><style>text{font-family:Arial,sans-serif;fill:#263238}</style><rect width="100%" height="100%" fill="white"/><text x="${W/2}" y="43" text-anchor="middle" font-size="27" font-weight="bold">Training augmentation preview · 5 images per class</text>`;
  for(let c=0;c<5;c++)svg+=`<text x="${x0+c*dx+112}" y="88" text-anchor="middle" font-size="16">Example ${c+1}</text>`;
  for(let r=0;r<4;r++){const y=y0+r*dy+105;svg+=`<rect x="18" y="${y-57}" width="6" height="115" fill="${classColors[r]}"/><text x="34" y="${y-7}" font-size="18" font-weight="bold">${classLabels[r][0]}</text><text x="34" y="${y+17}" font-size="18" font-weight="bold">${classLabels[r][1]}</text>`;}
  svg+='</svg>';composites.unshift({input:Buffer.from(svg),left:0,top:0});
  const grid=await sharp({create:{width:W,height:H,channels:3,background:'#fff'}}).composite(composites).png().toBuffer();fs.writeFileSync(path.join(assetDir,'augmentation_train_5_per_class.png'),grid);
  return manifest;
}
async function main(){
  const splitFile=path.join(root,'outputs_modes_v3_td','splits.csv'),rows=parseCsv(fs.readFileSync(splitFile,'utf8'));
  const pad=rows.find(r=>r.image_id==='san_phraphrom/san_phraphrom_0001'&&r.split==='train');if(!pad)throw Error('Requested train example is not in the canonical train split');
  const padding=await paddingPanel(pad),aug=await augmentationGrid(rows);
  const manifest={created_utc:new Date().toISOString(),split_csv_sha256:hashFile(splitFile),source_policy:'shrine_classifier/data.py letterbox; shrine_classifier/training.py ImageDataset transforms',rendering_note:'Deterministic JavaScript illustration of the configured policy. Source images are actual train-split images; this is not a logged training minibatch or bit-identical torchvision RNG output.',padding,augmentation:aug};
  fs.writeFileSync(path.join(__dirname,'preprocessing_preview_sources.json'),JSON.stringify(manifest,null,2)+'\n');
  console.log(`Generated large padding example and ${aug.length} augmentation previews from canonical train images.`);
}
main().catch(e=>{console.error(e);process.exitCode=1;});


In [ ]:
%%project_source report/generate_selected_model_calibration.js
// Report-only figure from saved diagnostics. Run: node report/generate_selected_model_calibration.js
const fs = require('fs');
const path = require('path');
const crypto = require('crypto');
const theme = require('./figure_theme');
const root = path.resolve(__dirname, '..');
const architecture = 'resnet50', mode = 'full', seed = '42';
const source = 'outputs_modes_v3_td/confidence/bin_metrics.csv';
const input = fs.readFileSync(path.join(root, source));
function csv(text) {
  const lines = text.replace(/^\uFEFF/, '').trim().split(/\r?\n/);
  const fields = line => line.split(',').map(v => v.replace(/^"|"$/g, ''));
  const header = fields(lines.shift());
  return lines.map(line => Object.fromEntries(fields(line).map((v, i) => [header[i], v])));
}
const bins = csv(input.toString('utf8')).filter(r => r.architecture === architecture && r.mode === mode && r.seed === seed);
if (bins.length !== 10 || bins.reduce((n, r) => n + Number(r.count), 0) !== 85) throw Error('Incomplete saved calibration bins');
const x = 90, y = 90, width = 580, height = 370;
let svg = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 570" role="img" aria-label="ResNet50 Full calibration on test, seed 42"><rect width="800" height="570" fill="white"/><style>text{font-family:Arial,"Noto Sans Thai","Leelawadee UI",Tahoma,sans-serif;fill:#1F2937;font-size:13px}.small{font-size:11px}</style>';
const text = (xx, yy, value, extra = '') => `<text x="${xx}" y="${yy}" text-anchor="middle" ${extra}>${value}</text>`;
svg += text(400, 34, 'ResNet50 / Full: test calibration, seed 42 (%)', 'style="font-size:23px;font-weight:700"');
for (const v of [0, 20, 40, 60, 80, 100]) {
  const py = y + height - v / 100 * height;
  svg += `<line x1="${x}" y1="${py}" x2="${x+width}" y2="${py}" stroke="#e7ebee"/>`;
  svg += text(x-19, py+4, v, 'class="small"') + text(x+v/100*width, y+height+18, v, 'class="small"');
}
svg += `<path d="M ${x} ${y} V ${y+height} H ${x+width}" fill="none" stroke="#87929b"/><line x1="${x}" y1="${y+height}" x2="${x+width}" y2="${y}" stroke="#c4cdd3" stroke-dasharray="7 5"/>`;
const seedEce = bins.reduce((sum, b) => sum + Number(b.count) / 85 * (Number(b.count) ? Math.abs(Number(b.mean_confidence)-Number(b.accuracy)) : 0), 0) * 100;
svg += text(265, 114, 'ECE (10 bins, this seed): ' + seedEce.toFixed(2), 'class="small"');
for (const b of bins) {
  if (!Number(b.count)) continue;
  const xx = x + Number(b.mean_confidence)*width, yy = y+height-Number(b.accuracy)*height;
  svg += `<circle cx="${xx}" cy="${yy}" r="${Math.max(5, Math.sqrt(Number(b.count))*2.6)}" fill="${theme.modeColors.full}" opacity=".65" stroke="${theme.ink}"/>` + text(xx, yy-13, b.count, 'class="small"');
}
svg += text(380, 499, 'Mean confidence (%)') + text(28, 275, 'Observed accuracy (%)', 'transform="rotate(-90 28 275)"') + text(400, 532, 'Dot area scales with bin count; label = images in bin (test n = 85)', 'class="small"') + '</svg>\n';
const figure = 'report/report_assets/latest/selected_resnet50_full_calibration.svg';
fs.writeFileSync(path.join(root, figure), svg);
const summaryPath = 'report/verified_pair_summary.csv';
const summaries = csv(fs.readFileSync(path.join(root, summaryPath), 'utf8'));
const metricLeaders = Object.fromEntries(['test_acc_mean','test_macro_precision_mean','test_macro_recall_mean','test_macro_f1_mean'].map(metric => {
  const best = [...summaries].sort((a,b) => Number(b[metric])-Number(a[metric]))[0];
  return [metric, {architecture:best.architecture, mode:best.mode, value:Number(best[metric])}];
}));
fs.writeFileSync(path.join(__dirname, 'selected_model_report_sources.json'), JSON.stringify({
  selected_report_model:{architecture,mode}, selection_context:'Post-hoc descriptive choice after examining multiple test metrics; requires confirmation on a new external holdout.',
  statistical_reference:{architecture:'mobilenet_v3_large',mode:'partial',original_selection_json_unchanged:true},
  metric_leaders:metricLeaders, calibration:{source,source_sha256:crypto.createHash('sha256').update(input).digest('hex'),seed:Number(seed),figure},
  pair_summary:{source:summaryPath,source_sha256:crypto.createHash('sha256').update(fs.readFileSync(path.join(root,summaryPath))).digest('hex')},
  gradcam_resnet50_full: JSON.parse(fs.readFileSync(path.join(__dirname, 'resnet_followup_sources.json'), 'utf8')).gradcam
},null,2)+'\n');
console.log('Generated ResNet50/Full calibration from saved bins and recorded the report selection separately from the experiment lock.');


In [ ]:
%%project_source report/render_preview.js
// Offline Markdown rendering and visual QA; no experiment execution.
const fs=require('fs'),path=require('path');
const {pathToFileURL}=require('url');
const {root,sharpRuntime}=require('./report_utils');
async function main(){
  const runtime=path.join(process.env.USERPROFILE,'.cache/codex-runtimes/codex-primary-runtime/dependencies/node/node_modules');
  const {marked}=await import(pathToFileURL(require.resolve(path.join(runtime,'marked'))).href);
  const {chromium}=require(path.join(runtime,'playwright'));
  const markdown=fs.readFileSync(path.join(root,'README.md'),'utf8');
  const html='<!doctype html><html lang="th"><meta charset="utf-8"><base href="'+pathToFileURL(root+path.sep).href+'"><title>README preview</title><style>body{font:16px/1.7 Arial,Tahoma,sans-serif;color:#1f2937;background:white;margin:40px auto;max-width:1000px}h1,h2,h3,h4{line-height:1.35;margin-top:1.7em}h1{font-size:30px}h2{font-size:25px;border-bottom:1px solid #ddd;padding-bottom:10px}h3{font-size:21px}img{max-width:100%;height:auto}table{border-collapse:collapse;width:100%;margin:20px 0;font-size:14px}th,td{border:1px solid #d1d5db;padding:9px;overflow-wrap:anywhere}th{background:#f3f4f6;text-align:left}pre{background:#f3f4f6;padding:16px;white-space:pre-wrap;overflow-wrap:anywhere}code{font-size:.9em}a{color:#0066a6}details{padding:12px;border:1px solid #ddd}sub{font-size:14px}</style><body>'+marked.parse(markdown)+'</body></html>';
  const htmlFile=path.join(__dirname,'qa/readme_preview.html');fs.writeFileSync(htmlFile,html);
  const executable=fs.existsSync(chromium.executablePath())?chromium.executablePath():'C:/Program Files (x86)/Microsoft/Edge/Application/msedge.exe';
  const browser=await chromium.launch({headless:true,executablePath:executable,args:['--allow-file-access-from-files']});
  try{
    const page=await browser.newPage({viewport:{width:1200,height:1000},deviceScaleFactor:1});
    await page.route(/^https?:/,route=>route.abort());
    await page.goto(pathToFileURL(htmlFile).href);
    await page.evaluate(async()=>{await document.fonts.ready;await Promise.all([...document.images].map(i=>i.decode().catch(()=>{})));});
    const result=await page.evaluate(()=>({images:[...document.images].length,broken_images:[...document.images].filter(i=>!i.complete||!i.naturalWidth).map(i=>i.src),horizontal_overflow:document.documentElement.scrollWidth>innerWidth}));
    if(result.broken_images.length||result.horizontal_overflow)throw Error(JSON.stringify(result));
    await page.screenshot({path:path.join(__dirname,'qa/readme_abstract.png')});
    await page.getByRole('heading',{name:'4.3 Selected configurations and final evaluation',exact:true}).scrollIntoViewIfNeeded();
    await page.screenshot({path:path.join(__dirname,'qa/readme_selected_configurations.png')});
    await page.getByRole('heading',{name:'6.2 Learning curves across five seeds',exact:true}).scrollIntoViewIfNeeded();
    await page.screenshot({path:path.join(__dirname,'qa/readme_learning_curves.png')});
    for(const [heading,file] of [
      ['Worked example: MobileNetV3-Large/Partial vs ResNet50/Full','readme_hypothesis'],
      ['6.7 Matched-image Grad-CAM comparison','readme_matched_cam'],
      ['6.8 Confidence and calibration','readme_calibration'],
      ['6.9 Final model selection','readme_model_selection']]){
      const target=page.getByRole('heading',{name:heading,exact:true});
      await target.evaluate(el=>el.scrollIntoView({block:'start'}));
      await page.screenshot({path:path.join(__dirname,'qa',file+'.png')});
    }
    fs.writeFileSync(path.join(__dirname,'qa/readme_render_validation.json'),JSON.stringify({rendered_utc:new Date().toISOString(),...result,external_network_requests_blocked:true},null,2)+'\n');
    console.log('Offline README render passed: '+result.images+' images, no broken images or horizontal overflow.');
  }finally{await browser.close();}
  const sharp=sharpRuntime();
  for(const name of ['image_dimensions','brightness','colour_modes','learning_curves_resnet50'])await sharp(path.join(__dirname,'report_assets/latest',name+'.svg')).greyscale().png().toFile(path.join(__dirname,'qa',name+'_grayscale.png'));
  console.log('Rendered four grayscale proofs for manual review.');
}
main().catch(error=>{console.error(error);process.exitCode=1;});


In [ ]:
%%project_source report/render_readme.js
// Report-only renderer. All percentages are display conversions of saved 0–1 metrics.
// Run: node report/render_readme.js. This never imports experiment code.
const fs = require('fs');
const path = require('path');
const { root, readCsv, hash } = require('./report_utils');
const names = {resnet50:'ResNet50', efficientnet_b0:'EfficientNet-B0', mobilenet_v3_large:'MobileNetV3-Large', densenet121:'DenseNet121', convnext_tiny:'ConvNeXt-Tiny'};
const modes = {full:'Full', frozen:'Frozen', partial:'Partial'};
const classes = {san_phraphrom:'พระพรหม', san_phraphum:'พระภูมิ', san_tayay:'ตายาย', tee_ju_eia:'ตี่จู้เอี๊ยะ'};
const setup = r => names[r.architecture] + '/' + modes[r.mode];
const pairName = id => { const [architecture, mode] = id.split('::'); return setup({architecture, mode}); };
const f = (n, d=2) => Number(n).toFixed(d);
const pct = n => f(Number(n) * 100);
const pm = (r, key) => pct(r[key+'_mean']) + ' ± ' + pct(r[key+'_sample_sd']);
const pval = n => Number(n) < 0.001 ? '<0.001' : f(n, 3);
const inputs = ['report/readme_template.md', 'report/verified_pair_summary.csv', 'report/verified_per_class_summary.csv',
  'report/verified_statistics.csv', 'report/verified_training_dynamics.csv', 'report/resnet_gradcam_cases.csv',
  'report/resnet_ablation_condition_summary.csv', 'report/resnet_ablation_statistical_tests.csv',
  'outputs_modes_v3_td/tuning/best_params_by_arch_mode.json', 'outputs_modes_v3_td/tuning/search_definition.json',
  'outputs_modes_v3_td/eyeball/eyeball_cases.csv', 'report/error_frequency.csv', 'report/final_checkpoint_summary.csv',
  'report/readme_revision_evidence.json', 'report/matched_cam_comparison.csv'];
const before = Object.fromEntries(inputs.map(p => [p, hash(p)]));
function table(headers, rows, numericColumns=[]) {
  if (rows.some(r => r.length !== headers.length)) throw Error('Markdown table column mismatch');
  return ['| '+headers.join(' | ')+' |', '| '+headers.map((_, i) => numericColumns.includes(i)?'---:':'---').join(' | ')+' |',
    ...rows.map(r => '| '+r.map(v => String(v).replaceAll('|', '\\|')).join(' | ')+' |')].join('\n');
}
const blocks = {};
const revision = JSON.parse(fs.readFileSync(path.join(root,'report/readme_revision_evidence.json'),'utf8'));
blocks.HYPOTHESIS_F1_TABLE = table(['Seed','MobileNet/Partial F1 (%)','ResNet/Full F1 (%)','Difference (pp)'],
  revision.paired_f1_example.matched.map(r=>[r.seed,f(r.mobile_f1_pct,4),f(r.resnet_f1_pct,4),f(r.difference_pp,4)]),[0,1,2,3]);
blocks.MATCHED_CAM_TABLE = table(['Image ID (stem)','True class','ResNet prediction','Confidence (%)','MobileNet prediction','Confidence (%)'],
  readCsv('report/matched_cam_comparison.csv').map(r=>[r.image_id.split('/')[1],classes[r.true_label],
    classes[r.resnet_pred]+' '+(r.resnet_pred===r.true_label?'✓':'✗'),pct(r.resnet_confidence),
    classes[r.mobile_pred]+' '+(r.mobile_pred===r.true_label?'✓':'✗'),pct(r.mobile_confidence)]),[3,5]);
const summary = readCsv('report/verified_pair_summary.csv');
if (summary.length !== 15 || summary.some(r => Number(r.n_runs || r.completed_seeds || r.n_seeds || 5) !== 5)) throw Error('Expected 15 five-seed summaries');
blocks.RESULT_TABLE = table(['Setup','Accuracy (%)','Macro precision (%)','Macro recall (%)','Macro F1 (%)'],
  [...summary].sort((a,b)=>Number(b.test_macro_f1_mean)-Number(a.test_macro_f1_mean)).map(r => [setup(r), ...['test_acc','test_macro_precision','test_macro_recall','test_macro_f1'].map(key => pm(r,key))]), [1,2,3,4]);
const classSummary = readCsv('report/verified_per_class_summary.csv');
const selectedClasses = classSummary.filter(r => (r.architecture==='resnet50'&&r.mode==='full') || (r.architecture==='mobilenet_v3_large'&&r.mode==='partial'));
if (selectedClasses.length !== 8) throw Error('Missing selected/reference per-class metrics');
blocks.PERCLASS_TABLE = table(['Setup','Class','Support','Precision (%)','Recall (%)','F1 (%)'], selectedClasses.map(r => [setup(r), classes[r.class], r.support, ...['precision','recall','f1'].map(key=>pm(r,key))]), [2,3,4,5]);
const stats = readCsv('report/verified_statistics.csv');
if(stats.length!==58)throw Error('Expected all 58 locked statistical comparisons');
const shownStats = stats.filter(r => r.significant.toLowerCase()==='true' || (r.reference==='mobilenet_v3_large::partial' && r.competitor==='resnet50::full'));
blocks.STATS_TABLE = table(['Family','A − B','Metric','Mean Δ (pp)','95% CI (pp)¹','Holm p'], shownStats.map(r => [r.family_id.startsWith('primary')?'Primary':'Secondary', pairName(r.reference)+'<br>− '+pairName(r.competitor), r.metric==='test_acc'?'Accuracy':'Macro F1', pct(r.mean_difference), '['+pct(r.ci95_lower)+', '+pct(r.ci95_upper)+']', pval(r.p_holm)]), [3,5]);
blocks.DYNAMICS_TABLE = table(['Setup','Mean selected epoch','Mean stop epoch','Training F1 (%)','Validation F1 (%)','Late loss divergence (runs)'], readCsv('report/verified_training_dynamics.csv').map(r => [setup(r),f(r.mean_best_epoch,1),f(r.mean_stop_epoch,1),pct(r.mean_train_f1_at_best),pct(r.mean_val_f1_at_best),r.late_divergence_runs+'/5']), [1,2,3,4,5]);
const winners = Object.values(JSON.parse(fs.readFileSync(path.join(root, inputs[8]),'utf8')).pairs);
if(winners.length!==15)throw Error('Expected all 15 independent tuning winners');
blocks.WINNER_TABLE = table(['Setup','Learning rate','Weight decay','Optimizer','Batch size','Tuning validation F1 (%)'], winners.map(r=>[setup(r),Number(r.learning_rate).toExponential(6),Number(r.weight_decay).toExponential(6),r.optimizer==='adamw'?'AdamW':r.optimizer==='sgd'?'SGD':'Adam',r.batch_size,pct(r.best_val_macro_f1)]), [1,2,4,5]);
const checkpointRows=readCsv('report/final_checkpoint_summary.csv');
if(checkpointRows.length!==75)throw Error('Expected 75 final checkpoint identities');
blocks.CHECKPOINT_SUMMARY_TABLE=table(['Setup','Final runs','Mean selected epoch','Selected epoch range','Seeds'],winners.map(w=>{const rows=checkpointRows.filter(r=>r.architecture===w.architecture&&r.mode===w.mode);if(rows.length!==5)throw Error('Missing checkpoint seeds');const epochs=rows.map(r=>Number(r.selected_epoch));return [setup(w),5,f(epochs.reduce((a,b)=>a+b,0)/5,1),Math.min(...epochs)+'–'+Math.max(...epochs),rows.map(r=>r.seed).join(', ')];}),[1,2]);
const search = JSON.parse(fs.readFileSync(path.join(root,inputs[9]),'utf8'));
blocks.SEARCH_TABLE = table(['Trial','Learning rate','Weight decay','Optimizer','Batch'], search.sampled_configs.map((r,i)=>[i+1,Number(r.learning_rate).toExponential(6),Number(r.weight_decay).toExponential(6),r.optimizer==='adamw'?'AdamW':'Adam',r.batch_size]), [0,1,2,4]);
const cases = readCsv('report/resnet_gradcam_cases.csv');
blocks.RESNET_CASE_TABLE = table(['Image ID (stem)','True class','Predicted class','Confidence (%)','Selection reason'], cases.map(r=>[r.image_id.split('/')[1],classes[r.true_label],classes[r.predicted_label]+' '+(r.true_label===r.predicted_label?'✓':'✗'),pct(r.confidence),r.selection_reason==='highest_confidence_correct'?'ถูก มั่นใจสูงสุดในคลาส':r.selection_reason==='highest_confidence_misclassified'?'ผิด มั่นใจสูงสุดในคลาส':'ถูก มั่นใจต่ำสุด; คลาสนี้ไม่มีภาพผิด']), [3]);
blocks.CASE_TABLE = table(['Image ID (stem)','True class','Fine-tuned prediction','Confidence (%)','ImageNet top-1'],readCsv(inputs[10]).map(r=>[r.image_id.split('/')[1],classes[r.true_label],classes[r.predicted_label]+' '+(r.true_label===r.predicted_label?'✓':'✗'),pct(r.confidence),r.imagenet_label]),[3]);
blocks.ERROR_TABLE = table(['Image ID (stem)','True class','Errors / 75','Reference errors / 5','ResNet Full errors / 5'],readCsv('report/error_frequency.csv').slice(0,10).map(r=>[r.image_id.split('/')[1],classes[r.true_label],r.wrong_all75,r.wrong_reference5,r.wrong_resnetfull5]),[2,3,4]);
const ablation = readCsv('report/resnet_ablation_condition_summary.csv');
const conditionNames = {baseline:'Baseline',no_augmentation:'ไม่ใช้ augmentation',no_class_weights:'ไม่ใช้ class weights'};
blocks.ABLATION_TABLE = table(['Condition','Accuracy (%)','Macro precision (%)','Macro recall (%)','Macro F1 (%)'],ablation.map(r=>[conditionNames[r.condition],...['test_acc','test_macro_precision','test_macro_recall','test_macro_f1'].map(key=>pm(r,key))]),[1,2,3,4]);
const ablationTests = readCsv('report/resnet_ablation_statistical_tests.csv');
blocks.ABLATION_DELTAS = table(['Removed component','Δ Accuracy (pp)','Δ Macro F1 (pp)','Holm p: Accuracy','Holm p: Macro F1'],['no_augmentation','no_class_weights'].map(condition=>{const a=ablationTests.find(r=>r.condition===condition&&r.metric==='test_acc'),b=ablationTests.find(r=>r.condition===condition&&r.metric==='test_macro_f1');return [condition==='no_augmentation'?'Augmentation':'Class weights',f(a.mean_difference_pp),f(b.mean_difference_pp),pval(a.p_holm),pval(b.p_holm)];}),[1,2,3,4]);
let text = fs.readFileSync(path.join(__dirname,'readme_template.md'),'utf8');
text = text.replace(/\{\{([A-Z][A-Z0-9_]+)\}\}/g,(_,name)=>{
  if(!blocks[name])throw Error('Unknown report block: '+name);
  return '<!-- generated:'+name+' -->\n\n'+blocks[name]+'\n\n<!-- /generated:'+name+' -->';
});
if(/\{\{|TODO|ย้ายเนื้อหาเดิม|คงตารางเดิมไว้|ถ้าต้องการใส่ weight/.test(text))throw Error('Unresolved editorial instruction');
for(const key of Object.keys(before))if(hash(key)!==before[key])throw Error('Source mutated: '+key);
// Check local links, images and table widths before replacing the report.
const links = [...text.matchAll(/!?\[[^\]]*\]\(([^)]+)\)/g)].map(m=>m[1]);
for(const href of links){if(/^(https?:|#)/.test(href))continue;const target=href.split('#')[0];if(target==='report/qa/presentation_validation.json')continue;if(!fs.existsSync(path.join(root,target)))throw Error('Broken README link: '+href);}
const lines=text.split(/\r?\n/);let columns=null,tables=0;
for(const line of lines){if(line.startsWith('|')){const count=line.split(/(?<!\\)\|/).length-2;if(columns===null){columns=count;tables++;}if(count!==columns)throw Error('Unequal Markdown table width: '+line);}else columns=null;}
fs.writeFileSync(path.join(root,'README.md'),text.trim()+'\n');
fs.writeFileSync(path.join(__dirname,'report_source.md'),'<!-- Generated mirror of ../README.md; edit report/readme_template.md and run node report/render_readme.js. -->\n'+text.trim().replaceAll('](report/','](')+'\n');
fs.mkdirSync(path.join(__dirname,'qa'),{recursive:true});
fs.writeFileSync(path.join(__dirname,'qa/presentation_validation.json'),JSON.stringify({
  checked_utc:new Date().toISOString(),source_sha256:before,readme_sha256:hash('README.md'),
  table_count:tables,local_and_external_links:links.length,broken_local_links:0,
  original_metric_units:'Saved CSV/JSON remain unchanged; only displayed metrics, differences and CIs are multiplied by 100. p/t/d_z are not rescaled.',
  model_selection:'ResNet50/Full post hoc; original MobileNetV3-Large/Partial statistical reference preserved.',
  execution_scope:'This renderer uses saved files only. The user separately authorized MobileNet Grad-CAM inference for eight fixed images; no new training or tuning was performed.',
  renderer_python_inference_training_performed:false
},null,2)+'\n');
console.log('Rendered README and report mirror; checked '+tables+' tables and '+links.length+' links.');


In [ ]:
%%project_source report/report_utils.js
const fs = require('fs');
const path = require('path');
const crypto = require('crypto');
const root = path.resolve(__dirname, '..');
function parseCsv(text) {
  const rows = []; let row = [], field = '', quoted = false;
  text = text.replace(/^\uFEFF/, '');
  for (let i = 0; i < text.length; i++) {
    const c = text[i];
    if (quoted) {
      if (c === '"' && text[i + 1] === '"') { field += '"'; i++; }
      else if (c === '"') quoted = false;
      else field += c;
    } else if (c === '"') quoted = true;
    else if (c === ',') { row.push(field); field = ''; }
    else if (c === '\n') { row.push(field.replace(/\r$/, '')); rows.push(row); row = []; field = ''; }
    else field += c;
  }
  if (quoted) throw Error('Unclosed CSV quote');
  if (field || row.length) { row.push(field.replace(/\r$/, '')); rows.push(row); }
  const header = rows.shift();
  return rows.filter(r => r.some(Boolean)).map(r => {
    if (r.length !== header.length) throw Error('CSV column mismatch');
    return Object.fromEntries(header.map((key, i) => [key, r[i]]));
  });
}
const readCsv = relative => parseCsv(fs.readFileSync(path.join(root, relative), 'utf8'));
const hash = relative => crypto.createHash('sha256').update(fs.readFileSync(path.join(root, relative))).digest('hex');
function sharpRuntime() {
  try { return require('sharp'); }
  catch { return require(path.join(process.env.USERPROFILE, '.cache/codex-runtimes/codex-primary-runtime/dependencies/node/node_modules/sharp')); }
}
const mean = values => values.reduce((a, b) => a + b, 0) / values.length;
const sd = values => { const m = mean(values); return Math.sqrt(values.reduce((a, b) => a + (b - m) ** 2, 0) / (values.length - 1)); };
module.exports = { root, parseCsv, readCsv, hash, sharpRuntime, mean, sd };


In [ ]:
%%project_source report/revise_readme_figures.js
// Report figures from saved outputs, including the explicitly authorized eight matched CAMs.
const fs=require('fs'),path=require('path'),assert=require('assert/strict');
const {root,readCsv,hash,sharpRuntime,mean,sd}=require('./report_utils');
const T=require('./figure_theme');
const assets='report/report_assets/latest/';
const out='outputs_modes_v3_td/';
const sources={};
function source(p){sources[p]=hash(p);return path.join(root,p);}
function csv(p){source(p);return readCsv(p);}
const resnet=csv('report/resnet_gradcam_cases.csv');
const latestPath=out+'gradcam/mobilenet_v3_large_partial_matched_resnet/seed42/latest.json';
const latest=JSON.parse(fs.readFileSync(source(latestPath),'utf8'));
for(const [p,h]of Object.entries(latest.artifact_sha256))assert.equal(hash(out+p),h);
const mobile=csv(out+latest.directory+'/selected_cases.csv');
assert.deepEqual(resnet.map(r=>r.image_id),mobile.map(r=>r.image_id));assert.equal(mobile.length,8);
const cls={san_phraphrom:'พระพรหม',san_phraphum:'พระภูมิ',san_tayay:'ตายาย',tee_ju_eia:'ตี่จู้เอี๊ยะ'};
const txt=(x,y,v,size=20,extra='')=>'<text x="'+x+'" y="'+y+'" text-anchor="middle" font-size="'+size+'" '+extra+'>'+T.esc(v)+'</text>';
const figures=[];
async function grid(rows,title,name){
 const sharp=sharpRuntime(),width=1520,height=1750,items=[];
 let labels='<svg xmlns="http://www.w3.org/2000/svg" width="'+width+'" height="'+height+'"><style>text{font-family:'+T.fontFamily+';fill:'+T.ink+'}</style>';
 labels+=txt(width/2,40,title,31,'font-weight="700"')+txt(width/2,77,'Same eight images · seed 42 · each overlay explains that model’s predicted class',20);
 for(let i=0;i<8;i++){
   const r=rows[i],left=20+(i%2)*760,top=108+Math.floor(i/2)*405;
   labels+='<rect x="'+left+'" y="'+top+'" width="720" height="390" rx="8" fill="none" stroke="'+T.grid+'"/>';
   labels+=txt(left+360,top+29,r.image_id.split('/')[1],22,'font-weight="700"');
   for(let j=0;j<2;j++){
     const input=source(out+r.sample_directory+'/'+['original.png','overlay.png'][j]);
     items.push({input:await sharp(input).resize(330,285,{fit:'contain',background:'white'}).png().toBuffer(),left:left+20+j*350,top:top+65});
     labels+=txt(left+185+j*350,top+56,['Original','Overlay'][j],20);
   }
   labels+=txt(left+360,top+378,'จริง: '+cls[r.true_label]+' → ทำนาย: '+cls[r.predicted_label]+' | '+(100*Number(r.confidence)).toFixed(2)+'% '+(r.true_label===r.predicted_label?'✓':'✗'),21);
 }
 labels+='</svg>';items.push({input:Buffer.from(labels),left:0,top:0});
 const rel=assets+name;await sharp({create:{width,height,channels:3,background:'white'}}).composite(items).png().toFile(path.join(root,rel));figures.push(rel);
}
function reliability(){
 const bins=csv(out+'confidence/bin_metrics.csv').filter(r=>r.architecture==='mobilenet_v3_large'&&r.mode==='partial'&&r.seed==='42');
 assert.equal(bins.length,10);assert.equal(bins.reduce((s,b)=>s+Number(b.count),0),85);
 let s='<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 570"><rect width="800" height="570" fill="white"/><style>text{font-family:'+T.fontFamily+';fill:'+T.ink+'}</style>';
 const x=90,y=90,w=580,h=370,px=v=>x+v*w,py=v=>y+h-v*h;
 s+=txt(400,34,'MobileNetV3-Large / Partial: seed 42 (%)',22,'font-weight="700"');
 for(const t of [0,.2,.4,.6,.8,1])s+='<path d="M '+x+' '+py(t)+' H '+(x+w)+'" stroke="'+T.grid+'"/>'+txt(x-24,py(t)+5,t*100,12)+txt(px(t),y+h+20,t*100,12);
 s+='<path d="M '+x+' '+y+' V '+(y+h)+' H '+(x+w)+'" fill="none" stroke="'+T.muted+'"/><path d="M '+x+' '+(y+h)+' L '+(x+w)+' '+y+'" stroke="#c4cdd3" stroke-dasharray="7 5"/>';
 const ece=bins.reduce((s,b)=>s+Number(b.count)/85*(Number(b.count)?Math.abs(Number(b.mean_confidence)-Number(b.accuracy)):0),0)*100;
 s+=txt(265,114,'ECE (10 bins, this seed): '+ece.toFixed(2),12);
 for(const b of bins)if(Number(b.count)){const xx=px(Number(b.mean_confidence)),yy=py(Number(b.accuracy));s+='<circle cx="'+xx+'" cy="'+yy+'" r="'+Math.max(5,Math.sqrt(Number(b.count))*2.6)+'" fill="'+T.modeColors.partial+'" opacity=".65" stroke="'+T.ink+'"/>'+txt(xx,yy-13,b.count,12);}
 s+=txt(380,499,'Mean confidence (%)',14)+txt(28,275,'Observed accuracy (%)',14,'transform="rotate(-90 28 275)"')+txt(400,532,'Dot area scales with bin count; label = images in bin (test n = 85)',12)+'</svg>';
 const rel=assets+'reliability_mobilenet_v3_large_partial_seed42.svg';fs.writeFileSync(path.join(root,rel),s);figures.push(rel);return {ece_seed42_pct:ece,bins};
}
async function main(){
 const sharp=sharpRuntime();
 for(let i=0;i<8;i++){const a=await sharp(source(out+resnet[i].sample_directory+'/original.png')).raw().toBuffer({resolveWithObject:true}),b=await sharp(source(out+mobile[i].sample_directory+'/original.png')).raw().toBuffer({resolveWithObject:true});assert.deepEqual(a.info,b.info);assert(a.data.equals(b.data),'CAM originals differ');}
 await grid(resnet,'ResNet50 / Full: original and Grad-CAM overlay','gradcam_resnet50_full_original_overlay.png');
 await grid(mobile,'MobileNetV3-Large / Partial: original and Grad-CAM overlay','gradcam_mobilenet_partial_matched_original_overlay.png');
 const calibration=reliability();
 const summary=csv(out+'final/pair_summary.csv');
 const leaders=Object.fromEntries(['test_acc','test_macro_precision','test_macro_recall','test_macro_f1'].map(k=>{const sorted=[...summary].sort((a,b)=>Number(b[k+'_mean'])-Number(a[k+'_mean']));assert.equal(sorted[0].architecture+'::'+sorted[0].mode,'resnet50::full');return [k,{leader:'resnet50::full',mean:Number(sorted[0][k+'_mean']),runner_up:sorted[1].architecture+'::'+sorted[1].mode,lead_pp:100*(Number(sorted[0][k+'_mean'])-Number(sorted[1][k+'_mean']))}];}));
 const finals=csv(out+'final/run_metrics.csv');
 const matched=[42,123,456,789,1024].map(seed=>{const find=(a,m)=>Number(finals.find(r=>r.architecture===a&&r.mode===m&&Number(r.seed)===seed).test_macro_f1)*100;return {seed,mobile_f1_pct:find('mobilenet_v3_large','partial'),resnet_f1_pct:find('resnet50','full'),difference_pp:find('mobilenet_v3_large','partial')-find('resnet50','full')};});
 const d=matched.map(r=>r.difference_pp),t=mean(d)/(sd(d)/Math.sqrt(5));
 const saved=csv('report/verified_statistics.csv').find(r=>r.reference==='mobilenet_v3_large::partial'&&r.competitor==='resnet50::full'&&r.metric==='test_macro_f1');assert(Math.abs(t-Number(saved.t))<1e-10);
 const comparison=resnet.map((r,i)=>({image_id:r.image_id,true_label:r.true_label,resnet_pred:r.predicted_label,resnet_confidence:Number(r.confidence),mobile_pred:mobile[i].predicted_label,mobile_confidence:Number(mobile[i].confidence)}));
 const keys=Object.keys(comparison[0]);fs.writeFileSync(path.join(__dirname,'matched_cam_comparison.csv'),[keys.join(','),...comparison.map(r=>keys.map(k=>r[k]).join(','))].join('\n')+'\n');
 const evidence={prepared_utc:new Date().toISOString(),leaders,paired_f1_example:{matched,mean_difference_pp:mean(d),sample_sd_difference_pp:sd(d),t,df:4,p_raw:Number(saved.p_raw),p_holm:Number(saved.p_holm),alpha:.05},calibration,confidence_pair_summary:csv(out+'confidence/pair_summary.csv'),cam_comparison:comparison,mean_selected_epochs:csv('report/mean_selected_epochs.csv'),cam_inference_authorized:true,new_training_runs:0,source_sha256:sources,output_sha256:Object.fromEntries(figures.map(p=>[p,hash(p)]))};
 fs.writeFileSync(path.join(__dirname,'readme_revision_evidence.json'),JSON.stringify(evidence,null,2)+'\n');
 console.log(JSON.stringify({figures:figures.length,all_four_metric_leaders:'ResNet50/Full',mobile_correct:mobile.filter(r=>r.true_label===r.predicted_label).length,resnet_correct:resnet.filter(r=>r.true_label===r.predicted_label).length,mobile_seed42_ece:calibration.ece_seed42_pct,paired_t:t}));
}
main().catch(e=>{console.error(e);process.exitCode=1;});


In [ ]:
%%project_source report/validate_report_presentation.js
// Read-only numerical/layout checks for the report; no model code, Python or training.
const fs=require('fs'),path=require('path');
const {root,readCsv,hash,mean,sd}=require('./report_utils');
const old=JSON.parse(fs.readFileSync(path.join(__dirname,'qa/report_revision_inputs.json'),'utf8'));
for(const [p,h] of Object.entries(old.source_sha256))if(hash(p)!==h)throw Error('Measured artifact changed: '+p);
const summary=readCsv('report/learning_curve_summary.csv'), registry=JSON.parse(fs.readFileSync(path.join(root,'outputs_modes_v3_td/run_registry.json'),'utf8'));
const runs=registry.final_runs.filter(r=>r.status==='completed').map(r=>({...r,history:readCsv(path.relative(root,path.join(r.run_dir,'history.csv'))).filter(h=>h.train_loss!=='')}));
let maxMeanError=0,maxSdError=0,singleRunEpochMetrics=0;
for(const row of summary){
  const observations=runs.filter(r=>r.architecture===row.architecture&&r.mode===row.mode).flatMap(r=>r.history.filter(h=>Number(h.epoch)===Number(row.epoch)).map(h=>Number(h[row.metric])));
  if(observations.length!==Number(row.n)||!observations.length)throw Error('Incorrect available-run count');
  maxMeanError=Math.max(maxMeanError,Math.abs(mean(observations)-Number(row.mean)));
  if(observations.length===1){singleRunEpochMetrics++;if(row.sample_sd!=='')throw Error('Sample SD present for one run');}
  else maxSdError=Math.max(maxSdError,Math.abs(sd(observations)-Number(row.sample_sd)));
}
if(maxMeanError>1e-12||maxSdError>1e-12)throw Error('Learning aggregation mismatch');
const hist=readCsv('report/image_property_histograms.csv');let histogramGroups=0;
for(const figure of ['aspect_ratio','brightness'])for(const c of ['san_phraphrom','san_phraphum','san_tayay','tee_ju_eia']){
  const group=hist.filter(r=>r.figure===figure&&r.class===c);if(!group.length)throw Error('Missing histogram group');
  const n=group.reduce((a,r)=>a+Number(r.count),0),pct=group.reduce((a,r)=>a+Number(r.proportion_pct),0);
  if(n!==Number(group[0].class_total)||Math.abs(pct-100)>1e-9)throw Error('Histogram omitted images or denominator mismatch');histogramGroups++;
}
const tuning=readCsv('report/tuning_checkpoint_observations.csv');if(tuning.length!==75||tuning.filter(r=>r.winner==='true').length!==15)throw Error('Incorrect tuning winner count');
const checkpoints=readCsv('report/final_checkpoint_summary.csv');if(checkpoints.length!==75)throw Error('Missing final checkpoint identities');
const readme=fs.readFileSync(path.join(root,'README.md'),'utf8');const lines=readme.split(/\r?\n/);
if(lines.some(l=>/^#+ /.test(l)&&/[ก-๙]/.test(l)))throw Error('Non-English heading');
let inTable=false,tables=0;for(const l of lines){if(l.startsWith('|')){if(!inTable){tables++;if(/[ก-๙]/.test(l))throw Error('Non-English table header');}inTable=true;}else inTable=false;}
if(/เรา|โจทย์|ผลการทดลองจาก.*ปรับปรุง|ปิด augmentation/.test(readme))throw Error('Outdated report language');
const mirror=fs.readFileSync(path.join(__dirname,'report_source.md'),'utf8').split('\n').slice(1).join('\n');
if(mirror!==readme.replaceAll('](report/',']('))throw Error('Report mirror differs');
const enhanced=JSON.parse(fs.readFileSync(path.join(__dirname,'enhanced_figure_sources.json'),'utf8'));
for(const [p,h]of Object.entries(enhanced.source_sha256))if(hash(p)!==h)throw Error('Figure input drift');
for(const [p,h]of Object.entries(enhanced.output_sha256))if(hash(p)!==h)throw Error('Figure output drift');
const evidence={checked_utc:new Date().toISOString(),unchanged_measured_csv_json_files:Object.keys(old.source_sha256).length,
  learning_curve_metric_rows:summary.length,final_runs:75,final_pairs:15,history_mean_max_error:maxMeanError,history_sample_sd_max_error:maxSdError,
  single_run_epoch_metric_rows_without_sd:singleRunEpochMetrics,histogram_groups_with_all_images_and_100pct:histogramGroups,
  tuning_trials:75,within_setup_winners:15,final_checkpoint_identities:75,english_headings_and_table_headers:true,personal_pronouns_removed:true,table_count:tables,
  selection_lock_unchanged:true,ablation_lock_unchanged:true,python_inference_training_performed:false};
fs.writeFileSync(path.join(__dirname,'qa/report_revision_validation.json'),JSON.stringify(evidence,null,2)+'\n');
console.log(JSON.stringify(evidence));


In [ ]:
%%project_source report/verify_readme_revision.js
// Report-only evidence checks. Does not import Python, train, or run inference.
const fs=require('fs'),path=require('path'),assert=require('assert/strict');
const {root,readCsv,hash,mean,sd}=require('./report_utils');
const read=p=>JSON.parse(fs.readFileSync(path.join(root,p),'utf8'));
const evidence=read('report/readme_revision_evidence.json');
const curves=read('report/enhanced_figure_sources.json');
const preservation=read('report/qa/evidence_preservation.json');
const previousFollowup=read('report/resnet_followup_sources.json');
const verifiedSources={};
for(const manifest of [evidence,curves])for(const [file,digest] of Object.entries(manifest.source_sha256)){
  assert.equal(hash(file),digest,'Saved source changed: '+file);verifiedSources[file]=digest;
}
for(const manifest of [evidence,curves])for(const [file,digest] of Object.entries(manifest.output_sha256))assert.equal(hash(file),digest,'Figure changed: '+file);
const base='outputs_modes_v3_td/';
const ablation=base+'ablation/resnet50_full_v1/';
assert.equal(hash(base+'selection.json'),preservation.selection_sha256);
assert.equal(hash(ablation+'ablation_lock.json'),preservation.ablation_lock_sha256);
assert.equal(hash(ablation+'run_registry.json'),previousFollowup.ablation.registry_sha256);
for(const [file,digest] of Object.entries(previousFollowup.ablation.artifact_sha256))assert.equal(hash(ablation+file),digest);
const selected=read(base+'selection.json');
assert.deepEqual(selected.reference_pair,{architecture:'mobilenet_v3_large',mode:'partial'});
const summary=readCsv(base+'final/pair_summary.csv');assert.equal(summary.length,15);
for(const key of ['test_acc','test_macro_precision','test_macro_recall','test_macro_f1']){
  const top=[...summary].sort((a,b)=>Number(b[key+'_mean'])-Number(a[key+'_mean']))[0];
  assert.equal(top.architecture+'::'+top.mode,'resnet50::full');
  assert.equal(Number(top[key+'_mean']),evidence.leaders[key].mean);
}
const checkpoints=readCsv('report/final_checkpoint_summary.csv');assert.equal(checkpoints.length,75);
const epochMeans=readCsv('report/mean_selected_epochs.csv');assert.equal(epochMeans.length,15);
for(const r of epochMeans){
  const group=checkpoints.filter(x=>x.architecture===r.architecture&&x.mode===r.mode);
  assert.equal(group.length,5);assert.equal(Number(r.mean_selected_epoch),mean(group.map(x=>Number(x.selected_epoch))));
  const svg=fs.readFileSync(path.join(root,'report/report_assets/latest/learning_curves_'+r.architecture+'.svg'),'utf8');
  assert.equal((svg.match(/Mean selected epoch = /g)||[]).length,3);
  assert.equal((svg.match(/stroke-dasharray="3 5"/g)||[]).length,3);
  assert(svg.includes('Mean selected epoch = '+Number(r.mean_selected_epoch).toFixed(1)));
}
const runs=readCsv(base+'final/run_metrics.csv');assert.equal(runs.length,75);
const differences=evidence.paired_f1_example.matched.map(r=>{
  const metric=(a,m)=>Number(runs.find(x=>Number(x.seed)===r.seed&&x.architecture===a&&x.mode===m).test_macro_f1)*100;
  assert.equal(metric('mobilenet_v3_large','partial'),r.mobile_f1_pct);
  assert.equal(metric('resnet50','full'),r.resnet_f1_pct);
  return r.mobile_f1_pct-r.resnet_f1_pct;
});
const example=evidence.paired_f1_example;
assert.equal(mean(differences),example.mean_difference_pp);assert.equal(sd(differences),example.sample_sd_difference_pp);
const t=mean(differences)/(sd(differences)/Math.sqrt(5));assert.equal(t,example.t);
// Exact two-sided t probability for df=4, independent of the saved p-value.
const u=Math.abs(t)/Math.sqrt(t*t+4),p=1-1.5*u+.5*u*u*u;
assert(Math.abs(p-example.p_raw)<1e-12);
const stats=readCsv('report/verified_statistics.csv');assert.equal(stats.length,58);
const primary=stats.filter(r=>r.family_id.startsWith('primary'));assert.equal(primary.length,28);
assert.equal(stats.filter(r=>r.family_id.startsWith('secondary')).length,30);
let adjusted=0;
for(const [i,r]of [...primary].sort((a,b)=>Number(a.p_raw)-Number(b.p_raw)).entries()){
  adjusted=Math.min(1,Math.max(adjusted,(28-i)*Number(r.p_raw)));
  if(r.competitor==='resnet50::full'&&r.metric==='test_macro_f1')assert.equal(adjusted,example.p_holm);
}
const bins=readCsv(base+'confidence/bin_metrics.csv');
const ece={};
for(const [arch,mode]of [['resnet50','full'],['mobilenet_v3_large','partial']]){
  const rows=bins.filter(r=>r.architecture===arch&&r.mode===mode&&r.seed==='42');
  assert.equal(rows.length,10);assert.equal(rows.reduce((s,r)=>s+Number(r.count),0),85);
  ece[arch]=100*rows.reduce((s,r)=>s+(Number(r.count)?Number(r.count)/85*Math.abs(Number(r.mean_confidence)-Number(r.accuracy)):0),0);
}
assert.equal(ece.resnet50.toFixed(2),'5.99');assert.equal(ece.mobilenet_v3_large.toFixed(2),'2.76');
const latest=read(base+'gradcam/mobilenet_v3_large_partial_matched_resnet/seed42/latest.json');
assert.equal(latest.examples,8);assert.equal(latest.training_runs_added,0);
for(const [file,digest]of Object.entries(latest.artifact_sha256))assert.equal(hash(base+file),digest);
const mobile=readCsv(base+latest.directory+'/selected_cases.csv');
const resnet=readCsv('report/resnet_gradcam_cases.csv');
assert.deepEqual(mobile.map(r=>r.image_id),resnet.map(r=>r.image_id));
assert.equal(new Set(mobile.map(r=>r.image_id)).size,8);
const classes=['san_phraphrom','san_phraphum','san_tayay','tee_ju_eia'];
for(const r of mobile){
  const probabilities=JSON.parse(r.cam_inference_probabilities);
  assert.equal(classes[probabilities.indexOf(Math.max(...probabilities))],r.predicted_label);
}
const markdown=fs.readFileSync(path.join(root,'README.md'),'utf8');
assert(!/1\.4 Duplicates|7\.2 Future work|ต้องประเมินยืนยันด้วยข้อมูลภายนอกชุดใหม่|\{\{/.test(markdown));
assert.equal((markdown.match(/### 6\.9 Final model selection/g)||[]).length,1);
assert(markdown.indexOf('### 6.9 Final model selection')>markdown.indexOf('### 6.8 Confidence and calibration'));
assert(markdown.includes('Holm-adjusted p = 1.0000 > α = 0.05'));
assert(markdown.includes('0.6514')&&markdown.includes('4.0327'));
const result={checked_utc:new Date().toISOString(),readme_sha256:hash('README.md'),template_sha256:hash('report/readme_template.md'),
  source_hashes_checked:Object.keys(verifiedSources).length,main_registry_sha256:hash(base+'run_registry.json'),
  selection_sha256:hash(base+'selection.json'),ablation_lock_sha256:hash(ablation+'ablation_lock.json'),
  preserved_main_final_runs:75,preserved_statistical_families:[28,30],mean_epoch_lines:15,
  metric_leaders:evidence.leaders,paired_f1:{t,raw_p_recomputed:p,holm_p:example.p_holm},seed42_ece_pct:ece,
  matched_cam_images:8,cam_prediction_classes_match_saved:true,
  maximum_cam_probability_difference:Math.max(...mobile.map(r=>Number(r.probability_max_abs_error))),
  execution_scope:'Report-only Node verification; eight unique MobileNet Grad-CAM images were generated under separate explicit authorization. No new training/tuning.',passed:true};
fs.writeFileSync(path.join(__dirname,'qa/readme_revision_checks.json'),JSON.stringify(result,null,2)+'\n');
console.log(JSON.stringify(result,null,2));


In [ ]:
%%project_source report/verify_resnet_followup.js
// Read-only recomputation of report evidence; writes one verification JSON.
// Does not invoke Python, model inference, or training.
const fs=require('fs'),path=require('path'),crypto=require('crypto');
const root=path.resolve(__dirname,'..'),out=path.join(root,'outputs_modes_v3_td');
const study=path.join(out,'ablation','resnet50_full_v1');
const classes=['san_phraphrom','san_phraphum','san_tayay','tee_ju_eia'],seeds=[42,123,456,789,1024];
const metrics=['test_acc','test_macro_precision','test_macro_recall','test_macro_f1'];
const json=p=>JSON.parse(fs.readFileSync(p,'utf8').replace(/^\uFEFF/,''));
const hash=p=>crypto.createHash('sha256').update(fs.readFileSync(p)).digest('hex');
const assert=(ok,message)=>{if(!ok)throw Error(message);};
function csv(file){const s=fs.readFileSync(file,'utf8').replace(/^\uFEFF/,'');let rows=[],row=[],f='',q=false;for(let i=0;i<s.length;i++){const c=s[i];if(q){if(c==='"'&&s[i+1]==='"'){f+='"';i++;}else if(c==='"')q=false;else f+=c;}else if(c==='"')q=true;else if(c===','){row.push(f);f='';}else if(c==='\n'){row.push(f.replace(/\r$/,''));rows.push(row);row=[];f='';}else f+=c;}if(f||row.length){row.push(f.replace(/\r$/,''));rows.push(row);}const h=rows.shift();return rows.filter(r=>r.length===h.length).map(r=>Object.fromEntries(h.map((k,i)=>[k,r[i]])));}
const mean=a=>a.reduce((s,x)=>s+x,0)/a.length;
const sd=a=>Math.sqrt(a.reduce((s,x)=>s+(x-mean(a))**2,0)/(a.length-1));
let maxMetricError=0,maxSummaryError=0,maxStatisticError=0;
const compare=(a,b,kind)=>{const d=Math.abs(Number(a)-Number(b));assert(Number.isFinite(d)&&d<1e-9,`${kind} discrepancy: ${a}, ${b}`);return d;};
const source=json(path.join(__dirname,'resnet_followup_sources.json'));
const lock=json(path.join(study,'ablation_lock.json')),registry=json(path.join(study,'run_registry.json'));
assert(source.ablation.registry_sha256===hash(path.join(study,'run_registry.json')),'Registry changed after analysis');
assert(source.ablation.study_fingerprint===lock.study_fingerprint,'Study fingerprint mismatch');
const copies={'condition_summary.csv':'resnet_ablation_condition_summary.csv','per_class_summary.csv':'resnet_ablation_per_class_summary.csv','matched_deltas.csv':'resnet_ablation_matched_deltas.csv','statistical_tests.csv':'resnet_ablation_statistical_tests.csv','ablation_summary.png':'report_assets/latest/resnet_ablation_summary.png'};
for(const [name,digest]of Object.entries(source.ablation.artifact_sha256)){assert(hash(path.join(study,name))===digest,'Ablation source hash: '+name);assert(hash(path.join(__dirname,copies[name]))===digest,'Ablation copy hash: '+name);}
const camdir=path.join(out,source.gradcam.directory);
for(const [name,digest]of Object.entries(source.gradcam.artifact_sha256))assert(hash(path.join(camdir,name))===digest,'CAM source hash: '+name);
assert(hash(path.join(__dirname,'report_assets/latest/gradcam_resnet50_full_seed42.png'))===source.gradcam.artifact_sha256['gradcam_grid.png'],'CAM grid copy hash');
assert(hash(path.join(__dirname,'resnet_gradcam_cases.csv'))===source.gradcam.artifact_sha256['selected_cases.csv'],'CAM cases copy hash');
const original=json(path.join(out,'run_registry.json'));
assert(json(path.join(out,'selection.json')).selection_fingerprint===lock.original_selection_fingerprint,'Original selection lock differs from the ablation plan');
const test=csv(path.join(out,'splits.csv')).filter(r=>r.split==='test');
const groups={baseline:original.final_runs.filter(r=>r.architecture==='resnet50'&&r.mode==='full'&&r.status==='completed')};
for(const c of ['no_augmentation','no_class_weights'])groups[c]=registry.attempts.filter(r=>r.condition===c&&r.status==='completed');
let predictionRows=0;
const calculated={};
for(const [condition,runs]of Object.entries(groups)){
 assert(runs.length===5&&seeds.every(s=>runs.filter(r=>Number(r.seed)===s).length===1),'Five matched seeds missing: '+condition);
 calculated[condition]={};
 for(const r of runs){
  const cfg=json(path.join(r.run_dir,'config.json'));
  for(const [key,value]of Object.entries(lock.baseline_config)){const expected=lock.variants[condition]?.[key]??value;assert(JSON.stringify(cfg[key])===JSON.stringify(expected),'Config factor mismatch: '+condition+'/'+key);}
  const pred=csv(path.join(r.run_dir,'test_predictions.csv'));predictionRows+=pred.length;
  assert(pred.length===85&&pred.every((p,i)=>p.image_id===test[i].image_id&&p.true_label===test[i].label),'Canonical prediction order mismatch');
  const mat=classes.map(()=>classes.map(()=>0));for(const p of pred)mat[classes.indexOf(p.true_label)][classes.indexOf(p.predicted_label)]++;
  const pc=classes.map((c,i)=>{const tp=mat[i][i],support=mat[i].reduce((a,b)=>a+b,0),col=mat.reduce((n,row)=>n+row[i],0),p=col?tp/col:0,rec=tp/support;return {class:c,precision:p,recall:rec,f1:p+rec?2*p*rec/(p+rec):0,support};});
  const vals={test_acc:mat.reduce((n,row,i)=>n+row[i],0)/85,test_macro_precision:mean(pc.map(v=>v.precision)),test_macro_recall:mean(pc.map(v=>v.recall)),test_macro_f1:mean(pc.map(v=>v.f1)),pc};
  for(const m of metrics)maxMetricError=Math.max(maxMetricError,compare(vals[m],r[m],'run metric'));
  calculated[condition][r.seed]=vals;
 }
}
for(const row of csv(path.join(study,'condition_summary.csv')))for(const m of metrics){const v=seeds.map(s=>calculated[row.condition][s][m]);maxSummaryError=Math.max(maxSummaryError,compare(mean(v),row[m+'_mean'],'mean'),compare(sd(v),row[m+'_sample_sd'],'sample SD'));}
for(const row of csv(path.join(study,'matched_deltas.csv')))for(const m of metrics)maxSummaryError=Math.max(maxSummaryError,compare(100*(calculated[row.condition][row.seed][m]-calculated.baseline[row.seed][m]),row[m+'_delta_pp'],'matched delta'));
for(const row of csv(path.join(study,'per_class_summary.csv'))){for(const m of ['precision','recall','f1']){const v=seeds.map(s=>calculated[row.condition][s].pc.find(p=>p.class===row.class)[m]);maxSummaryError=Math.max(maxSummaryError,compare(mean(v),row[m+'_mean'],'per-class mean'),compare(sd(v),row[m+'_sample_sd'],'per-class SD'));}assert(Number(row.support)===calculated[row.condition][42].pc.find(p=>p.class===row.class).support,'Per-class support mismatch');}
const tests=csv(path.join(study,'statistical_tests.csv'));assert(tests.length===4,'Expected four tests');
for(const row of tests){const diff=seeds.map(s=>calculated[row.condition][s][row.metric]-calculated.baseline[s][row.metric]),t=mean(diff)/(sd(diff)/Math.sqrt(5)),u=Math.abs(t)/Math.sqrt(t*t+4),p=.5*(1-u)**2*(u+2);maxStatisticError=Math.max(maxStatisticError,compare(mean(diff),row.mean_difference,'paired mean'),compare(t,row.t_statistic,'paired t'),compare(p,row.p_value,'paired p'));assert(row.status==='ok'&&Number(row.holm_family_size)===4&&Number(row.paired_n)===5,'Statistical family mismatch');}
let running=0;[...tests].sort((a,b)=>Number(a.p_value)-Number(b.p_value)).forEach((r,i)=>{running=Math.max(running,Number(r.p_value)*(4-i));maxStatisticError=Math.max(maxStatisticError,compare(Math.min(1,running),r.p_holm,'Holm p'));assert((Number(r.p_holm)<.05)===(r.significant_holm==='True'),'Significance flag mismatch');});
const cases=csv(path.join(camdir,'selected_cases.csv'));
assert(cases.length===8&&cases.every(r=>r.architecture==='resnet50'&&r.mode==='full'&&Number(r.seed)===42),'CAM identities mismatch');
const seed42=csv(path.join(groups.baseline.find(r=>r.seed===42).run_dir,'test_predictions.csv'));
for(const c of cases){const p=seed42.find(p=>p.image_id===c.image_id);assert(p&&p.predicted_label===c.predicted_label&&p.true_label===c.true_label&&p.confidence===c.confidence,'CAM case mismatch');}
const mainRuns=[...original.trials,...original.final_runs].filter(r=>r.status==='completed');
const newRuns=registry.attempts.filter(r=>r.status==='completed');
const result={verified_utc:new Date().toISOString(),method:'Node.js recomputation from saved prediction files; no Python/inference/training',baseline_reused:5,new_ablation_runs:10,unique_new_seeds_per_condition:5,matched_prediction_rows:predictionRows,cam_examples:cases.length,cam_correct:cases.filter(c=>c.true_label===c.predicted_label).length,cam_wrong:cases.filter(c=>c.true_label!==c.predicted_label).length,completed_ablation_epochs:newRuns.reduce((n,r)=>n+r.epochs_completed,0),completed_main_runs:mainRuns.length,total_completed_training_runs:mainRuns.length+newRuns.length,total_completed_training_epochs:[...mainRuns,...newRuns].reduce((n,r)=>n+r.epochs_completed,0),max_run_metric_error:maxMetricError,max_summary_error:maxSummaryError,max_statistics_error:maxStatisticError,significant_holm_tests:tests.filter(r=>r.significant_holm==='True').length,artifact_hashes_match:true,original_selection_matches_ablation_lock:true,effective_tracking_modes:[...new Set(registry.attempts.map(r=>r.tracking_mode))],cloud_completeness_verified:false};
fs.writeFileSync(path.join(__dirname,'resnet_followup_verification.json'),JSON.stringify(result,null,2)+'\n');console.log(JSON.stringify(result,null,2));


In [ ]:
%%project_source report/write_report.ps1
$ErrorActionPreference='Stop'
. (Join-Path $PSScriptRoot 'build_report.ps1')
$readmePath=Join-Path $root 'README.md'
$text=[IO.File]::ReadAllText($readmePath)
function Insert($name,$value){
 $tag='@@'+$name+'@@';$block="<!-- generated:$name -->`n$value`n<!-- /generated:$name -->"
 if($script:text.Contains($tag)){$script:text=$script:text.Replace($tag,$block)}else{$pattern='(?s)<!-- generated:'+ $name +' -->.*?<!-- /generated:'+ $name +' -->';$script:text=[regex]::Replace($script:text,$pattern,[Text.RegularExpressions.MatchEvaluator]{param($match) $block})}
}
function Rows { [Collections.Generic.List[object]]::new() }
$rr=[Collections.Generic.List[object]]::new()
$manifest=Get-Content (Join-Path $out 'source_manifest.json') -Raw|ConvertFrom-Json
foreach($g in ($manifest.images|Group-Object source_domain|Sort-Object Count -Descending|Select-Object -First 10)){$name=if($g.Name){$g.Name}else{'Unknown'};$rr.Add(@($name,$g.Count))}
Insert DOMAINS (Table @('Source domain','จำนวนภาพ') $rr)
$classThai=@{san_phraphrom='ศาลพระพรหม';san_phraphum='ศาลพระภูมิ';san_tayay='ศาลตายาย';tee_ju_eia='ศาลตี่จู้เอี๊ยะ'}
$rr.Clear();foreach($c in $classes){$n=@($split|Where-Object label -eq $c).Count;$rr.Add(@($c,$classThai[$c],$n,(F (100*$n/560) 'F2')))};Insert CLASS_COUNTS (Table @('Class','ประเภทศาล','จำนวนภาพ','สัดส่วน (%)') $rr)
$rr.Clear();foreach($s in @('train','validation','test')){$group=@($split|Where-Object split -eq $s);$row=@($s,$group.Count);foreach($c in $classes){$row+=@($group|Where-Object label -eq $c).Count};$rr.Add($row)};Insert SPLIT_TABLE (Table @('Split','Total','พระพรหม','พระภูมิ','ตายาย','ตี่จู้') $rr)
$archRows=@(
 @('ResNet50','Residual blocks; fc→Linear(in_features,4)','ไม่มี head dropout เพิ่ม','Residual connections ช่วยการฝึก backbone ลึก; มีparametersมากกว่าMobileNetในโมเดลที่บันทึก'),
 @('EfficientNet-B0','MBConv; classifier[-1]→Linear(in_features,4)','คง Dropout .2','MBConv/compound design เน้นประสิทธิภาพเชิงโครงสร้าง; ไม่ได้จับเวลาbenchmarkในงานนี้'),
 @('MobileNetV3-Large','InvertedResidual; classifier[-1]→Linear(in_features,4)','คง hidden Linear/Hardswish/Dropout .2','ออกแบบสำหรับงานทรัพยากรจำกัด; มีmeanสูงหรือต่ำขึ้นกับmode ไม่ใช่ดีที่สุดทุกเงื่อนไข'),
 @('DenseNet121','Dense connections; classifier→Linear(in_features,4)','ไม่มี head dropout เพิ่ม; default dense drop_rate0','ใช้feature reuse; ข้อดีของการเชื่อมต่อไม่แทนผลเชิงสถิติ'),
 @('ConvNeXt-Tiny','ConvNeXt stages; classifier[-1]→Linear(in_features,4)','คง normalization/flatten; ไม่มี head dropoutเพิ่ม','CNN designที่ปรับให้ทันสมัย; จำนวนparametersสูงสุดใน5โมเดลนี้')
)
Insert ARCH_TABLE (Table @('Architecture','Backbone/classifier replacement','Head regularization','เหตุผลและข้อพิจารณา') $archRows)
$rr.Clear();foreach($a in $archs){$row=@($display[$a]);foreach($m in $modes){$r=$runs|Where-Object{$_.architecture-eq$a-and$_.mode-eq$m}|Select-Object -First 1;$cfg=$configs[$r.identity];if($m-eq'full'){$row+=(F $cfg.mode_stats.total_parameters 'N0')};$row+=(F $cfg.mode_stats.trainable_parameters 'N0')};$rr.Add($row)};Insert PARAM_TABLE (Table @('Architecture','Total parameters','Trainable Full','Trainable Frozen','Trainable Partial') $rr)
$search=Get-Content (Join-Path $out 'tuning/search_definition.json') -Raw|ConvertFrom-Json
$rr.Clear();$i=0;foreach($v in $search.sampled_configs){$i++;$rr.Add(@($i,(F $v.learning_rate 'G7'),(F $v.weight_decay 'G7'),$v.optimizer,$v.batch_size))};Insert SEARCH_TABLE (Table @('Trial','LR','Weight decay','Optimizer','Batch') $rr)
$winners=Get-Content (Join-Path $out 'tuning/best_params_by_arch_mode.json') -Raw|ConvertFrom-Json
$rr.Clear();foreach($p in $winners.pairs.PSObject.Properties){$v=$p.Value;$rr.Add(@($display[$v.architecture],$v.mode,$v.winning_trial,(F $v.best_val_macro_f1)))};Insert WINNER_TABLE (Table @('Architecture','Mode','Trial ที่เลือก','Validation Macro F1') $rr)
$rr.Clear();$i=0;foreach($v in ($summary|Sort-Object test_macro_f1_mean -Descending)){$i++;$rr.Add(@($i,$display[$v.architecture],$v.mode,"$(F (100*$v.test_acc_mean) 'F2') ± $(F (100*$v.test_acc_sample_sd) 'F2')","$(F $v.test_macro_precision_mean) ± $(F $v.test_macro_precision_sample_sd)","$(F $v.test_macro_recall_mean) ± $(F $v.test_macro_recall_sample_sd)","$(F $v.test_macro_f1_mean) ± $(F $v.test_macro_f1_sample_sd)"))};Insert RESULT_TABLE (Table @('Rank','Architecture','Mode','Accuracy (%)','Macro precision','Macro recall','Macro F1') $rr)
$rr.Clear();foreach($v in ($classSummary|Where-Object{($_.architecture-eq'mobilenet_v3_large'-and$_.mode-eq'partial')-or($_.architecture-eq'resnet50'-and$_.mode-eq'full')})){$rr.Add(@(($display[$v.architecture]+'/'+$v.mode),$v.class,$v.support,"$(F $v.precision_mean) ± $(F $v.precision_sample_sd)","$(F $v.recall_mean) ± $(F $v.recall_sample_sd)","$(F $v.f1_mean) ± $(F $v.f1_sample_sd)"))};Insert PERCLASS_TABLE (Table @('Pair','Class','Support','Precision','Recall','F1') $rr)
$rr.Clear();foreach($v in ($checked|Where-Object{$_.significant-or($_.reference-eq'mobilenet_v3_large::partial'-and$_.competitor-eq'resnet50::full')})){
 $a=$v.reference-split'::';$b=$v.competitor-split'::'
 $comparison=$display[$a[0]]+'/'+$a[1]+'<br>− '+$display[$b[0]]+'/'+$b[1]
 $metric=if($v.metric-eq'test_acc'){'Accuracy'}else{'Macro F1'}
 $rr.Add(@($v.family_id.Replace('primary_reference_pair_28','Primary').Replace('secondary_within_arch_mode_30','Secondary'),$comparison,$metric,(F $v.mean_difference),"[$(F $v.ci95_lower), $(F $v.ci95_upper)]",(F $v.cohen_dz 'F3'),(F $v.p_holm 'F6')))
};Insert STATS_TABLE (Table @('Family','A − B','Metric','Mean Δ','95% CI (unadjusted)','d_z','Holm p') $rr)
$rr.Clear();$dynamics=[Collections.Generic.List[object]]::new()
foreach($a in $archs){foreach($m in $modes){$bb=@();$ee=@();$tf=@();$vf=@();$late=0;foreach($r in ($runs|Where-Object{$_.architecture-eq$a-and$_.mode-eq$m})){$h=$histories[$r.identity];$best=$h|Where-Object{[int]$_.epoch-eq[int]$r.best_epoch}|Select-Object -First 1;$end=$h[-1];$bb+=[int]$r.best_epoch;$ee+=[int]$r.epochs_completed;$tf+=[double]$best.train_macro_f1;$vf+=[double]$best.val_macro_f1;if([double]$end.train_loss-lt[double]$best.train_loss-and[double]$end.val_loss-gt[double]$best.val_loss){$late++}};$rr.Add(@($display[$a],$m,(F (Mean $bb) 'F1'),(F (Mean $ee) 'F1'),(F (Mean $tf)),(F (Mean $vf)),"$late/5"));$dynamics.Add([pscustomobject]@{architecture=$a;mode=$m;mean_best_epoch=(Mean $bb);mean_stop_epoch=(Mean $ee);mean_train_f1_at_best=(Mean $tf);mean_val_f1_at_best=(Mean $vf);late_divergence_runs=$late})}}
Insert DYNAMICS_TABLE (Table @('Architecture','Mode','Mean best epoch','Mean stop epoch','Train F1@best','Val F1@best','Loss แยกตัวช่วงท้าย (runs)') $rr)
$dense=$dynamics|Where-Object{$_.architecture-eq'densenet121'-and$_.mode-eq'full'}
Insert DYNAMICS_TEXT "สำหรับ DenseNet121/Full ทั้ง5histories มี mean train F1 ณ checkpoint=$(F $dense.mean_train_f1_at_best), validation F1=$(F $dense.mean_val_f1_at_best), mean best/stop epochs=$(F $dense.mean_best_epoch 'F1')/$(F $dense.mean_stop_epoch 'F1') และ late-divergence ตามนิยามข้างต้น $($dense.late_divergence_runs)/5 runs จึงมีหลักฐานการแยกตัวช่วงท้ายในบางseed ไม่ควรสรุป underfitting จากoverallscoreเพียงอย่างเดียว ดูเส้นlossและvalidationplateauประกอบทุกseed"
$dynamics|Export-Csv (Join-Path $PSScriptRoot 'verified_training_dynamics.csv') -NoTypeInformation -Encoding utf8
$rr.Clear();foreach($v in (Import-Csv (Join-Path $out 'eyeball/eyeball_cases.csv'))){$rr.Add(@($v.image_id,$v.true_label,$v.predicted_label,(F $v.confidence),$v.imagenet_label))};Insert CASE_TABLE (Table @('Image ID','True','Fine-tuned prediction','Confidence (0–1)','ImageNet top1') $rr)
$errors=@{};foreach($id in $testIDs){$errors[$id]=[pscustomobject]@{image_id=$id;true_label=($split|Where-Object image_id -eq $id).label;wrong_all75=0;wrong_reference5=0;wrong_resnetfull5=0}}
foreach($r in $runs){foreach($p in (Import-Csv (Join-Path $r.run_dir 'test_predictions.csv'))){if($p.true_label-ne$p.predicted_label){$e=$errors[$p.image_id];$e.wrong_all75++;if($r.architecture-eq'mobilenet_v3_large'-and$r.mode-eq'partial'){$e.wrong_reference5++};if($r.architecture-eq'resnet50'-and$r.mode-eq'full'){$e.wrong_resnetfull5++}}}}
$errorRows=@($errors.Values|Sort-Object wrong_all75 -Descending);$errorRows|Export-Csv (Join-Path $PSScriptRoot 'error_frequency.csv') -NoTypeInformation -Encoding utf8
$rr.Clear();foreach($e in ($errorRows|Select-Object -First 10)){$rr.Add(@($e.image_id,$e.true_label,$e.wrong_all75,$e.wrong_reference5,$e.wrong_resnetfull5))};Insert ERROR_TABLE (Table @('Image ID','True class','Wrong/75','Wrong reference/5','Wrong ResNet Full/5') $rr)
if($text-match'@@[A-Z_]+@@'){throw 'Unresolved placeholder'}
Put $readmePath $text
# Run-level map makes architecture/mode/seed/split/metrics explicit.
$runMap=foreach($stage in @('trials','final_runs')){foreach($r in $reg.$stage){[pscustomobject]@{stage=$stage;identity=$r.identity;architecture=$r.architecture;mode=$r.mode;seed=$r.seed;status=$r.status;split=if($stage-eq'trials'){'train/validation'}else{'train/validation/test'};run_dir=$r.run_dir.Substring($root.Length+1).Replace('\','/');raw_or_summary='per-run raw history, predictions(test only), structured metrics';metric_names=if($stage-eq'trials'){'train/val loss,accuracy,macro_f1'}else{'train/val loss,accuracy,macro_f1; test loss,accuracy,macro precision/recall/f1; per-class P/R/F1'};report_sections='2,3,4,5,6,7'}}}
$runMap|Export-Csv (Join-Path $PSScriptRoot 'run_evidence_map.csv') -NoTypeInformation -Encoding utf8
$sourceText="<!-- Generated mirror of ../README.md; edit the canonical README narrative. -->`n"+$text.Replace('](report/', '](')
Put (Join-Path $PSScriptRoot 'report_source.md') $sourceText
# Keep the reviewed narrative, display units and appendix order after rebuilding evidence.
& node (Join-Path $PSScriptRoot 'render_readme.js')
if ($LASTEXITCODE -ne 0) { throw 'Report presentation rendering failed' }
"README filled from latest evidence and the reviewed presentation template"


In [ ]:
%%project_source requirements.txt
numpy>=1.26
Pillow>=10.0
scipy>=1.11
scikit-learn>=1.4
matplotlib>=3.8
wandb>=0.18


## 7. ยืนยันว่าเตรียม source ครบและกำหนดตัวช่วยรัน

ตัวช่วยใช้ subprocess แบบไม่ผ่าน shell และรัน CLI จากไฟล์ในเครื่อง ทำให้ Windows multiprocessing มี __main__ guard เดิม การเรียก stage ต้องเป็นคำสั่งที่ผู้ใช้เลือกเอง Flags ทั้งหมดเริ่มเป็น False


In [ ]:
missing_sources = sorted(set(BUNDLE_MANIFEST) - set(MATERIALIZED))
if missing_sources:
    raise RuntimeError("Run every source cell first: " + ", ".join(missing_sources))
for relative, record in MATERIALIZED.items():
    actual = hashlib.sha256((PROJECT_ROOT / relative).read_bytes()).hexdigest()
    if actual != record["sha256"]:
        raise RuntimeError("Prepared source changed: " + relative)
(PROJECT_ROOT / ".notebook_sources.json").write_text(json.dumps({"notebook": "thai_shrine_classifier_project.ipynb", "sources": MATERIALIZED}, indent=2), encoding="utf-8")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Prepared source files:", len(MATERIALIZED), "/", len(BUNDLE_MANIFEST))

def run_cli(script, arguments):
    command = [sys.executable, str(PROJECT_ROOT / script), *map(str, arguments)]
    print("Command:", " ".join(command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONIOENCODING"] = "utf-8"
    environment["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + environment.get("PYTHONPATH", "")
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

def run_modes(stage):
    arguments = ["--stage", stage, "--data-root", DATA_ROOT, "--base-out-dir", BASE_OUT_DIR, "--out-dir", OUT_DIR,
                 "--architectures", *ARCHITECTURES, "--modes", *MODES, "--trials-per-pair", 5,
                 "--runs", len(SEEDS), "--seeds", *SEEDS, "--epochs", 30, "--input-size", 224,
                 "--seed", 42, "--split-seed", 42, "--device", DEVICE, "--num-workers", NUM_WORKERS,
                 "--wandb-project", WANDB_PROJECT, "--wandb-mode", WANDB_MODE]
    if RESUME:
        arguments.append("--resume")
    run_cli("pipeline_modes.py", arguments)

def run_resnet(stage):
    if stage == "report" and not ALLOW_README_UPDATE:
        raise ValueError("This stage edits local README. Set ALLOW_README_UPDATE=True only when you intend to update it.")
    arguments = ["--stage", stage, "--data-root", DATA_ROOT, "--out-dir", OUT_DIR, "--device", DEVICE,
                 "--num-workers", NUM_WORKERS, "--wandb-project", WANDB_PROJECT, "--wandb-mode", WANDB_MODE]
    if RESUME:
        arguments.append("--resume")
    run_cli("resnet_followup.py", arguments)

def saved_json(relative):
    file = PROJECT_ROOT / relative
    return json.loads(file.read_text(encoding="utf-8-sig")) if file.is_file() else None

def saved_csv(file):
    file = Path(file)
    if not file.is_file():
        print("Missing saved artifact:", file)
        return []
    with file.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(rows, columns=None, limit=None):
    if not rows:
        return
    columns = columns or list(rows[0])
    visible = rows if limit is None else rows[:limit]
    def clean(value):
        return str(value).replace("|", "/").replace("\n", " ")
    content = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    content += ["| " + " | ".join(clean(row.get(key, "")) for key in columns) + " |" for row in visible]
    display(Markdown("\n".join(content)))

def show_figure(relative):
    file = PROJECT_ROOT / relative
    if not file.is_file():
        print("Missing saved figure:", file)
        return
    display(SVG(filename=str(file)) if file.suffix.lower() == ".svg" else Image(filename=str(file)))


## 8. Data preparation และ class imbalance

ข้อมูล 560 ภาพ / 550 กลุ่ม ใช้ canonical grouped split seed 42: train 390, validation 85, test 85 กลุ่มภาพซ้ำไม่ข้าม partitions ทุก architecture/mode/seed ใช้ assignment เดิม

Pipeline คือ decode → EXIF transpose → RGB/alpha composite → LANCZOS resize รักษาสัดส่วน → neutral canvas 224×224 สี (124,116,104) → train-only augmentation → tensor → ImageNet normalization ไม่ใช้ edge-replicate padding หรือสุ่ม split ใหม่

Train augmentation: rotation ±12°, translation ±5%, scale .92–1.08, perspective distortion .10/p=.20, horizontal flip .5, jitter .10/.10/.10/hue .02 และ erasing p=.05/area .02–.08 ส่วน validation/test deterministic

Class weights คำนวณจาก train เท่านั้น: weight[c] = 390/(4×n_train[c]) ตาม class order ได้ประมาณ 1.392857, 1.095506, 1.071429, 0.696429 ใช้ weighted cross-entropy แบบเฉลี่ย per image ไม่มี oversampling


In [ ]:
if RUN["prepare"]:
    run_modes("prepare")
split_rows = saved_csv(BASE_OUT_DIR / "splits.csv")
if split_rows:
    counts = Counter((row["label"], row["split"]) for row in split_rows)
    show_table([{"class": label, **{part: counts[(label, part)] for part in ("train", "validation", "test")}} for label in CLASSES])
show_figure("report/report_assets/latest/class_distribution.svg")
show_figure("report/report_assets/latest/padding_large.png")


## 9. ตรวจ mode policies และ smoke — ผู้ใช้เลือกเอง

Full ฝึกทั้งโมเดล; Frozen ฝึก classifier และตรึง backbone+eval; Partial ฝึก final semantic stage กับ classifier โดยตรึงส่วนอื่น ต้อง apply mode ก่อน optimizer และหลัง model.train() ทุก training phase Smoke เป็นการตรวจระบบแยกจากผลการทดลองจริง ไม่ใช่ผลประเมินโมเดลบน test


In [ ]:
if RUN["smoke"]:
    run_modes("smoke")
else:
    print("Smoke is disabled. Source is included; no test/smoke was run by this cell.")


## 10. Finding best parameters — 75 independently evaluated trials

ใช้ local seeded random search 5 configurations ร่วมกันให้ทุก setup ทดลอง แต่เลือก winner ของแต่ละ architecture × mode แยกกัน รวม 15×5=75 trials ไม่ใช้สูตร Full winner แทน Frozen/Partial

Search space: LR log-uniform [1e-5,1e-3], weight decay log-uniform [1e-6,1e-2], optimizer AdamW/Adam/SGD และ batch 16/32 ในชุดสุ่มจริงได้ Adam 4 configurations กับ AdamW 1 จึงไม่มีผล SGD รอบนี้ การบันทึก W&B runs ไม่ใช่ cloud sweep service

ต่อ run ใช้สูงสุด 30 epochs, early stopping patience 7/min_delta=1e-4, ReduceLROnPlateau(mode=max,factor=.5,patience=3) บน validation Macro F1 เลือก checkpoint จาก F1 สูงสุด → loss ต่ำกว่า → epoch ก่อน ส่วน tuning winner ใช้ F1 → Accuracy → loss → trial index


In [ ]:
if RUN["tune"]:
    run_modes("tune")
search = saved_json(str(OUT_DIR / "tuning/search_definition.json"))
if search:
    print("Search method:", search["method"])
    show_table([{"trial": index + 1, **config} for index, config in enumerate(search["sampled_configs"])])
winners = saved_json(str(OUT_DIR / "tuning/best_params_by_arch_mode.json"))
if winners:
    show_table(list(winners["pairs"].values()), ["architecture", "mode", "winning_trial", "optimizer", "learning_rate", "weight_decay", "batch_size", "best_val_macro_f1"])
show_figure("report/report_assets/latest/tuning_setup_summary.svg")


## 11. Lock ก่อน test

ล็อกสูตรครบ 15 winners, fingerprints, seeds, budgets และ statistical families ใน selection.json ก่อน final โดยเลือก validation reference จากทั้ง 15 tuned winners ตัวที่ล็อกไว้คือ MobileNetV3-Large/Partial ไม่เปลี่ยน reference ตามอันดับ test


In [ ]:
if RUN["lock"]:
    run_modes("lock")
selection = saved_json(str(OUT_DIR / "selection.json"))
if selection:
    print("Validation reference:", selection["reference_pair"])
    print("Representative seed:", selection["representative_seed"])
    print("Reference validation:", selection["reference_validation"])
    print("Holm families:", selection["primary_holm_family"], selection["secondary_holm_family"])


## 12. Final training — 75 fresh ImageNet initializations

แต่ละ setup ใช้สูตรของตัวเอง เริ่มจาก ImageNet weights ใหม่ด้วย matched seeds 42,123,456,789,1024 เลือก/reload checkpoint ด้วย validation แล้ว test ครั้งเดียวต่อ run ไม่เริ่มต่อจาก tuning checkpoint และไม่เลือก seed จาก test score สูงสุด

ผลเดิมครบแล้ว ใช้ RESUME ตรวจ identity ก่อน reuse การตั้ง RUN['final']=True อาจฝึก runs ที่ไม่มีหลักฐาน matching ครบ Checkpoints ไม่อยู่ใน Git ต้องมีไฟล์ local ที่ตรงกันเมื่อต้อง resume หรือทำ CAM


In [ ]:
if RUN["final"]:
    run_modes("final")
registry = saved_json(str(OUT_DIR / "run_registry.json"))
if registry:
    for label, key in (("tuning", "trials"), ("final", "final_runs")):
        completed = [row for row in registry[key] if row.get("status") == "completed"]
        print(label, "completed:", len(completed), "actual epochs:", sum(row["epochs_completed"] for row in completed))


## 13. Evaluation, statistics และ model selection

ใช้ Accuracy, Macro precision/recall/F1, per-class metrics และ confusion สรุป mean ± sample SD (ddof=1) แยกตาม architecture+mode จับคู่ seeds สำหรับ two-sided paired t-tests และปรับ Holm แยก Primary 28 / Secondary 30 comparisons

ResNet50/Full มี mean test Accuracy 93.65% และ Macro F1 92.56% สูงสุดใน 15 setups รายงานเลือก ResNet หลังดู test ด้วยความได้เปรียบหลาย metrics, Accuracy/F1 SD ต่ำกว่า reference และ Recall พระภูมิสูงกว่า แต่เทียบ MobileNet/Partial ได้ Holm p=1.000 ทั้งสอง metrics จึงไม่อ้างความเหนือกว่าที่พิสูจน์แล้ว SD/tests วัด seed variation บน split เดียว ไม่ใช่ external generalization


In [ ]:
if RUN["analyze"]:
    run_modes("analyze")
summary = saved_csv(OUT_DIR / "final/pair_summary.csv")
if summary:
    metric_columns = ["test_acc", "test_macro_precision", "test_macro_recall", "test_macro_f1"]
    formatted = [{"setup": row["architecture"] + "/" + row["mode"], **{
        metric + " (%) ± SD (pp)": f"{100 * float(row[metric + '_mean']):.2f} ± {100 * float(row[metric + '_sample_sd']):.2f}"
        for metric in metric_columns}} for row in sorted(summary, key=lambda row: float(row["test_macro_f1_mean"]), reverse=True)]
    show_table(formatted)
stats = saved_csv(OUT_DIR / "final/statistical_tests.csv")
if stats:
    significant = [row for row in stats if str(row["significant_holm"]).lower() == "true"]
    print("Statistical rows:", len(stats), "significant after Holm:", len(significant))
    show_table(significant, ["family_id", "reference", "competitor", "metric", "p_holm"])
show_figure("report/report_assets/latest/mode_comparison.svg")
show_figure("report/report_assets/latest/confusion_resnet50_full.svg")


## 14. Learning dynamics และ calibration

อ่าน histories จริง ไม่เติมค่าหลัง early stopping กราฟเฉลี่ยช่วงท้ายมีจำนวน runs ลดลงได้ตาม available runs ส่วน ECE ใช้ 10 bins และต้องพิจารณาควบคู่กับจำนวนภาพในแต่ละ bin Confidence สูงไม่รับประกันว่าจะทายถูก


In [ ]:
if RUN["learning_curves"]:
    run_cli("plot_learning_curves.py", ["--out-dir", OUT_DIR])
show_figure("report/report_assets/latest/learning_curves_resnet50.svg")
show_figure("report/report_assets/latest/selected_resnet50_full_calibration.svg")
show_table(saved_csv(OUT_DIR / "confidence/pair_summary.csv"))


## 15. Grad-CAM และ eyeball

Main pipeline ทำ CAM/eyeball ของ validation-selected MobileNet/Partial seed 42 ส่วน ResNet follow-up ทำ CAM ของ ResNet/Full seed 42 มี matched-image MobileNet CAM บน 8 ภาพเดียวกันเพิ่มเติมแล้ว

Eyeball เปรียบเทียบป้าย ImageNet เดิมกับ fine-tuned label ของภาพเดียวกัน ชุดที่มีอยู่เป็น MobileNet supplementary evidence Grad-CAM แสดงบริเวณที่สัมพันธ์กับคำทำนาย ไม่พิสูจน์เหตุและผล และภาพคัดเลือกไม่ใช่ตัวแทนคะแนน test ทั้งชุด การสร้างใหม่เป็น inference จึงต้องเปิด flag เอง


In [ ]:
if RUN["gradcam"]:
    run_modes("gradcam")
elif RUN["eyeball"]:
    run_modes("eyeball")  # ทั้งสอง stages ใช้ combined CAM/eyeball routine เดียวกัน
if RUN["resnet_gradcam"]:
    run_resnet("gradcam")
show_figure("report/report_assets/latest/gradcam_resnet50_full_original_overlay.png")
show_figure("report/report_assets/latest/gradcam_mobilenet_partial_matched_original_overlay.png")
show_figure("report/report_assets/latest/eyeball_reference_percent.png")
show_table(saved_csv(OUT_DIR / "eyeball/eyeball_cases.csv"), ["image_id", "true_label", "predicted_label", "confidence", "imagenet_label", "imagenet_confidence"])


## 16. ResNet50/Full ablation — แยกจากงานหลัก

ล็อกสอง variants ก่อนทดสอบ: ไม่ใช้ augmentation และไม่ใช้ class weights คง recipe/split/checkpoint rule/seeds ใช้ baseline final ห้า seeds เดิม ไม่ retrain baseline และไม่ retune เพิ่ม 10 runs สูงสุด 300 epochs; ผลที่เสร็จแล้วใช้จริง 145 epochs

การนำ augmentation ออกลด mean Accuracy/F1 ประมาณ 0.94/1.32 pp ส่วนเอา weights ออกลด 0.47/0.67 pp แต่ไม่มี comparison ใด significant หลัง Holm family 4 (min adjusted p≈0.9002) ไม่ใช่หลักฐานยืนยันประโยชน์หรือความเทียบเท่ากัน


In [ ]:
if RUN["ablation_lock"]:
    run_resnet("lock")
if RUN["ablation"]:
    run_resnet("ablation")
if RUN["ablation_analyze"]:
    run_resnet("analyze")
study = OUT_DIR / "ablation/resnet50_full_v1"
show_table(saved_csv(study / "condition_summary.csv"), ["condition", "completed_seeds", "test_acc_mean", "test_acc_sample_sd", "test_macro_f1_mean", "test_macro_f1_sample_sd"])
show_table(saved_csv(study / "statistical_tests.csv"), ["condition", "metric", "mean_difference_pp", "p_holm", "significant_holm"])
show_figure("report/report_assets/latest/resnet_ablation_summary.svg")


## 17. Report export และหลักฐาน

อ่าน README และ report assets เดิมได้ทันที Main report stage สร้าง HTML ใน output ส่วน ResNet report stage และ Node renderer บางตัวเขียน README ในเครื่อง จึงปิดไว้และใช้ ALLOW_README_UPDATE guard ไม่มีการ commit/push จาก notebook

Report utilities ถูกฝังครบ แต่การจัดหน้าเดิมยังต้องใช้ files ใน report และเครื่องมือ Node/Sharp/PowerShell ที่รองรับ Source maps และ verification records เป็นหลักฐานของการทดลองที่บันทึกไว้ ไม่ใช่การอ้างว่า notebook นี้รันสำเร็จแล้ว


In [ ]:
if RUN["html_report"]:
    run_modes("report")
if RUN["readme_export"]:
    run_resnet("report")
for relative in ("report/verification.json", "report/resnet_followup_verification.json", "report/repository_contents.json"):
    file = PROJECT_ROOT / relative
    print(relative, "present" if file.is_file() else "missing")
print("README:", PROJECT_ROOT / "README.md")
print("No Git commit or push is performed by this notebook.")


## 18. วิธีใช้ไฟล์เดียวนี้และขอบเขตผลการตรวจ

1. เก็บ notebook พร้อมข้อมูล/ผลที่ต้องใช้ใน project folder แล้วตั้ง PROJECT_ROOT ในข้อ 0
2. รัน setup และ source cells ให้ครบ ระบบสร้างไฟล์ Python/JS/PowerShell ที่ฝังไว้ให้ในเครื่อง หาก source เดิมตรงกันจะไม่เขียนซ้ำ
3. คง flags เป็น False เพื่อดู saved results หรือเปิดเฉพาะ stages ที่ต้องการรัน แล้ว execute เซลล์ของขั้นตอนนั้น
4. ไม่ต้อง push ไฟล์ Python ที่สร้างแล้ว เพราะ .gitignore กัน root runners, packages และ tests ไว้ Git ใช้ notebook นี้เป็น canonical source ส่วน report/generate_matched_mobilenet_cam.py คงไว้หนึ่งไฟล์เพราะ README ลิงก์ตรงอยู่
5. Dataset, canonical split, logs/predictions และ report assets ยังต้องเก็บแยกเพื่อรองรับหลักฐาน Checkpoints อยู่เฉพาะ local จึงต้องมีไฟล์ที่ตรง identity ก่อนทำ inference/CAM หรือ resume ที่ต้องตรวจ checkpoint

ตรวจแบบ static ด้วย Node: notebook JSON/schema structure, code-cell execution states, แหล่ง source ครบ และคืน bytes/newlines แล้วตรง SHA256 ของทั้ง 45 ไฟล์ ไม่มีการ execute Python, tests, smoke, training หรือ inference เพื่อจัดทำ notebook นี้ การเปลี่ยนรูปแบบจัดเก็บจึงไม่ได้เพิ่ม training runs หรือแก้ผลการทดลองเดิม

ไฟล์ validation ของการจัดแพ็ก: report/notebook_validation.json
